In [ ]:
# Setup: in Google Colab, download and install the repository (a few minutes, once per session); elsewhere, work from the
# repository. Pages that read the derived catalogues need Zenodo bundle 1 (python scripts/fetch_release.py).
import os, subprocess, sys
os.environ['MPLBACKEND'] = 'Agg'   # as in a plain script run: Jupyter's inline settings would change the saved figures
if 'google.colab' in sys.modules:
    if not os.path.exists('/content/lmc-ancient-disc'):
        subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/vasilybelokurov/lmc-ancient-disc', '/content/lmc-ancient-disc'], check=True)
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '/content/lmc-ancient-disc'], check=True)
    os.chdir('/content/lmc-ancient-disc')
else:
    while not os.path.exists('pyproject.toml'):
        os.chdir('..')
sys.path[:0] = ['src', 'scripts', 'kb']
if not os.path.exists('data/giants_apogee.parquet'):   # the catalogues of Zenodo bundle 1, checked against data/sources.yaml
    subprocess.run([sys.executable, 'scripts/fetch_release.py'], check=True)

# Getting the data and writing the tables

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/lmc-ancient-disc/blob/main/kb/notebooks/chapters/getting_data.ipynb)

The previous chapters start from catalogues and result tables that are already on disk. This chapter shows where those files come
from and how the paper's tables and the numbers in its text are written from them. None of its scripts is run on this page: the
downloads need network access, credentials or many gigabytes of disk, and the table and text scripts are run with the rest of the
analysis (`python scripts/manifest.py run <script>`). Each script is shown cell by cell, with a note saying what it needs.

## Two routes to the data

Every input of the paper is listed in `data/sources.yaml`, in three tiers. Tier A holds the public inputs: survey catalogues,
published tables, isochrones and simulations. Each entry gives the origin and release of the file, its URL or query, the script that
fetches it, its number of rows where known, the paper it comes from and its sha256 checksum. Tier B holds the derived catalogues
(the cleaned and cross-matched samples, the selection weights and the posterior samples of the kinematic fits), each with the script
that builds it. Tier C holds results kept in `results/`. The function `lmcold.sources.path` refuses to return a file whose sha256
differs from the one recorded, so a script never reads an input that is not the one the paper was made with.

A reader can take one of two routes. The quick route downloads the tier-B catalogues as Zenodo bundle 1; with them and the tables in
`results/`, every figure, table and number of the paper can be remade. The full route fetches the tier-A inputs from the public
archives with the `scripts/fetch_*.py` scripts below and rebuilds the tier-B catalogues with the build scripts of the earlier
chapters; the rebuilt files are checked against the same checksums.

### The quick route: Zenodo bundle 1

`scripts/fetch_release.py` downloads the zip file of bundle 1 from its Zenodo record and checks it against the md5 that Zenodo
gives. Before it writes a member into `data/`, it compares the member's sha256 with the entry for that file in `data/sources.yaml`;
a member that is not listed, has another checksum or would land outside `data/` stops the installation, and nothing in `data/` is
overwritten by a file that fails its check. The first cell of every notebook of this site runs this script when the catalogues are
missing.

<details><summary>About <code>scripts/fetch_release.py</code></summary>

```text
Download Zenodo bundle 1 (scripts/build_bundle.py) and install it in data/, so that every figure, table and number of the paper
can be remade (the quick route of README.md).

The zip is downloaded from the Zenodo record RECORD (its md5 checked against the record) or, when it is set, from URL; then every
member is checked against the sha256 of its file in data/sources.yaml before it is written: a member that is not listed, has another checksum, or would land
outside data/ stops the installation, and a file already present with the recorded checksum is kept. Nothing in data/ is
overwritten by a file that fails its check.

Bundle 1 holds the tier-B catalogues and the posterior samples of the kinematic fits, about 250 MB (docs/release.md).
Run: .venv/bin/python scripts/fetch_release.py [--sandbox] [RECORD]   (--sandbox: the Zenodo sandbox, for testing the release)
```
</details>

*Shown, not run here: network access to Zenodo; bundle 1 is about 250 MB (docs/release.md). The cells below are the script itself; run it from the repository with* `python scripts/fetch_release.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The Zenodo record of bundle 1 (set at release), an optional direct link, and the name of the zip file.

```python
import hashlib
import sys
import tempfile
import zipfile
from pathlib import Path

import requests
import yaml

from lmcold.sources import DATA, resolve, sha256

RECORD = None   # Zenodo record id of bundle 1, set at release
URL = None   # a direct link to the bundle, used instead of RECORD (a test build of the release: an asset of the test repository)
BUNDLE = 'lmc-old-population-bundle1.zip'  # file name of the bundle in the Zenodo record
```

#### The recorded checksums
The sha256 of every file under data/ that data/sources.yaml lists, against which each member of the bundle is checked.

```python
def recorded():
    """{data/-relative path: sha256} of the files under data/ listed in data/sources.yaml: an entry's own sha256, or, for an entry made
    of many files ('per file: data/checksums/<list>'), the lines 'sha256  path' of that list (paths relative to the entry's file)."""
    out = {}
    # every entry of data/sources.yaml; p: its file (absolute), h: its sha256 field as a string
    for e in yaml.safe_load((DATA / 'sources.yaml').read_text()).values():
        p, h = resolve(str(e['file'])).resolve(), str(e.get('sha256'))
        # skip entries whose file lies outside data/ (e.g. the public tables kept elsewhere)
        if not p.is_relative_to(DATA.resolve()):
            continue
        # a single file: its 64-character hex sha256
        if len(h) == 64:
            out[str(p.relative_to(DATA.resolve()))] = h
        # many files: read the checksum list named in the field ('per file: data/checksums/<list>')
        elif h.startswith('per file: data/checksums/'):
            for line in (DATA.parent / h.split()[2]).read_text().splitlines():
                # each line: 'sha256  path' (two spaces), path relative to the entry's folder
                sha, rel = line.split('  ', 1)
                out[str((p / rel).relative_to(DATA.resolve()))] = sha
    return out
```

#### Install the bundle
Each member is extracted to a temporary directory, checked, and only then moved into data/.

```python
def install(zip_path, dest=DATA, sums=None):
    """Check every member of zip_path against sums (default: recorded()) and write the members to dest; returns the paths written."""
    # sums: {data/-relative path: sha256}
    sums = recorded() if sums is None else sums
    written = []
    with zipfile.ZipFile(zip_path) as z, tempfile.TemporaryDirectory() as tmp:
        # every member of the zip, by its path inside the archive (relative to data/)
        for name in z.namelist():
            # refuse members that would land outside dest (e.g. '../' paths)
            target = (dest / name).resolve()
            if not target.is_relative_to(dest.resolve()):
                raise ValueError(f'{name}: outside {dest}')
            # refuse members that sources.yaml does not list
            if name not in sums:
                raise ValueError(f'{name}: not listed in data/sources.yaml')
            # extract to the temporary directory and check before anything is written to dest
            staged = Path(z.extract(name, tmp))
            if sha256(staged) != sums[name]:
                raise ValueError(f'{name}: sha256 differs from data/sources.yaml')
            # a file already in place with the recorded checksum is kept
            if target.exists() and sha256(target) == sums[name]:
                continue
            # move the checked member into place
            target.parent.mkdir(parents=True, exist_ok=True)
            staged.replace(target)
            written.append(target)
    return written
```

#### Download the bundle
From a direct link, or from the Zenodo record with the md5 that Zenodo gives checked.

```python
def fetch(url):
    """Path of the bundle downloaded from `url` (install() then checks every member against data/sources.yaml)."""
    # written to the system temporary folder
    out = Path(tempfile.gettempdir()) / BUNDLE
    with requests.get(url, stream=True, timeout=60) as r, open(out, 'wb') as fo:
        r.raise_for_status()
        # blocks of 1 << 20 bytes = 1 MiB
        for block in r.iter_content(1 << 20):
            fo.write(block)
    return out


def download(record, sandbox):
    """Path of the bundle downloaded from Zenodo record `record` (md5 checked against the record)."""
    # Zenodo records API; the sandbox has its own host
    api = f"https://{'sandbox.' if sandbox else ''}zenodo.org/api/records/{record}"
    # the bundle's entry in the record's file list (its download link and 'md5:<hex>' checksum)
    f = next(f for f in requests.get(api, timeout=60).json()['files'] if f['key'] == BUNDLE)
    out = Path(tempfile.gettempdir()) / BUNDLE
    # md5 accumulated while streaming, compared with the record afterwards
    md5 = hashlib.md5()
    with requests.get(f['links']['self'], stream=True, timeout=60) as r, open(out, 'wb') as fo:
        r.raise_for_status()
        for block in r.iter_content(1 << 20):
            fo.write(block); md5.update(block)
    if 'md5:' + md5.hexdigest() != f['checksum']:
        raise ValueError('download md5 differs from the Zenodo record')
    return out
```

#### Run
Download from the record given on the command line (or RECORD, or URL), then install.

```python
# command-line arguments other than the --sandbox flag; the first, if any, is the record id
args = [a for a in sys.argv[1:] if a != '--sandbox']
record = args[0] if args else RECORD
# neither a record nor a direct link: stop with a message
if record is None and URL is None:
    sys.exit('give the Zenodo record id of bundle 1')
# download from Zenodo if a record is known, else from URL; then check and install every member
path = download(record, '--sandbox' in sys.argv) if record else fetch(URL)
print(f'{len(install(path))} files written to {DATA}')
```

### The full route: the public archives

The sections below follow the tiers of `data/sources.yaml` source by source. Each fetch script prints the sha256 of what it wrote,
to be compared with the entry of `data/sources.yaml`. Large files (survey summary catalogues, simulation snapshots) are written to
`~/data`, outside the repository; small ones go to `data/`.

## Surveys

### OGLE-IV RR Lyrae

The RR Lyrae sample starts from the OGLE-IV Collection of Variable Stars of the LMC
([Soszyński et al. 2016](https://doi.org/10.48550/arXiv.1606.02727)): the identifications and positions of all RR Lyrae, and the
$I$-band light-curve parameters of the RRab stars. `scripts/build_rrab.py` reads both.

<details><summary>About <code>scripts/fetch_ogle4_rrab.py</code></summary>

```text
Download the OGLE-IV LMC RR Lyrae identification and RRab parameter files (Soszynski et al. 2016) into data/raw/ogle4/.

sources.yaml entries written: ogle4_ident (identifications and positions of all LMC RR Lyrae) and ogle4_rrab (I-band light-curve
parameters of the RRab stars). Both are read by scripts/build_rrab.py (through lmcold.ogle).

Needs network access to the OGLE web server.
Run: .venv/bin/python scripts/fetch_ogle4_rrab.py
```
</details>

*Shown, not run here: network access to the OGLE web server. The cells below are the script itself; run it from the repository with* `python scripts/fetch_ogle4_rrab.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import urllib.request

from lmcold.sources import DATA, sha256
```

#### Download
Two files of the OGLE-IV Collection of Variable Stars, LMC RR Lyrae, each with its sha256 printed.

```python
URL = "https://www.astrouw.edu.pl/ogle/ogle4/OCVS/lmc/rrlyr/"   # OGLE-IV OCVS directory of the LMC RR Lyrae

out = DATA / "raw" / "ogle4"
out.mkdir(parents=True, exist_ok=True)
for name in ("ident.dat", "RRab.dat"):
    urllib.request.urlretrieve(URL + name, out / name)
    print(name, sha256(out / name))
```

### OGLE-IV fields

The lists of OGLE-IV field centres ([Udalski et al. 2015](https://doi.org/10.48550/arXiv.1504.05966)) define the footprint of the
RR Lyrae completeness estimate (`lmcold.ogle`).

<details><summary>About <code>scripts/fetch_ogle4_fields.py</code></summary>

```text
Download the OGLE-IV field centre lists for the LMC and the Magellanic Bridge (Udalski et al. 2015) into data/raw/ogle4/.

sources.yaml entries written: ogle4_fields_lmc and ogle4_fields_mbr (tier A). The field centres define the OGLE-IV footprint
used for the RRab completeness (lmcold.ogle.field_centres, in_footprint). Note: data/raw/ogle4/ must exist (created by
fetch_ogle4_rrab.py).

Needs network access to the OGLE web server; two small text files.
Run: .venv/bin/python scripts/fetch_ogle4_fields.py
```
</details>

*Shown, not run here: network access to the OGLE web server; run after fetch_ogle4_rrab.py, which creates data/raw/ogle4/. The cells below are the script itself; run it from the repository with* `python scripts/fetch_ogle4_fields.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import urllib.request

from lmcold.sources import DATA, sha256
```

#### Download
The field-centre list of the LMC and of the Magellanic Bridge, each with its sha256 printed.

```python
# (file suffix, directory on the OGLE sky pages); the Bridge directory is ogle4-MB but the file is named MBR.
for region, page in (("LMC", "ogle4-LMC"), ("MBR", "ogle4-MB")):
    out = DATA / "raw" / "ogle4" / f"fields-OGLE-IV-{region}.txt"
    with urllib.request.urlopen(f"https://ogle.astrouw.edu.pl/sky/{page}/fields-OGLE-IV-{region}.txt", timeout=30) as r:
        out.write_bytes(r.read())
    print(out.name, sha256(out))
```

### *Gaia* DR3 RR Lyrae

Two queries of the *Gaia* archive ([Gaia Collaboration 2023](https://doi.org/10.1051/0004-6361/202243940)) give the *Gaia* part of
the data. The first selects the RR Lyrae of the DR3 variability catalogue
([Clementini et al. 2023](https://doi.org/10.1051/0004-6361/202243964)) in a cone about the LMC centre, with their astrometry; it is
joined with OGLE-IV in `scripts/build_rrab.py`.

<details><summary>About <code>scripts/fetch_gaia_rrlyrae.py</code></summary>

```text
Gaia DR3 RR Lyrae (vari_rrlyrae, Clementini et al. 2023) within 20 deg of the LMC centre, with gaia_source astrometry.

Public Gaia archive, anonymous asynchronous job. Writes data/gaia_rrlyrae_lmc20.parquet.

sources.yaml entry written: gaia_rrlyrae_lmc20 (tier A; 40,970 rows). Read by scripts/build_rrab.py, which combines it with
OGLE-IV into the release RRab catalogue (rrab.parquet, release_rrab).
Output columns: from vari_rrlyrae the classification (best_classification: RRab, RRc or RRd), fundamental and first-overtone
periods pf, p1_o [d], the G-band Fourier parameters phi31_g [rad] and r21_g, peak_to_peak_g [mag], the number of clean G epochs,
the intensity-averaged G, BP, RP magnitudes and the photometric metallicity (Clementini et al. 2023) with its error; from
gaia_source the astrometry (ra, dec [deg], parallax [mas], pmra, pmdec [mas/yr] with errors and correlation), ruwe and the mean
G, BP, RP magnitudes.

Needs network access to the Gaia archive; no run time is recorded.
Run: .venv/bin/python scripts/fetch_gaia_rrlyrae.py
```
</details>

*Shown, not run here: network access to the Gaia archive. The cells below are the script itself; run it from the repository with* `python scripts/fetch_gaia_rrlyrae.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
from astroquery.gaia import Gaia

from lmcold.constants import LMC
from lmcold.sources import DATA, sha256
```

#### The query
Gaia DR3 vari_rrlyrae joined with gaia_source on source_id, in a cone of 20 deg about the LMC centre.

```python
# WHERE: the source lies within 20 deg of the adopted LMC centre (constants.LMC ra0, dec0 in deg); no other cut, so every
# Gaia DR3 RR Lyrae type (RRab, RRc, RRd) and the SMC side of the cone are kept and selected later in build_rrab.py.
QUERY = f"""
SELECT v.source_id, v.best_classification, v.pf, v.pf_error, v.p1_o, v.phi31_g, v.phi31_g_error, v.r21_g,
       v.peak_to_peak_g, v.num_clean_epochs_g, v.int_average_g, v.int_average_bp, v.int_average_rp,
       v.metallicity, v.metallicity_error,
       g.ra, g.dec, g.parallax, g.parallax_error, g.pmra, g.pmra_error, g.pmdec, g.pmdec_error, g.pmra_pmdec_corr,
       g.ruwe, g.phot_g_mean_mag, g.phot_bp_mean_mag, g.phot_rp_mean_mag
FROM gaiadr3.vari_rrlyrae AS v JOIN gaiadr3.gaia_source AS g USING (source_id)
WHERE 1 = CONTAINS(POINT(g.ra, g.dec), CIRCLE({LMC['ra0']}, {LMC['dec0']}, 20))
"""
```

#### Run it and save
One anonymous asynchronous job; the rows are saved as data/gaia_rrlyrae_lmc20.parquet and its sha256 printed.

```python
# Submit the ADQL query as an asynchronous job, wait, and convert the result to a pandas DataFrame
t = Gaia.launch_job_async(QUERY).get_results().to_pandas()
# Save under data/ and print the row count and sha256 (to compare with sources.yaml)
out = DATA / "gaia_rrlyrae_lmc20.parquet"
t.to_parquet(out, index=False)
print(len(t), "rows;", sha256(out))
```

### *Gaia* DR3 astrometry of the giants

The second query fetches the astrometry and $G$ magnitude of every spectroscopic giant, uploading the source identifiers in batches;
`scripts/build_giants.py` joins it to the spectra.

<details><summary>About <code>scripts/fetch_gaia_giants.py</code></summary>

```text
Gaia DR3 astrometry and photometry for the spectroscopic giants, from the public Gaia archive (anonymous TAP upload of the
source_ids, in batches). Reads data/spec_giants.parquet (column source_id; written by scripts/build_spec_giants.py from the public
SDSS files); writes data/gaia_giants.parquet.

sources.yaml: data/spec_giants.parquet is listed as release_spec_giants; data/gaia_giants.parquet (release_gaia_giants) is what
scripts/build_giants.py joins to the spectra to produce giants_apogee.parquet and giants_boss.parquet (release_giants_apogee,
release_giants_boss; both list this script among their builders).
Output columns: those of COLS (Gaia DR3 gaia_source: ra, dec [deg]; parallax and errors [mas]; pmra, pmdec and errors [mas/yr];
pmra_pmdec_corr; ruwe; phot_g_mean_mag [mag]). Source ids with no Gaia DR3 row are simply absent.

Needs network access to the Gaia archive (one asynchronous job per batch); no run time is recorded.
Run: .venv/bin/python scripts/fetch_gaia_giants.py   (after build_spec_giants.py)
```
</details>

*Shown, not run here: network access to the Gaia archive, and data/spec_giants.parquet from scripts/build_spec_giants.py. The cells below are the script itself; run it from the repository with* `python scripts/fetch_gaia_giants.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import numpy as np
import pandas as pd
from astropy.table import Table
from astroquery.gaia import Gaia

from lmcold.sources import DATA, sha256
```

#### Settings
The Gaia DR3 columns fetched and the number of source_ids per uploaded table.

```python
# gaia_source columns fetched for each giant (units as in Gaia DR3: deg, mas, mas/yr, mag).
COLS = "source_id, ra, dec, parallax, parallax_error, pmra, pmdec, pmra_error, pmdec_error, pmra_pmdec_corr, ruwe, phot_g_mean_mag"
BATCH = 50000  # source_ids per uploaded table (one asynchronous job per batch)
```

#### Load the data
The Gaia DR3 source_ids of the spectroscopic giants, each once.

```python
ids = np.unique(pd.read_parquet(DATA / "spec_giants.parquet").source_id.to_numpy())
ids = ids[ids > 0]  # valid Gaia DR3 source_ids are positive; anything else is a spectrum without a Gaia id
```

#### Query the Gaia archive
The ids are uploaded in batches and joined to gaia_source; the rows of all batches are saved together.

```python
# One DataFrame of Gaia rows per batch.
out = []
for k in range(0, len(ids), BATCH):
    # Upload the batch as TAP_UPLOAD.ids and inner-join it to gaia_source on source_id (no other condition).
    up = Table({"source_id": ids[k:k + BATCH]})
    job = Gaia.launch_job_async(f"SELECT {', '.join('g.' + c.strip() for c in COLS.split(','))} FROM tap_upload.ids AS u "
                                "JOIN gaiadr3.gaia_source AS g ON g.source_id = u.source_id",
                                upload_resource=up, upload_table_name="ids", verbose=False)
    out.append(job.get_results().to_pandas())
    print(f"batch {k // BATCH}: {len(out[-1])} rows", flush=True)
# All batches in one table, written with its sha256.
g = pd.concat(out, ignore_index=True)
g.to_parquet(DATA / "gaia_giants.parquet", index=False)
print(len(g), "rows of", len(ids), "ids;", sha256(DATA / "gaia_giants.parquet"))
```

### APOGEE DR17 and SDSS-V DR20 spectra

The giants' spectra come from two summary catalogues of the SDSS Science Archive Server: the APOGEE DR17 allStarLite file
([Abdurro'uf et al. 2022](https://doi.org/10.3847/1538-4365/ac4414)) and the SDSS-V DR20 Astra BOSSNet summary. Where the release
publishes a SHA-1 checksum, the download is checked against it.

<details><summary>About <code>scripts/fetch_giants.py</code></summary>

```text
Download the public giant catalogues into ~/data (large files stay outside the repository): APOGEE DR17 allStarLite
(ASPCAP; Abdurro'uf et al. 2022) and the SDSS-V DR20 Astra BOSSNet summary (astraAllStarBossNet 0.8.1, SHA-1 from the release).
Prints sha256 for data/sources.yaml.

sources.yaml entries written: apogee_dr17_allstarlite and boss_dr20_bossnet (tier A). Both are read by
scripts/build_spec_giants.py (spectroscopic giant catalogue); allStarLite also by build_apogee_parent.py and build_giant_mass.py,
the BOSSNet summary by build_boss_observed.py. The allStarLite release gives no checksum file, so only the BOSSNet summary
is checked against its release SHA-1; lmcold.download.fetch resumes an interrupted download.

Needs network access to data.sdss.org and room for both files in ~/data; no size or run time is recorded.
Run: .venv/bin/python scripts/fetch_giants.py
```
</details>

*Shown, not run here: network access to data.sdss.org and disk space in ~/data for two large summary catalogues. The cells below are the script itself; run it from the repository with* `python scripts/fetch_giants.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
from pathlib import Path

from lmcold.download import fetch
from lmcold.sources import sha256
```

#### The files
Local path, URL on the SDSS Science Archive Server (SAS) and, where the release gives one, the SHA-1 to check against.

```python
SAS = "https://data.sdss.org/sas"   # root URL of the SDSS Science Archive Server
# local path -> (URL, release SHA-1 or None)
FILES = {
    "~/data/sdss/dr17/apogee/allStarLite-dr17-synspec_rev1.fits":
        (f"{SAS}/dr17/apogee/spectro/aspcap/dr17/synspec_rev1/allStarLite-dr17-synspec_rev1.fits", None),
    "~/data/sdss/dr20/astra/0.8.1/astraAllStarBossNet-0.8.1.fits.gz":
        (f"{SAS}/dr20/spectro/astra/0.8.1/summary/astraAllStarBossNet-0.8.1.fits.gz", "371090ec4cef023f59099926b99a81f2217cf7f8"),
}
```

#### Download
Each file is downloaded (resuming an interrupted download), checked against its SHA-1 where given, and its sha256 printed.

```python
for out, (url, sha1) in FILES.items():
    out = Path(out).expanduser()   # ~ -> home folder
    fetch(url, out, sha1)   # download (resumed if partial); checked against sha1 when it is not None
    # Path, size [bytes] and sha256, for data/sources.yaml.
    print(out, out.stat().st_size, "sha256", sha256(out), flush=True)
```

### Targeting records

The selection functions of the chapter on selection need the targeting records as well: for BOSS, the spAll-lite file (the carton
under which each fibre was assigned); for APOGEE, the allVisit file and the APOGEE-2 targeting files of the LMC fields.

<details><summary>About <code>scripts/fetch_targeting.py</code></summary>

```text
Download the public targeting inputs of the giant selection functions into ~/data:
SDSS-V DR20 mwmTargets (carton memberships of observed targets, Astra 0.8.1, SHA-1 from the release) and spAll-lite (BOSS
spectra with the carton each fibre was assigned under, v6_2_1); APOGEE DR17 allVisit (per-visit targeting flags) and the
APOGEE-2 targeting files of the LMC fields (apogee2Field, apogee2Object_LMC1-35). Prints sha256 for data/sources.yaml.

sources.yaml entries written: sdss5_mwmtargets, sdss5_spall_lite, apogee_dr17_allvisit, apogee2_field and
apogee2_object_lmc1 ... apogee2_object_lmc35 (tier A). spAll-lite is read by build_boss_observed.py (BOSS selection
function); allVisit, apogee2Field and apogee2Object by build_apogee_parent.py (APOGEE selection function; allVisit also by
apogee_selection.py). No script in this repository reads mwmTargets at present. Only mwmTargets has a release SHA-1 to
check against.

Needs network access to data.sdss.org and room for the files in ~/data; no size or run time is recorded.
Run: .venv/bin/python scripts/fetch_targeting.py
```
</details>

*Shown, not run here: network access to data.sdss.org and disk space in ~/data for the targeting summaries. The cells below are the script itself; run it from the repository with* `python scripts/fetch_targeting.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
from pathlib import Path

import requests

from lmcold.download import fetch
from lmcold.sources import sha256
```

#### The files
The release SHA-1 list of the Astra 0.8.1 summaries, then local path, URL and SHA-1 (where there is one) of each file.

```python
SAS = "https://data.sdss.org/sas"
# Astra 0.8.1 summary checksum file: lines '<sha1>  <name>' -> {name: sha1}.
sha = {l.split()[1]: l.split()[0] for l in requests.get(f"{SAS}/dr20/spectro/astra/0.8.1/summary/spectro_astra_0.8.1_summary.sha1sum",
                                                        timeout=60).text.splitlines()}
# local path -> (URL, release SHA-1 or None)
FILES = {
    "~/data/sdss/dr20/astra/0.8.1/mwmTargets-0.8.1.fits.gz":
        (f"{SAS}/dr20/spectro/astra/0.8.1/summary/mwmTargets-0.8.1.fits.gz", sha["mwmTargets-0.8.1.fits.gz"]),
    "~/data/sdss/dr20/boss/spAll-lite-v6_2_1.fits.gz": (f"{SAS}/dr20/spectro/boss/redux/v6_2_1/summary/daily/spAll-lite-v6_2_1.fits.gz", None),
    "~/data/sdss/dr17/apogee/allVisit-dr17-synspec_rev1.fits":
        (f"{SAS}/dr17/apogee/spectro/aspcap/dr17/synspec_rev1/allVisit-dr17-synspec_rev1.fits", None),
    "~/data/sdss/dr17/apogee/target/apogee2Field.fits": (f"{SAS}/dr17/apogee/target/apogee2Field.fits", None),
}
FILES.update({f"~/data/sdss/dr17/apogee/target/apogee2Object_LMC{i}.fits": (f"{SAS}/dr17/apogee/target/apogee2Object/apogee2Object_LMC{i}.fits", None)
              for i in range(1, 36)})
```

#### Download
The 36 small APOGEE-2 targeting files first, then the large summary files; each sha256 is printed.

```python
for out, (url, sha1) in sorted(FILES.items(), key=lambda kv: "apogee2" not in kv[0]):  # small targeting files first
    out = Path(out).expanduser()
    fetch(url, out, sha1)
    print(out, out.stat().st_size, "sha256", sha256(out), flush=True)
```

### SDSS-V targeting database

Five tables of the SDSS-V targeting database give every *Gaia* star of the BOSS parent its carton memberships and priorities, and
the XP metallicities that the SDSS-V cartons were selected on; `scripts/build_boss_cartons.py` reads them.

<details><summary>About <code>scripts/fetch_targetdb.py</code></summary>

```text
Download the SDSS-V DR20 targeting database tables (targetdb 2.0.0, parquet; data.sdss.org/sas/dr20/mos/target/2.0.0/) needed to
give every Gaia star its carton memberships and priorities: mos_carton (names, plans), mos_carton_to_target (carton, target,
priority), mos_target (target -> catalogid), mos_catalog_to_gaia_dr3_source (catalogid -> Gaia DR3 source_id),
mos_xpfeh_gaia_dr3 (the Andrae et al. 2023 XP [M/H] used by the SDSS-V XP cartons). SHA-1 from the
release. Into ~/data/sdss/dr20/targetdb/; prints sha256 for data/sources.yaml.

Read by scripts/build_boss_cartons.py (carton memberships and XP [M/H] of the BOSS parent stars). sources.yaml entries:
sdss5_targetdb_<table> for each of the five tables (tier A).

Needs network access to data.sdss.org and room for the tables in ~/data; no size or run time is recorded.
Run: .venv/bin/python scripts/fetch_targetdb.py
```
</details>

*Shown, not run here: network access to data.sdss.org and disk space in ~/data for five targeting-database tables. The cells below are the script itself; run it from the repository with* `python scripts/fetch_targetdb.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
from pathlib import Path

import requests

from lmcold.download import fetch
from lmcold.sources import sha256
```

#### Download and check
The release checksum file gives the SHA-1 of each table; each table is downloaded, checked and its sha256 printed.

```python
URL = "https://data.sdss.org/sas/dr20/mos/target/2.0.0/parquet"
# Release checksum file: lines '<sha1>  <path>/<name>.parquet' -> {name.parquet: sha1}; fetch checks each download.
sha = {l.split()[1].split("/")[-1]: l.split()[0] for l in requests.get(f"{URL}/mos_target_parquet_2.0.0.sha1sum", timeout=60).text.splitlines()}
for name in ("mos_carton", "mos_catalog_to_gaia_dr3_source", "mos_xpfeh_gaia_dr3", "mos_carton_to_target", "mos_target"):
    out = Path(f"~/data/sdss/dr20/targetdb/{name}.parquet").expanduser()
    fetch(f"{URL}/{name}.parquet", out, sha[f"{name}.parquet"])
    print(out, out.stat().st_size, "sha256", sha256(out), flush=True)
```

### 2MASS

The infrared test of the BOSS parent sample in regions of high extinction uses the 2MASS Point Source Catalog
([Skrutskie et al. 2006](https://doi.org/10.1086/498708)) near the LMC centre, queried from the IRSA archive.

<details><summary>About <code>scripts/fetch_twomass_centre.py</code></summary>

```text
2MASS Point Source Catalog within 5 deg of the LMC centre, 12 <= H <= 14, from the public IRSA TAP service (table fp_psc).
For the infrared test of the BOSS parent in high extinction (scripts/paper_parent_extinction.py). Writes data/twomass_centre.parquet.

sources.yaml entry written: twomass_centre (259,488 rows). Columns: ra, dec [deg]; j_m, h_m, k_m (2MASS J, H, Ks [mag]);
ph_qual (photometric quality flags, JHK) and cc_flg (contamination and confusion flags).

Needs network access to the IRSA TAP service; no run time is recorded.
Run: .venv/bin/python scripts/fetch_twomass_centre.py
```
</details>

*Shown, not run here: network access to the IRSA TAP service. The cells below are the script itself; run it from the repository with* `python scripts/fetch_twomass_centre.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import pyvo

from lmcold.constants import LMC
from lmcold.sources import DATA, sha256
```

#### Query and save
One asynchronous TAP query of the 2MASS Point Source Catalog; the rows are saved as data/twomass_centre.parquet.

```python
# WHERE: 12 <= H <= 14 mag (both ends inclusive) and within 5 deg of the adopted LMC centre (constants.LMC ra0, dec0).
QUERY = (f"SELECT ra, dec, j_m, h_m, k_m, ph_qual, cc_flg FROM fp_psc WHERE h_m BETWEEN 12 AND 14 AND "
         f"CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {LMC['ra0']}, {LMC['dec0']}, 5.0)) = 1")
t = pyvo.dal.TAPService("https://irsa.ipac.caltech.edu/TAP").run_async(QUERY, maxrec=5_000_000).to_table().to_pandas()
t.to_parquet(DATA / "twomass_centre.parquet", index=False)
print(len(t), "rows;", sha256(DATA / "twomass_centre.parquet"))
```

## Published catalogues

### *Gaia* XP metallicities

The XP metallicities of [Andrae, Rix & Chandra (2023)](https://doi.org/10.3847/1538-4365/acd53e) give the metallicity of the stars
in the parent samples of the selection functions. Their table is downloaded from its Zenodo record
([doi:10.5281/zenodo.7945154](https://doi.org/10.5281/zenodo.7945154)); `scripts/build_xp_andrae.py` keeps the stars of the parents.

<details><summary>About <code>scripts/fetch_andrae23.py</code></summary>

```text
Download Table 1 of Andrae, Rix & Chandra (2023, ApJS 267, 8): XGBoost [M/H], Teff and log g from Gaia DR3 XP spectra for
175 million stars (Zenodo record 7945154, doi:10.5281/zenodo.7945154) into ~/data/gaia/andrae2023. Prints sha256 for
data/sources.yaml.

sources.yaml entry written: andrae23_table1 (tier A). Read by scripts/build_xp_andrae.py, which keeps the stars of the giant
selection-function parents and writes data/xp_andrae.parquet.
The file (~10^8 rows) stays outside the repository; lmcold.download.fetch resumes an interrupted download. The Zenodo record gives
an MD5 (checked by hand, see sources.yaml), not a SHA-1, so no checksum is passed to fetch here.

Needs network access to zenodo.org and room for the file in ~/data; no run time is recorded.
Run: .venv/bin/python scripts/fetch_andrae23.py
```
</details>

*Shown, not run here: network access to Zenodo and disk space in ~/data for a table of about 10^8 rows. The cells below are the script itself; run it from the repository with* `python scripts/fetch_andrae23.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
from pathlib import Path

from lmcold.download import fetch
from lmcold.sources import sha256
```

#### Download the table
One file from the Zenodo record into ~/data/gaia/andrae2023, then its size [bytes] and sha256 are printed for
data/sources.yaml.

```python
# Destination outside the repository; the path matches the `file` field of andrae23_table1.
out = Path("~/data/gaia/andrae2023/table_1_catwise.fits.gz").expanduser()
fetch("https://zenodo.org/records/7945154/files/table_1_catwise.fits.gz", out)
print(out, out.stat().st_size, "sha256", sha256(out), flush=True)
```

### Reddening map

The red-clump reddening map of the Magellanic Clouds of [Skowron et al. (2021)](https://doi.org/10.3847/1538-4365/abcb81) gives the
reddening of the BOSS parent in the test of extinction.

<details><summary>About <code>scripts/fetch_skowron21.py</code></summary>

```text
Download the OGLE-IV red-clump E(V-I) map of the Magellanic Clouds (Skowron et al. 2021, ApJS 252, 23) from CDS
(J/ApJS/252/23, table1.dat.gz; format in the catalogue ReadMe) into data/raw/skowron21/.

sources.yaml entry written: skowron21_table1 (81,684 rows). The map gives the reddening of the BOSS parent in the infrared test
of figures/paper_parent_extinction.png (scripts/paper_parent_extinction.py, via lmcold.extinction).

Needs network access to CDS.
Run: .venv/bin/python scripts/fetch_skowron21.py
```
</details>

*Shown, not run here: network access to CDS. The cells below are the script itself; run it from the repository with* `python scripts/fetch_skowron21.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import urllib.request

from lmcold.sources import DATA, sha256
```

#### Download
The gzipped table of the reddening map, with its sha256 printed.

```python
for name in ("table1.dat.gz",):
    # Local path data/raw/skowron21/<name>; fetched from the CDS FTP area of J/ApJS/252/23 (NAOJ mirror), its sha256 printed.
    out = DATA / "raw" / "skowron21" / name
    out.parent.mkdir(parents=True, exist_ok=True)
    # Download (timeout 120 s), write the bytes and print the file's sha256.
    with urllib.request.urlopen(f"http://vizier.nao.ac.jp/ftp/J/ApJS/252/23/{name}", timeout=120) as r:
        out.write_bytes(r.read())
    print(name, sha256(out))
```

### Spectroscopic metallicities of RR Lyrae

The photometric metallicities of the RR Lyrae are compared with spectroscopic ones: [Gratton et al. (2004)](https://doi.org/10.1051/0004-6361:20035840),
with positions from [Di Fabrizio et al. (2005)](http://vizier.nao.ac.jp/ftp/J/A+A/430/603/), and
[Borissova et al. (2004](https://doi.org/10.1051/0004-6361:20034494), [2006)](http://vizier.nao.ac.jp/ftp/J/A+A/460/459/), whose
stars are located through SIMBAD.

<details><summary>About <code>scripts/fetch_spec_rrl.py</code></summary>

```text
Spectroscopic [Fe/H] of LMC RR Lyrae: static CDS files (data + ReadMe) into data/raw/spec/<catalogue>/.

Borissova et al. 2004 (J/A+A/423/97) and 2006 (J/A+A/460/459): positions and [Fe/H].
Gratton et al. 2004 (J/A+A/421/937): [Fe/H] by star name; positions from Di Fabrizio et al. 2005 (J/A+A/430/603, field A in
table5, field B in table6).

sources.yaml entries written: spec_borissova04(_readme), spec_borissova06(_readme), spec_gratton04(_readme),
spec_difabrizio05_A, spec_difabrizio05_B and spec_simbad_macho. The Di Fabrizio ReadMe is downloaded too but not registered.
They are read through lmcold.spectra; these spectroscopic [Fe/H] are the reference for the RRab photometric [Fe/H] scales
(figures/paper_rrab_feh.png, scripts/paper_rrab_feh.py).

Needs network access to CDS and SIMBAD.
Run: .venv/bin/python scripts/fetch_spec_rrl.py
```
</details>

*Shown, not run here: network access to CDS and SIMBAD. The cells below are the script itself; run it from the repository with* `python scripts/fetch_spec_rrl.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import urllib.request

from lmcold.sources import DATA, sha256
```

#### The CDS files
The data table(s) and ReadMe of each catalogue, into data/raw/spec/<catalogue>/.

```python
# CDS catalogue -> data files (from the VizieR mirror at NAOJ); each directory also gets its ReadMe (CDS byte-by-byte format).
FILES = {"J/A+A/423/97": ["macho.dat"], "J/A+A/460/459": ["table9.dat"], "J/A+A/421/937": ["table7.dat"],
         "J/A+A/430/603": ["table5.dat", "table6.dat"]}

for cat, names in FILES.items():
    # e.g. J/A+A/423/97 -> data/raw/spec/J_AA_423_97
    out = DATA / "raw" / "spec" / cat.replace("/", "_").replace("+", "")
    out.mkdir(parents=True, exist_ok=True)
    for name in ("ReadMe", *names):
        with urllib.request.urlopen(f"http://vizier.nao.ac.jp/ftp/{cat}/{name}", timeout=30) as r:
            (out / name).write_bytes(r.read())
        print(out.name, name, sha256(out / name))
```

#### Positions from SIMBAD
The stars of Borissova et al. (2004, 2006) are named, not positioned; SIMBAD gives their positions.

```python
# Borissova et al. identify stars by MACHO name only: resolve them in SIMBAD (main_id is the OGLE-IV name for most).
from astropy.io import ascii  # noqa: E402
from astroquery.simbad import Simbad  # noqa: E402

names = []
for cat, f in (("J_AA_423_97", "macho.dat"), ("J_AA_460_459", "table9.dat")):
    d = DATA / "raw" / "spec" / cat
    t = ascii.read(d / f, readme=d / "ReadMe", format="cds")
    # Borissova+04 lists MACHO ids (queried as 'MACHO <id>'); Borissova+06 gives full names in column Name.
    names += [f"MACHO {n}" if cat == "J_AA_423_97" else str(n) for n in t["MACHO" if cat == "J_AA_423_97" else "Name"]]
Simbad.TIMEOUT = 30
Simbad.server = "simbad.harvard.edu"   # SIMBAD mirror at Harvard (the same database as simbad.cds.unistra.fr)
# Output columns: user_specified_id (the name queried), main_id, ra, dec (SIMBAD ICRS, deg).
r = Simbad.query_objects(sorted(set(names)))[["user_specified_id", "main_id", "ra", "dec"]].to_pandas()
out = DATA / "raw" / "spec" / "simbad_macho.csv"
r.to_csv(out, index=False)
print(out.name, len(r), sha256(out))
```

### Star clusters

The LMC clusters with Ca II triplet metallicities of [Grocholski et al. (2006)](https://doi.org/10.1086/507303), and the *Gaia* DR3
stars around each, are the clusters of the mass comparison in the chapter on stellar mass.

<details><summary>About <code>scripts/fetch_clusters.py</code></summary>

```text
LMC star clusters with Ca II triplet [Fe/H] (Grocholski et al. 2006, VizieR J/AJ/132/1630, 2006AJ....132.1630G) and the
Gaia DR3 sources within RADIUS of each, G <= 18, from the public Gaia archive TAP service. For the colour calibration of the
isochrones on the giant branch (scripts/fig_cluster_colours.py). Writes data/clusters_grocholski06.csv and data/clusters_gaia.parquet.

sources.yaml entries written: clusters_grocholski06 (VizieR table, one row per cluster, with columns `feh`, `e_feh`, `ra`, `dec`
in deg added from the sexagesimal RAJ2000/DEJ2000) and clusters_gaia (Gaia DR3 sources with G <= 18 within RADIUS of each
cluster; column `cluster` gives the Grocholski name). clusters_grocholski06 is also read by scripts/paper_mass.py (cluster
[Fe/H] in panel c of figures/paper_mass.png).
Row order of clusters_gaia may differ between fetches (see sources.yaml), so its sha256 can change without a change of content.

Needs network access to VizieR and the Gaia archive (one TAP query per cluster); no run time is recorded.
Run: .venv/bin/python scripts/fetch_clusters.py
```
</details>

*Shown, not run here: network access to VizieR and the Gaia archive. The cells below are the script itself; run it from the repository with* `python scripts/fetch_clusters.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The cone radius around each cluster and the VizieR query of the cluster table.

```python
import io
import urllib.request

import pandas as pd
import pyvo
from astropy.coordinates import SkyCoord
import astropy.units as u

from lmcold.sources import DATA, sha256

RADIUS = 6. / 60  # deg; clusters have diameters of 0.8-4.7 arcmin, the rest is the field annulus
# VizieR ASU query of table1 of J/AJ/132/1630: all columns, tab-separated, at most 100 rows (the table has 29).
URL = "https://vizier.cds.unistra.fr/viz-bin/asu-tsv?-source=J/AJ/132/1630/table1&-out.all&-out.max=100"
```

#### The cluster table
Table 1 of Grocholski et al. (2006) from VizieR: [Fe/H] of each cluster and its position, converted to degrees.

```python
# VizieR TSV: comment lines start with '#'; then a header line, a units line and a dashes line before the data rows.
raw = urllib.request.urlopen(URL).read().decode()
lines = [x for x in raw.splitlines() if x and not x.startswith('#')]
c = pd.read_csv(io.StringIO('\n'.join([lines[0]] + lines[3:])), sep='\t')
c.columns = [x.strip() for x in c.columns]
# Each column becomes numeric if every non-empty value parses as a number; otherwise it stays a stripped string.
for col in c:
    v = c[col].astype(str).str.strip().replace('', None)
    num = pd.to_numeric(v, errors='coerce')
    c[col] = num if num.notna().sum() == v.notna().sum() else v
c = c.rename(columns={'[Fe/H]': 'feh', 'e_[Fe/H]': 'e_feh'})
pos = SkyCoord(c.RAJ2000, c.DEJ2000, unit=(u.hourangle, u.deg))
c['ra'], c['dec'] = pos.ra.deg, pos.dec.deg
c.to_csv(DATA / 'clusters_grocholski06.csv', index=False)
```

#### Gaia DR3 stars around each cluster
One cone search per cluster on the Gaia archive; the stars are labelled with their cluster's name and saved.

```python
# One cone search per cluster on the public Gaia archive TAP service (synchronous queries).
tap = pyvo.dal.TAPService("https://gea.esac.esa.int/tap-server/tap")
parts = []
for r in c.itertuples():
    # WHERE: G <= 18 (giant-branch stars of the clusters; the limit of the stored sample) and the source lies within RADIUS deg
    # of the cluster centre (CONTAINS(POINT, CIRCLE) = 1 is the ADQL cone test).
    q = ("SELECT source_id, ra, dec, phot_g_mean_mag, phot_bp_mean_mag, phot_rp_mean_mag, phot_bp_rp_excess_factor, pmra, pmdec, "
         "parallax, parallax_error, ruwe FROM gaiadr3.gaia_source WHERE phot_g_mean_mag <= 18 AND "
         f"CONTAINS(POINT('ICRS', ra, dec), CIRCLE('ICRS', {r.ra}, {r.dec}, {RADIUS})) = 1")
    t = tap.search(q).to_table().to_pandas()
    parts.append(t.assign(cluster=r.Cluster))
g = pd.concat(parts, ignore_index=True)
g.to_parquet(DATA / 'clusters_gaia.parquet', index=False)
print(len(c), 'clusters;', len(g), 'Gaia rows;', sha256(DATA / 'clusters_grocholski06.csv'), sha256(DATA / 'clusters_gaia.parquet'))
```

### Published metallicity distributions

The metallicity distributions of field stars are compared with three published catalogues:
[Cole et al. (2005)](https://doi.org/10.1086/428007), [Carrera et al. (2008)](https://doi.org/10.1088/0004-6256/135/3/836) and
[Omkumar et al. (2025)](https://doi.org/10.1051/0004-6361/202452510), all from VizieR.

<details><summary>About <code>scripts/fetch_mdf_literature.py</code></summary>

```text
Published LMC field-star metallicities for the MDF comparison (scripts/paper_mdf_literature.py), from VizieR.

- Cole et al. (2005, 2005AJ....129.1465C, J/AJ/129/1465/stars): Ca II triplet [Fe/H] of 373 bar giants.
- Carrera et al. (2008, 2008AJ....135..836C, J/AJ/135/836/table1): Ca II triplet index (sum of the three equivalent widths), V, I and
  radial velocity of 702 stars in four fields 3-8 deg north of the bar (no [Fe/H] in the table; paper_mdf_literature.py applies
  their equation 1).
- Omkumar et al. (2025, 2025A&A...700A..74O, J/A+A/700/A74/lmc-rgb): Stroemgren photometric [Fe/H] from Gaia DR3 XP of LMC red giants.
Fetched from the VizieR mirror at IDIA (the Strasbourg server did not respond, 2026-10-28); same catalogues.
Writes data/mdf_cole05.csv, data/mdf_carrera08.csv and data/mdf_omkumar25.parquet.

sources.yaml entries written: mdf_cole05, mdf_carrera08 and mdf_omkumar25 (tier A). They serve figures/paper_mdf_literature.png
(appendix E). Columns are renamed to lower case: ra, dec [deg], feh, e_feh [dex], rv, e_rv [km/s], sum_ca, e_sum_ca [Angstrom]
(Carrera), rho [deg] (Omkumar: their angular distance from the LMC centre), source_id (Gaia DR3).
Note: sources.yaml records 635 rows for mdf_carrera08 (the VizieR table); the 702 above is the number of stars in the paper.

Needs network access to the VizieR mirror; no run time is recorded.
Run: .venv/bin/python scripts/fetch_mdf_literature.py
```
</details>

*Shown, not run here: network access to the VizieR mirror at IDIA. The cells below are the script itself; run it from the repository with* `python scripts/fetch_mdf_literature.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import io
import urllib.request

import pandas as pd

from lmcold.sources import DATA, sha256
```

#### Reading a VizieR table
One function downloads the chosen columns of a VizieR table and parses the tab-separated text.

```python
# VizieR ASU TSV query at the IDIA mirror: all rows (-out.max=unlimited) of one table, the listed columns only.
VIZIER = "https://vizier.idia.ac.za/viz-bin/asu-tsv?-source={}&-out.max=unlimited&-out={}"


def vizier(table, cols):
    """Download columns of one VizieR table as a DataFrame.

    Parameters
    ----------
    table : str
        VizieR table name, e.g. 'J/AJ/129/1465/stars' ('+' must be URL-encoded as %2B).
    cols : list of str
        VizieR column names; computed columns such as _RA, _DE (J2000 position in deg) are allowed.

    Returns
    -------
    pandas.DataFrame
        One row per catalogue row. A column is numeric if every non-empty value parses as a number, otherwise a stripped
        string with empty values as None.
    """
    raw = urllib.request.urlopen(VIZIER.format(table, ','.join(cols)), timeout=600).read().decode()
    # TSV layout: '#' comment lines, then header, units and dashes lines, then the data rows.
    lines = [x for x in raw.splitlines() if x and not x.startswith('#')]
    t = pd.read_csv(io.StringIO('\n'.join([lines[0]] + lines[3:])), sep='\t', dtype=str)
    t.columns = [x.strip() for x in t.columns]
    # Per column: numeric if every non-empty value parses, else stripped strings (empty -> None).
    for col in t:
        v = t[col].str.strip().replace('', None)
        num = pd.to_numeric(v, errors='coerce')
        t[col] = num if num.notna().sum() == v.notna().sum() else v
    return t
```

#### The three catalogues
Cole et al. (2005), Carrera et al. (2008) and Omkumar et al. (2025), with their columns renamed, saved in data/.

```python
# Cole et al. (2005): 2MASS id, field, V, I [mag], RV [km/s], CaT [Fe/H] and error [dex], J2000 position [deg].
cole = vizier('J/AJ/129/1465/stars', ['2MASS', 'Field', 'Vmag', 'Imag', 'RV', '[Fe/H]', 'e_[Fe/H]', '_RA', '_DE'])
cole = cole.rename(columns={'[Fe/H]': 'feh', 'e_[Fe/H]': 'e_feh', '_RA': 'ra', '_DE': 'dec', 'RV': 'rv'})
# Carrera et al. (2008): position [deg], CaT index sum and error [Angstrom], V, I [mag], RV and error [km/s], comment.
carrera = vizier('J/AJ/135/836/table1', ['_RAJ2000', '_DEJ2000', 'W.CaT', 'e_W.CaT', 'Vmag', 'Imag', 'RV', 'e_RV', 'Comm'])
carrera = carrera.rename(columns={'_RAJ2000': 'ra', '_DEJ2000': 'dec', 'W.CaT': 'sum_ca', 'e_W.CaT': 'e_sum_ca', 'RV': 'rv', 'e_RV': 'e_rv'})
# Omkumar et al. (2025): Gaia DR3 id, position [deg], distance from the LMC centre [deg], XP [Fe/H] and error [dex].
omk = vizier('J/A%2BA/700/A74/lmc-rgb', ['GaiaDR3', 'RA_ICRS', 'DE_ICRS', 'Dist-deg', '[Fe/H]', 'e_[Fe/H]'])
omk = omk.rename(columns={'GaiaDR3': 'source_id', 'RA_ICRS': 'ra', 'DE_ICRS': 'dec', 'Dist-deg': 'rho', '[Fe/H]': 'feh', 'e_[Fe/H]': 'e_feh'})
# Save in data/ and print row count and sha256 of each file (the numbers recorded in sources.yaml).
cole.to_csv(DATA / 'mdf_cole05.csv', index=False)
carrera.to_csv(DATA / 'mdf_carrera08.csv', index=False)
omk.to_parquet(DATA / 'mdf_omkumar25.parquet', index=False)
for f, t in [('mdf_cole05.csv', cole), ('mdf_carrera08.csv', carrera), ('mdf_omkumar25.parquet', omk)]:
    print(f, len(t), sha256(DATA / f))
```

### Published LMC masses

The total stellar masses of the LMC published by others are typed into `lmcold.constants.LMC_MASS_LITERATURE`, each with the section or
table it comes from. This script archives the source papers from arXiv, so that the table of masses (Table 4.2) can check
each source against its checksum.

<details><summary>About <code>scripts/fetch_lmc_mass_literature.py</code></summary>

```text
Archive primary references used by the LMC stellar-mass comparison table.

Numerical entries are manually transcribed with section/table references in
constants.py and docs/lmc_mass_literature.md; PDFs are retained for audit.

sources.yaml entries written: lmc_mass_reference_<name> for each key of `references` (data/raw/literature/<name>.pdf).
The numbers taken from these papers (constants.LMC_MASS_LITERATURE) are compared with this work's total mass in
figures/paper_mass_summary.png and Table 3 (scripts/paper_mass_summary.py; scripts/paper_table_mass_summary.py, which also reads
the lmc_mass_reference_* entries).

Needs network access to arxiv.org; no run time is recorded.
Run: .venv/bin/python scripts/fetch_lmc_mass_literature.py
```
</details>

*Shown, not run here: network access to arxiv.org. The cells below are the script itself; run it from the repository with* `python scripts/fetch_lmc_mass_literature.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import requests
from lmcold.sources import DATA,sha256
```

#### The papers
The arXiv identifier of each paper whose LMC stellar mass is quoted in the comparison.

```python
# file stem -> arXiv identifier (bibcodes are in sources.yaml)
references={
    'keim2023':'2207.14277',
    'mazzi2021':'2108.07225',
    'harris_zaritsky2009':'0908.1422',
    'mcconnachie2012':'1204.1562',
    'vanderMarel2002':'astro-ph/0205161',
    'eskew2012':'1204.0524',
    'skibba2012':'1210.7812',
    'madau_dickinson2014':'1403.0007',
}
```

#### Download
Each PDF goes to data/raw/literature/<name>.pdf; its sha256 is printed for data/sources.yaml.

```python
for name,identifier in references.items():
    url='https://arxiv.org/pdf/'+identifier
    p=DATA/'raw/literature'/f'{name}.pdf'
    p.parent.mkdir(parents=True,exist_ok=True)
    r=requests.get(url,timeout=45);r.raise_for_status()
    assert r.content.startswith(b'%PDF'),url  # arXiv can return an HTML page instead of the PDF
    p.write_bytes(r.content)
    print(name,url,sha256(p),flush=True)
```

### Isochrones

The conversion of giant counts into stellar mass uses PARSEC v1.2S isochrones with COLIBRI asymptotic-giant-branch tracks
([Bressan et al. 2012](https://doi.org/10.1111/j.1365-2966.2012.21948.x); [Marigo et al. 2017](https://doi.org/10.3847/1538-4357/835/1/77))
and the [Kroupa (2001)](https://doi.org/10.1046/j.1365-8711.2001.04022.x) initial mass function, in the *Gaia* and 2MASS passbands,
downloaded from the CMD web service.

<details><summary>About <code>scripts/fetch_isochrones.py</code></summary>

```text
Public PARSEC v1.2S+COLIBRI grid for the BOSS/APOGEE mass conversion.

Install the pinned ezpadova revision in pyproject.toml's isochrones extra.
Request the Gaia EDR3 or 2MASS+Spitzer passbands, Kroupa IMF, zero extinction,
log age 8.6..10.13 at 0.05 steps and [M/H] -2.2..+0.4 at 0.1 steps.
--mh requests one metallicity for public/research reproducibility checks.
Per-metallicity downloads are cached so the full fetch can resume.

The grid is downloaded from the CMD web service (https://stev.oapd.inaf.it/cgi-bin/cmd) through ezpadova. Tracks: PARSEC
parsec_CAF09_v1.2S with COLIBRI parsec_CAF09_v1.2S_S_LMC_08_web for the TP-AGB; IMF: Kroupa (tab_imf/imf_kroupa_orig.dat).
The two photometric systems give the Gaia (G, BP, RP) and 2MASS (J, H, Ks) magnitudes used in the mass conversion of the
BOSS and APOGEE giants (scripts/build_giant_mass.py, build_present_mass.py, paper_mass_method.py).

sources.yaml entries written: parsec_gaiaEDR3 (--system gaiaEDR3) and parsec_2mass_spitzer (--system 2mass_spitzer), each
data/raw/parsec/<system>.parquet with 289,596 rows (all CMD columns, one row per isochrone point; MH is [M/H], logAge is
log10 of the age in yr). A JSON sidecar <system>.json records the request. Each metallicity is cached as
data/raw/parsec/<system>/<[M/H]>.parquet; with --mh only that cache file is written (for the reproducibility check of the
research download described in sources.yaml) and the combined file is not touched.

Run: .venv/bin/python scripts/fetch_isochrones.py --system gaiaEDR3 [--mh -1.0]
Requires the `isochrones` extra of pyproject.toml (pinned ezpadova).
```
</details>

*Shown, not run here: network access to the CMD web service and the isochrones extra of pyproject.toml (pinned ezpadova); run once per photometric system. The cells below are the script itself; run it from the repository with* `python scripts/fetch_isochrones.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import argparse
import json

import ezpadova
import numpy as np
import pandas as pd

from lmcold.sources import DATA
```

#### Arguments
--system: the photometric system (Gaia EDR3 or 2MASS+Spitzer); --mh: one metallicity only.

```python
# Command-line options; the help text is the module docstring.
parser = argparse.ArgumentParser(description=__doc__)
parser.add_argument('--system', choices=['gaiaEDR3', '2mass_spitzer'], required=True)
parser.add_argument('--mh', type=float)  # one [M/H] [dex] only (cache file, no combined grid)
args = parser.parse_args()
# Per-metallicity cache folder data/raw/parsec/<system>/.
folder = DATA / 'raw/parsec' / args.system
folder.mkdir(parents=True, exist_ok=True)
```

#### Download one metallicity at a time
Each [M/H] of the grid is requested from the CMD service unless its cache file exists.

```python
parts = []  # one table per [M/H]
# [M/H] grid -2.2, -2.1, ..., +0.4 (27 values), rounded to avoid floating-point labels such as -0.30000000000000004.
for mh in ([args.mh] if args.mh is not None else np.round(np.arange(-2.2, .41, .1), 2)):
    file = folder / f'{mh:+.2f}.parquet'  # cache file named by [M/H], e.g. -1.00.parquet
    # Read from the cache if present (resumable fetch), otherwise download and cache.
    if file.exists():
        t = pd.read_parquet(file)
    else:
        # One metallicity, all ages log(age/yr) = 8.6 ... 10.1 in steps of 0.05, zero extinction (A_V = 0).
        t = ezpadova.get_isochrones(logage=(8.6, 10.13, .05), MH=(float(mh), float(mh), 0.),
                                   photsys_file=args.system, track_parsec='parsec_CAF09_v1.2S',
                                   track_colibri='parsec_CAF09_v1.2S_S_LMC_08_web',
                                   imf_file='tab_imf/imf_kroupa_orig.dat', extinction_av=0.)
        t.to_parquet(file, index=False)
    # Progress: system, [M/H], number of isochrone points, and the MH values returned by the service.
    print(args.system, mh, len(t), sorted(t.MH.unique()), flush=True)
    parts.append(t)
```

#### Combine
The full grid is written as one parquet file with a JSON record of the request.

```python
# Only the full grid is combined into the file registered in sources.yaml.
if args.mh is None:
    # data/raw/parsec/<system>.parquet, plus a <system>.json sidecar recording the request.
    output = folder.parent / f'{args.system}.parquet'
    pd.concat(parts, ignore_index=True).to_parquet(output, index=False)
    output.with_suffix('.json').write_text(json.dumps(dict(system=args.system, logage=[8.6,10.13,.05],
        requested_mh=[-2.2,.4,.1], track='PARSEC v1.2S + COLIBRI S_LMC_08', imf='Kroupa', extinction_av=0), indent=2))
```

## Simulations

The simulation chapters compare the LMC with analogues from three suites and with one pair of dark-matter-only runs. The snapshot
directories are read by `lmcold.sims` from `~/data/sims`; they are too large to hash as a whole, so the Auriga and FIRE-2 files are
checked one by one against the lists in `data/checksums/`.

### TNG50 analogues

The TNG50-1 analogues ([Pillepich et al. 2019](https://doi.org/10.1093/mnras/stz2338);
[Nelson et al. 2019a](https://doi.org/10.1093/mnras/stz2306)) are every central subhalo in the stellar- and total-mass ranges of the
LMC at $z=0$. This script downloads two star-particle cutouts of each from the TNG web service: one with positions, velocities,
masses, metallicities and formation times, and one with the birth positions and velocities. It needs a registered key for the TNG
web service.

<details><summary>About <code>scripts/fetch_tng50.py</code></summary>

```text
Download the TNG50-1 LMC analogues (paper Section 6): star-particle cutouts at z = 0 of every central subhalo with bound stellar
mass 1.5-5e9 Msun and bound total mass 5e10-5e11 Msun (Nelson et al. 2019, MNRAS 490, 3234; Pillepich et al. 2019, MNRAS 490, 3196;
API https://www.tng-project.org/api/, which needs a registered key).

Fields: STAR_FIELDS of PartType4. Each cutout goes to lmcold.sims.TNG_DIR/cutout_<id>.hdf5 (cutouts already there are kept), and the
subhalo metadata of the selected objects with a cutout (position, velocity, masses) to TNG_DIR/sample.json, which lmcold.sims reads;
sample.json is rewritten (atomically) after every new cutout, so that scripts/sims_observe.py can work on the analogues already on
disk while the rest arrive; objects the server keeps failing are left out and fetched by a rerun. A second cutout of each object
with BIRTH_FIELDS (birth position and velocity, for Fig. sims_birth) goes to lmcold.sims.TNG_BIRTH_DIR/birth_<id>.hdf5; its particles
are in the order of the first. The first
120 analogues of the paper's earlier draft were a random subset (seed 1) of the same selection.

Needs a registered TNG API key (environment variable TNG_API_KEY) and network access; the z = 0 cutouts take about 2.5 GB
(docs/release.md). No run time is recorded.
Run (the key from the macOS Keychain, never printed):
    TNG_API_KEY="$(security find-generic-password -a "$USER" -s illustris-tng-api -w)" .venv/bin/python scripts/fetch_tng50.py
```
</details>

*Shown, not run here: a registered TNG API key (environment variable TNG_API_KEY); about 2.5 GB (docs/release.md). The cells below are the script itself; run it from the repository with* `python scripts/fetch_tng50.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The API address, the particle fields of the two cutouts, the mass ranges of the analogues and the download settings.

```python
import json
import os
import threading
import time
import urllib.request
from concurrent.futures import ThreadPoolExecutor

from lmcold.sims import TNG_BIRTH_DIR, TNG_DIR, TNG_H

BASE = 'https://www.tng-project.org/api/TNG50-1/snapshots/99/subhalos/'   # subhalo endpoint of TNG50-1 at snapshot 99 (z = 0)
# PartType4 fields of the z = 0 cutout: positions, velocities, masses, element abundances, formation scale factor
STAR_FIELDS = 'Coordinates,Velocities,Masses,GFM_Metals,GFM_StellarFormationTime'
# Fields of the second cutout: ids, formation scale factor, birth position and velocity (Fig. sims_birth)
BIRTH_FIELDS = 'ParticleIDs,GFM_StellarFormationTime,BirthPos,BirthVel'
MSTAR = (1.5e9, 5e9)    # Msun, bound stellar mass of the analogues
MTOT = (5e10, 5e11)     # Msun, bound total (subhalo) mass
# Subhalo metadata kept in sample.json (TNG units as served by the API).
META = ('id', 'pos_x', 'pos_y', 'pos_z', 'vel_x', 'vel_y', 'vel_z', 'mass', 'mass_stars', 'mass_gas', 'mass_dm', 'halfmassrad_stars',
        'sfr', 'len_stars', 'starmetallicity')
WORKERS = 3   # parallel downloads
TRIES = 8     # attempts per request
LOCK = threading.Lock()   # serialises the rewriting of sample.json between threads
```

#### Requests and downloads
A request with retries, the rewriting of sample.json, and the download of the two cutouts and the metadata of one subhalo.

```python
def request(url):
    """Open url with the API key header (TNG_API_KEY): up to TRIES attempts, waiting 10 s x 2^k (at most 300 s) after a failure (the
    server answers 503 when busy)."""
    # Request with the key in the 'api-key' header; each attempt has a 600-s timeout
    req = urllib.request.Request(url, headers={'api-key': os.environ['TNG_API_KEY']})
    for attempt in range(TRIES):
        try:
            return urllib.request.urlopen(req, timeout=600)
        except OSError:
            # Give up after the last attempt; otherwise back off exponentially
            if attempt == TRIES - 1:
                raise
            time.sleep(min(10 * 2 ** attempt, 300))


def write_sample():
    """sample.json of the objects in KNOWN that have a cutout, written to a temporary file and renamed."""
    with LOCK:
        # Metadata of the objects whose z = 0 cutout is on disk, sorted by subhalo id
        meta = [KNOWN[i] for i in sorted(KNOWN) if (TNG_DIR / f'cutout_{i}.hdf5').exists()]
        tmp = TNG_DIR / 'sample.json.tmp'
        # Selection (catalogue count, query URL, number downloaded) and the subhalo list; the rename is atomic
        json.dump(dict(selection=dict(count=COUNT, query=QUERY, n_downloaded=len(meta)), subhalos=meta), open(tmp, 'w'), indent=1)
        tmp.rename(TNG_DIR / 'sample.json')


def fetch(i):
    """Metadata of subhalo i, and its cutout unless already on disk (then sample.json is rewritten); None if the server keeps
    failing (rerun to retry)."""
    # A subhalo whose requests all fail is reported and skipped
    try:
        m = _fetch(i)
    except OSError as e:
        print(f'subhalo {i}: {e}', flush=True)
        return None
    # A newly downloaded object enters KNOWN and sample.json is rewritten
    if i not in KNOWN:
        KNOWN[i] = m
        write_sample()
    return m


def _fetch(i):
    """Download the star and birth cutouts of subhalo i that are not on disk; return its metadata (from KNOWN or the API)."""
    # Both cutouts; each written to a '.part' file and renamed once complete, so a partial download is never used
    for out, fields in ((TNG_DIR / f'cutout_{i}.hdf5', STAR_FIELDS), (TNG_BIRTH_DIR / f'birth_{i}.hdf5', BIRTH_FIELDS)):
        if not out.exists():
            tmp = out.with_suffix('.part')
            with request(f'{BASE}{i}/cutout.hdf5?stars={fields}') as r, open(tmp, 'wb') as f:
                f.write(r.read())
            tmp.rename(out)
    if i in KNOWN:  # already in sample.json
        return KNOWN[i]
    # Metadata from the subhalo's API page, the META keys only
    meta = json.load(request(f'{BASE}{i}/'))
    return {k: meta[k] for k in META}
```

#### Select the analogues
A search of the z = 0 subhalo catalogue: centrals (primary_flag = 1) in both mass ranges, converted to the API's units of
1e10 Msun/h (TNG_H is the Hubble parameter h). Subhaloes already in sample.json are kept.

```python
# Search URL: masses converted from Msun to the API's 1e10 Msun/h; centrals only; up to 2000 results
TNG_BIRTH_DIR.mkdir(exist_ok=True)
QUERY = q = (f'{BASE}?mass_stars__gt={MSTAR[0] * TNG_H / 1e10}&mass_stars__lt={MSTAR[1] * TNG_H / 1e10}&mass__gt={MTOT[0] * TNG_H / 1e10}'
     f'&mass__lt={MTOT[1] * TNG_H / 1e10}&primary_flag=1&limit=2000')
# KNOWN: metadata already in sample.json, by subhalo id (empty on a first run)
old = TNG_DIR / 'sample.json'
KNOWN = {m['id']: m for m in json.load(open(old))['subhalos']} if old.exists() else {}
# Run the search; ids: the selected subhalo ids, COUNT: the number the catalogue reports
res = json.load(request(q))
ids, COUNT = sorted(r['id'] for r in res['results']), res['count']
print(f'{res["count"]} LMC-mass centrals; {sum((TNG_DIR / f"cutout_{i}.hdf5").exists() for i in ids)} already on disk', flush=True)
```

#### Download
The cutouts of every selected subhalo, WORKERS at a time; a rerun fetches the ones that failed.

```python
# Download WORKERS subhaloes at a time; failures (None) are dropped; final rewrite of sample.json
with ThreadPoolExecutor(WORKERS) as ex:
    meta = [m for m in ex.map(fetch, ids) if m is not None]
write_sample()
print(f'{len(meta)} of {len(ids)} cutouts in {TNG_DIR}' + ('' if len(meta) == len(ids) else ': rerun for the rest'))
```

### TNG50 progenitors

For the evolution of the old stars, this script follows the main progenitors of a subset of the analogues back in time: their merger
trees and cutouts of stars, gas and dark matter at earlier snapshots. It also needs the TNG key.

<details><summary>About <code>scripts/fetch_tng50_progenitors.py</code></summary>

```text
Download the main progenitors of a subset of the TNG50-1 LMC analogues through cosmic time (paper Section 6, Fig. sims_heating).

Samples (z = 0 subhalo ids, fixed here so the figure can be rebuilt):
- SAMPLE: 60 analogues, 11 drawn at random (numpy seed 7) from each quintile of today's v_phi/sigma_z of the stars older than 10 Gyr
  within 10 kpc, plus the five closest to the LMC in Fig. sims_kinematics (708636, 721624, 745872, 746429, 698969).
- FULL: 20 of them for every-snapshot cutouts: 708636, 746429 and 609461, and 17 chosen to span the fraction of birth
  v_phi/sigma_z kept by their oldest cohorts after the first resolved snapshot (0.01-0.88).
Products (all from https://www.tng-project.org/api/TNG50-1/, which needs a registered key), each file kept if already on disk:
- sims.TNG_TREE_DIR/sublink_<id>.hdf5: the SubLink tree (sublink/full.hdf5; the main branch is read by lmcold.sims.main_branch);
- sims.TNG_TRACK_DIR/<id>/snap_<s>.hdf5: stars (TRACK_FIELDS) of the main progenitor at every third snapshot TRACK_SNAPS;
- sims.TNG_GAS_DIR/<id>/gas_<s>.hdf5: gas (GAS_FIELDS) of the main progenitor at the full snapshots GAS_SNAPS;
- sims.TNG_FULL_DIR/<id>/snap_<s>.hdf5: stars, gas and dark matter (FULL_QUERY) of the main progenitor of the FULL sample at every
  snapshot FULL_SNAPS (z = 4.0-1.5);
- sims.TNG_DM_DIR/<id>/snap_<s>.hdf5: dark-matter positions (DM_QUERY) where there is no full cutout, for Fig. sims_settle: the FULL
  sample at DM_SNAPS_FULL (z = 1.4-1.0; gas positions and masses were fetched with them and are not used), the others at the track
  snapshots up to 51 (DM_SNAPS).

Needs a registered TNG API key (environment variable TNG_API_KEY) and network access; no size or run time is recorded.
Run (the key from the macOS Keychain, never printed):
    TNG_API_KEY="$(security find-generic-password -a "$USER" -s illustris-tng-api -w)" .venv/bin/python scripts/fetch_tng50_progenitors.py
```
</details>

*Shown, not run here: a registered TNG API key (environment variable TNG_API_KEY), and the z = 0 sample of fetch_tng50.py. The cells below are the script itself; run it from the repository with* `python scripts/fetch_tng50_progenitors.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The two samples of z = 0 subhalo ids, the snapshots of each product and the particle fields requested.

```python
import os
from concurrent.futures import ThreadPoolExecutor

from fetch_tng50 import request
from lmcold import sims

API = 'https://www.tng-project.org/api/TNG50-1/snapshots/'   # base URL of the TNG50-1 snapshot API
# z = 0 subhalo ids of the 60 analogues (module docstring: SAMPLE).
SAMPLE = ('691655 702198 696385 714398 670355 650101 709065 717017 719882 673019 674596 663033 665506 682820 688075 670104 697949 '
          '662101 677785 679800 693067 744886 746475 710426 691358 651520 757614 710982 647687 678193 685995 651449 627683 661260 '
          '645213 721469 737577 609461 706042 657308 699432 676067 661855 737735 720707 667858 672870 727582 689767 652044 627888 '
          '687031 706344 713155 700338 708636 721624 745872 746429 698969').split()
# The 20 analogues of SAMPLE with every-snapshot cutouts (module docstring: FULL).
FULL = ('609461 627683 657308 662101 670104 676067 687031 691358 693067 697949 702198 708636 709065 713155 720707 721469 721624 '
        '744886 746429 757614').split()
TRACK_SNAPS = tuple(range(21, 100, 3))   # every third snapshot, 21-99 (99 is z = 0)
GAS_SNAPS = (21, 25, 33, 40, 50, 59)   # snapshots of the gas cutouts (full snapshots of TNG)
FULL_SNAPS = tuple(range(21, 41))   # every snapshot, z = 4.0-1.5
# Particle fields requested in each cutout (TNG field names; GFM_StellarFormationTime is the scale factor at birth).
TRACK_FIELDS = 'stars=ParticleIDs,Coordinates,Velocities,Masses,GFM_StellarFormationTime'
GAS_FIELDS = 'gas=Coordinates,Velocities,Masses,StarFormationRate,InternalEnergy'
FULL_QUERY = (TRACK_FIELDS + '&gas=ParticleIDs,Coordinates,Velocities,Masses,StarFormationRate,Density,InternalEnergy'
              '&dm=Coordinates,Velocities')
# Dark-matter snapshots: track snapshots up to 51 for most analogues; 42-51 (z = 1.4-1.0) for the FULL sample.
DM_SNAPS, DM_SNAPS_FULL = tuple(range(21, 52, 3)), (42, 45, 48, 51)
# Dark-matter queries; for FULL also gas positions and masses (fetched, not used).
DM_QUERY, DM_QUERY_FULL = 'dm=Coordinates', 'dm=Coordinates&gas=Coordinates,Masses'
WORKERS = 3  # more parallel requests make the server return errors
```

#### The files to fetch
The SubLink tree of each analogue gives the subhalo id of its main progenitor at each snapshot, from which the cutout
addresses are made.

```python
def save(url, out):
    """Write the response of url to out (via a temporary file), unless out exists."""
    if out.exists():
        return
    out.parent.mkdir(parents=True, exist_ok=True)
    # Download to <name>.part and rename, so that an interrupted file is never taken as complete.
    tmp = out.with_suffix('.part')
    with request(url) as r, open(tmp, 'wb') as f:
        f.write(r.read())
    os.replace(tmp, out)


def jobs():
    """(url, path) of every file: trees first (the cutouts need the main branch's subhalo id at each snapshot)."""
    # The SubLink tree of each analogue, from its z = 0 subhalo (snapshot 99); downloaded here, before any cutout.
    for name in SAMPLE:
        save(f'{API}99/subhalos/{name}/sublink/full.hdf5', sims.TNG_TREE_DIR / f'sublink_{name}.hdf5')
    for name in SAMPLE:
        # Snapshot -> subhalo id of the main progenitor at that snapshot.
        sub = dict(zip(*(lambda b: (b['SnapNum'], b['SubfindID']))(sims.main_branch(name))))
        # Star tracks and gas cutouts, at the snapshots where the main branch exists.
        yield from ((f'{API}{s}/subhalos/{sub[s]}/cutout.hdf5?{TRACK_FIELDS}', sims.TNG_TRACK_DIR / name / f'snap_{s:03d}.hdf5')
                    for s in TRACK_SNAPS if s in sub)
        yield from ((f'{API}{s}/subhalos/{sub[s]}/cutout.hdf5?{GAS_FIELDS}', sims.TNG_GAS_DIR / name / f'gas_{s:03d}.hdf5')
                    for s in GAS_SNAPS if s in sub)
        # Stars, gas and dark matter at every snapshot 21-40, for the FULL sample only.
        if name in FULL:
            yield from ((f'{API}{s}/subhalos/{sub[s]}/cutout.hdf5?{FULL_QUERY}', sims.TNG_FULL_DIR / name / f'snap_{s:03d}.hdf5')
                        for s in FULL_SNAPS if s in sub)
        # Dark-matter positions where there is no full cutout.
        snaps, query = (DM_SNAPS_FULL, DM_QUERY_FULL) if name in FULL else (DM_SNAPS, DM_QUERY)
        yield from ((f'{API}{s}/subhalos/{sub[s]}/cutout.hdf5?{query}', sims.TNG_DM_DIR / name / f'snap_{s:03d}.hdf5')
                    for s in snaps if s in sub)
```

#### Download
Every file not yet on disk, WORKERS at a time.

```python
todo = [j for j in jobs() if not j[1].exists()]   # (url, path) of the files still missing
print(f'{len(todo)} files to fetch', flush=True)
# WORKERS downloads at a time.
with ThreadPoolExecutor(WORKERS) as ex:
    list(ex.map(lambda j: save(*j), todo))
print('done', flush=True)
```

### Auriga

The Auriga analogues are the level-3 dwarf-galaxy zooms of [Grand et al. (2024)](https://doi.org/10.1093/mnras/stae1598): their
$z=0$ snapshots, group catalogues and lists of accreted stars, and the early snapshots used for the scrambling of orbits. They are
transferred from the public Globus collection of the Auriga project.

<details><summary>About <code>scripts/fetch_auriga.py</code></summary>

```text
Download the Auriga files the paper uses (Grand et al. 2024, MNRAS 532, 1814): for the five level-3 dwarf zooms halo_2, 8, 9, 10
and 11 (lmcold.sims.AURIGA), the z = 0 snapshot (snapdir_251), its group catalogue (groups_251: the centre and velocity of the main
subhalo) and the lists of accreted stars (lists/accretedstardata/halo_<n>: in-situ / ex-situ), read by lmcold.sims for Figs
sims_kinematics and sims_birth (scripts/sims_observe.py, build_sims_birth.py); and, for scripts/auriga_scramble.py (Letter, Section 3),
the early full snapshots EARLY_SNAPS (snapdir_050, 060, 070, 090) and the star-only high-cadence outputs EARLY_OUTS
(snapdir_highfreqstars_280-1200, every 20th).

Source: the public Globus collection 'The Auriga Simulations' (GLOBUS; https://wwwmpa.mpa-garching.mpg.de/auriga/data.html), with
globus-cli (checksums verified by Globus) into the same layout under lmcold.sims.SIMS; directories already present are not
transferred again. All files used are then checked against the per-file sha256 of data/checksums/sims_auriga_l3.sha256
(sources.yaml: sims_auriga_l3), written from the files the paper was made with.
Run: .venv/bin/python scripts/fetch_auriga.py   (globus-cli logged in and Globus Connect Personal running, allowed to write under
SIMS; about 120 GB)
```
</details>

*Shown, not run here: globus-cli logged in and Globus Connect Personal running; about 120 GB (script header). The cells below are the script itself; run it from the repository with* `python scripts/fetch_auriga.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The Globus collection, the remote directory of the level-3 dwarf zooms, and the snapshot directories to transfer.

```python
import glob
import subprocess
import tempfile

from lmcold import sims
from lmcold.checksums import check_files
from lmcold.sources import DATA

GLOBUS = '02a2dbb8-f64d-4440-bafe-44b60b964501'   # The Auriga Simulations (Globus collection id)
REMOTE = '/level3/Halos_1e11msol'   # level-3 dwarf zooms (haloes of about 1e11 Msun) in that collection
EARLY_SNAPS = (50, 60, 70, 90)        # scripts/auriga_scramble.SNAPS
EARLY_OUTS = range(280, 1201, 20)     # scripts/auriga_scramble.OUTS
DIRS = ('snapdir_251', 'groups_251') + tuple(f'snapdir_{s:03d}' for s in EARLY_SNAPS) + tuple(
    f'snapdir_highfreqstars_{i:03d}' for i in EARLY_OUTS)
# Per directory of each halo: the z = 0 snapshot and its group catalogue, the early full snapshots and the star-only outputs.
CHECKSUMS = DATA / 'checksums/sims_auriga_l3.sha256'   # lines 'sha256  path', paths relative to lmcold.sims.SIMS
```

#### The files used
The snapshot, group and accreted-star files of one halo that the analysis reads; only these are checked.

```python
def used(d):
    """The files of halo directory d that lmcold.sims and scripts/auriga_scramble.py read.

    Parameters
    ----------
    d : pathlib.Path
        Local directory of one halo (a value of lmcold.sims.AURIGA).

    Returns
    -------
    list of str
        Paths of the HDF5 files: z = 0 snapshot chunks, subhalo catalogue chunks, accreted-star lists, early snapshots and
        star-only outputs.
    """
    # z = 0 snapshot chunks and subhalo-catalogue chunks of snapshot 251.
    return (glob.glob(str(d / 'snapdir_251/snapshot_251.*.hdf5')) + glob.glob(str(d / 'groups_251/fof_subhalo_tab_251.*.hdf5'))
            # Accreted-star lists, stored next to the halo directories under lists/accretedstardata/<halo>.
            + glob.glob(str(d.parent / 'lists/accretedstardata' / d.name / '*.hdf5'))
            # Early full snapshots (EARLY_SNAPS), then the star-only high-cadence outputs (EARLY_OUTS).
            + [f for s in EARLY_SNAPS for f in glob.glob(str(d / f'snapdir_{s:03d}/snapshot_{s:03d}.*.hdf5'))]
            + [f for i in EARLY_OUTS for f in glob.glob(str(d / f'snapdir_highfreqstars_{i:03d}/snapshot_highfreqstars_{i:03d}.*.hdf5'))])
```

#### Transfer the missing directories
One Globus batch transfer (remote path -> local path) of every directory not yet on disk, with checksums verified by
Globus; the script waits for the transfer to finish, polling every 120 s.

```python
# (remote, local) pairs: every directory in DIRS of every halo, plus each halo's accreted-star list directory.
jobs = [(f'{REMOTE}/{d.name}/{s}', d / s) for d in sims.AURIGA.values() for s in DIRS]
jobs += [(f'{REMOTE}/lists/accretedstardata/{d.name}', d.parent / 'lists/accretedstardata' / d.name) for d in sims.AURIGA.values()]
# Only directories not yet on disk are transferred.
todo = [(src, dst) for src, dst in jobs if not dst.exists()]
if todo:
    # Id of the local Globus Connect Personal endpoint (the destination).
    local = subprocess.run(['globus', 'endpoint', 'local-id'], capture_output=True, text=True, check=True).stdout.strip()
    # Batch file: one recursive transfer per line, 'source destination'.
    with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as f:
        f.writelines(f'--recursive {src} {dst}\n' for src, dst in todo)
    # Submit the batch with checksum verification and keep the task id; then block until the task is done.
    task = subprocess.run(['globus', 'transfer', GLOBUS, local, '--batch', f.name, '--verify-checksum', '--label', 'Auriga L3 dwarfs z0',
                           '--jmespath', 'task_id', '-F', 'unix'], capture_output=True, text=True, check=True).stdout.strip()
    subprocess.run(['globus', 'task', 'wait', task, '--polling-interval', '120'], check=True)
```

#### Check the files
Every file used is compared with its sha256 in data/checksums/sims_auriga_l3.sha256; check_files returns the number checked.

```python
# Every used file of the five haloes, checked against the stored sha256 (paths relative to SIMS).
files = [f for d in sims.AURIGA.values() for f in used(d)]
print(check_files(files, sims.SIMS, CHECKSUMS), 'files checked against', CHECKSUMS.relative_to(DATA.parent))
```

### FIRE-2

The FIRE-2 analogues are the LMC-mass runs m11d, m11e and m11h of the public release
([Wetzel et al. 2023](https://doi.org/10.3847/1538-4365/acb99a)): the $z=0$ snapshot of each and the track of its host's centre
and velocity through time, downloaded over HTTPS.

<details><summary>About <code>scripts/fetch_fire2.py</code></summary>

```text
Download the FIRE-2 public-release files the paper uses (Wetzel et al. 2023, ApJS 265, 44): for the LMC-mass runs m11d, m11e and
m11h at 7100 Msun (lmcold.sims.FIRE), the z = 0 snapshot (output/snapshot_600.hdf5) and the host's centre and velocity through time
(track/host_coordinates.hdf5), read by lmcold.sims for Figs sims_kinematics and sims_birth (scripts/sims_observe.py).

Source: the public release over HTTPS (BASE, the Flatiron mirror listed at https://flathub.flatironinstitute.org/fire). Files already
present are not downloaded again; a download goes to a .part file renamed when complete. All files are then checked against the
per-file sha256 of data/checksums/sims_fire2.sha256 (sources.yaml: sims_fire2), written from the files the paper was made with.
Run: .venv/bin/python scripts/fetch_fire2.py   (about 8 GB)
```
</details>

*Shown, not run here: network access to the Flatiron Institute server; about 8 GB (script header). The cells below are the script itself; run it from the repository with* `python scripts/fetch_fire2.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
The public-release address, the two files per run, and the checksum list.

```python
import os

import requests

from lmcold import sims
from lmcold.checksums import check_files
from lmcold.sources import DATA

BASE = 'https://users.flatironinstitute.org/~mgrudic/fire2_public_release/core'  # root of the public release over HTTPS
FILES = ('output/snapshot_600.hdf5', 'track/host_coordinates.hdf5')  # the two files per run, relative to the run's folder
CHECKSUMS = DATA / 'checksums/sims_fire2.sha256'  # per-file sha256 of the files the paper used
```

#### Download helper

```python
def download(url, out):
    """Stream `url` to the path `out` in blocks of 16 MiB through a .part file, renamed when the download is complete."""
    # create the run's folder if needed; download into <name>.part so an interrupted download never looks complete
    out.parent.mkdir(parents=True, exist_ok=True)
    part = out.with_suffix(out.suffix + '.part')
    with requests.get(url, stream=True, timeout=60) as r, open(part, 'wb') as f:
        r.raise_for_status()
        # blocks of 1 << 24 bytes = 16 MiB
        for b in r.iter_content(1 << 24):
            f.write(b)
    # rename the .part file only after the last block is written
    os.replace(part, out)
```

#### Download the missing files
For each run (lmcold.sims.FIRE: m11d, m11e, m11h at 7100 Msun), the files not yet on disk.

```python
# sims.FIRE: run name -> local folder of that run
for d in sims.FIRE.values():
    for f in FILES:
        # skip files already on disk
        if not (d / f).exists():
            download(f'{BASE}/{d.name}/{f}', d / f)
            print('downloaded', d / f)
```

#### Check the files
Every file is compared with its sha256 in data/checksums/sims_fire2.sha256.

```python
# all six files (3 runs x 2 files), downloaded now or before
files = [d / f for d in sims.FIRE.values() for f in FILES]
# check_files compares each file's sha256 with the list; prints the number of files checked
print(check_files(files, sims.SIMS, CHECKSUMS), 'files checked against', CHECKSUMS.relative_to(DATA.parent))
```

### SIDM Concerto

The comparison of a dark-matter halo with and without self-interaction uses Halo104 of the SIDM Concerto suite
([Nadler et al. 2025](https://doi.org/10.3847/1538-4357/adf553)), an LMC-mass host run with cold dark matter (CDM) and with
self-interacting dark matter (SIDM) from the same initial conditions. The $z=0$ snapshots and halo catalogues of both runs are public
on Zenodo ([doi:10.5281/zenodo.14933624](https://doi.org/10.5281/zenodo.14933624)). The snapshots at $z>0$ are not public; they are
available from the simulation authors on request, as a Globus collection whose identifier the script reads from the environment
variable `CONCERTO_HIGHZ_GLOBUS`. Without it, the script downloads the $z=0$ data only.

<details><summary>About <code>scripts/fetch_concerto.py</code></summary>

```text
Download SIDM Concerto Halo104, an LMC-mass host run in CDM and GroupSIDM from the same initial conditions (Nadler, Kong, Yang & Yu
2025, ApJ 991; arXiv:2503.10748), for Fig. sims_sidm (scripts/sims_sidm_shapes.py).

- z = 0 (snapshot 235): the public Zenodo record 10.5281/zenodo.14933624 (ZENODO: particles and Rockstar+consistent-trees catalogues of
  both runs, and the GroupSIDM parametric model), each tar checked against its Zenodo md5 and unpacked in CONCERTO
  (-> central/groups/carnegie_poc/enadler/zoomins/Halo104/{output_32K, output_GroupSIDM_32K}).
- z = 4, 3, 2, 1.5 (snapshots HIGHZ): not public; available from the simulation authors on request, as a Globus collection whose id
  is given in the environment variable CONCERTO_HIGHZ_GLOBUS. Transferred with globus-cli (checksums verified) to HIGHZ_DIR (the
  local Globus collection must allow writes there), then each Gadget format-1 snapshot file is cut to its header and position block
  (the shapes need positions only; masses come from the header), with the block markers checked. Skipped if the variable is unset.
Run: .venv/bin/python scripts/fetch_concerto.py   (globus-cli logged in and Globus Connect Personal running for the z > 0 part)
```
</details>

*Shown, not run here: network access to Zenodo; for the z > 0 snapshots, the Globus collection id from the simulation authors (CONCERTO_HIGHZ_GLOBUS), globus-cli and Globus Connect Personal. The cells below are the script itself; run it from the repository with* `python scripts/fetch_concerto.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
Local directories, the two runs (CDM and GroupSIDM), the Zenodo tar files with their md5, and the snapshots at z > 0.

```python
import hashlib
import os
import subprocess
import tarfile
import tempfile
from pathlib import Path

import numpy as np
import requests

from lmcold import sims

CONCERTO = sims.SIMS / 'concerto'   # local root of the Concerto data (lmcold.sims.SIMS, in ~/data)
HIGHZ_DIR = CONCERTO / 'highz'      # destination of the z > 0 snapshots and hlists
Z0_DIR = CONCERTO / 'central/groups/carnegie_poc/enadler/zoomins/Halo104'   # as unpacked from the Zenodo tars
RUNS = ('output_32K', 'output_GroupSIDM_32K')   # run directories: CDM, GroupSIDM
SIDM_SIGMA0_M, SIDM_W = 147.0, 120.0   # cm^2/g, km/s: the GroupSIDM cross-section of the LMC suite (Nadler et al. 2025, ApJ 991, 69)
ZENODO = 'https://zenodo.org/api/records/14933624/files'   # Zenodo API files endpoint of the z = 0 record
# Zenodo tar file -> its md5 as listed on the record.
TARS = {'LMC_Halo104_CDM_RCT.tar': 'b5bf67f0b317e0f3c5422833ac71a41a', 'LMC_Halo104_GroupSIDM_RCT.tar': '33070b3ae839673e7b9f9c7aeffa0082',
        'LMC_Halo104_GroupSIDM_parametric.tar': 'c956fe941b068921cb633a991842b483',
        'LMC_Halo104_CDM_particles.tar': 'bd2b3208ffeadc765167049d01a9d5fc',
        'LMC_Halo104_GroupSIDM_particles.tar': '37d87d2ab67dd2b5d8576e14fabfa68d'}
GLOBUS = os.environ.get('CONCERTO_HIGHZ_GLOBUS')   # private collection of the z > 0 snapshots (on request)
# Snapshots at z = 4, 3, 2, 1.5 (16 files each, k = 0..15) and the scale factor a = 1/(1+z) in the name of each Rockstar hlist.
HIGHZ = {109: '0.20066', 126: '0.24922', 149: '0.33413', 163: '0.39942'}   # snapshot: scale factor of its Rockstar hlist
```

#### Helpers
The md5 of a file, the download of the public z = 0 data, the cut of a snapshot file to its positions, and the transfer of
the z > 0 snapshots.

```python
def md5(path):
    """Hexadecimal md5 of the file at `path`, read in blocks of 16 MiB."""
    h = hashlib.md5()
    # Read 1 << 24 bytes (16 MiB) at a time until an empty block (end of file).
    with open(path, 'rb') as f:
        for b in iter(lambda: f.read(1 << 24), b''):
            h.update(b)
    return h.hexdigest()


def zenodo():
    """Download the five Zenodo tars into CONCERTO (via a .part file), check each against its md5 and unpack it; nothing is done if
    the z = 0 snapshot of both runs is already unpacked."""
    if all((Z0_DIR / r / 'snapshot_235.0').exists() for r in RUNS):
        return
    CONCERTO.mkdir(parents=True, exist_ok=True)
    for name, sum_ in TARS.items():
        out = CONCERTO / name
        # Stream the tar in 16 MiB blocks to name.part, renamed only when complete (an interrupted download leaves no tar).
        if not out.exists():
            with requests.get(f'{ZENODO}/{name}/content', stream=True, timeout=60) as r, open(out.with_suffix('.part'), 'wb') as f:
                r.raise_for_status()
                for b in r.iter_content(1 << 24):
                    f.write(b)
            os.replace(out.with_suffix('.part'), out)
        # Stop on an md5 mismatch; otherwise unpack (filter='data' refuses absolute paths and links leaving CONCERTO).
        if md5(out) != sum_:
            raise SystemExit(f'md5 mismatch: {out}')
        with tarfile.open(out) as t:
            t.extractall(CONCERTO, filter='data')


def strip(path):
    """Cut a Gadget format-1 file to its header and position block, checking the Fortran record markers.

    The file starts with a 256-byte header record (4-byte markers before and after), then the position record of 3 float32 per
    particle (12 bytes); the six particle counts are the first 24 bytes of the header. The file is rewritten in place.
    """
    b = path.read_bytes()
    # Leading marker of the header record must be its length, 256 bytes.
    assert np.frombuffer(b[0:4], 'i4')[0] == 256
    o = 4 + 256 + 4                                            # byte offset of the position record's leading marker
    npos = np.frombuffer(b[o:o + 4], 'i4')[0]                  # length of the position record [bytes]
    assert npos == 12 * np.frombuffer(b[4:28], 'i4').sum()     # = 12 bytes x total of the six particle counts
    # Header record + position record with both markers; the trailing marker must repeat the length.
    keep = b[:o + 4 + npos + 4]
    assert np.frombuffer(keep[-4:], 'i4')[0] == npos
    # Write to a .tmp file, then rename over the original.
    path.with_suffix(path.suffix + '.tmp').write_bytes(keep)
    os.replace(path.with_suffix(path.suffix + '.tmp'), path)


def highz():
    """Transfer the z > 0 snapshot files and Rockstar hlists of both runs from the Globus collection CONCERTO_HIGHZ_GLOBUS (given
    by the simulation authors on request) to HIGHZ_DIR and cut each snapshot file to its positions; prints a note and does
    nothing if the variable is unset."""
    # (remote path, local path) of the 16 files of each snapshot and of each hlist, both runs; only missing files are fetched.
    jobs = [(f'/{r}/snapshot_{s}.{k}', HIGHZ_DIR / r / f'snapshot_{s}.{k}') for r in RUNS for s in HIGHZ for k in range(16)]
    jobs += [(f'/{r}/rockstar/hlists/hlist_{a}.list', HIGHZ_DIR / r / 'rockstar/hlists' / f'hlist_{a}.list') for r in RUNS for a in HIGHZ.values()]
    todo = [(src, dst) for src, dst in jobs if not dst.exists()]
    if todo and GLOBUS is None:
        print('z > 0 snapshots: not public; set CONCERTO_HIGHZ_GLOBUS to the collection id given by the simulation authors')
    elif todo:
        # Id of the local Globus collection, a batch file of 'src dst' lines, one transfer task with checksum verification.
        local = subprocess.run(['globus', 'endpoint', 'local-id'], capture_output=True, text=True, check=True).stdout.strip()
        with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False) as f:
            f.writelines(f'{src} {dst}\n' for src, dst in todo)
        task = subprocess.run(['globus', 'transfer', GLOBUS, local, '--batch', f.name, '--verify-checksum', '--label', 'Concerto Halo104 z1.5-4',
                               '--jmespath', 'task_id', '-F', 'unix'], capture_output=True, text=True, check=True).stdout.strip()
        # Wait for the task (polled every 120 s), then cut each transferred snapshot file to its positions.
        subprocess.run(['globus', 'task', 'wait', task, '--polling-interval', '120'], check=True)
        for _, dst in todo:
            if 'snapshot_' in dst.name:
                strip(dst)
```

#### Download
The public z = 0 data from Zenodo, then the z > 0 snapshots if the collection id is set.

```python
zenodo()
highz()
```

## How the tables are written

Every table of the paper is written by a script `scripts/paper_table_*.py` from the result tables in `results/`. Each script
writes only the rows of its table, as LaTeX, to `results/table_*.tex`; the column headings, the caption and the notes stay in the
paper's source, which reads the rows with `\input`. The values are formatted by the helpers of `lmcold.tex` (for example a
posterior median with its 16th and 84th percentiles), so no cell is typed or edited by hand. A change in a result changes the table
when the script is run again.

### Ages of the mass priors

The ages of the formed mass in each [Fe/H] range under the reference weighting and the three age priors (Table 4.1).

<details><summary>About <code>scripts/paper_table_ages.py</code></summary>

```text
Paper table (Section 2, tab:ages): ages [Gyr] of the formed mass per prior [Fe/H] range.

Row AMR (reference): 16th-84th percentile ages and the mean of the reference weighting (age-metallicity relation of Povick et al.
2024), results/age_weighting.json from scripts/paper_mass_method.py. Rows A, B (older), C (younger): the age windows of uniform
formed mass, lmcold.mass_conversion.AGE_RANGES.

Writes results/table_ages.tex (the rows; copied to the paper's tables/table_ages.tex).
Run: .venv/bin/python scripts/manifest.py run paper_table_ages.py
```
</details>

**Data read by `scripts/paper_table_ages.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/age_weighting.json` | repository | written by `python scripts/paper_mass_method.py` |

*Shown, not run here: results/age_weighting.json from scripts/paper_mass_method.py. The cells below are the script itself; run it from the repository with* `python scripts/paper_table_ages.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import json

from lmcold.mass_conversion import AGE_RANGES
from lmcold.sources import ROOT
```

#### Load the data
The ages of the reference weighting per [Fe/H] range (scripts/paper_mass_method.py).

```python
w = json.loads((ROOT / 'results/age_weighting.json').read_text())
RANGES = ['<-1.5', '-1.5..-1', '-1..-0.5', '>-0.5']   # keys of age_weighting.json, in the order of AGE_RANGES
```

#### Write the rows
One row for the reference weighting, then one per age prior; the file holds only the rows, the paper's table the rest.

```python
# AMR cells: 'p16--p84 (mean)' ages [Gyr] per [Fe/H] range, one decimal.
amr = [f'{w[k]["p16_p84"][0]:.1f}--{w[k]["p16_p84"][1]:.1f} ({w[k]["mean"]:.1f})' for k in RANGES]
rows = ['AMR (reference) & ' + ' & '.join(amr) + r'\\']
# Prior rows: the age window 'lo--hi' [Gyr] of each [Fe/H] range from AGE_RANGES; LaTeX row ends with \\.
for prior, label in (('A', 'A'), ('B', 'B (older)'), ('C', 'C (younger)')):
    rows.append(label + ' & ' + ' & '.join(f'{lo:g}--{hi:g}' for lo, hi in AGE_RANGES[prior]) + r'\\')
(ROOT / 'results/table_ages.tex').write_text('\n'.join(rows) + '\n')
print('\n'.join(rows))
```

### Selection counts

The counts of the parent samples, observed stars, good spectra and mass samples of the two surveys (Table 2.1).

<details><summary>About <code>scripts/paper_table_cuts.py</code></summary>

```text
Paper table (Section 2, tab:cuts): the count rows of the selection-function table, BOSS and APOGEE-2.

Parent stars, observed, good spectra and parent in covered groups: results/boss_selection.json 'summary' (eligible, good, covered;
scripts/boss_selection.py) with the observed count from data/boss_weights.parquet (column observed, same script), and
results/apogee_selection.json 'summary' (candidates, observed, quality, covered; scripts/apogee_selection.py). Mass sample:
results/giant_counts.json (boss_members, apogee_members; scripts/paper_giant_counts.py). The cut rows above them are text.

Writes results/table_cuts.tex (the count rows; copied to the paper's tables/table_cuts.tex).
Run: .venv/bin/python scripts/manifest.py run paper_table_cuts.py
```
</details>

**Data read by `scripts/paper_table_cuts.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `results/apogee_selection.json` | repository | fixed input |
| `results/boss_selection.json` | repository | fixed input |
| `results/giant_counts.json` | repository | written by `python scripts/paper_giant_counts.py` |

*Shown, not run here: the selection summaries in results/ and data/boss_weights.parquet (Zenodo bundle 1). The cells below are the script itself; run it from the repository with* `python scripts/paper_table_cuts.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import json

import pandas as pd

from lmcold.sources import DATA, ROOT
from lmcold.tex import count
```

#### Load the data
The selection summaries of both surveys, the mass-sample counts and the observed flag of the BOSS parent.

```python
# Count summaries: b for BOSS, a for APOGEE-2 (dicts of integers), m the mass-sample counts of both
b = json.loads((ROOT / 'results/boss_selection.json').read_text())['summary']
a = json.loads((ROOT / 'results/apogee_selection.json').read_text())['summary']
m = json.loads((ROOT / 'results/giant_counts.json').read_text())
# Number of BOSS parent stars with a spectrum (sum of the boolean column observed)
b_observed = int(pd.read_parquet(DATA / 'boss_weights.parquet', columns=['observed']).observed.sum())
```

#### Write the rows
Label, BOSS count, APOGEE-2 count; lmcold.tex.count formats integers as the paper prints them.

```python
# (label, BOSS count, APOGEE-2 count) of each row, in the order of the paper's table
rows = [('Parent stars', b['eligible'], a['candidates']), ('Observed', b_observed, a['observed']),
        ('Good spectra', b['good'], a['quality']), ('Parent in covered groups', b['covered'], a['covered']),
        ('Mass sample$^b$', m['boss_members'], m['apogee_members'])]
# LaTeX rows 'label & N_BOSS & N_APOGEE \\'; written to results/table_cuts.tex and printed
rows = [f'{label} & {count(nb)} & {count(na)} \\\\' for label, nb, na in rows]
(ROOT / 'results/table_cuts.tex').write_text('\n'.join(rows) + '\n')
print('\n'.join(rows))
```

### Stellar mass per metallicity

The formed and present stellar mass per [Fe/H] bin (Table 4.3).

<details><summary>About <code>scripts/paper_table_mass.py</code></summary>

```text
Paper table (Sect. 2.3): formed and present mass per [Fe/H] bin and below each upper edge, with bootstrap percentiles and model
ranges. Reads results/giant_present_mass.csv (build_giant_mass.py, build_present_mass.py); writes results/table_mass.tex.

One LaTeX row per reporting [Fe/H] bin (lmcold.mass_samples.FEH_EDGES, open ends shown as < -2 and >= 0), with 10 columns:
bin; formed mass and its model range; present mass (living stars and remnants) and its model range; upper edge of the bin;
cumulative formed mass below that edge and its range; cumulative present mass and its range. Masses in 10^6 Msun, as
value^{+(p84 - value)}_{-(value - p16)} (16th-84th percentiles of the pixel and response bootstrap) and [model min, max].
Run: .venv/bin/python scripts/paper_table_mass.py
```
</details>

**Data read by `scripts/paper_table_mass.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/giant_present_mass.csv` | repository | written by `python scripts/build_present_mass.py` |

*Shown, not run here: results/giant_present_mass.csv. The cells below are the script itself; run it from the repository with* `python scripts/paper_table_mass.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import numpy as np
import pandas as pd

from lmcold.sources import ROOT
```

#### Load the data
Formed and present mass per reporting [Fe/H] bin, with bootstrap percentiles and model ranges.

```python
t = pd.read_csv(ROOT / "results/giant_present_mass.csv")   # one row per bin; masses in Msun
```

#### Cell format

```python
def cell(v, lo, hi, mlo, mhi):
    """value_{-lo}^{+hi} [model range], in 1e6 Msun.

    Parameters
    ----------
    v, lo, hi : float
        Value and its 16th and 84th bootstrap percentiles [Msun].
    mlo, mhi : float
        Minimum and maximum over the model alternatives [Msun].

    Returns
    -------
    str
        Two LaTeX cells ('value with errors & [min, max]') in 10^6 Msun, with 2, 1 or 0 decimals for values below 10, below 100
        and above.
    """
    # all inputs from Msun to 10^6 Msun
    v, lo, hi, mlo, mhi = (x / 1e6 for x in (v, lo, hi, mlo, mhi))
    # decimals: 2 below 10, 1 below 100, 0 above
    d = 2 if v < 10 else (1 if v < 100 else 0)
    # format string with d decimals, e.g. '{:.2f}'
    f = f"{{:.{d}f}}"
    # value with the upper (p84 - v) and lower (v - p16) errors, then the model range in brackets
    return f"${f.format(v)}^{{+{f.format(hi - v)}}}_{{-{f.format(v - lo)}}}$ & [{f.format(mlo)}, {f.format(mhi)}]"
```

#### Write the rows
One row per bin: the bin, its formed and present mass, its upper edge, and the cumulative masses below that edge.

```python
# one LaTeX row (string) per reporting [Fe/H] bin
rows = []
for r in t.itertuples():
    # bin label: open lower edge (feh_lo = -inf) -> '< -2', open upper edge (feh_hi = +inf) -> '>= 0', else 'lo to hi' [dex]
    lo = "$<-2$" if not np.isfinite(r.feh_lo) else (r"$\geq 0$" if not np.isfinite(r.feh_hi) else f"${r.feh_lo:.2f}$ to ${r.feh_hi:.2f}$")
    # upper edge of the bin [dex], or 'all' for the last (open) bin
    up = "all" if not np.isfinite(r.feh_hi) else f"${r.feh_hi:.2f}$"
    # columns: bin, formed mass, present mass, upper edge, cumulative formed mass, cumulative present mass (with ranges)
    rows.append(" & ".join([lo, cell(r.formed_mass, r.mass_p16, r.mass_p84, r.mass_model_min, r.mass_model_max),
                            cell(r.present_mass, r.present_mass_p16, r.present_mass_p84, r.present_mass_model_min, r.present_mass_model_max),
                            up, cell(r.cumulative, r.cumulative_p16, r.cumulative_p84, r.cumulative_model_min, r.cumulative_model_max),
                            cell(r.present_cumulative, r.present_cumulative_p16, r.present_cumulative_p84, r.present_cumulative_model_min,
                                 r.present_cumulative_model_max)]) + r" \\")
# the table body only; the table environment and header are in the paper source
(ROOT / "results/table_mass.tex").write_text("\n".join(rows) + "\n")
print("\n".join(rows))
```

### Stellar mass against published estimates

The summary of this work against the published estimates (Table 4.2).

<details><summary>About <code>scripts/paper_table_mass_summary.py</code></summary>

```text
Paper table (Sect. 2.3): summary of the LMC stellar mass, this work and published estimates.

This work: formed, present and living mass within 12 deg, and formed and present mass below [Fe/H] = -2, -1.5 and -1, each with the
16th-84th bootstrap percentiles and the range of the model alternatives (results/giant_present_mass.csv from build_giant_mass.py
and build_present_mass.py), and the fraction of formed mass above -0.5 (bootstrap: results/giant_mass_bootstrap.csv; models:
results/giant_mass_variants.csv). Published: lmcold.constants.LMC_MASS_LITERATURE, each as published (formed or present), with its
aperture, method and initial mass function; their source files are checked against data/sources.yaml.
Writes results/table_mass_summary.tex (table rows).

Inputs: results/giant_present_mass.csv (one row per reporting [Fe/H] bin; formed, present and living masses [Msun], their
cumulative sums, bootstrap _p16/_p84 and model _model_min/_model_max), results/giant_mass_bootstrap.csv (formed mass per bin
[Msun] for each bootstrap draw, columns = bin centres), results/giant_mass_variants.csv (formed mass per bin for each model
alternative: conversion, prior, alpha_model, foreground, filled). Published values: lmcold.constants.LMC_MASS_LITERATURE; each
source file is checked through lmcold.sources.path('lmc_mass_reference_<key>'), which raises if its sha256 differs.
Output rows (five LaTeX columns: label & formed & present & aperture/method & IMF), mass cells as
$(v^{+}_{-}) x 10^n$ [model min--max] in Msun; the metal-rich fraction in per cent. Paper Table 3 (Sect. 2.3).
Run: .venv/bin/python scripts/paper_table_mass_summary.py
```
</details>

**Data read by `scripts/paper_table_mass_summary.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/raw/literature/eskew2012.pdf` | public archive | Converting from 3.6 and 4.5 μm Fluxes to Stellar Mass |
| `data/raw/literature/harris_zaritsky2009.pdf` | public archive | The Star Formation History of the Large Magellanic Cloud |
| `data/raw/literature/keim2023.pdf` | public archive | The large magellanic cloud revealed in gravitational waves with LISA |
| `data/raw/literature/mazzi2021.pdf` | public archive | The VMC survey - XLIII. The spatially resolved star formation history across the Large Magellanic Cloud |
| `data/raw/literature/mcconnachie2012.pdf` | public archive | The Observed Properties of Dwarf Galaxies in and around the Local Group |
| `data/raw/literature/skibba2012.pdf` | public archive | The Spatial Distribution of Dust and Stellar Emission of the Magellanic Clouds |
| `data/raw/literature/vanderMarel2002.pdf` | public archive | New Understanding of Large Magellanic Cloud Structure, Dynamics, and Orbit from Carbon Star Kinematics |
| `results/giant_mass_bootstrap.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_variants.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_present_mass.csv` | repository | written by `python scripts/build_present_mass.py` |

*Shown, not run here: the mass tables in results/ and the archived source papers of fetch_lmc_mass_literature.py. The cells below are the script itself; run it from the repository with* `python scripts/paper_table_mass_summary.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import numpy as np
import pandas as pd
from lmcold.constants import LMC_MASS_LITERATURE
from lmcold.sources import ROOT, path
```

#### Load the data
Each published source is checked against data/sources.yaml; the masses per [Fe/H] bin come from
results/giant_present_mass.csv.

```python
for r in LMC_MASS_LITERATURE:  # checksum check of each published source (the return value is not needed)
    path('lmc_mass_reference_' + r.get('source_key', r['key']))
# One row per reporting [Fe/H] bin (feh: centre, feh_hi: upper edge [dex]); masses [Msun].
t = pd.read_csv(ROOT / 'results/giant_present_mass.csv')
```

#### Cell formats
A mass with its bootstrap errors and model range in powers of ten, and the cumulative mass below a bin edge.

```python
def sci(v, lo, hi, mlo, mhi):
    """$(v^{+}_{-})\\times10^n$ [model range] with two significant decimals.

    Parameters
    ----------
    v, lo, hi : float
        Value and its 16th and 84th bootstrap percentiles [Msun].
    mlo, mhi : float
        Minimum and maximum over the model alternatives [Msun].

    Returns
    -------
    str
        LaTeX cell; all numbers scaled by 10^n with n = floor(log10 v) and given to two decimals.
    """
    # Common power of ten n; the value, its errors (p84 - v, v - p16) and the model range in units of 10^n.
    n = int(np.floor(np.log10(v)))
    v, lo, hi, mlo, mhi = (x / 10 ** n for x in (v, lo, hi, mlo, mhi))
    # LaTeX: value with upper and lower errors, power of ten, then the model range in brackets.
    return (rf"$({v:.2f}^{{+{hi - v:.2f}}}_{{-{v - lo:.2f}}})\times10^{{{n}}}$ [{mlo:.2f}--{mhi:.2f}]")


def row(i, pre):
    """Cumulative mass below the upper edge of bin i, for prefix '' (formed), 'present_' or 'living_'.

    Parameters
    ----------
    i : int
        Row of results/giant_present_mass.csv (reporting bin); the last row gives the total.
    pre : str
        Column prefix: '' (formed), 'present_' (living stars and remnants) or 'living_'.

    Returns
    -------
    str
        LaTeX cell from `sci`.
    """
    # The column stem: 'cumulative' for formed mass, '<pre>cumulative' otherwise; suffixes _p16, _p84, _model_min, _model_max.
    r = t.iloc[i]
    # Cumulative mass below the bin's upper edge, with its bootstrap percentiles and model range.
    key = 'cumulative' if pre == '' else pre + 'cumulative'
    return sci(r[key], r[key + '_p16'], r[key + '_p84'], r[key + '_model_min'], r[key + '_model_max'])
```

#### This work
Total formed, present and living mass, the mass below [Fe/H] = -1, -1.5 and -2, and the metal-rich fraction.

```python
# Last row: the cumulative mass of all bins, i.e. the total within 12 deg.
last = len(t) - 1
below = {f'{t.feh_hi[i]:g}': i for i in range(last)}  # upper bin edge as text ('-2', '-1.5', ...) -> row index
# Section header, the totals (formed, present; living stars only), then the cumulative masses below -1, -1.5 and -2.
rows = [r"\multicolumn{5}{l}{\textit{This work: BOSS giants within $12^\circ$, PARSEC conversion}}\\",
        rf"All & {row(last, '')} & {row(last, 'present_')} & $\rho<12^\circ$ & Kroupa\\",
        rf"Living stars & -- & {row(last, 'living_')} & & \\"]
for edge in ('-1', '-1.5', '-2'):
    rows.append(rf"$\feh<{edge}$ & {row(below[edge], '')} & {row(below[edge], 'present_')} & & \\")
# Fraction of formed mass in the bins above [Fe/H] = -0.5: reference, bootstrap 16th-84th percentiles, and the range over the
# model alternatives (filled, i.e. with the full-coverage extrapolation).
rich = t.feh.to_numpy() > -.5  # bin centres above -0.5
# Bootstrap draws (n_boot, n_bins) [Msun]; fb: the metal-rich fraction in each draw.
boot = pd.read_csv(ROOT / 'results/giant_mass_bootstrap.csv').to_numpy()
fb = boot[:, rich].sum(1) / boot.sum(1)
# Metal-rich fraction for each model alternative (conversion, prior, alpha_model, foreground); f0: the reference value.
v = pd.read_csv(ROOT / 'results/giant_mass_variants.csv').query('filled')
fm = v.groupby(['conversion', 'prior', 'alpha_model', 'foreground']).apply(
    lambda d: d.mass[d.feh > -.5].sum() / d.mass.sum(), include_groups=False)
f0 = t.formed_mass[rich].sum() / t.formed_mass.sum()
p16, p84 = np.percentile(fb, [16, 84])
# Fraction in per cent, integer: reference, bootstrap errors, [model min--max].
rows.append(rf"$\feh>-0.5$, fraction & ${100 * f0:.0f}^{{+{100 * (p84 - f0):.0f}}}_{{-{100 * (f0 - p16):.0f}}}$ per cent "
            rf"[{100 * fm.min():.0f}--{100 * fm.max():.0f}] & & & \\")
```

#### Published estimates
One row per entry of lmcold.constants.LMC_MASS_LITERATURE, as published (formed or present mass).

```python
rows.append(r"\hline")
rows.append(r"\multicolumn{5}{l}{\textit{Published estimates, as published}}\\")
for r in LMC_MASS_LITERATURE:
    # Published mass [Msun] in units of 1e9, put in the formed or the present column according to its kind.
    m = rf"${r['mass'] / 1e9:.2f}\times10^{{9}}$"
    formed, present = (m, '--') if r['kind'] == 'formed' else ('--', m)
    rows.append(rf"\citet{{{r['bibkey']}}} & {formed} & {present} & {r['summary']} & {r['imf_short']}\\")
# Write the rows (the paper's table file holds the rest) and print them.
(ROOT / 'results/table_mass_summary.tex').write_text('\n'.join(rows) + '\n')
print('\n'.join(rows))
```

### Shared kinematic parameters

The shared parameters of the joint fits (Table 6.2).

<details><summary>About <code>scripts/paper_table_kin_shared.py</code></summary>

```text
Paper table (Section 4, tab:kin_shared): shared parameters of the joint (stage-2) fits.

Columns: Level 1 (symmetric disc, sky-aligned dispersions: results/kin_mcmc_data_centre.csv, stage 2 'shared') and Level 2
(velocity ellipsoid: results/kin_mcmc_data_ellipsoid.csv, stage 2 'shared'), each the posterior median with the 16th-84th
percentile half-range (as the caption states); then the range (min--max) of the stage-1 Level-2 medians of the five giant groups
(kin_mcmc_data_ellipsoid.csv), with the RR Lyrae stage-1 median in brackets (kin_mcmc_data_ellipsoid_outer.csv; none for v_los,0,
which the RR Lyrae do not constrain). Decimals per parameter in DEC (value, stage-1 range).

Writes results/table_kin_shared.tex (the rows; copied to the paper's tables/table_kin_shared.tex).
Run: .venv/bin/python scripts/manifest.py run paper_table_kin_shared.py
```
</details>

**Data read by `scripts/paper_table_kin_shared.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/kin_mcmc_data_centre.csv` | repository | posterior summary of the free-centre model; MCMC; written by `python scripts/fit_kinematics_mcmc.py data centre` |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_mcmc_data_ellipsoid_outer.csv` | repository | the same fit at 4-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid outer` |

*Shown, not run here: the posterior summaries of the kinematic fits in results/. The cells below are the script itself; run it from the repository with* `python scripts/paper_table_kin_shared.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import pandas as pd

from lmcold.sources import ROOT
from lmcold.tex import pm_sym
```

#### Load the data
The posterior summaries of the Level-1 and Level-2 joint fits (2-9 deg) and of the RR Lyrae fit at 4-9 deg.

```python
# One row per stage x population x parameter: median, p16, p84 (scripts/fit_kinematics_mcmc.py).
L1 = pd.read_csv(ROOT / 'results/kin_mcmc_data_centre.csv')  # Level 1: sky-aligned model, 2-9 deg
L2 = pd.read_csv(ROOT / 'results/kin_mcmc_data_ellipsoid.csv')  # Level 2: velocity-ellipsoid model, 2-9 deg
RR = pd.read_csv(ROOT / 'results/kin_mcmc_data_ellipsoid_outer.csv')  # Level 2 at 4-9 deg (the RR Lyrae values)
```

#### Formats
Row label and decimals of each shared parameter, and the two cell formats.

```python
# parameter: (row label, decimals of the stage-2 values, decimals of the stage-1 range)
DEC = {'dx': (r'$\Delta x$ (deg)', 3, 2), 'dy': (r'$\Delta y$ (deg)', 3, 2),
       'pmra0': (r'$\mu_{\alpha*,0}$ (mas\,yr$^{-1}$)', 4, 3), 'pmdec0': (r'$\mu_{\delta,0}$ (mas\,yr$^{-1}$)', 3, 3),
       'vlos0': (r'$v_{\rm los,0}$ (\kms)', 1, 1), 'inc': (r'$i$ (deg)', 1, 1), 'theta': (r'$\theta$ (deg)', 1, 1)}


def shared(t, p, d):
    """Stage-2 shared value of p in table t as median +- half the 16th-84th range, or '--' if t does not fit p."""
    r = t[(t.stage == 2) & (t.population == 'shared') & (t.param == p)]
    return pm_sym(r['median'].iloc[0], (r.p84.iloc[0] - r.p16.iloc[0]) / 2, d) if len(r) else '--'


def num(x, d):
    """Plain number, in math mode if negative (as the paper prints minus signs)."""
    return f'${x:.{d}f}$' if round(x, d) < 0 else f'{x:.{d}f}'
```

#### Write the rows
One row per shared parameter, then the m = 2 terms and the radial-motion terms of the Level-2 model.

```python
rows = []  # LaTeX table rows
for p, (label, d, d1) in DEC.items():
    # Stage-1 Level-2 medians of the five giant groups, and the RR Lyrae stage-1 median of the outer run.
    g = L2[(L2.stage == 1) & L2.population.str.startswith('giants') & (L2.param == p)]['median']
    r = RR[(RR.stage == 1) & (RR.population == 'RR Lyrae') & (RR.param == p)]['median']
    # Stage-1 range 'min--max (RR Lyrae)'; no RR Lyrae value for vlos0 (no line-of-sight velocities).
    rng = f'{num(g.min(), d1)}--{num(g.max(), d1)}' + (f' ({num(r.iloc[0], d1)})' if p != 'vlos0' else '')
    # Columns: label, Level 1, Level 2, stage-1 range.
    rows.append(' & '.join([label, shared(L1, p, d), shared(L2, p, d), rng]) + r' \\')
# Streaming terms (Level 2 only; not in Level 1): m = 2 modulation of v_phi, then mean and m = 2 radial velocity.
rows.append(r'$m_{2c}$, $m_{2s}$ & -- & ' + ', '.join(shared(L2, p, 3) for p in ('m2c', 'm2s')) + r' & \\')
rows.append(r'$v_{R,0}$, $r_{2c}$, $r_{2s}$ & -- & ' + ', '.join(shared(L2, p, 3) for p in ('vr0', 'r2c', 'r2s')) + r' & \\')
# Write the rows (copied to the paper's tables/) and print them.
(ROOT / 'results/table_kin_shared.tex').write_text('\n'.join(rows) + '\n')
print('\n'.join(rows))
```

### Rotation and velocity ellipsoid

The rotation and velocity ellipsoid of each group fitted on its own (Table 7.1).

<details><summary>About <code>scripts/paper_table_kin.py</code></summary>

```text
Paper table (Section 4, tab:kin): rotation and velocity ellipsoid of each group fitted on its own (stage 1), at R_sigma = 4 kpc.

Rows: the five giant [Fe/H] groups (2 < rho < 9 deg) and the RR Lyrae (4 < rho < 9 deg); then the APOGEE giants of the three
metal-rich groups alone. Columns: N; v_rot(4 kpc), sigma_R, sigma_phi, sigma_z [km/s] and v_rot/sigma_R, v_rot/sigma_z (posterior
median and 16th-84th percentiles of the draws, results/kin_support_ellipsoid.csv from scripts/paper_kin_support_ellipsoid.py);
gamma = dlnsig_dR [1/kpc], k and the outlier fraction f [%] (stage-1 posterior, results/kin_mcmc_data_ellipsoid.csv, RR Lyrae
results/kin_mcmc_data_ellipsoid_outer.csv). APOGEE rows: maximum-likelihood values with inverse-Hessian errors (the *_apogee
columns of kin_support_ellipsoid.csv). Format: lmcold.tex.pm (1 decimal for velocities, 3 for gamma, 2 for ratios and k, 1 for f).
The RR Lyrae have no line-of-sight velocities: sigma_z and v_rot/sigma_z are '--'.

Writes results/table_kin.tex (the rows; copied to the paper's tables/table_kin.tex).
Run: .venv/bin/python scripts/manifest.py run paper_table_kin.py
```
</details>

**Data read by `scripts/paper_table_kin.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_mcmc_data_ellipsoid_outer.csv` | repository | the same fit at 4-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid outer` |
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |

*Shown, not run here: results/kin_support_ellipsoid.csv and the posterior summaries of the kinematic fits. The cells below are the script itself; run it from the repository with* `python scripts/paper_table_kin.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import numpy as np
import pandas as pd

from lmcold.sources import ROOT
from lmcold.tex import GROUP_LABEL, count, fout_percent, pm, pm_sym
```

#### Load the data
The velocity-ellipsoid summary per population and the stage-1 posteriors of the giants' and the RR Lyrae fits.

```python
# One row per population: n, posterior medians and p16/p84 of vrot4, sigma_R, sigma_phi, sigma_z, ratio_R, ratio_z [km/s or
# dimensionless], and the APOGEE-only maximum-likelihood values (*_apogee, *_apogee_err; NaN for the other groups).
S = pd.read_csv(ROOT / 'results/kin_support_ellipsoid.csv').set_index('population')
# Stage-1 posterior summaries (columns stage, population, param, median, p16, p84) of the giants' and the RR Lyrae fits.
FITS = {'giants': pd.read_csv(ROOT / 'results/kin_mcmc_data_ellipsoid.csv'),
        'RR Lyrae': pd.read_csv(ROOT / 'results/kin_mcmc_data_ellipsoid_outer.csv')}


def stage1(pop, param):
    """(median, p16, p84) of a stage-1 parameter of population pop."""
    t = FITS['RR Lyrae' if pop == 'RR Lyrae' else 'giants']   # the table holding pop
    r = t[(t.stage == 1) & (t.population == pop) & (t.param == param)].iloc[0]
    return r['median'], r.p16, r.p84
```

#### Write the rows
One row per population (MCMC posteriors), then the APOGEE-only rows of the metal-rich groups (maximum likelihood).

```python
rows = []   # LaTeX table rows
for pop, s in S.iterrows():
    # q(column, decimals): median and its 16th, 84th percentiles from the summary table, formatted by lmcold.tex.pm.
    q = lambda c, d: pm(s[c], s[f'{c}_p16'], s[f'{c}_p84'], d)
    rr = pop == 'RR Lyrae'   # no line-of-sight velocities: sigma_z columns are '--'
    # Columns: group, N, vrot4, sigma_R, sigma_phi, sigma_z, gamma, vrot/sigma_R, vrot/sigma_z, k, outlier fraction [%].
    rows.append(' & '.join([GROUP_LABEL[pop], count(s.n), q('vrot4', 1), q('sigma_R', 1), q('sigma_phi', 1),
                            '--' if rr else q('sigma_z', 1), pm(*stage1(pop, 'dlnsig_dR'), 3), q('ratio_R', 2),
                            '--' if rr else q('ratio_z', 2), pm(*stage1(pop, 'k'), 2),
                            f'{fout_percent(stage1(pop, "logit_fout")[0]):.1f}']) + r' \\')
rows.append(r'\hline')
# APOGEE-only rows: groups with a finite APOGEE fit; value +- inverse-Hessian error; gamma, k and f left empty.
for pop, s in S[np.isfinite(S.vrot4_apogee)].iterrows():
    a = lambda c, d: pm_sym(s[f'{c}_apogee'], s[f'{c}_apogee_err'], d)
    rows.append(' & '.join(['APOGEE ' + GROUP_LABEL[pop], count(s.n_apogee), a('vrot4', 1), a('sigma_R', 1), a('sigma_phi', 1),
                            a('sigma_z', 1), '', a('ratio_R', 2), a('ratio_z', 2), '', '']) + r' \\')
(ROOT / 'results/table_kin.tex').write_text('\n'.join(rows) + '\n')
print('\n'.join(rows))
```

### Sky-aligned dispersions

The fits with sky-aligned dispersions (Table 6.3).

<details><summary>About <code>scripts/paper_table_kin_sky.py</code></summary>

```text
Paper table (Appendix, tab:kin_sky): Level-2 parameters of each group with sky-aligned dispersions, fitted on its own (stage 1).

Rows: the five giant [Fe/H] groups and the RR Lyrae at 2 < rho < 9 deg; then the APOGEE giants of the three metal-rich groups
alone. Columns: N; v_0 [km/s], sigma_los [km/s], v_0/sigma_los (results/kin_support.csv from scripts/paper_kin_support.py: posterior
median and 16th-84th percentiles, the ratio from the draws); R_0 [kpc] and the outlier fraction f [%] (stage-1 posterior,
results/kin_mcmc_data_stream.csv); sigma_PM [km/s] and v_0/sigma_PM from the 4 < rho < 9 deg fits (kin_support.csv columns sigma,
ratio_pm). APOGEE rows: maximum-likelihood v_0, sigma_los and their ratio with inverse-Hessian errors. Format: lmcold.tex.pm
(1 decimal for velocities, 2 for R_0 and ratios, 1 for f). The RR Lyrae have no line-of-sight velocities ('--').

Writes results/table_kin_sky.tex (the rows; copied to the paper's tables/table_kin_sky.tex).
Run: .venv/bin/python scripts/manifest.py run paper_table_kin_sky.py
```
</details>

**Data read by `scripts/paper_table_kin_sky.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/kin_mcmc_data_stream.csv` | repository | posterior summary of the sky-aligned dispersion model with m = 2 streaming, 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data stream` |
| `results/kin_support.csv` | repository | written by `python scripts/paper_kin_support.py` |

*Shown, not run here: results/kin_support.csv and the posterior summaries of the sky-aligned fits. The cells below are the script itself; run it from the repository with* `python scripts/paper_table_kin_sky.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import numpy as np
import pandas as pd

from lmcold.sources import ROOT
from lmcold.tex import GROUP_LABEL, count, fout_percent, pm, pm_sym
```

#### Load the data
The rotational-support summary per population and the stage-1 posteriors of the sky-aligned fits.

```python
# One row per population: v0, sigma_los, sigma [km/s], ratios, with percentiles; APOGEE-only ML columns *_apogee.
S = pd.read_csv(ROOT / 'results/kin_support.csv').set_index('population')
# Stage-1 posterior summaries (each population on its own) of the sky-aligned 2-9 deg fit: population, param, median, p16, p84.
F = pd.read_csv(ROOT / 'results/kin_mcmc_data_stream.csv').query('stage == 1')


def stage1(pop, param):
    """(median, p16, p84) of a stage-1 parameter of population pop in the 2-9 deg sky-aligned fit."""
    r = F[(F.population == pop) & (F.param == param)].iloc[0]   # the one row of this population and parameter
    return r['median'], r.p16, r.p84
```

#### Write the rows
One row per population (MCMC posteriors), then the APOGEE-only rows of the metal-rich groups (maximum likelihood).

```python
rows = []   # LaTeX table rows
for pop, s in S.iterrows():
    # q(column, decimals): median with its percentile columns (suffixes _p16/_p84, or _lo/_hi for the ratios) via lmcold.tex.pm.
    q = lambda c, d, lo='_p16', hi='_p84': pm(s[c], s[c + lo], s[c + hi], d)
    rr = pop == 'RR Lyrae'   # no line-of-sight velocities: '--' in the sigma_los and ratio columns
    # Columns: group, N, v0, R0, sigma_los, v0/sigma_los, sigma_PM, v0/sigma_PM, outlier fraction [%].
    rows.append(' & '.join([GROUP_LABEL[pop], count(s.n), q('v0', 1), pm(*stage1(pop, 'R0'), 2),
                            '--' if rr else q('sigma_los', 1), '--' if rr else q('ratio', 2, '_lo', '_hi'), q('sigma', 1),
                            q('ratio_pm', 2, '_lo', '_hi'), f'{fout_percent(stage1(pop, "logit_fout")[0]):.1f}']) + r' \\')
rows.append(r'\hline')
# APOGEE-only rows (populations with a finite v0_apogee): N, ML v0, sigma_los, ratio with symmetric errors; other columns empty.
for pop, s in S[np.isfinite(S.v0_apogee)].iterrows():
    rows.append(' & '.join(['APOGEE ' + GROUP_LABEL[pop], count(s.n_apogee), pm_sym(s.v0_apogee, s.v0_apogee_err, 1), '',
                            pm_sym(s.sigma_los_apogee, s.sigma_los_apogee_err, 1), pm_sym(s.ratio_apogee, s.ratio_apogee_err, 2),
                            '', '', '']) + r' \\')
# Write the rows (copied to the paper's tables/table_kin_sky.tex) and print them.
(ROOT / 'results/table_kin_sky.tex').write_text('\n'.join(rows) + '\n')
print('\n'.join(rows))
```

### Axis ratios

The axis ratio implied by the kinematics of each giant bin in the three potentials (Table 8.1).

<details><summary>About <code>scripts/paper_table_shape.py</code></summary>

```text
Paper table (Section 5, tab:shape): the axis ratio q = c/a implied by the kinematics of each giant [Fe/H] bin in the potentials
P1-P3 (results/shape_virial.csv, no central floor): q of the posterior-median model with the 16th-84th percentiles of the joint draws
as a range, and 2K/|W| of the median model at that q.

Writes results/table_shape.tex (the tabular; copied to the paper's tables/table_shape.tex).

Run: .venv/bin/python scripts/paper_table_shape.py   (after scripts/shape_virial.py)
```
</details>

**Data read by `scripts/paper_table_shape.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/shape_virial.csv` | repository | axis ratio implied by the motions (tensor virial theorem) per giant bin and potential; written by `python scripts/shape_virial.py` |

*Shown, not run here: results/shape_virial.csv from scripts/shape_virial.py. The cells below are the script itself; run it from the repository with* `python scripts/paper_table_shape.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import pandas as pd

from lmcold.sources import ROOT

POTS = ('P1', 'P2', 'P3')   # the three potentials of scripts/shape_virial.py, one column each
```

#### Load the data
The axis ratios of scripts/shape_virial.py; only the giant bins without a central floor (r_floor = 0) enter the table.

```python
# One row per (population, potential, r_floor) with q, its percentiles q_p16, q_p84 and balance = 2K/|W|
v = pd.read_csv(ROOT / 'results/shape_virial.csv')
# Keep the giant [Fe/H] bins with no central floor
v = v[(v.r_floor == 0) & v.population.str.startswith('giants')]
```

#### Write the table
One row per [Fe/H] bin; each cell is q [p16--p84]; 2K/|W|.

```python
# Table header: [Fe/H] bin and one column per potential
rows = [r'\begin{tabular}{lccc}', r'\hline', r'\feh & P1 & P2 & P3\\', r'\hline']
for pop, g in v.groupby('population', sort=False):
    # Rows of one bin ordered P1, P2, P3; label: the bin's [Fe/H] range in LaTeX
    g = g.set_index('potential').loc[list(POTS)]
    label = '$' + pop.removeprefix('giants ').replace('[Fe/H]', r'\feh') + '$'
    # Cell: q [p16--p84]; 2K/|W|, all to two decimals
    cells = [f'{r.q:.2f} [{r.q_p16:.2f}--{r.q_p84:.2f}]; {r.balance:.2f}' for _, r in g.iterrows()]
    rows.append(label + ' & ' + ' & '.join(cells) + r'\\')
# Close the tabular, write results/table_shape.tex and print it
rows += [r'\hline', r'\end{tabular}']
(ROOT / 'results/table_shape.tex').write_text('\n'.join(rows) + '\n')
print('\n'.join(rows))
```

### Analogues that reach the LMC

The number of simulated analogues that reach the LMC's rotational support under each definition (Table 9.1).

<details><summary>About <code>scripts/paper_table_sims_definitions.py</code></summary>

```text
Paper table (Appendix, tab:sims_definitions): number of simulated analogues whose rotational support at R = 4 kpc reaches the
LMC's, per giant [Fe/H] group, for v_rot over sigma_z, sigma_R or sigma_RMS, fitted (fit) or from the mock giants' particles (part.),
and under any definition. Counts from results/sims_definitions.csv (scripts/paper_sims_kinematics.py, which also defines the LMC
thresholds).

Writes results/table_sims_definitions.tex (the rows; copied to the paper's tables/table_sims_definitions.tex).
Run: .venv/bin/python scripts/manifest.py run paper_table_sims_definitions.py
```
</details>

**Data read by `scripts/paper_table_sims_definitions.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/sims_definitions.csv` | repository | written by `python scripts/paper_sims_kinematics.py` |

*Shown, not run here: results/sims_definitions.csv from scripts/paper_sims_kinematics.py. The cells below are the script itself; run it from the repository with* `python scripts/paper_table_sims_definitions.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

```python
import pandas as pd

from lmcold.sources import ROOT
from lmcold.tex import GROUP_LABEL
```

#### Load the data
The counts of analogues per group and definition (scripts/paper_sims_kinematics.py).

```python
# One row per giant group (population) with the number of analogues reaching the LMC under each definition (columns of COLS).
t = pd.read_csv(ROOT / 'results/sims_definitions.csv')
```

#### Write the rows
One row per giant group: the counts for each definition, in the column order of the paper.

```python
# Definitions in the paper's column order: v_rot over sigma_z, sigma_R, sigma_RMS, fitted or from particles; 'any' of them.
COLS = ['z', 'z particles', 'R', 'R particles', 'RMS', 'RMS particles', 'any']
# Row: group label without the \feh macro, then the integer counts; LaTeX row end.
rows = [' & '.join([GROUP_LABEL[r.population].replace(r'\feh', '')] + [str(int(r[c])) for c in COLS]) + r' \\'
        for _, r in t.iterrows()]
(ROOT / 'results/table_sims_definitions.tex').write_text('\n'.join(rows) + '\n')
print('\n'.join(rows))
```

## How the numbers in the text are written

No number in the text of the paper or of this site is typed by hand. Each is a LaTeX macro, such as `\daNUnion` for the number of RR
Lyrae in the union of OGLE-IV and *Gaia*, defined in a file `results/macros_*.tex`. A script `scripts/paper_text_*.py` writes each
file. Settings (a radius, a bin edge, a clipping threshold) are read from the code that uses them: module constants of a script are
read from its source without running it, defaults of a function from its signature, and constants of `lmcold` by import. Results are
computed from the tables and catalogues the figures are drawn from. The paper reads the macro files with `\input`; on this site,
`kb/make_notebooks.py` puts the value of each macro in place of every `{glue:text}` field. Where the text states something about a
result (for example that only one fit has divergent transitions, or that no analogue reaches the LMC today), the script stops if the
result no longer agrees with the wording.

### Shared helpers

`scripts/textnum.py` holds the functions the generators share: reading settings from the code, and writing numbers as the text
writes them (integers without a decimal point, approximate values to one significant figure).

<details><summary>About <code>scripts/textnum.py</code></summary>

```text
Helpers of the paper-text generators (scripts/paper_text_*.py): read settings from the code without running it, and write numbers
as they appear in the text.
```
</details>

*Shown, not run here: nothing beyond the repository (a module of helpers, imported by the generators). The cells below are the script itself; run it from the repository with* `python scripts/textnum.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Reading settings from the code
Module constants of a script and default arguments of a function, read without running anything, so that a number in the
text follows the code that uses it.

```python
import ast
import inspect

import numpy as np

from lmcold.sources import ROOT


def const(script):
    """Module-level constants of scripts/<script>.py whose values are literals (also tuple assignments), without running it."""
    # walk the top-level statements of the script's syntax tree; keep single-target assignments only
    out = {}
    for n in ast.parse((ROOT / 'scripts' / f'{script}.py').read_text()).body:
        # only plain 'NAME = value' or 'A, B = value' statements
        if not isinstance(n, ast.Assign) or len(n.targets) != 1:
            continue
        t = n.targets[0]
        # literal_eval accepts numbers, strings, tuples, lists, dicts; anything computed (calls, names) is skipped
        try:
            v = ast.literal_eval(n.value)
        except ValueError:
            continue
        # NAME = literal
        if isinstance(t, ast.Name):
            out[t.id] = v
        # A, B = (literal, literal): one entry per name
        elif isinstance(t, ast.Tuple) and isinstance(v, tuple) and len(v) == len(t.elts):
            out.update({e.id: x for e, x in zip(t.elts, v)})
    return out


def default(f, arg):
    """Default value of argument `arg` of function `f`."""
    # the default as written in the function's signature (inspect.Parameter.empty if it has none)
    return inspect.signature(f).parameters[arg].default
```

#### Writing numbers as in the text
Integers without a decimal point, approximate values to one significant figure, lists joined with 'and', and powers of ten.

```python
def num(x):
    """A number as it is written in the text: integers without a point, others as given (results are rounded where computed)."""
    # whole numbers (below 1e6) without a decimal point; anything else in the shortest general format
    x = float(x)
    return f'{x:.0f}' if x == round(x) and abs(x) < 1e6 else f'{x:g}'


def sigfig(x, n=1):
    """x rounded to n significant figures, as written in the text (approximate values, scatters, correlations)."""
    # zero has no logarithm; written as '0'
    x = float(x)
    if x == 0:
        # d: number of decimals that keeps n significant figures (negative d rounds to tens, hundreds, ...)
        return '0'
    d = n - 1 - int(np.floor(np.log10(abs(x))))
    # never print a negative number of decimals
    return f'{round(x, d):.{max(d, 0)}f}'


def listing(xs):
    """Strings xs joined as 'a, b and c'."""
    # all but the last joined with commas, the last with 'and'
    return ', '.join(xs[:-1]) + ' and ' + xs[-1]


def tex_sci(x):
    # decimal exponent e and mantissa x/10^e rounded to one decimal
    """x as m\\times10^e (two significant figures)."""
    e = int(np.floor(np.log10(x)))
    return rf'{num(round(x / 10 ** e, 1))}\times10^{{{e}}}'
```

### The Letter

The numbers of the Letter's account of how its figures were made (`results/macros_letter.tex`, prefix `\lt`).

<details><summary>About <code>scripts/paper_text_letter.py</code></summary>

```text
Letter text: every number of the Letter's account of how its five figures were made, as LaTeX macros (prefix \\lt), so that none is
typed by hand.

Two kinds of number.
- Settings: read from the code that uses them, never retyped here. Constants of the scripts are read from their source with ast
  (module-level assignments of literals; the scripts are not run), constants of lmcold by import, defaults of lmcold functions with
  inspect. Each macro names its source below.
- Results: computed from the results tables the figures are drawn from (and, for the Concerto particle mass, from the snapshot header).
Inputs: scripts/{paper_letter_signal, fit_kinematics_mcmc, sims_observe, paper_sims_kinematics, fetch_tng50, build_sims_birth,
paper_sims_birth, fetch_tng50_progenitors, sims_scramble, build_sims_vcirc, paper_letter_scramble, sims_halo_settle, fetch_concerto,
sims_sidm_shapes, paper_sims_sidm, kin_feh_lowest, kin_vc_drift}.py; results/{letter_signal_counts, kin_support_ellipsoid, kin_feh_lowest, kin_feh_lowest_null,
kin_vc_drift_h, giant_mass, sims_observed, sims_birth,
sims_scramble, sims_heating_gas, sims_vcirc, sims_halo_settle, sims_sidm_shapes}.csv; the z = 0 Concerto snapshot header.
Writes results/macros_letter.tex (copied to the paper's tables/; \\input in the Letter's preamble).
Run: .venv/bin/python scripts/manifest.py run paper_text_letter.py
```
</details>

**Data read by `scripts/paper_text_letter.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/giants_apogee.parquet` | Zenodo bundle 1 | APOGEE DR17 giants with Gaia DR3 |
| `data/giants_boss.parquet` | Zenodo bundle 1 | SDSS-V DR20 BOSS giants with Gaia DR3, on the APOGEE scale |
| `results/giant_mass.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_present_mass.csv` | repository | written by `python scripts/build_present_mass.py` |
| `results/kin_feh_lowest.csv` | repository | written by `python scripts/kin_feh_lowest.py` |
| `results/kin_feh_lowest_null.csv` | repository | written by `python scripts/kin_feh_lowest.py` |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |
| `results/kin_vc_drift_h.csv` | repository | fixed input |
| `results/letter_signal_counts.csv` | repository | written by `python scripts/paper_letter_signal.py` |
| `results/macros_sims_suites.tex` | repository | written by `python scripts/paper_text_sims_suites.py` |
| `results/research/fire_early_birth.csv` | repository | written by `python scripts/fire2_birth.py` |
| `results/sims_birth.csv` | repository | written by `python scripts/build_sims_birth.py` |
| `results/sims_birth_radii.csv` | repository | written by `python scripts/sims_birth_radii.py` |
| `results/sims_halo_settle.csv` | repository | written by `python scripts/sims_halo_settle.py` |
| `results/sims_heating_gas.csv` | repository | written by `python scripts/build_sims_heating.py` |
| `results/sims_kinematics_summary.csv` | repository | written by `python scripts/paper_sims_kinematics.py` |
| `results/sims_observed.csv` | repository | written by `python scripts/sims_observe.py merge` |
| `results/sims_scramble.csv` | repository | written by `python scripts/sims_scramble.py` |
| `results/sims_sidm_particle_mass.csv` | repository | written by `python scripts/sims_sidm_shapes.py` |
| `results/sims_sidm_shapes.csv` | repository | written by `python scripts/sims_sidm_shapes.py` |
| `results/sims_vcirc.csv` | repository | written by `python scripts/build_sims_vcirc.py` |

*Shown, not run here: the result tables in results/, the release samples (Zenodo bundle 1) and the header of the z = 0 Concerto snapshot (fetch_concerto.py). The cells below are the script itself; run it from the repository with* `python scripts/paper_text_letter.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
Imports, the dictionary M of macros (macro name without its prefix: the text it stands for) and the functions that format
numbers as the text writes them. The inputs listed in the header are read in the step that uses them.

```python
import re

import numpy as np
import pandas as pd

from fit_kinematics_mcmc import ring
from lmcold import samples, sims
from fetch_tng50_progenitors import FULL, FULL_SNAPS, TRACK_SNAPS
from lmcold.constants import LMC, LMC_MASS_LITERATURE
from lmcold.footprint import RHO_MAX
from lmcold.kinfit import R_SIG
from lmcold.mass_samples import FEH_EDGES
from lmcold.samples import FEH_BINS
from lmcold.sources import ROOT
from textnum import const, default, listing, num, sigfig, tex_sci


M = {}   # macro name (without the \lt prefix) -> the text it stands for
# Literal module-level constants of each script, read from its source with textnum.const (dicts name -> value)
sig, kin, obs, ksim = const('paper_letter_signal'), const('fit_kinematics_mcmc'), const('sims_observe'), const('paper_sims_kinematics')
tng, birth, pbirth = const('fetch_tng50'), const('build_sims_birth'), const('paper_sims_birth')
scr, vc, lsc, settle = const('sims_scramble'), const('build_sims_vcirc'), const('paper_letter_scramble'), const('sims_halo_settle')
conc, shp, psidm = const('fetch_concerto'), const('sims_sidm_shapes'), const('paper_sims_sidm')
```

#### Fig. 1: the signal in the data
Settings and sample sizes of the Letter's Fig. 1.

```python
# Fig. 1 (scripts/paper_letter_signal.py)
# Numbers of BOSS and APOGEE giants in Fig. 1 (one-row table)
counts = pd.read_csv(ROOT / 'results/letter_signal_counts.csv').iloc[0]
edges = [hi for lo, hi in FEH_BINS[:-1]]                              # lmcold.samples.FEH_BINS: -1.5, -1.1, -0.8, -0.5
# Settings of Fig. 1 (footprint radius, [Fe/H] edges, ring, pixel sizes, clipping, reference grid) and the
# sample sizes with thin spaces as thousands separators
M.update(RrlRho=num(RHO_MAX), FehA=num(edges[0]), FehB=num(edges[1]), FehC=num(edges[2]), FehD=num(edges[3]),
         RingIn=num(kin['RING'][0]), RingOut=num(kin['RING'][1]), RrlFitIn=num(kin['OUTER_INNER']), Pix=num(sig['PIX']), PixAlpha=num(100 * sig['PIX_ALPHA']),
         NBoss=f"{int(counts.boss):,}".replace(',', r'\,'), NApogee=f"{int(counts.apogee):,}".replace(',', r'\,'),
         RrPix=num(sig['RR_PIX']), RrSmooth=num(sig['RR_SMOOTH']), RrLevLo=num(100 * sig['RR_LEVELS'][0]),
         RrLevHi=num(100 * sig['RR_LEVELS'][1]), NLit=num(len(LMC_MASS_LITERATURE)),
         MassBin=num(np.diff(FEH_EDGES[1:-1])[0]), Clip=num(sig['CLIP']), ClipErr=num(sig['CLIP_ERR']),
         ProfIn=num(sig['PROFILE_RING'][0]), ProfOut=num(sig['PROFILE_RING'][1]), Sector=num(sig['SECTOR']),
         MedianSE=f"{sig['MEDIAN_SE']:.2f}", NMinPA=num(sig['NMIN']), RefDpa=num(sig['REF_DPA']),
         RefRhoA=num(sig['REF_RHO'][0]), RefRhoB=num(sig['REF_RHO'][1]), RefRhoStep=num(sig['REF_RHO'][2]))
```

#### Fig. 2: rotational support, LMC and simulations
Settings of the mock observation, the fitted centre, the rotational support of the LMC bins against the simulated analogues,
the early formed mass and the recovery test.

```python
# Fig. 2 (scripts/sims_observe.py, paper_sims_kinematics.py, fetch_tng50.py)
# Per-bin fits; apo: True for the giant bins with an APOGEE-only v_rot (vrot4_apogee not NaN)
ks = pd.read_csv(ROOT / 'results/kin_support_ellipsoid.csv')
apo = ks[ks.population.str.startswith('giants')].reset_index(drop=True).vrot4_apogee.notna().to_numpy()
mf = pd.read_csv(ROOT / 'results/giant_mass.csv')                       # LMC cumulative formed mass, as sims_observe.lmc_knots
# Fraction of the LMC's formed mass below the upper edge of the most metal-poor bin
first = np.interp(edges[0], mf.feh_hi, mf.cumulative) / mf.cumulative.iloc[-1]
# Number of simulated analogues per suite
so = pd.read_csv(ROOT / 'results/sims_observed.csv', dtype={'name': str}).groupby('suite').name.nunique()
# Settings of Fig. 2: the radius of sigma, the analogue samples and their selection (stellar and total mass
# [1e9 Msun, Msun]), the mock-observation grid and windows
M.update(RSig=num(R_SIG), ApoFeh=num(FEH_BINS[int(np.argmax(apo))][0]), NTng=num(so['TNG50']), NAu=num(so['Auriga']),
         NFire=num(so['FIRE-2']), MstarLo=num(tng['MSTAR'][0] / 1e9), MstarHi=num(tng['MSTAR'][1] / 1e9),
         MtotLo=tex_sci(tng['MTOT'][0]), MtotHi=tex_sci(tng['MTOT'][1]),
         AuNames=listing([k.replace('Au', '') for k in sims.AURIGA]), FireNames=listing(list(sims.FIRE)),
         RMax=num(sims.R_MAX), AlignR=num(sims.ALIGN_R), Dist=num(LMC['D0']), FirstFrac=num(round(100 * first)),
         CellR=num(obs['CELL_R']), CellPhi=num(obs['CELL_PHI']), VlosLo=num(obs['VLOS_WINDOW'][0]), VlosHi=num(obs['VLOS_WINDOW'][1]),
         NMaxFit=num(obs['NMAX']), NMinSim=num(ksim['NMIN_SIM']))
# The fitted centre: its distance [deg] from the adopted (van der Marel & Kallivayalil 2014) centre, from the stage-2 shared dx, dy
# (tangent-plane offsets) of the joint ellipsoid fit (results/kin_mcmc_data_ellipsoid.csv), to one decimal.
kc = pd.read_csv(ROOT / 'results/kin_mcmc_data_ellipsoid.csv').query("stage == 2 and population == 'shared'").set_index('param')['median']
M.update(CentreOffset=f"{np.hypot(kc['dx'], kc['dy']):.1f}")
# Fig. 2 results. LMC (results/kin_support_ellipsoid.csv, stage-1 ellipsoid fits, posterior medians at R_SIG): the most metal-poor
# (MP) and most metal-rich (MR) giant bins, the MR bin also for APOGEE alone, and the RR Lyrae (sigma_R: proper motions alone do not
# measure sigma_z). Simulations (results/sims_kinematics_summary.csv, written by paper_sims_kinematics.py): medians, 16th-84th
# percentiles and maxima over the analogues in the MP bin; per cent of TNG50 analogues at or above the lower edge of the LMC band in
# v_rot/sigma_z per bin (as printed in Fig. 2c); analogues at or above it in the second bin.
g = ks[ks.population.str.startswith('giants')].reset_index(drop=True)
rl = ks[ks.population == 'RR Lyrae'].iloc[0]
sk = pd.read_csv(ROOT / 'results/sims_kinematics_summary.csv').set_index(['suite', 'population', 'quantity'])
# pct: per cent of TNG50 analogues at or above the LMC band, one value per giant bin
pct = [100 * sk.loc[('TNG50', k, 'ratio_z'), 'n_above_lmc'] / sk.loc[('TNG50', k, 'ratio_z'), 'n_analogues'] for k in range(len(g))]
M.update(KinVrotMP=num(round(g.vrot4.iloc[0])), KinVrotMR=num(round(g.vrot4.iloc[-1])),
         KinSigMP=num(round(g.sigma_z.iloc[0])), KinSigMR=num(round(g.sigma_z.iloc[-1])),
         KinRatioMP=num(round(g.ratio_z.iloc[0], 1)), KinRatioMR=num(round(g.ratio_z.iloc[-1], 1)),
         KinRatioMRApo=num(round(g.ratio_z_apogee.iloc[-1], 1)), KinSigRMP=num(round(g.sigma_R.iloc[0])),
         KinRatioRMP=num(round(g.ratio_R.iloc[0], 1)), KinRrlVrot=num(round(rl.vrot4)), KinRrlSigR=num(round(rl.sigma_R)),
         KinRrlRatioR=num(round(rl.ratio_R, 1)),
         TngVrotMP=num(round(sk.loc[('TNG50', 0, 'vrot4'), 'median'])), TngSigMP=num(round(sk.loc[('TNG50', 0, 'sigma_z'), 'median'])),
         TngSigMinMP=num(round(sk.loc[('TNG50', 0, 'sigma_z'), 'p16'])),
         TngRatioMP=f"{sk.loc[('TNG50', 0, 'ratio_z'), 'median']:.1f}",
         TngRatioMPLo=f"{sk.loc[('TNG50', 0, 'ratio_z'), 'p16']:.1f}",
         TngRatioMPHi=f"{sk.loc[('TNG50', 0, 'ratio_z'), 'p84']:.1f}",
         TngRatioMPMax=f"{sk.loc[('TNG50', 0, 'ratio_z'), 'max']:.1f}",
         AuRatioMP=f"{sk.loc[('Auriga', 0, 'ratio_z'), 'median']:.1f}",
         FireRatioMP=f"{sk.loc[('FIRE-2', 0, 'ratio_z'), 'median']:.1f}",
         AuRatioMRMax=f"{sk.loc[('Auriga', len(g) - 1, 'ratio_z'), 'max']:.1f}",
         FireRatioMRMax=f"{sk.loc[('FIRE-2', len(g) - 1, 'ratio_z'), 'max']:.1f}",
         TngNAboveB=num(sk.loc[('TNG50', 1, 'ratio_z'), 'n_above_lmc']),
         TngPctC=num(round(pct[2])), TngPctD=num(round(pct[3])), TngPctE=num(round(pct[4])))
# Early formed mass: the LMC's cumulative formed mass below the upper edge of the most metal-poor bin (results/giant_present_mass.csv,
# the table of Fig. 1b; bootstrap 16th-84th percentiles), an upper bound on the mass formed when those stars were born; and its ratio
# to the stellar mass at which a simulated Milky Way progenitor forms its disc, M_DISC_SEMENOV (Semenov et al. 2025a, ApJ, sections
# 3.3 and 4: 'when the galaxy stellar mass reaches ~2e9 Msun', in TNG50 and in their cold-ISM resimulation alike).
M_DISC_SEMENOV = 2e9
M_MW_SPINUP = 5e8   # Msun formed when the Milky Way spun up its disc (Belokurov & Kravtsov 2023; Kurbatov et al. 2024), as in the Introduction
gp = pd.read_csv(ROOT / 'results/giant_present_mass.csv')
# Cumulative formed mass [Msun] (median, p16, p84) at the upper edge of the most metal-poor bin;
# e7: its power of ten, used to write the mass as m x 10^e
em = {c: np.interp(edges[0], gp.feh_hi, gp[c]) for c in ('cumulative', 'cumulative_p16', 'cumulative_p84')}
e7 = 10 ** np.floor(np.log10(em['cumulative']))
M.update(EarlyMass=rf"{em['cumulative'] / e7:.0f}\times10^{{{int(np.log10(e7))}}}",
         EarlyMassLo=f"{em['cumulative_p16'] / e7:.0f}", EarlyMassHi=f"{em['cumulative_p84'] / e7:.0f}",
         EarlyExp=num(np.log10(e7)), EarlyRatio=num(10 * round(M_DISC_SEMENOV / em['cumulative'] / 10)),
         EarlyRatioMW=num(round(M_MW_SPINUP / em['cumulative'])))
# The most metal-poor bin's metallicities: its median [Fe/H] (results/kin_support_ellipsoid.csv) and the per cent of its fitted giants
# (lmcold.samples.giants at RING) below FEH_LOW, to the nearest 5.
FEH_LOW = -2.0
# [Fe/H] [dex] of the most metal-poor bin's giants within the fitted ring
mp = np.asarray(ring(samples.giants()[0][1])['feh'])
M.update(MPFehMed=f'{g.feh_med.iloc[0]:.1f}', MPFehLow=num(FEH_LOW), MPPctLow=num(5 * round(100 * (mp < FEH_LOW).mean() / 5)))
# Giants with measured [Fe/H] < FEH_LOW (scripts/kin_feh_lowest.py, results/kin_feh_lowest.csv and kin_feh_lowest_null.csv): their
# number, the formed mass below FEH_LOW as a power of ten (giant_present_mass.csv), v_rot(4 kpc) / sigma_z of the narrow bins below
# FEH_LOW (lowest and highest, 1 decimal), the per cent of BOSS-APOGEE overlap stars with calibrated BOSS [Fe/H] < FEH_LOW that have
# APOGEE [Fe/H] < FEH_LOW (1 significant figure), and the lowest purity of the null test at which every non-rotating mock (true
# ratio 0) falls below the 16th percentile of the measured value, and its complement, the largest interloper fraction.
kl = pd.read_csv(ROOT / 'results/kin_feh_lowest.csv').set_index('sample')
kn = pd.read_csv(ROOT / 'results/kin_feh_lowest_null.csv')
# low: fit of the whole [Fe/H] < FEH_LOW sample; narrow_low: v_rot/sigma_z of its two narrow bins
low = kl.loc[f'[Fe/H] < {FEH_LOW}']
narrow_low = kl.loc[['[Fe/H] < -2.2', '-2.2 < [Fe/H] < -2.0'], 'ratio_z']
# BOSS-APOGEE overlap stars (feh_a: APOGEE, feh_b: calibrated BOSS [Fe/H]); pur: fraction confirmed by APOGEE
ov = samples.overlap()
pur = (ov.feh_a[ov.feh_b < FEH_LOW] < FEH_LOW).mean()
# Null mocks with true ratio 0: the largest fitted v_rot/sigma_z at each purity
n0 = kn[(kn.bin == f'[Fe/H] < {FEH_LOW}') & (kn.ratio == 0)].groupby('purity').fit_ratio.max()
M.update(LowFeh=num(FEH_LOW), LowN=num(int(low.n)), LowMass=rf"10^{{{round(np.log10(np.interp(FEH_LOW, gp.feh_hi, gp.cumulative)))}}}",
         LowRatioLo=f'{narrow_low.min():.1f}', LowRatioHi=f'{narrow_low.max():.1f}', LowPurity=sigfig(100 * pur),
         LowPurityMin=num(round(100 * n0[n0 < low.ratio_z_p16].index.min())))
M.update(LowInterMax=num(100 - int(M['LowPurityMin'])))   # per cent of interlopers the conclusions allow: 100 - LowPurityMin
# The interlopers' APOGEE [Fe/H] (lowest and highest, 1 decimal) among those overlap stars, and settings of kin_feh_lowest.py read
# from its source: the lowest narrow-bin edge, the S/N of the best subset, the interloper bin, the hot fraction and dispersion.
inter = ov.feh_a[(ov.feh_b < FEH_LOW) & (ov.feh_a >= FEH_LOW)]
kfl = const('kin_feh_lowest')
M.update(LowInterLo=f'{inter.min():.1f}', LowInterHi=f'{inter.max():.1f}', LowNarrowFeh=num(kfl['BINS'][0][1]),
         LowSnr=num(kfl['SNR_BEST']), LowInterBinLo=num(kfl['INTERLOPER_BIN'][0]), LowInterBinHi=num(kfl['INTERLOPER_BIN'][1]),
         LowHotPct=num(round(100 * max(kfl['HOT']))), LowHotSigma=num(kfl['HOT_SIGMA']))
# Sizes: exponential scale lengths h [kpc] of the selection-weighted surface densities over H_FIT (scripts/kin_vc_drift.py,
# results/kin_vc_drift_h.csv) for the most metal-rich and most metal-poor giant bins and the RR Lyrae, and the half-mass radii
# R_e = RE_PER_H h that the exponential would give if it continued inside H_FIT[0] (projected exponential disc: R_e / h = 1.678).
RE_PER_H = 1.678
# Scale length h [kpc] per population, and the fitted radial range H_FIT [kpc]
hs = pd.read_csv(ROOT / 'results/kin_vc_drift_h.csv').set_index('population').h
h_fit = const('kin_vc_drift')['H_FIT']
# h and R_e [kpc] of the most metal-poor and most metal-rich giant bins and the RR Lyrae, 1 decimal
for key, pop in (('MP', f'giants [Fe/H] < {num(edges[0])}'), ('MR', f'giants [Fe/H] > {num(edges[3])}'), ('Rrl', 'RR Lyrae')):
    M.update({f'H{key}': f'{hs[pop]:.1f}', f'Re{key}': f'{RE_PER_H * hs[pop]:.1f}'})
M.update(HFitLo=num(h_fit[0]), HFitHi=num(h_fit[1]))
# Recovery (paper_sims_kinematics.py, figures/paper_sims_recovery.png): fitted against the particle values of the same mock giants,
# all analogues and bins of results/sims_observed.csv; median |v_rot - truth| [km/s, integer] and |sigma_z / truth - 1| [per cent].
so_all = pd.read_csv(ROOT / 'results/sims_observed.csv', dtype={'name': str})
M.update(RecVrot=num(max(1, round(abs(np.nanmedian(so_all.vrot4 - so_all.t_vphi))))),
         RecSig=num(max(1, round(100 * abs(np.nanmedian(so_all.sigma_z / so_all.t_sigma_z) - 1)))))
# TNG50 analogues as cold as the LMC in the MP bin (sigma_z at or below the upper edge of the LMC band, results/sims_observed.csv),
# and the sigma_z and v_rot of the coldest.
t0 = pd.read_csv(ROOT / 'results/sims_observed.csv', dtype={'name': str}).query("suite == 'TNG50' and population == 0")
# The TNG50 analogue with the lowest sigma_z [km/s] in the most metal-poor bin
cold = t0.loc[t0.sigma_z.idxmin()]
M.update(TngNColdMP=num((t0.sigma_z <= g.sigma_z_p84.iloc[0]).sum()), TngColdSig=num(round(cold.sigma_z)),
         TngColdVrot=num(round(cold.vrot4)))
```

#### Fig. 3: the simulated stars at birth
Settings and results for the old cohorts at birth and today.

```python
# Fig. 3 (scripts/build_sims_birth.py, paper_sims_birth.py, lmcold.sims)
# Birth cohorts of TNG50 and Auriga: v_phi and sigma_z [km/s] at birth and today, per cohort age [Gyr]
sb = pd.read_csv(ROOT / 'results/sims_birth.csv', dtype={'halo': str})
# Fig. 3 results: cohorts of OLD_AGES Gyr (the ages of the two most metal-poor LMC giant bins in Fig. 3), at birth and today. Each
# analogue's median over its cohorts in that range, then the median and 16th-84th percentiles over the analogues of each suite.
OLD_AGES = (10.0, 12.0)
# Cohorts of 10-12 Gyr; rb, rt: v_phi/sigma_z at birth and today (same stars)
ob = sb[(sb.age >= OLD_AGES[0]) & (sb.age < OLD_AGES[1])].assign(rb=lambda d: d.vphi_birth / d.sz_birth,
                                                                     rt=lambda d: d.vphi_same / d.sz_same)
# Per-analogue medians, then the 16th, 50th and 84th percentiles over the analogues of each suite
og = ob.groupby(['suite', 'halo'])[['rb', 'rt', 'vphi_birth', 'sz_birth', 'vphi_same', 'sz_same']].median()
oq = {s: og.loc[s].quantile([0.16, 0.5, 0.84]) for s in ('TNG50', 'Auriga')}
M.update(OldAgeLo=num(OLD_AGES[0]), OldAgeHi=num(OLD_AGES[1]),
         TngRbMed=num(round(oq['TNG50'].rb[0.5], 1)), TngRbLo=num(round(oq['TNG50'].rb[0.16], 1)),
         TngRbHi=num(round(oq['TNG50'].rb[0.84], 1)), TngRtMed=f"{oq['TNG50'].rt[0.5]:.1f}",
         TngRtLo=f"{oq['TNG50'].rt[0.16]:.1f}", TngRtHi=f"{oq['TNG50'].rt[0.84]:.1f}",
         TngVbMed=num(round(oq['TNG50'].vphi_birth[0.5])), TngVtMed=num(round(oq['TNG50'].vphi_same[0.5])),
         TngSbMed=num(round(oq['TNG50'].sz_birth[0.5])), TngStMed=num(round(oq['TNG50'].sz_same[0.5])),
         AuRbMed=num(round(oq['Auriga'].rb[0.5], 1)), AuRtMed=num(round(oq['Auriga'].rt[0.5], 1)),
         KinRatioB=num(round(g.ratio_z.iloc[1], 1)))
# Numbers of analogues and the settings of build_sims_birth.py, lmcold.sims and paper_sims_birth.py
M.update(NTngBirth=num(sb[(sb.suite == 'TNG50')].dropna(subset=['vphi_birth']).halo.nunique()),
         NAuBirth=num(sb[sb.suite == 'Auriga'].halo.nunique()), RToday=num(birth['R_TODAY']), Cohort=num(birth['COHORT']),
         MinCohort=num(birth['MIN_COHORT']), TrackGroup=num(default(sims.progenitor_track, 'group')),
         TrackSmooth=num(default(sims.progenitor_track, 'smooth')), RBirth=num(birth['R_BIRTH']),
         RMinBirth=num(default(sims.birth_phase_space, 'r_min')), MinIn=num(birth['MIN_IN']), PlotBin=num(pbirth['PLOT_BIN']),
         AgeMax=num(pbirth['AGE_MAX']))
```

#### Fig. 4: scrambling and the settling halo
Settings and results of the three panels and the settling timescale.

```python
# Fig. 4 (scripts/sims_scramble.py, build_sims_vcirc.py, sims_halo_settle.py, paper_letter_scramble.py, fetch_tng50_progenitors.py)
# Halo state per (progenitor, snapshot); zsnap: median redshift of each snapshot
hs = pd.read_csv(ROOT / 'results/sims_halo_settle.csv', dtype={'halo': str})
zsnap = hs.groupby('snap').z.median()
# Cohorts of the real stars (variant A), the star-forming gas at z >= Z_GAS, the orbital times, the time bins [Gyr]
sc = pd.read_csv(ROOT / 'results/sims_scramble.csv', dtype={'halo': str})
sc = sc[sc.variant == 'A']
gas = pd.read_csv(ROOT / 'results/sims_heating_gas.csv', dtype={'halo': str}).query(f"z >= {lsc['Z_GAS']}")
v = pd.read_csv(ROOT / 'results/sims_vcirc.csv', dtype={'halo': str})
te = lsc['TIME_EDGES']
# Settings of the scramble, orbital-time, settling and plotting scripts, and the sample sizes
M.update(NFull=num(len(FULL)), SnapA=num(FULL_SNAPS[0]), SnapB=num(FULL_SNAPS[-1]),
         ZSnapA=num(round(zsnap[FULL_SNAPS[0]], 1)), ZSnapB=num(round(zsnap[FULL_SNAPS[-1]], 1)),
         Window=num(1000 * scr['WINDOW']), RCohort=num(scr['R_COHORT']), MinStars=num(scr['MIN_STARS']),
         BirthEvery={2: 'second'}.get(scr['BIRTH_EVERY'], f"{scr['BIRTH_EVERY']}th"), DtMax=num(scr['DTMAX']),
         NCohorts=num(sc.groupby('halo').s_birth.nunique().sum()), NCohortGal=num(sc.halo.nunique()), RVel=num(scr['R_VEL']),
         RSupIn=num(scr['R_SUP'][0]), RSupOut=num(scr['R_SUP'][1]), Lmax=num(scr['LMAX']), Grid=num(scr['GRID']), RMinPot=num(scr['RMIN']),
         RMaxPot=num(scr['RMAX']), TimeEnd=num(te[-1]), TimeA=num(te[1]), TimeB=num(te[2]), TimeC=num(te[3]),
         ZGas=num(lsc['Z_GAS']), NGas=num(gas.halo.nunique()), ROrb=num(vc['R_ORB']), NOrbGal=num(v.halo.nunique()),
         TOrb=sigfig(v.t_orb.median()), TAfterA=num(lsc['T_AFTER'][0]), TAfterB=num(lsc['T_AFTER'][1]),
         NEps={4: 'quartiles'}.get(lsc['N_EPS'], f"{lsc['N_EPS']} quantiles"), SetLmax=num(settle['LMAX']), SetGrid=num(settle['GRID']),
         SetRMin=num(settle['RMIN']), SetRPot=num(settle['R_POT']), SetRZero=num(settle['R0']), SetRIn=num(settle['R_IN']),
         NSample=num(hs.halo.nunique()), ZSetHi=num(round(hs.z.max())), ZSetLo=num(round(hs.z.min())),
         TrackStep={3: 'third'}[TRACK_SNAPS[1] - TRACK_SNAPS[0]], NSettle=num(len(hs)), Epoch=num(lsc['EPOCH']))
# Fig. 4a results, as plotted by paper_letter_scramble.py: v_phi/sigma_z about the birth spin in the TIME_EDGES bins (per galaxy the
# median, then the median over galaxies) for the real stars (variant A, measured), frozen gravity (E), dark matter non-round only
# (G_DM) and axisymmetric (D0) test particles; the bins' median times; the star-forming gas (median over galaxies).
# All variants, plus the real stars as their own variant 'real' (measured v_phi/sigma_z); tb: median time
# [Gyr] of the TIME_EDGES bin of each row; va: median over galaxies of the per-galaxy medians per bin, for each variant
sa = pd.read_csv(ROOT / 'results/sims_scramble.csv', dtype={'halo': str}).replace([np.inf, -np.inf], np.nan)
sa = pd.concat([sa, sa[sa.variant == 'A'].assign(variant='real', vs_sim_birth=lambda d: d.vs_meas_birth)])
sa['tb'] = sa.groupby(pd.cut(sa.t, np.r_[-1e-9, te[1:-1], te[-1] + 1e-6])).t.transform('median')
va = {k: sa[sa.variant == k].groupby(['tb', 'halo']).vs_sim_birth.median().groupby(level=0).median()
      for k in ('real', 'E', 'G_DM', 'D0')}
# Values at the first, second and last time bins; ScrOrbEnd: the last time in units of the median orbital time
tb = va['real'].index
M.update(ScrStart=num(round(va['real'].iloc[0], 1)), ScrSecond=f"{va['real'].iloc[1]:.1f}",
         ScrEnd=num(round(va['real'].iloc[-1], 1)), ScrTSecond=f'{tb[1]:.2f}', ScrTEnd=sigfig(tb[-1]),
         ScrOrbEnd=num(round(tb[-1] / v.t_orb.median(), 1)), FrozenEnd=num(round(va['E'].iloc[-1], 1)),
         DmEnd=num(round(va['G_DM'].iloc[-1], 1)), AxiEnd=num(round(va['D0'].iloc[-1], 1)),
         GasRatio=num(round((gas.vphi / gas.sz).groupby(gas.halo).median().median(), 1)))
# Fig. 4b results, as paper_letter_scramble.py: each cohort's median v_phi/sigma_z T_AFTER Gyr after birth against eps2 at its birth
# snapshot (results/sims_halo_settle.csv); medians of eps2 and v_phi/sigma_z in the lowest and highest N_EPS-quantiles; Spearman rank
# correlation.
# af: median v_phi/sigma_z of each real cohort between T_AFTER[0] and T_AFTER[1] Gyr after birth;
# kb: joined to the halo state at its birth snapshot; qb: medians in N_EPS quantiles of eps2
re_ = sa[sa.variant == 'real']
af = re_[(re_.t > lsc['T_AFTER'][0]) & (re_.t <= lsc['T_AFTER'][1])].groupby(['halo', 's_birth']).vs_sim_birth.median().rename('vs')
kb = af.reset_index().merge(hs, left_on=['halo', 's_birth'], right_on=['halo', 'snap'], how='inner')
qb = kb.groupby(pd.qcut(kb.eps2, lsc['N_EPS']), observed=True).agg(x=('eps2', 'median'), m=('vs', 'median'))
M.update(EpsLow=sigfig(qb.x.iloc[0]), EpsHigh=sigfig(qb.x.iloc[-1]), VsEpsLow=num(round(qb.m.iloc[0], 1)),
         VsEpsHigh=sigfig(qb.m.iloc[-1]), EpsSpearman=sigfig(kb.vs.corr(kb.eps2, method='spearman')),
         NCohortsB=num(len(kb)))
# Fig. 4c results, as paper_letter_scramble.by_epoch: per galaxy the median in EPOCH-Gyr bins of lookback time, then the median over
# the progenitors; the earliest and latest bins (angles to the nearest 5 deg, the precision of a median over galaxies).
# Median over progenitors of the per-progenitor medians of psi [deg], eps2 and c/a in EPOCH-Gyr lookback bins
ep = hs.assign(tb=(hs.lookback / lsc['EPOCH']).round() * lsc['EPOCH']).groupby(['tb', 'halo'])[['misalign', 'eps2', 'c_a']].median()
ep = ep.groupby(level=0).median()
# Settling timescale (Fig. 4c medians): from the peak of psi until psi falls below half of that peak (alignment), and from the earliest
# bin until eps2 is within 25 per cent of its value in the latest bin (quadrupole); lookback bins are descending in time.
lbs = ep.sort_index(ascending=False)
# t_psi, t_eps [Gyr]: the alignment and quadrupole settling times defined above
pk = lbs.misalign.idxmax()
t_psi = pk - lbs[(lbs.index < pk) & (lbs.misalign < 0.5 * lbs.misalign.max())].index.max()
t_eps = lbs.index.max() - lbs[lbs.eps2 < 1.25 * lbs.eps2.iloc[-1]].index.max()
M.update(SettleLo=num(round(t_psi)), SettleHi=num(round(t_eps)))
# Earliest (largest lookback) and latest bins: psi to the nearest 5 deg, eps2 to 1 significant figure, c/a
M.update(LbEarly=num(ep.index.max()), LbLate=num(ep.index.min()), PsiEarly=num(5 * round(ep.misalign.iloc[-1] / 5)),
         PsiLate=num(5 * round(ep.misalign.iloc[0] / 5)), EpsEarly=sigfig(ep.eps2.iloc[-1]), EpsLate=sigfig(ep.eps2.iloc[0]),
         CaEarly=f'{ep.c_a.iloc[-1]:.2f}', CaLate=f'{ep.c_a.iloc[0]:.2f}')
```

#### Fig. 5: SIDM
Settings and results of the CDM and SIDM Concerto runs, and the birth radii of the simulated early discs.

```python
# Fig. 5 (scripts/fetch_concerto.py, sims_sidm_shapes.py, paper_sims_sidm.py)
# Shape profiles of the CDM and SIDM runs per redshift; zs: the redshifts in descending order, 1 decimal
ss = pd.read_csv(ROOT / 'results/sims_sidm_shapes.csv')
zs = [num(round(z, 1)) for z in sorted(ss.z.unique(), reverse=True)]
mpart = pd.read_csv(ROOT / 'results/sims_sidm_particle_mass.csv').set_index('run').m_part['CDM']   # from the snapshot header
# Settings of the Concerto runs (cross-section, velocity scale) and of the shape and plotting scripts
M.update(MPart=tex_sci(mpart), SigmaZero=num(conc['SIDM_SIGMA0_M']), WSidm=num(conc['SIDM_W']), ZList=listing(zs),
         CoreBox=num(shp['CORE_BOX']), NPeaks={5: 'five'}.get(shp['N_PEAKS'], num(shp['N_PEAKS'])), ShrinkA=num(shp['CORE_SHRINK'][0]),
         ShrinkB=num(shp['CORE_SHRINK'][1]), CoreR=num(shp['CORE_R']), RShellMin=num(shp['R_SHELL_MIN']),
         NShellMin=f"10^{{{int(np.log10(psidm['NMIN']))}}}", RKeep=num(shp['R_KEEP']), SLmax=num(shp['LMAX']), SGrid=num(shp['GRID']),
         SRMin=num(shp['RMIN']), FrameA=num(shp['FRAME_SHELL'][0]), FrameB=num(shp['FRAME_SHELL'][1]), REps=num(shp['R_EPS']),
         ZBin=num(psidm['Z_BIN']), ZMin=num(psidm['Z_MIN']), MinGal=num(psidm['MIN_GAL']))
# Fig. 5 results (results/sims_sidm_shapes.csv): eps2,all and eps2 (non-axisymmetric about the minor axis) at R_EPS for each run and
# redshift; c/a at R_EPS interpolated over the shells of at least NMIN particles (as plotted).
# e2: the rows at R_EPS per (run, z); ca: c/a at R_EPS per (run, z) from the shells with >= NMIN particles
e2 = ss[np.isclose(ss.r, shp['R_EPS'])].set_index(['run', 'z'])
ca = {k: np.interp(shp['R_EPS'], g.r, g.s) for k, g in ss[ss.N >= psidm['NMIN']].groupby(['run', 'z'])}
ca = pd.Series(ca)
zhi, zlate = ss.z.max(), 1.6   # the first redshift; 'late': z <= 1.5 (the 1.5 and 0 outputs)
# Ranges of eps2 over redshift for CDM; SIDM at the first redshift and late; CDM/SIDM ratio late (nearest 5); c/a
M.update(SiCdmEpsLo=sigfig(e2.loc['CDM'].eps2.min()), SiCdmEpsHi=sigfig(e2.loc['CDM'].eps2.max()),
         SiCdmNaLo=sigfig(e2.loc['CDM'].eps2_na.min()), SiCdmNaHi=sigfig(e2.loc['CDM'].eps2_na.max()),
         SiEpsFirst=sigfig(e2.loc[('SIDM', zhi)].eps2), SiNaFirst=sigfig(e2.loc[('SIDM', zhi)].eps2_na),
         SiEpsLate=sigfig(e2.loc['SIDM'].query(f'z < {zlate}').eps2.max()),
         SiRatioLate=num(5 * round(e2.loc['CDM'].query(f'z < {zlate}').eps2.min() / e2.loc['SIDM'].query(f'z < {zlate}').eps2.max() / 5)),
         SiCaCdmLo=f"{ca['CDM'].min():.1f}", SiCaCdmHi=f"{ca['CDM'].max():.1f}", SiCaSidmMin=f"{ca['SIDM'].min():.1f}",
         SiCaSidmMed=f"{ca['SIDM'].median():.1f}", SiZFirst=num(round(zhi)))

assert all(k.isalpha() for k in M), [k for k in M if not k.isalpha()]   # TeX command names: letters only
# Why the simulated early discs are measured at 0.5-3 kpc (results/sims_birth_radii.csv, scripts/sims_birth_radii.py; TNG50):
# star particles of the old cohorts at 3-5 kpc relative to 0.5-3 kpc; per cent of the cohorts with enough particles at 0.5-3 kpc
# that still have enough at 3-5 kpc (nearest 5); of the old stars at 3-5 kpc today, the median per cent born at 0.5-3 kpc and at
# 3-5 kpc (nearest 5).
# One row per TNG50 analogue: particle and cohort counts at 0.5-3 and 3-5 kpc, birth-radius fractions
br = pd.read_csv(ROOT / 'results/sims_birth_radii.csv', dtype={'halo': str}).query("suite == 'TNG50'")
M.update(RadPartRatio=f'{br.n_outer.sum() / br.n_inner.sum():.1f}',
         RadCohKept=num(5 * round(100 * br.coh_outer.sum() / br.coh_inner.sum() / 5)),
         RadBornInner=num(5 * round(100 * br.f_born_inner.median() / 5)),
         RadBornOuter=num(5 * round(100 * br.f_born_outer.median() / 5)))
```

#### Auriga and FIRE-2
The Auriga and FIRE-2 numbers of the Letter, rounded from results/macros_sims_suites.tex, and FIRE-2 at birth.

```python
# Section 3 summary: Auriga and FIRE-2 (results/macros_sims_suites.tex, written by scripts/paper_text_sims_suites.py from
# results/sims_birth.csv and the research tables; definitions there), rounded for the Letter.
# Macro name (without the \ss prefix) -> value, parsed from the \newcommand lines
ssm = dict(re.findall(r'\\newcommand\{\\ss(\w+)\}\{([^}]*)\}', (ROOT / 'results/macros_sims_suites.tex').read_text()))
M.update({f'Su{k}': f'{float(ssm[k]):.1f}' for k in ('OldAuAfter', 'OldFireAfter')})   # v_phi/sigma_z: 1 decimal
# Settings of paper_text_sims_suites.py: the age limit [Gyr] of the oldest stars and the time after birth [Gyr]
sus = const('paper_text_sims_suites')
M.update(SuOldLb=num(sus['OLD_LB']), SuTAfter=num(sus['T_AFTER']))
# FIRE-2 at birth with the definition of Fig. 3 (OLD_AGES cohorts; per galaxy the median, then the median over the galaxies):
# results/research/fire_early_birth.csv, cohorts of 0.1 Gyr (lb: lookback time at birth).
fb = pd.read_csv(ROOT / 'results/research/fire_early_birth.csv').query('width == 0.1')
# Cohorts born 10-12 Gyr ago; per-galaxy median v_phi/sigma_z, then the median over galaxies
fb = fb[(fb.lb >= OLD_AGES[0]) & (fb.lb < OLD_AGES[1])]
M.update(FireRbMed=f'{(fb.vphi / fb.sz).groupby(fb.halo).median().median():.1f}')
M.update({f'Su{k}': sigfig(float(ssm[k])) for k in ('AuRound', 'AuAxi')})   # fractions of the loss
# Number of Auriga cohorts in the halo tests
M.update(SuAuN=ssm['AuN'])
```

#### Write the macros
One \newcommand per entry of M, with the prefix \lt, into results/macros_letter.tex; the text shows them as macros, so no
number is typed by hand.

```python
# Header line, then one \newcommand per macro; written to results/macros_letter.tex and printed
lines = ['% Numbers of the Letter (scripts/paper_text_letter.py); do not edit.']
lines += [rf'\newcommand{{\lt{k}}}{{{v}}}' for k, v in M.items()]
(ROOT / 'results/macros_letter.tex').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
```

### Sampling of the kinematic fits

The sentences on the sampling of the kinematic fits (`results/text_mcmc_*.tex`), written whole from the run settings and the
convergence diagnostics.

<details><summary>About <code>scripts/paper_text_mcmc.py</code></summary>

```text
Paper text: the sentences on the NUTS sampling of the kinematic fits (Section 4.2 'Posteriors' and Appendix app:kin_sky), from
the run settings (scripts/fit_kinematics_mcmc.py: CHAINS, DRAWS, WARMUP, WARMUP_ELLIPSOID) and the diagnostics in the result tables
(r_hat, n_eff, divergences per stage and population).

- results/text_mcmc_ellipsoid.tex: velocity-ellipsoid fits (results/kin_mcmc_data_ellipsoid.csv, giants and RR Lyrae at 2-9 deg;
  kin_mcmc_data_ellipsoid_outer.csv, 4-9 deg, RR Lyrae dispersions).
- results/text_mcmc_stream.tex: sky-aligned fits (kin_mcmc_data_stream.csv, kin_mcmc_data_stream_outer.csv).
Bounds: max r_hat rounded up to 3 decimals, min n_eff rounded down. Each sentence names the fits with divergent transitions; the
script stops if their number differs from what the sentence's wording covers (one fit), so the text cannot silently go wrong.
Copied to the paper's tables/ and \\input in the text.
Run: .venv/bin/python scripts/manifest.py run paper_text_mcmc.py
```
</details>

**Data read by `scripts/paper_text_mcmc.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_mcmc_data_ellipsoid_outer.csv` | repository | the same fit at 4-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid outer` |
| `results/kin_mcmc_data_stream.csv` | repository | posterior summary of the sky-aligned dispersion model with m = 2 streaming, 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data stream` |
| `results/kin_mcmc_data_stream_outer.csv` | repository | the same fit at 4-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data stream outer` |

*Shown, not run here: the posterior summaries of the kinematic fits in results/. The cells below are the script itself; run it from the repository with* `python scripts/paper_text_mcmc.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
The run settings of the fits and two helpers: the diagnostics of a set of runs, and a check that exactly one fit has divergent
transitions.

```python
import math

import pandas as pd

from fit_kinematics_mcmc import CHAINS, DRAWS, WARMUP, WARMUP_ELLIPSOID
from lmcold.sources import ROOT
from lmcold.tex import GROUP_LABEL, count

WORD = {4: 'four', 8: 'eight'}  # number of chains as written in the text


def diagnostics(*runs):
    """(max r_hat rounded up to 3 decimals, min n_eff rounded down, [(stage, population, divergences) with divergences > 0])."""
    # All rows of the given runs (one per stage x population x parameter), tagged with the run name.
    t = pd.concat([pd.read_csv(ROOT / f'results/kin_mcmc_data_{r}.csv').assign(run=r) for r in runs])
    # Divergences of each fit (the same value on every row of a fit), and the fits that have any.
    d = t.groupby(['run', 'stage', 'population']).divergences.max()
    div = [(s, p, int(n)) for (_, s, p), n in d[d > 0].items()]
    # Bounds safe for the text: largest r_hat rounded up, smallest n_eff rounded down.
    return math.ceil(t.r_hat.max() * 1000) / 1000, math.floor(t.n_eff.min()), div


def one_fit(div, what):
    """The single fit with divergences; stop if there is not exactly one (the wording assumes it)."""
    # Stop rather than write a wrong sentence.
    if len(div) != 1:
        raise SystemExit(f'{what}: {len(div)} fits with divergent transitions, the sentence is written for one: {div}')
    return div[0]
```

#### Velocity-ellipsoid fits
The sentence on the sampling of the velocity-ellipsoid fits.

```python
# r: max r_hat, n: min n_eff over the two ellipsoid runs; the one fit with divergences (stage, population, number).
r, n, div = diagnostics('ellipsoid', 'ellipsoid_outer')
stage, pop, nd = one_fit(div, 'ellipsoid')
# Name of that fit in the text (giant groups by their paper label, lmcold.tex.GROUP_LABEL).
who = 'the RR Lyrae' if pop == 'RR Lyrae' else 'giants, ' + GROUP_LABEL[pop]
# The sentence: sampler, chains, draws and warm-up from fit_kinematics_mcmc.py, then the diagnostics.
ellipsoid = (f'We sample the posteriors with NUTS \\citep{{Hoffman2014}} in {{\\sc numpyro}} \\citep{{Phan2019}}, {WORD[CHAINS]} chains '
             f'of {count(DRAWS)} draws after {count(WARMUP_ELLIPSOID)} warm-up steps ({count(WARMUP)} for the RR Lyrae fit at '
             f'$4<\\rho<9^\\circ$). Every run has $\\hat R\\leq{r:.3f}$ and at least {count(n)} effective draws per parameter; one '
             f'stage-{stage} fit ({who}) has {nd} divergent transitions among its {count(CHAINS * DRAWS)} draws, the others none.')
```

#### Sky-aligned fits
The sentence on the sky-aligned fits; the script stops if the divergences are not in the RR Lyrae fit the sentence names.

```python
# The same diagnostics for the two sky-aligned runs; exactly one fit with divergences, and it must be the RR Lyrae fit.
r, n, div = diagnostics('stream', 'stream_outer')
stage, pop, nd = one_fit(div, 'stream')
if pop != 'RR Lyrae':
    raise SystemExit(f'stream: the sentence names the RR Lyrae fit at 4-9 deg, but the divergences are in {pop}')
# The sentence of the sky-aligned fits.
stream = (f'The sky-aligned fits sampled the posteriors with $\\hat R\\leq{r:.3f}$ and at least {count(n)} effective draws per '
          f'parameter; the RR Lyrae fit at $4<\\rho<9^\\circ$ had {nd} divergent transitions.')
```

#### Write the sentences
Each sentence into results/text_mcmc_<name>.tex.

```python
# One .tex file per sentence (copied to the paper's tables/), and print it.
for name, text in (('ellipsoid', ellipsoid), ('stream', stream)):
    (ROOT / f'results/text_mcmc_{name}.tex').write_text(text + '\n')
    print(text)
```

### The data chapters

The numbers of the chapters on the samples, the selection functions and the calibration (`results/macros_data.tex`, prefix `\da`).

<details><summary>About <code>scripts/paper_text_data.py</code></summary>

```text
Knowledge-base text, the three data chapters ('Data: the samples', 'Data: selection functions', 'Data: RR Lyrae and BOSS
calibration'; extended draft Sections 2.1-2.2 and the data appendices): every number of their text as a LaTeX macro (prefix \\da),
so that none is typed by hand.

As scripts/paper_text_shape.py: settings are read from the code (module constants by import or scripts/textnum.const, function
defaults with textnum.default, and literals inside functions with grab() from the source), results from the tables and catalogues
the figures are drawn from. Literature values quoted with their citation stay in the text.
Inputs: results/{boss_selection, apogee_selection, boss_calibration, boss_hot_bias, boss_mock, selection_gaps, snr_cut_check,
snr_threshold_sensitivity, boss_response, giant_counts}.json; results/{rrab_membership, rrab_independence, rrab_pw_fit,
rrab_feh_spec, rrab_feh_transform, parent_extinction, kin_mcmc_data_ellipsoid}.csv; data/{rrab, giants_apogee, giants_boss,
boss_weights, boss_cartons, apogee_parent, apogee_weights, spec_giants}.parquet; scripts/{build_rrab, build_spec_giants, build_giants,
boss_selection, apogee_selection, boss_mock, build_xp_andrae, fig_rrab_membership, paper_letter_signal, paper_maps, paper_selection,
paper_selection_checks, paper_selection_gaps, paper_boss_calibration, paper_boss_hot_bias, fit_kinematics_mcmc}.py; lmcold.{constants,
footprint, rrlyrae, giants, magcloud, selection, apogee_target, mass_samples, maps, samples, kinfit}.
Writes results/macros_data.tex.
Run: .venv/bin/python scripts/manifest.py run paper_text_data.py
```
</details>

**Data read by `scripts/paper_text_data.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_parent.parquet` | Zenodo bundle 1 | APOGEE-2 LMC1-35 RGB-polygon candidates (field x star) with Gaia DR3 (2MASS best neighbour), observed flags and allStarLite columns |
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_cartons.parquet` | Zenodo bundle 1 | SDSS-V carton memberships and priorities of the BOSS parent stars (targetdb 2.0.0) |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/giants_apogee.parquet` | Zenodo bundle 1 | APOGEE DR17 giants with Gaia DR3 |
| `data/giants_boss.parquet` | Zenodo bundle 1 | SDSS-V DR20 BOSS giants with Gaia DR3, on the APOGEE scale |
| `data/raw/ogle4/fields-OGLE-IV-LMC.txt` | public archive | OGLE-IV field centres, LMC (LMC); Udalski et al. 2015, AcA 65, 1 |
| `data/raw/ogle4/fields-OGLE-IV-MBR.txt` | public archive | OGLE-IV field centres, MBR (Magellanic Bridge); Udalski et al. 2015, AcA 65, 1 |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/apogee_selection.json` | repository | fixed input |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |
| `results/boss_hot_bias.json` | repository | written by `python scripts/paper_boss_hot_bias.py` |
| `results/boss_mock.json` | repository | mock tests of the BOSS selection function; written by `python scripts/boss_mock.py` |
| `results/boss_response.json` | repository | written by `python scripts/boss_response.py` |
| `results/boss_selection.json` | repository | fixed input |
| `results/giant_counts.json` | repository | written by `python scripts/paper_giant_counts.py` |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/parent_extinction.csv` | repository | written by `python scripts/paper_parent_extinction.py` |
| `results/rrab_feh_spec.csv` | repository | fixed input |
| `results/rrab_feh_transform.csv` | repository | RR Lyrae [Fe/H] scale transformation; written by `python scripts/fig_rrab_feh_transform.py` |
| `results/rrab_independence.csv` | repository | completeness of the OGLE-IV and Gaia RR Lyrae catalogues and their union; written by `python scripts/check_rrab_independence.py` |
| `results/rrab_membership.csv` | repository | RR Lyrae foreground fraction per ring; written by `python scripts/fig_rrab_membership.py` |
| `results/rrab_pw_fit.csv` | repository | fixed input |
| `results/selection_gaps.json` | repository | written by `python scripts/paper_selection_gaps.py` |
| `results/snr_cut_check.json` | repository | fixed input |
| `results/snr_threshold_sensitivity.json` | repository | fixed input |

*Shown, not run here: the result tables in results/ and the catalogues of Zenodo bundle 1. The cells below are the script itself; run it from the repository with* `python scripts/paper_text_data.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
Imports, the dictionary M of macros (macro name without its prefix: the text it stands for) and the functions that format
numbers as the text writes them. The inputs listed in the header are read in the step that uses them.

```python
import json
import re

import healpy as hp
import numpy as np
import pandas as pd

from fit_kinematics_mcmc import ring as kin_ring
from lmcold import apogee_target, giants, kinfit, ogle, rrlyrae, samples, selection
from lmcold.constants import KAPPA, LMC
from lmcold.footprint import DMU_WINDOW, RHO_MAX, SMC_MASK_RADIUS, angular_distance
from lmcold.maps import pixel_map
from lmcold.mass_samples import TRUE_EDGES
from lmcold.sources import DATA, ROOT
from lmcold.tex import count
from textnum import const, default, num, sigfig

R = ROOT / 'results'  # folder of the result tables read here and of the macro file written
M = {}  # macro name (without the \da prefix): text it stands for


def js(name):
    """results/<name>.json as a dict."""
    return json.loads((R / f'{name}.json').read_text())  # parsed JSON


def grab(path, pattern, k=1):
    """Number in group k of the regular expression `pattern` in the source file `path` (relative to the repository root): a
    literal inside a function, read from the code rather than retyped."""
    # First match of the pattern in the file text; group k converted to float.
    return float(re.search(pattern, (ROOT / path).read_text()).group(k))


def f2(x):
    """x with two decimals."""
    return f'{x:.2f}'


def signed(x, fmt=f2):
    """x formatted by fmt with an explicit sign ('+' added to non-negative values)."""
    s = fmt(x)
    return s if s.startswith('-') else '+' + s


def pct(x, d=0):
    """Fraction x in per cent with d decimals."""
    return f'{100 * x:.{d}f}'


def pm_err(x, e):
    """Value and error, the error to one significant figure (two if its first digit is 1) and the value to the same decimals."""
    # Error string: one significant figure, or two if the error's leading digit is 1.
    s = sigfig(e)
    s = sigfig(e, 2) if f'{e:e}'.startswith('1') else s
    d = len(s.split('.')[1]) if '.' in s else 0  # decimals of the error string
    return f'{x:.{d}f}', s  # (value, error) strings


def edges(xs):
    """A list of bin edges as written in the text, e.g. '12, 15.5, 16.25'."""
    return ', '.join(num(x) for x in xs)
```

#### Chapter 1: the RR Lyrae sample
Query radius and cross-match of the RRab catalogues, the counts of the union and its members, the metallicity scales, the
period-Wesenheit fit and the foreground contamination (data/rrab.parquet and the RR Lyrae result tables).

```python
# ------------------------------------------------------------------------------------------------------------------------------
# Chapter 1: the samples
# ------------------------------------------------------------------------------------------------------------------------------
M.update(RaZero=num(LMC['ra0']), DecZero=num(LMC['dec0']), RrlQuery=num(grab('scripts/build_rrab.py', r't\.rho < (\d+)\]')),
         Match=num(grab('scripts/build_rrab.py', r'sep\.arcsec < ([\d.]+)')), FirstAdded=num(rrlyrae.FIRST_GAIA_ADDED))
# RRab union (data/rrab.parquet) and its LMC members: counts by catalogue, proper motions and [Fe/H] source.
t = pd.read_parquet(DATA / 'rrab.parquet')
mem = t[t.member]
M.update(NUnion=count(len(t)), NBoth=count((t.in_ogle & t.in_gaia).sum()), NOwn=count(t.ogle_own.sum()),
         NMembers=count(len(mem)), NMembersPM=count((mem.pmra.notna() & mem.pmra_pmdec_corr.notna()).sum()),
         NFehI=count((mem.feh_src == 'I').sum()), NFehG=count((mem.feh_src == 'G').sum()), NFehNone=count((mem.feh_src == '').sum()))
# Photometric [Fe/H] transformation (shift and Deming slope) and the offsets from three spectroscopic samples (G04, B04, B06).
tr = pd.read_csv(R / 'rrab_feh_transform.csv').iloc[0]
M.update(FehShift=f2(tr['shift']), NShift=count(tr.N), Deming=f'{tr.deming_b:.3f}', DemingErr=f'{tr.deming_b_err:.3f}')
sp = pd.read_csv(R / 'rrab_feh_spec.csv').query("estimator == 'feh_I'").set_index('spec')
M.update(GrattonOff=f2(sp.d_median.G04), GrattonErr=f2(sp.d_median_err.G04), NGratton=num(sp.N.G04),
         BorA=f2(sp.d_median.B04), BorAErr=f2(sp.d_median_err.B04), NBorA=num(sp.N.B04),
         BorB=f2(sp.d_median.B06), BorBErr=f2(sp.d_median_err.B06), NBorB=num(sp.N.B06))
# Period-Wesenheit fit: core radius and Wesenheit colour coefficients read from build_rrab.py, scatter per Wesenheit index,
# membership window in distance modulus.
pw = pd.read_csv(R / 'rrab_pw_fit.csv').set_index('wesenheit')
M.update(PwRho=num(grab('scripts/build_rrab.py', r'core = \(\(t\.rho < (\d+)\)')),
         WICoef=f2(grab('scripts/build_rrab.py', r't\.I - ([\d.]+) \* \(t\.V - t\.I\)')),
         WGCoef=f2(grab('scripts/build_rrab.py', r't\.int_average_g - ([\d.]+) \*')),
         ScatI=f2(pw.rsd.I), NPwI=count(pw.N.I), ScatG=f2(pw.rsd.G),
         RhoMax=num(RHO_MAX), SmcMask=num(SMC_MASK_RADIUS), DmuWin=num(DMU_WINDOW), DmuScat=sigfig(pw.rsd.G),
         DmuRatio=num(round(DMU_WINDOW / float(sigfig(pw.rsd.G)))))
# Foreground contamination per rho bin (results/rrab_membership.csv): inner, 2-8 deg, outer and beyond RHO_MAX.
side = const('fig_rrab_membership')['SIDE']
cm = pd.read_csv(R / 'rrab_membership.csv')
mid = cm[(cm.rho_lo >= 2) & (cm.rho_hi <= 8)]
M.update(SideLo=num(side[0]), SideHi=num(side[1]), ContIn=pct(cm.contam.iloc[0], 1), RhoContIn=num(cm.rho_hi.iloc[0]),
         ContMidLo=pct(mid.contam.min(), 1), ContMidHi=pct(mid.contam.max(), 1), RhoContMid=num(mid.rho_hi.max()),
         ContOut=pct(cm.contam[cm.rho_hi == RHO_MAX].iloc[0], 1),
         ContMax=pct(cm.contam[cm.rho_hi <= RHO_MAX].max(), 1), ContBeyond=num(round(100 * cm.contam[cm.rho_lo == RHO_MAX].iloc[0])),
         RhoBeyond=num(cm.rho_hi[cm.rho_lo == RHO_MAX].iloc[0]))
```

#### Chapter 1: the giant samples
The cuts of the giant samples and the velocity clip of the Letter's Fig. 1 with the number of stars it drops in each
population.

```python
# Giants: sample definition (scripts/build_spec_giants.py, build_giants.py; lmcold.giants)
win = re.search(r'< (\d+)\) & \(t\.v > (\d+)\) & \(t\.v < (\d+)\)', (ROOT / 'scripts/build_spec_giants.py').read_text())
sig = const('paper_letter_signal')   # the velocity clip of the Letter's Fig. 1 (c)
M.update(Clip=num(sig['CLIP']), ClipErr=num(sig['CLIP_ERR']))
# Stars the clip drops per population, as internal() of scripts/paper_letter_signal.py (which prints them): residuals from the
# population's stage-2 disc model of the joint ellipsoid fit, proper-motion components against CLIP + CLIP_ERR x the star's error
# (KAPPA x disc distance x error [km/s]), line of sight against CLIP.
km = pd.read_csv(R / 'kin_mcmc_data_ellipsoid.csv')  # posterior summaries of the joint ellipsoid fit
rr = samples.rrab()
nan = np.full(len(rr['ra']), np.nan)
clip = []   # (stars, dropped): RR Lyrae, then the giant bins from metal-poor to metal-rich
for name, c in [('RR Lyrae', kin_ring(rr | dict(vlos=nan, evlos=nan)))] + [(f'giants {k}', kin_ring(c)) for k, c in samples.giants()]:
    # Parameter vector of the population: stage-2 medians of the shared and of its own parameters.
    p = kinfit.start()
    for _, r in km[(km.stage == 2) & km.population.isin(['shared', name])].iterrows():
        p[kinfit.PARAMS.index(r.param)] = r['median']
    # Residuals of the proper-motion velocity components [km/s] from the disc model.
    _, rt, rr_ = (np.asarray(x) for x in kinfit.residual_polar(p, c))
    d = kinfit.data(*(c[k] for k in sig['COLS']))
    mod = kinfit.model(p, d['rhat'], d['ea'], d['ed'])
    # Clip limit per star [km/s]: CLIP + CLIP_ERR x proper-motion error converted at the disc distance mod[3] [kpc].
    lim = sig['CLIP'] + sig['CLIP_ERR'] * KAPPA * np.asarray(mod[3]) * np.hypot(c['pmra_error'], c['pmdec_error'])
    ok = (np.abs(rt) < lim) & (np.abs(rr_) < lim) & ~(np.abs(np.nan_to_num(np.asarray(c['vlos']) - np.asarray(mod[2]))) > sig['CLIP'])
    clip.append((len(ok), (~ok).sum()))
drop = [x / n for n, x in clip[1:]]  # fraction dropped in each giant bin
M.update(ClipNRr=count(clip[0][0]), ClipRr=num(clip[0][1]), ClipRrPct=sigfig(100 * clip[0][1] / clip[0][0]),
         ClipGLo=sigfig(100 * min(drop)), ClipGHi=sigfig(100 * max(drop)), ClipNMp=count(clip[1][0]), ClipMp=num(clip[1][1]))
# Giant-sample cuts: from lmcold constants and function defaults, and from the build scripts' sources.
M.update(NFields=num(len(apogee_target.FIELDS)), GiantRho=win.group(1), VlosLo=win.group(2), VlosHi=win.group(3),
         GLim=num(grab('src/lmcold/magcloud.py', r'ok & \(g <= ([\d.]+)\)')),
         SnrApoKin=num(default(giants.apogee_spectrum_ok, 'snr_min')), SnrApoSel=num(const('apogee_selection')['SNR_MIN']),
         SnrBoss=num(giants.BOSS_SNR_MIN), EvBoss=num(default(giants.boss_spectrum_ok, 'ev_max')),
         TeffMax=num(default(giants.is_giant, 'teff_max')), LoggMax=num(default(giants.is_giant, 'logg_max')),
         EvApo=num(default(giants.rv_ok, 'ev_max')), VscatApo=num(default(giants.rv_ok, 'vscatter_max')),
         Ruwe=num(default(giants.gaia_ok, 'ruwe_max')),
         PlxZero=num(grab('scripts/build_giants.py', r'\(s\.parallax \+ ([\d.]+)\) / s\.parallax_error < (\d+)')),
         PlxSig=num(grab('scripts/build_giants.py', r'\(s\.parallax \+ ([\d.]+)\) / s\.parallax_error < (\d+)', 2)))
# BOSS [Fe/H] slope against APOGEE per S/N bin (results/snr_cut_check.json): bins below the BOSS S/N cut, and the top bin.
sn = pd.DataFrame(js('snr_cut_check')['by_snr'])
lo, top = sn[sn.snr_hi <= giants.BOSS_SNR_MIN], sn.iloc[-1]
M.update(SnrA=num(lo.snr_lo.iloc[0]), SnrB=num(lo.snr_hi.iloc[0]), SnrC=num(lo.snr_hi.iloc[1]), SnrTop=num(top.snr_lo),
         SlopeA=f2(lo.slope.iloc[0]), SlopeAErr=f2(lo.slope_err.iloc[0]), NSlopeA=num(lo.n.iloc[0]),
         SlopeB=f2(lo.slope.iloc[1]), SlopeBErr=f2(lo.slope_err.iloc[1]), NSlopeB=num(lo.n.iloc[1]),
         SlopeTop=f2(top.slope), SlopeTopErr=f2(top.slope_err),
         CalHighA=f2(lo.calibrated_minus_apogee.iloc[0]), CalHighB=f2(lo.calibrated_minus_apogee.iloc[1]))
# Bins above the cut: the first, the second, and those at S/N >= SNR_HIGH quoted as a range.
above = sn[sn.snr_lo >= giants.BOSS_SNR_MIN]
SNR_HIGH = 30   # BOSS S/N above which the slopes are quoted together
high = above[above.snr_lo >= SNR_HIGH]
r16 = above.iloc[0]
M.update(SlopeC=f2(r16.slope), SlopeCErr=f2(r16.slope_err), NSlopeC=num(r16.n), SnrD=num(r16.snr_hi),
         SlopeD=f2(above.slope.iloc[1]), SlopeDErr=f2(above.slope_err.iloc[1]), SnrE=num(above.snr_hi.iloc[1]),
         SnrF=num(SNR_HIGH), SlopeFLo=f2(high.slope.min()), SlopeFHi=f2(high.slope.max()), SlopeFErr=f2(high.slope_err.max()),
         CalAbove=f2(above.calibrated_minus_apogee.abs().max()))
```

#### Chapter 1: BOSS calibration
The calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both.

```python
# BOSS calibration (results/boss_calibration.json; lmcold.giants.regressors)
cal, ho = js('boss_calibration')['calibration'], js('boss_calibration')['held_out_test']
# Calibration coefficients (velocity d0, d1; [Fe/H] c0-c3), pivots read from lmcold.giants, and the held-out test.
M.update(NOverlap=count(cal['N']), TPivot=num(giants.T_PIVOT), TScale=num(grab('src/lmcold/giants.py', r'T_PIVOT\) / ([\d.]+)')),
         SPivot=num(grab('src/lmcold/giants.py', r'([\d.]+) / np\.asarray\(snr, float\) - 1')),
         VcalA=f2(cal['d0']), VcalB=f2(-cal['d1']), FcalA=f'{cal["c0"]:.3f}', FcalB=f'{cal["c1"]:.3f}', FcalC=f'{-cal["c2"]:.3f}',
         FcalD=f'{cal["c3"]:.3f}', Compress=f2(cal['c1']),
         HoMed=f'{ho["dfeh_median"]:.3f}', HoRsd=f'{ho["dfeh_rsd"]:.3f}', HoDv=signed(ho['dv_median']),
         HoBin=sigfig(max(abs(v) for k in ('dfeh_median_by_snr', 'dfeh_median_by_teff') for v in ho[k].values())))
# Residual bias of the calibrated BOSS [Fe/H] in the APOGEE range BIAS_FEH.
bias = js('boss_response')['bias']
BIAS_FEH = (-2.0, -1.5)   # APOGEE [Fe/H] range [dex] of the residual bias quoted (results/boss_response.json, 0.25-dex bins)
bmid = [m for f, m in zip(bias['feh'], bias['median']) if BIAS_FEH[0] < f < BIAS_FEH[1]]
M.update(BiasLo=f2(-max(bmid)), BiasHi=f2(-min(bmid)), BiasFehLo=num(BIAS_FEH[0]), BiasFehHi=num(BIAS_FEH[1]))
# Sizes of the two kinematic giant catalogues and their overlap.
ga, gb = pd.read_parquet(DATA / 'giants_apogee.parquet'), pd.read_parquet(DATA / 'giants_boss.parquet')
M.update(NKinApo=count(len(ga)), NKinBoss=count(len(gb)), NKinBoth=count(ga.in_both.sum()))
```

#### Chapter 2: the RR Lyrae and BOSS selection functions
Completeness of the RR Lyrae union, the BOSS parent and its groups, the pass factors, the coverage and the tests of the XP
bins.

```python
# ------------------------------------------------------------------------------------------------------------------------------
# Chapter 2: selection functions
# ------------------------------------------------------------------------------------------------------------------------------
# Carton cuts read from the source of lmcold.magcloud: proper-motion window, parallax cut, SFD extinction rule, A_V scaling.
mc = (ROOT / 'src/lmcold/magcloud.py').read_text()
pmw = re.search(r'np\.hypot\(pmL - ([\d.]+), pmB - ([\d.]+)\) < ([\d.]+)', mc)
plx = re.search(r'\(parallax \+ ([\d.]+)\) / parallax_error > (\d+)', mc)
sfd = re.search(r'\(lmcrad <= ([\d.]+)\).*?\(ebv >= ([\d.]+)\), ([\d.]+) \* ebv', mc)
av = re.search(r'av = ([\d.]+) \* ([\d.]+) \* rescaled_ebv', mc)
# Carton colour and proper-motion cuts; APOGEE targeting (H limits, field radius, bright neighbours); velocity window of the
# mass samples; coverage threshold NMIN_GOOD.
M.update(BpRp=num(grab('src/lmcold/magcloud.py', r'\(bp - rp >= ([\d.]+)\)')), PmBox=num(grab('src/lmcold/magcloud.py', r'np\.abs\(pmra\) <= ([\d.]+)')),
         PmL=pmw.group(1), PmB=pmw.group(2), PmWin=pmw.group(3), PlxOff=plx.group(1), PlxSigC=plx.group(2),
         SfdRad=sfd.group(1), SfdEbv=sfd.group(2), SfdScale=sfd.group(3), AvA=av.group(1), AvB=av.group(2),
         HBright=num(grab('src/lmcold/apogee_target.py', r'\(h >= ([\d.]+) - tol\)')),
         HFaint=num(grab('src/lmcold/apogee_target.py', r'np\.where\(h < ([\d.]+), 1, 2\)')),
         HlimLo=num(min(apogee_target.MAGLIM.values())), HlimMid=num(sorted(set(apogee_target.MAGLIM.values()))[1]),
         HlimHi=num(max(apogee_target.MAGLIM.values())), FieldR=num(apogee_target.RADIUS),
         NbrMag=num(default(apogee_target.bright_neighbour, 'dmag')), NbrSep=num(default(apogee_target.bright_neighbour, 'radius')),
         VMassLo=num(grab('src/lmcold/mass_samples.py', r'\(b\.vlos > (\d+)\) & \(b\.vlos < (\d+)\)')),
         VMassHi=num(grab('src/lmcold/mass_samples.py', r'\(b\.vlos > (\d+)\) & \(b\.vlos < (\d+)\)', 2)),
         NminGood=num(selection.NMIN_GOOD))

# RR Lyrae completeness (results/rrab_independence.csv; data/rrab.parquet as scripts/paper_selection.py, 1-deg bins, >= 20 stars)
ri = pd.read_csv(R / 'rrab_independence.csv')
M.update(SplitMax=pct(max((ri.C_union_by_P - ri.C_union_by_mag).abs().max(), (ri.C_union_by_amp - ri.C_union_by_mag).abs().max()), 1),
         UnionIn=pct(ri.C_union_by_mag.iloc[0]), UnionRhoA=num(ri.rho_hi.iloc[0]), UnionRhoB=num(ri.rho_hi.iloc[2]),
         UnionMidLo=pct(ri.C_union_by_mag.iloc[1:3].min(), 1), UnionMidHi=pct(ri.C_union_by_mag.iloc[1:3].max(), 1),
         UnionOutLo=pct(ri.C_union_by_mag.iloc[3:].min(), 1), UnionOutHi=pct(ri.C_union_by_mag.iloc[3:].max(), 1))
t['in_fp'] = ogle.in_footprint(t.ra.to_numpy(), t.dec.to_numpy())
RHO_BINS = np.arange(0, 16.1, 1.0)


def complete(s, flag):
    """Fraction of s with `flag` per 1-deg bin of rho (NaN below 20 stars), as panel (a) of figures/paper_selection.png."""
    k = np.digitize(s.rho, RHO_BINS) - 1  # 1-deg rho bin index per star
    return np.array([s[flag][k == i].mean() if (k == i).sum() >= 20 else np.nan for i in range(len(RHO_BINS) - 1)])


# Completeness of each catalogue measured on the other: Gaia on OGLE's own RRab, OGLE on Gaia RRab inside the OGLE footprint.
cg, co = complete(t[t.ogle_own], 'in_gaia'), complete(t[t.in_gaia & t.in_fp], 'ogle_own')
GAIA_FULL, OGLE_IN = 9, 6   # deg: Gaia completeness quoted beyond GAIA_FULL; OGLE's own search quoted inside OGLE_IN
M.update(GaiaMissIn=pct(1 - cg[0]), GaiaRhoOut=num(GAIA_FULL), GaiaMissOut=pct(1 - np.nanmin(cg[GAIA_FULL:])),
         OgleRho=num(OGLE_IN), OgleLo=pct(co[:OGLE_IN].min()), OgleHi=pct(co[:OGLE_IN].max()))

# BOSS parent and groups (results/boss_selection.json; data/boss_weights.parquet; scripts/boss_selection.py)
bs = js('boss_selection')
summ = bs['summary']
bw = pd.read_parquet(DATA / 'boss_weights.parquet')
cart = pd.read_parquet(DATA / 'boss_cartons.parquet')
# Parent stars in the mwm_magcloud_rgb_boss carton.
in_cart = bw[['source_id']].merge(cart, on='source_id').cartons.str.contains('mwm_magcloud_rgb_boss')
sel = const('boss_selection')
ngood_g = bw.group.map(bw.groupby('group').good.sum())  # good spectra in each star's selection group
# Parent, carton share, observed and good counts; selection-group settings of boss_selection.py; groups; bootstrap nside.
M.update(NParent=count(summ['eligible']), InCarton=pct(in_cart.sum() / summ['eligible']), NObs=count(bw.observed.sum()),
         NGood=count(summ['good']), Nside=num(sel['LEVELS'][0][0]), PixArea=f2(hp.nside2pixarea(sel['LEVELS'][0][0], degrees=True)),
         GEdges=edges(sel['GEDGES']), NminGroup=num(sel['NMIN']), NsideCoarse=', '.join(num(n) for n, _ in sel['LEVELS'][2:-1]),
         NsideLast=num(sel['LEVELS'][-1][0]), NGroups=count(bw.group.nunique()), FinestShare=pct(bw.group.str.startswith('0:').mean()),
         NsideBoot=num(grab('scripts/boss_selection.py', r'pix8, pix16 = hp\.ang2pix\(\d+, theta, phi, nest=True\), hp\.ang2pix\((\d+),')))
# Members lost to the astrometric cuts (results/selection_gaps.json).
sg = js('selection_gaps')
al = sg['astrometric_loss_of_members']
M.update(NAstroB=count(al['LMC18-35']['n']), LossB=pct(al['LMC18-35']['fail'], 1), NAstroA=count(al['LMC1-17']['n']),
         LossA=pct(al['LMC1-17']['fail'], 1))
pe = pd.read_csv(R / 'parent_extinction.csv').query("model == 'region'")
DUSTY = 0.15   # E(V-I) [mag] above which the central parent loses stars (results/parent_extinction.csv, region model)
dz = pe[pe.evi_lo >= DUSTY]  # E(V-I) bins above DUSTY; q is the retained fraction, so n_g (1/q - 1) stars are missing
M.update(DustMissing=pct((dz.n_g * (1 / dz.q - 1)).sum() / pe.n_g.sum(), 1))

# Pass factors (results/boss_selection.json P_pass_factor; lmcold.selection; scripts/build_xp_andrae.py)
pf = bs['P_pass_factor']
xa = const('build_xp_andrae')
nx, nr = len(pf['xp_edges']), len(pf['ring_edges']) + 1
i0 = ((len(pf['g_edges']) - 2) * nx + 0) * nr   # faintest G bin, lowest XP bin, ring 0
# Pass factors with errors in the three rings, for the faintest G bin and lowest XP bin.
pa = [pm_err(pf['factor'][i0 + r], pf['err'][i0 + r]) for r in range(nr)]
M.update(XpTeff=num(xa['TEFF_MAX']), XpLogg=num(xa['LOGG_MAX']), XpShare=pct(summ['with_xp'] / summ['eligible']),
         PgEdges=edges(selection.PG_EDGES), XpEdges=edges(selection.XP_EDGES[1:-1]), XpLow=num(selection.XP_EDGES[1]),
         RingA=num(selection.RING_EDGES[0]), RingB=num(selection.RING_EDGES[1]), RingOut=M['GiantRho'],
         PassG=num(selection.PG_EDGES[1]), PassXp=num(selection.XP_EDGES[3]),
         GFaintLo=num(pf['g_edges'][-2]), GFaintHi=num(pf['g_edges'][-1]),
         PassA=pa[0][0], PassAErr=pa[0][1], PassB=pa[1][0], PassBErr=pa[1][1], PassC=pa[2][0], PassCErr=pa[2][1])
# Ring closure ([M/H] < -1.5): pass factor common to all rings ('common') against the final weights.
rc = bs['R_ring_closure']
M.update(CommonIn=f2(rc['common']['0-1.5']['<-1.5']), CommonOut=f2(rc['common']['6-12']['<-1.5']), RingIn=list(rc['final'])[0].split('-')[1],
         RingFinA=f2(rc['final']['0-1.5']['<-1.5']), RingFinB=f2(rc['final']['1.5-3']['<-1.5']),
         RingFinC=f2(rc['final']['3-6']['<-1.5']), RingFinD=f2(rc['final']['6-12']['<-1.5']))

# Coverage (results/selection_gaps.json)
cf, sparse = sg['coverage_factor'], sg['sparse_groups']
M.update(NCovGroups=count(bw[ngood_g >= selection.NMIN_GOOD].group.nunique()), NCovParent=count(summ['covered']),
         CovShare=pct(summ['covered'] / summ['eligible'], 1), NUncov=count(sg['uncovered']), UncovShare=pct(sg['uncovered_share'], 1),
         UncovIn=pct(sg['uncovered_by_ring']['0'] / sg['uncovered']),
         CovRingLo=f2(min(cf['per_ring'])), CovRingHi=f2(max(cf['per_ring'])), CovMax=f2(cf['ring0'][0]), CovMaxG=num(sel['GEDGES'][1]),
         SparseFrac=num(100 * grab('scripts/paper_selection_gaps.py', r'gs\.good / gs\.parent < ([\d.]+)')), SparseN=num(sparse['groups']),
         SparseShare=pct(sparse['parent_share'], 1), SparseGood=num(sparse['good']), SparseRho=f'{sparse["median_rho"]:.1f}',
         MaxW=count(round(sparse['max_w'])))
# Weighted non-member fraction of BOSS good spectra, overall and per ring; share of good spectra that are not giants.
nm = sg['boss_nonmember']
gd = bw[bw.good]
M.update(NonMem=pct(nm['all'], 1), NonMemLo=pct(min(nm['by_ring'].values()), 1), NonMemHi=pct(max(nm['by_ring'].values()), 1),
         Dwarfs=pct((~gd.giant).mean(), 1))

# Tests: the share of each XP bin in the good spectra over that in the covered parent (as figures/paper_selection.png b)
XP_BINS = np.array([-2.5, -2.0, -1.75, -1.5, -1.25, -1.0, -0.75, -0.5, 0.0])   # edges of panel (b), scripts/paper_selection.py
cov = bw[(ngood_g >= selection.NMIN_GOOD) & bw.mh_xp.notna()]  # covered parent with XP [M/H]
# XP bin per star; par: covered parent per bin; unw: share of good spectra over share of parent (unweighted, 1 = no bias).
k = np.digitize(cov.mh_xp, XP_BINS) - 1
ok = (k >= 0) & (k < len(XP_BINS) - 1)
par = np.bincount(k[ok], minlength=len(XP_BINS) - 1)
s_ = ok & cov.good.to_numpy()
unw = np.bincount(k[s_], minlength=len(XP_BINS) - 1) / s_.sum() / (par / par.sum())
OVER_XP = -1.25   # [dex]: the unweighted over-representation is quoted for the XP bins below this edge
poor = XP_BINS[1:] <= OVER_XP
# Over-representation range, closure (final and group weights), held-out sky, and the parent fraction below -1.5.
cl = bs['C1_closure']
M.update(OverLo=pct(unw[poor].min() - 1), OverHi=pct(unw[poor].max() - 1), OverXp=num(OVER_XP), Under=pct(1 - unw[-1]),
         FinA=f'{cl["final"]["<-1.0"]["diff"]:.4f}', FinAErr=f'{cl["final"]["<-1.0"]["err"]:.4f}',
         FinB=f'{cl["final"]["<-1.5"]["diff"]:.4f}', FinBErr=f'{cl["final"]["<-1.5"]["err"]:.4f}',
         FinC=f'{cl["final"]["<-2.0"]["diff"]:.4f}', FinCErr=f'{cl["final"]["<-2.0"]["err"]:.4f}',
         GrpA=f'{cl["group"]["<-1.0"]["diff"]:.4f}', GrpB=f'{cl["group"]["<-1.5"]["diff"]:.4f}', GrpC=f'{cl["group"]["<-2.0"]["diff"]:.4f}',
         HeldShort=f'{-bs["H_held_out"]["final"][1]:.4f}', HeldNoPass=f'{-bs["H_held_out"]["group"][1]:.4f}',
         ParentFrac=f'{sg["parent_xp_fraction_covered"]["<-1.5"]:.3f}',
         HeldRatio=pct(-bs['H_held_out']['final'][1] / sg['parent_xp_fraction_covered']['<-1.5']), NsideHeld=num(grab('scripts/boss_selection.py', r'pix8, pix16 = hp\.ang2pix\((\d+),')))
# Mock of the BOSS selection (results/boss_mock.json; scripts/boss_mock.py): cumulative recovery and coverage fill per ring.
mk = js('boss_mock')
bm = const('boss_mock')
c0 = mk['obs_bias_1.0']['final']['cumulative']
ring = mk['fill']['central']['standard/ring']
M.update(MockA=f2(c0['mean'][0]), MockAErr=f2(c0['sd'][0]), MockB=f2(c0['mean'][1]), MockBErr=f2(c0['sd'][1]),
         FillA=f2(ring['ring0']['mean'][0]), FillAErr=f2(ring['ring0']['sd'][0]), FillB=f2(ring['ring1']['mean'][0]),
         FillBErr=f2(ring['ring1']['sd'][0]), FillC=f2(ring['ring2']['mean'][0]), FillCErr=f2(ring['ring2']['sd'][0]))
xg = {round(r['below'], 3): r for r in sg['xp_gaps']}  # XP gap rows keyed by threshold [dex]


def rel(x):
    """Relative difference of the uncovered and covered parent fractions below XP [M/H] = x."""
    return abs(1 - xg[x]['uncovered'] / xg[x]['covered_same_mix'])


# Gap differences, sensitivity to the coverage threshold (NMIN_GOOD = 2, 5), and the S/N cut check.
ct = sg['coverage_threshold']
sc = js('snr_cut_check')
M.update(GapA=sigfig(100 * rel(-1.0)), GapB=sigfig(100 * rel(-1.5)), CovTwo=f'{100 * (ct["nmin=2"]["all"] - 1):.1f}',
         CovFive=f'{100 * (ct["nmin=5"]["all"] - 1):.1f}', SnrDiff=f'{sc["diff"]:.3f}', SnrDiffHi=f'{sc["diff_p84"] - sc["diff"]:.3f}',
         SnrDiffLo=f'{sc["diff"] - sc["diff_p16"]:.3f}', SnrN=num(sc['n_rejected']))
```

#### Chapter 2: the APOGEE selection function
Candidates, observed stars, ASPCAP failures and the field-bootstrap error of the APOGEE weights.

```python
# APOGEE (results/apogee_selection.json; data/apogee_parent.parquet, apogee_weights.parquet; scripts/apogee_selection.py)
asel = js('apogee_selection')
ap = pd.read_parquet(DATA / 'apogee_parent.parquet').sort_values(['apogee_id', 'observed'], ascending=[True, False]).drop_duplicates('apogee_id')
late = ap[(ap.field.str[3:].astype(int) >= 18) & ~ap.nbr]  # LMC18-35 candidates without a bright neighbour
aw = pd.read_parquet(DATA / 'apogee_weights.parquet')
# ASPCAP failures: bright (BrtRGB) observed candidates with S/N above the selection threshold, so that a failure is STAR_BAD or no
# [Fe/H] (lmcold.giants.apogee_spectrum_ok), not low S/N
snr_sel = const('apogee_selection')['SNR_MIN']
bright = aw[aw.observed & (aw.cls == 1)].merge(ap[['apogee_id', 'snr', 'extratarg', 'aspcapflag', 'feh']], on='apogee_id', suffixes=('_w', ''))
bright = bright[bright.snr >= snr_sel]
bright['fail'] = ~giants.apogee_spectrum_ok(bright, snr_sel)
ASP_XP = (-1.5, -1.0)   # XP [M/H] [dex] below and above which the ASPCAP failure rates of bright observed stars are quoted
# Coverage rule of lmcold.apogee_target, read from its source (minimum counts and observed fractions).
cov_rule = re.search(r'n_q >= (\d+), \(n_q >= (\d+)\) & \(f_obs >= ([\d.]+)\)\) \| \(\(f_q >= ([\d.]+)\)',
                     (ROOT / 'src/lmcold/apogee_target.py').read_text())
asum = asel['summary']
# Candidate and observed counts, astrometric failures, targeting zones, coverage rule, ASPCAP failure rates and the tests T5-T7.
M.update(NPoly=count(len(ap)), NNbr=count(ap.nbr.sum()), InPoly=pct(asel['T1']['wd_in_parent'], 2), InPolyB=pct(asel['T1']['lmc18_35_in_parent']),
         NCand=count(asum['candidates']), AstroFail=pct(1 - late.astro_ok.mean()), AstroGiant=pct(al['LMC18-35']['fail'], 1),
         ZoneR=num(grab('scripts/apogee_selection.py', r'\(c\.rfield >= ([\d.]+)\)')),
         HStep=num(grab('scripts/apogee_selection.py', r'\(c\.h - 12\.8\) / ([\d.]+)\)')), NColour=num(len(const('apogee_selection')['XEDGES']) + 1),
         NApoObs=count(asum['observed']), NApoGood=count(asum['quality']), CovBright=cov_rule.group(1), CovFaint=cov_rule.group(2),
         CovFobs=cov_rule.group(3), CovHalf=cov_rule.group(4), NApoCov=count(asum['covered']),
         PgHEdges=edges(const('apogee_selection')['PG_H_EDGES']), AspXpLo=num(ASP_XP[0]), AspXpHi=num(ASP_XP[1]),
         AspMP=pct(bright.fail[bright.mh_xp < ASP_XP[0]].mean()), AspMR=pct(bright.fail[bright.mh_xp > ASP_XP[1]].mean(), 1),
         VelA=pct(sg['apogee_nonmember']['LMC1-17']['weighted'], 1), VelB=pct(sg['apogee_nonmember']['LMC18-35']['weighted'], 1),
         Jk=f'{max(abs(v) for k_, v in asel["T5_dJK"].items() if k_ != "pass"):.4f}',
         Fdiff=f'{np.ptp([v["f_below_m15"] for k_, v in asel["T6_snr_neutrality"].items() if k_ != "pass"]):.4f}',
         ApoLateFin=f2(asel['T7_xp_closure']['LMC18-35 final']['<-1.5']), ApoLateNo=f2(asel['T7_xp_closure']['LMC18-35 no_pass']['<-1.5']),
         ApoEarlyFin=f2(asel['T7_xp_closure']['LMC1-17 final']['<-1.5']), ApoEarlyNo=f2(asel['T7_xp_closure']['LMC1-17 no_pass']['<-1.5']))
# Field-bootstrap error of the LMC1-17 ratio before the pass factor (T7 of scripts/apogee_selection.py): weights without the pass
# factor, 1 / (quality fraction of the group), for quality stars in covered groups; fields drawn with replacement, weights fixed.
NBOOT_T7 = 1000
x = aw[aw.mh_xp.notna() & (aw.field.str[3:].astype(int) <= 17)]  # LMC1-17 candidates with XP [M/H]
w0 = np.where(x.covered & x.quality, 1 / x.groupby('group')['quality'].transform('mean'), 0.)
low = x.mh_xp < -1.5
# Per field: weighted and covered counts, all and with XP [M/H] < -1.5; the ratio is (weighted fraction) / (covered fraction).
fs = pd.DataFrame(dict(field=x.field, a=w0 * low, b=w0, c=x.covered & low, d=x.covered)).groupby('field').sum()
rng = np.random.default_rng(1)  # fixed seed
t7 = [(s.a.sum() / s.b.sum()) / (s.c.sum() / s.d.sum()) for s in (fs.iloc[rng.integers(0, len(fs), len(fs))] for _ in range(NBOOT_T7))]
M.update(ApoEarlyNoErr=f2(np.std(t7)))
# Members of the two mass samples.
gc = js('giant_counts')
M.update(NMassBoss=count(gc['boss_members']), NMassApo=count(gc['apogee_members']))
```

#### Chapter 2: appendix checks
The mock test of the BOSS selection function, the coverage gaps and the non-members among the weighted good BOSS spectra.

```python
# Appendix checks: mock (results/boss_mock.json; scripts/boss_mock.py) and gaps (results/selection_gaps.json)
grp = mk['obs_bias_1.0']['group']['binned']['mean']  # recovery per [Fe/H] bin with group weights only
fe = mk['feh_edges']  # [Fe/H] bin edges of the mock
b13 = mk['obs_bias_1.3']['final']['cumulative']['mean'][0]
xp_ = mk['fill']['central']['standard/xp']
# Inner-ring fill for the sparse-gap variants, and the total recovery per ring.
spg = [mk['fill']['central'][k]['ring0']['mean'][0] for k in ('sparse_gap/ring', 'sparse_gap/xp')]
tot = [ring[f'ring{r}']['mean'][2] for r in range(3)]
M.update(GaiaBrightRho=re.search(r'\(2, (\d+)\), \(4,', (ROOT / 'scripts/paper_selection_checks.py').read_text()).group(1), NReal=num(bm['NREAL']), NFill=num(bm['NFILL']),
         GroupLoss=pct(1 - min(m for m, hi in zip(grp, fe[1:]) if hi <= -2)), BiasFac=[k.split('_')[-1] for k in mk if k.startswith('obs_bias_')][-1], BiasHigh=pct(b13 - 1),
         XpFillA=f2(xp_['ring0']['mean'][0]), XpFillAErr=f2(xp_['ring0']['sd'][0]), XpFillB=f2(xp_['ring1']['mean'][0]),
         XpFillBErr=f2(xp_['ring1']['sd'][0]), XpFillC=f2(xp_['ring2']['mean'][0]), XpFillCErr=f2(xp_['ring2']['sd'][0]),
         SparseLo=f2(min(spg)), SparseHi=f2(max(spg)), TotA=f2(tot[0]), TotB=f2(tot[1]), TotC=f2(tot[2]),
         PixCoarse=num(round(np.sqrt(hp.nside2pixarea(sel['LEVELS'][-1][0], degrees=True)))))
# XP gap fractions (uncovered, covered same mix, bootstrap half-width) below -1, -1.5 and -1.75.
for name, x, d in (('One', -1.0, 3), ('Two', -1.5, 4), ('Three', -1.75, 4)):
    r = xg[x]
    M.update({f'GapU{name}': f'{r["uncovered"]:.{d}f}', f'GapC{name}': f'{r["covered_same_mix"]:.{d}f}',
              f'GapE{name}': f'{(r["uncovered_p84"] - r["uncovered_p16"]) / 2:.{d}f}'})
M.update(GapXpThree=num(-1.75), Deficit=pct(rel(-1.75) * sg['uncovered_share'], 1))
# Non-members of the weighted good BOSS spectra per ring x G bin, as panel (c) of figures/paper_selection_gaps.png
v = pd.read_parquet(DATA / 'spec_giants.parquet')
vb = v[v.survey == 'boss'][['source_id', 'v', 'teff', 'logg']].drop_duplicates('source_id')
gg = bw[bw.good & (bw.w > 0)].merge(vb, on='source_id', how='left')
# Ring (RING_EDGES) and G bin per star; non-member = fails the giant cuts or the mass-sample velocity window.
gg['ring'] = np.digitize(angular_distance(gg.ra.to_numpy(), gg.dec.to_numpy(), LMC['ra0'], LMC['dec0']), selection.RING_EDGES)
gg['gb'] = np.digitize(gg.g, const('paper_selection_gaps')['G_EDGES'])
gg['nm'] = ~((gg.teff < default(giants.is_giant, 'teff_max')) & (gg.logg < default(giants.is_giant, 'logg_max'))
             & (gg.v > float(M['VMassLo'])) & (gg.v < float(M['VMassHi'])))
per = gg.groupby(['ring', 'gb']).apply(lambda s: np.average(s.nm, weights=s.w), include_groups=False)
M.update(NonMemCellLo=pct(per.min(), 1), NonMemCellHi=pct(per.max(), 1),
         ApoNonMemMax=pct(max(x['weighted'] for x in sg['apogee_nonmember'].values()), 1))
```

#### Chapter 3: RR Lyrae membership and metallicity; BOSS calibration
The numbers of the calibration chapter.

```python
# ------------------------------------------------------------------------------------------------------------------------------
# Chapter 3: RR Lyrae membership and metallicity; BOSS calibration
# ------------------------------------------------------------------------------------------------------------------------------
# BOSS S/N threshold 16 (adopted) against 20: change of formed masses [per cent], scale lengths h and the [Fe/H] profile.
st = js('snr_threshold_sensitivity')
a16, a20 = st['16'], st['20']  # results with S/N >= 16 and S/N >= 20
prof = np.abs(np.array(st['median_feh_by_R_20']) - np.array(st['median_feh_by_R_16']))[1:]   # profile bins beyond the first kpc
# Alternative threshold (second numeric key), relative changes of total, [Fe/H] < -1.5 and < -2 formed mass, largest change
# of h, largest change of the median [Fe/H] profile beyond the first kpc.
M.update(SnrAlt=[k for k in st if k.isdigit()][1], DTot=f'{100 * (a20["formed_total"] / a16["formed_total"] - 1):.1f}',
         DMP=num(round(100 * (a20['formed_below_m15'] / a16['formed_below_m15'] - 1))),
         DVMP=signed(round(100 * (a20['formed_below_m2'] / a16['formed_below_m2'] - 1)), num),
         DH=f2(max(abs(a20['h'][k] - a16['h'][k]) for k in a16['h'])), DProf=f'{prof.max():.3f}', ProfR=num(1))
# BOSS [Fe/H] bias of hot giants (results/boss_hot_bias.json; scripts/paper_boss_hot_bias.py).
hb = js('boss_hot_bias')
te = hb['t_edges']  # Teff bin edges [K]
T_HOT = grab('scripts/paper_boss_hot_bias.py', r'hot = \(b\.teff > (\d+)\)')   # K: 'hot' giants of the figure script
cool = [j for j in range(len(te) - 1) if te[j + 1] <= T_HOT]  # Teff bins entirely below T_HOT
warm = te.index(T_HOT)  # index of the first bin above T_HOT; the next one (warm + 1) is the 'hot' bin


def offs(j):
    """BOSS - APOGEE [Fe/H] offsets of Teff bin j at low and high S/N."""
    return [hb['offset_low'][j], hb['offset_high'][j]]  # [dex]


red = max(j for j, f in enumerate(hb['frac_boss_low']) if np.isfinite(f))   # reddest dereddened BP-RP bin of panel (b) with stars
# S/N split, Teff edges, offset ranges per Teff group, fractions of BOSS stars in the bluest and reddest colour bins at low and
# high S/N, APOGEE range, and the cuts of the figure script read from its source (G range, rho, minimum stars, metal-rich bin).
M.update(SnrSplit=num(hb['snr_split']), TCool=num(T_HOT), TWarm=num(te[warm + 1]), THot=num(te[warm + 2]),
         CoolMax=f2(max(abs(x) for j in cool for x in offs(j))), WarmLo=f2(min(offs(warm))), WarmHi=f2(max(offs(warm))),
         HotLo=f2(min(offs(warm + 1))), HotHi=f2(max(offs(warm + 1))),
         BlueLow=pct(hb['frac_boss_low'][0]), BlueHigh=pct(hb['frac_boss_high'][0]), BlueEdge=num(hb['c_edges'][1]),
         RedLow=pct(hb['frac_boss_low'][red]), RedHigh=pct(hb['frac_boss_high'][red]), RedLo=num(hb['c_edges'][red]),
         RedHi=num(hb['c_edges'][red + 1]),
         ApoRichLo=pct(np.nanmin(hb['frac_apogee'])), ApoRichHi=pct(np.nanmax(hb['frac_apogee'])),
         HotGLo=num(grab('scripts/paper_boss_hot_bias.py', r't\.g\.between\(([\d.]+), ([\d.]+)\)')),
         HotGHi=num(grab('scripts/paper_boss_hot_bias.py', r't\.g\.between\(([\d.]+), ([\d.]+)\)', 2)),
         HotRho=num(grab('scripts/paper_boss_hot_bias.py', r't\.rho >= (\d+)\)')),
         HotNMin=num(grab('scripts/paper_boss_hot_bias.py', r'T_EDGES, mid, (\d+)\)')),
         TBin=num(np.diff(te)[0]),
         RichLo=num(grab('scripts/paper_boss_hot_bias.py', r'rich = o\[\(o\.feh_a >= ([-.\d]+)\) & \(o\.feh_a < ([-.\d]+)\)\]')),
         RichHi=num(grab('scripts/paper_boss_hot_bias.py', r'rich = o\[\(o\.feh_a >= ([-.\d]+)\) & \(o\.feh_a < ([-.\d]+)\)\]', 2)),
         SplitFeh=num(TRUE_EDGES[-2]), CalNMin=num(grab('scripts/paper_boss_calibration.py', r'def running\(ax, x, y, edges, nmin=(\d+)\)')))
```

#### Settings quoted in captions and tests
Values read from the figure scripts and lmcold that the captions and tests quote.

```python
# Settings quoted in the captions and tests (figure scripts, lmcold.maps, scripts/boss_selection.py, boss_mock.py, apogee_selection.py)
# The three (inner, outer) rings [deg] of scripts/paper_selection_checks.py, read from its source.
rings = re.search(r'zip\(\(\((\d+), (\d+)\), \((\d+), (\d+)\), \((\d+), (\d+)\)\)', (ROOT / 'scripts/paper_selection_checks.py').read_text())
# Map pixel size and minimum stars; closure thresholds of boss_selection.py and boss_mock.py; APOGEE S/N thresholds of T6;
# bin size, minimum stars and HEALPix nside of paper_selection.py; magnitude step, rings, minimum stars and colour limit of
# paper_selection_checks.py.
M.update(MapPix=num(grab('scripts/paper_maps.py', r'np\.arange\(-L, L \+ 0\.01, ([\d.]+)\)')), MapNMin=num(default(pixel_map, 'nmin')),
         CutA=num(sel['CUTS'][0]), CutB=num(sel['CUTS'][1]), CutC=num(sel['CUTS'][2]),
         MockCutA=num(bm['CUTS'][0]), MockCutB=num(bm['CUTS'][1]),
         SnrT=', '.join(k.split('>=')[1] for k in asel['T6_snr_neutrality'] if k != 'pass'),
         SelBin=num(grab('scripts/paper_selection.py', r'np\.arange\(0, 16\.1, ([\d.]+)\)\)')),
         SelNBin=num(grab('scripts/paper_selection.py', r'if n\[i\] >= (\d+) else')), SelNMin=num(const('paper_selection')['NMIN']),
         SelNside=num(grab('scripts/paper_selection.py', r'hp\.ang2pix\((\d+),')),
         ChkMag=num(grab('scripts/paper_selection_checks.py', r'np\.arange\(17\.75, 20\.6, ([\d.]+)\)')),
         ChkRA=rings.group(2), ChkRB=rings.group(4), ChkRC=rings.group(6), ChkNB=num(grab('scripts/paper_selection_checks.py', r'if nB >= (\d+):')),
         ChkVmin=num(grab('scripts/paper_selection_checks.py', r'vmin=([\d.]+), vmax')))
```

#### Write the macros
One \newcommand per entry of M, with the prefix \da, into results/macros_data.tex; the page text shows them through
{glue:text}, so no number is typed by hand.

```python
# Header line, then one \newcommand per macro; written to results/macros_data.tex and printed.
lines = ['% Numbers of the KB data chapters 1-3 (scripts/paper_text_data.py); do not edit.']
lines += [rf'\newcommand{{\da{k}}}{{{x}}}' for k, x in M.items()]
(R / 'macros_data.tex').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
```

### Stellar mass

The numbers of the chapter on stellar mass per metallicity (`results/macros_mass.tex`, prefix `\ms`).

<details><summary>About <code>scripts/paper_text_mass.py</code></summary>

```text
Knowledge-base text, chapter 'Stellar mass per metallicity' (extended draft Section 2.3 and its appendix on the mass): every number
of the text as a LaTeX macro (prefix \\ms), so that none is typed by hand.

As scripts/paper_text_shape.py: settings are read from the code (scripts/textnum.const, lmcold by import, or the source line where a
setting is written inline), results from the tables the figures and tables are drawn from. Literature values quoted with their
citation stay in the text. Three comparisons not written to any table are counted here from the catalogues: the Gaia XP metallicities
of the BOSS parent and the LMC clusters (as scripts/paper_mass.py, panel c), and the model foreground fraction of the outermost ring
(as scripts/fig_giant_foreground.py, panel a).
Inputs: results/{giant_present_mass, giant_mass_bootstrap, giant_mass_variants, giant_mass_response, giant_mass_common,
giant_mass_fields, mass_foreground_reference, giant_foreground_sideband, mass_alpha_trend, mass_regions, mass_conversion_factors,
giant_conversion, parent_extinction}.csv and {giant_mass, giant_present_mass, giant_counts, boss_response, boss_hot_bias}.json;
data/{boss_parent, xp_andrae, giant_foreground}.parquet, data/clusters_grocholski06.csv and the BOSS giants of
lmcold.mass_samples.samples(); settings of scripts/{build_giant_mass, boss_response, fig_giant_foreground, paper_parent_extinction}.py,
lmcold.{constants, extinction, isochrones, magcloud, mass_conversion, mass_samples, selection, foreground}.
Writes results/macros_mass.tex.
Run: .venv/bin/python scripts/manifest.py run paper_text_mass.py
```
</details>

**Data read by `scripts/paper_text_mass.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_parent.parquet` | Zenodo bundle 1 | Gaia DR3 stars within 12 deg of the LMC passing the rough cuts of mwm_magcloud_rgb_boss, with SFD E(B-V) and `eligible` |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/clusters_grocholski06.csv` | public archive | LMC clusters with Ca II triplet [Fe/H] and radial velocities (Grocholski et al. 2006, 2006AJ....132.1630G) |
| `data/giant_foreground.parquet` | Zenodo bundle 1 | Milky Way foreground model per giant, from the mass analysis |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `data/xp_andrae.parquet` | Zenodo bundle 1 | Andrae et al. 2023 Gaia XP [M/H] of the selection-function parents |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |
| `results/boss_hot_bias.json` | repository | written by `python scripts/paper_boss_hot_bias.py` |
| `results/boss_response.json` | repository | written by `python scripts/boss_response.py` |
| `results/giant_conversion.csv` | repository | written by `python scripts/paper_mass_method.py` |
| `results/giant_counts.json` | repository | written by `python scripts/paper_giant_counts.py` |
| `results/giant_foreground_sideband.csv` | repository | written by `python scripts/fig_giant_foreground.py` |
| `results/giant_mass.json` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_bootstrap.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_common.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_fields.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_response.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_mass_variants.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/giant_present_mass.csv` | repository | written by `python scripts/build_present_mass.py` |
| `results/giant_present_mass.json` | repository | written by `python scripts/build_present_mass.py` |
| `results/mass_alpha_trend.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/mass_conversion_factors.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/mass_foreground_reference.csv` | repository | Fixed input from an earlier analysis (not rebuilt here): in-field Milky Way foreground fractions per [Fe/H] bin (feh_med, f_mw, f_mw_err); transferred reference, not a new weighted membership fit; the transfer sensitivity uses zero and twice the reference amplitude |
| `results/mass_regions.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/mdf_literature.json` | repository | written by `python scripts/paper_mdf_literature.py` |
| `results/parent_extinction.csv` | repository | written by `python scripts/paper_parent_extinction.py` |

*Shown, not run here: the result tables in results/ and the catalogues of Zenodo bundle 1. The cells below are the script itself; run it from the repository with* `python scripts/paper_text_mass.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
Imports, the dictionary M of macros (macro name without its prefix: the text it stands for) and the functions that format
numbers as the text writes them. The inputs listed in the header are read in the step that uses them.

```python
import json
import re

import healpy as hp
import numpy as np
import pandas as pd

from lmcold import foreground, isochrones, magcloud, mass_conversion, mass_samples, selection
from lmcold.constants import LMC, LMC_AMR_POVICK24, LMC_DEPTH_MAG, LMC_DISTANCE_MODULUS, LMC_MASS_LITERATURE
from lmcold.extinction import EVI_PER_EBV
from lmcold.mass_samples import FEH_CENTRES, FEH_EDGES
from lmcold.sources import DATA, ROOT
from textnum import const, default, listing, num, sigfig

R = ROOT / 'results'   # folder of the result tables
M = {}   # macro name (without the \ms prefix) -> text


def f1(x):
    """x with one decimal."""
    return f'{x:.1f}'


def f2(x):
    """x with two decimals."""
    return f'{x:.2f}'


def pct(x):
    """Fraction x as a whole-number percentage."""
    return num(round(100 * x))


def src(path, pattern):
    """The number captured by `pattern` in the source file `path` (a setting written inline)."""
    return float(re.search(pattern, (ROOT / path).read_text()).group(1))


def bounds(v, lo, hi, fmt=f2):
    """Value with its upper and lower distances to the 16th/84th percentiles, as three strings."""
    return fmt(v), fmt(hi - v), fmt(v - lo)


def sci2(v, lo, hi, e):
    """v, hi - v and v - lo in units of 10^e, v to two significant figures and the distances to the same decimals."""
    d = max(1 - int(np.floor(np.log10(v / 10 ** e))), 0)   # decimals that give v two significant figures
    return tuple(f'{x / 10 ** e:.{d}f}' for x in (v, hi - v, v - lo))


def binname(i):
    """Reporting [Fe/H] bin i as text, e.g. '−0.75 to −0.5'."""
    return f'{num(FEH_EDGES[i])} to {num(FEH_EDGES[i + 1])}'


# Result tables used in several steps: formed and present mass per [Fe/H] bin (t), mass summary (gm), present-day mass (gp),
# member counts (gc), BOSS response (br), hot-giant bias (hb).
t = pd.read_csv(R / 'giant_present_mass.csv')
gm = json.loads((R / 'giant_mass.json').read_text())
gp = json.loads((R / 'giant_present_mass.json').read_text())
gc = json.loads((R / 'giant_counts.json').read_text())
br = json.loads((R / 'boss_response.json').read_text())
hb = json.loads((R / 'boss_hot_bias.json').read_text())
# Module-level constants of four scripts, read from their source without running them.
bgm, bresp, ffg, ppe = const('build_giant_mass'), const('boss_response'), const('fig_giant_foreground'), const('paper_parent_extinction')
```

#### Samples, bins and foreground
The mass samples and [Fe/H] bins, the Milky Way foreground, and the stars with both spectra.

```python
# Mass samples and bins (lmcold.mass_samples)
# Velocity window [km/s] read from the line of lmcold.mass_samples that applies it.
vlos = re.search(r'\(b\.vlos > (\d+)\) & \(b\.vlos < (\d+)\)', (ROOT / 'src/lmcold/mass_samples.py').read_text()).groups()
# Window, member counts (thin-space thousands), radius [deg], reporting-bin width and the edges of the inner bins [dex].
M.update(VlosLo=vlos[0], VlosHi=vlos[1], NBoss=f"{gc['boss_members']:,}".replace(',', '\\,'),
         NApogee=f"{gc['apogee_members']:,}".replace(',', '\\,'), Rad=num(gm['full_radius_deg']), BinW=num(FEH_EDGES[2] - FEH_EDGES[1]),
         FehLo=num(FEH_EDGES[1]), FehHi=num(FEH_EDGES[-2]))

# Corrected counts: Milky Way foreground (results/mass_foreground_reference.csv, giant_foreground_sideband.csv)
fr = pd.read_csv(R / 'mass_foreground_reference.csv')   # foreground fraction f_mw per [Fe/H] bin (feh_med)
# Reference ring of lmcold.foreground.radial_fraction (default argument) [deg].
ref = default(foreground.radial_fraction, 'reference')
sb = pd.read_csv(R / 'giant_foreground_sideband.csv')   # sideband density and error per ring, n_raw stars
mean = np.sum(sb.density / sb.err ** 2) / np.sum(1 / sb.err ** 2)   # as scripts/fig_giant_foreground.py
top = sb.loc[sb.n_raw.idxmax()]   # the ring with the most sideband stars
# Foreground fractions [%] (metal-poor range, metal-rich mean), reference ring, sideband width, chi2 about the weighted mean.
M.update(FmwPoorLo=pct(fr[fr.feh_med < -1.5].f_mw.min()), FmwPoorHi=pct(fr[fr.feh_med < -1.5].f_mw.max()),
         FmwRich=sigfig(100 * fr[fr.feh_med > -1].f_mw.mean()), RefLo=num(ref[0]), RefHi=num(ref[1]), Side=num(ffg['SIDE']),
         SideChi=f1(np.sum(((sb.density - mean) / sb.err) ** 2)), SideDof=num(len(sb) - 1), SideN=num(top.n_raw),
         SideRhoLo=num(top.rho_lo), SideRhoHi=num(top.rho_hi))
# Model foreground fraction in the outermost 1-deg ring, per [Fe/H] range (scripts/fig_giant_foreground.py, panel a)
b, a, _ = mass_samples.samples()   # BOSS and APOGEE mass samples
# Stars with both spectra (lmcold.mass_samples.overlap): their median BOSS S/N against that of the BOSS giants in the faintest G bin of
# the response; and, at G >= XP_G, the median XP [M/H] - APOGEE [Fe/H] of metal-rich giants in the APOGEE [Fe/H] bins XP_RICH.
XP_G, XP_RICH = 16.5, (-0.9, -0.7, -0.5)   # mag; dex
o = mass_samples.overlap(b, a).merge(b[['source_id', 'mh_xp']], on='source_id')   # overlap stars with their XP [M/H]
faint = b.g >= mass_samples.RESPONSE_G_EDGES[-2]   # BOSS giants in the faintest G bin of the response
of = o[(o.g >= XP_G) & o.mh_xp.notna()]   # faint overlap stars with an XP [M/H]
# Median XP [M/H] - APOGEE [Fe/H] [dex] in each APOGEE [Fe/H] bin of XP_RICH.
dxp = [np.median((of.mh_xp - of.feh_a)[(of.feh_a >= lo) & (of.feh_a < hi)]) for lo, hi in zip(XP_RICH[:-1], XP_RICH[1:])]
M.update(SnrBoth=num(round(o.snr.median())), SnrFaint=num(round(b.snr[faint].median())), SnrGLo=num(mass_samples.RESPONSE_G_EDGES[-2]),
         SnrGHi=num(mass_samples.RESPONSE_G_EDGES[-1]), XpOffLo=f2(-max(dxp)), XpOffHi=f2(-min(dxp)), XpG=num(XP_G),
         XpRichLo=num(XP_RICH[0]), XpRichHi=num(XP_RICH[-1]))
b = b.merge(pd.read_parquet(DATA / 'giant_foreground.parquet')[['source_id', 'rho', 'fill', 'f_mw']], on='source_id')
outer = b[b.rho >= foreground.RINGS[-2]]   # BOSS giants in the outermost ring of lmcold.foreground.RINGS
w = outer.w * outer['fill']   # selection weight x coverage fill
# Weighted mean foreground fraction for [Fe/H] < -1.5, -1.5 to -1 and >= -1; quoted rounded to 10 per cent.
fo = [np.average(outer.f_mw[m], weights=w[m]) for m in (outer.feh < -1.5, (outer.feh >= -1.5) & (outer.feh < -1), outer.feh >= -1)]
M.update(FgOutLo=num(10 * round(10 * min(fo))), FgOutHi=num(10 * round(10 * max(fo))), FgRingLo=num(foreground.RINGS[-2]),
         FgRingHi=num(foreground.RINGS[-1]))
```

#### Giants per unit mass
The PARSEC conversion from giants to formed mass, the age windows and the efficiency against age.

```python
# Giants per unit mass (PARSEC conversion)
# IMF mass range [Msun] from the mass-type string of results/giant_mass.json.
imf = re.search(r'Kroupa ([0-9.]+)\D+([0-9.]+) Msun', gm['mass_type']).groups()
reg = pd.read_csv(R / 'mass_regions.csv')   # sky regions of the conversion, with their distance-modulus offset dmu [mag]
# Extinction coefficients written inline in lmcold.isochrones: the two factors of A_V per E(B-V), and A_H per E(B-V).
av = re.search(r'av, av_carton = ([.0-9]+) \* ([.0-9]+) \* ebv', (ROOT / 'src/lmcold/isochrones.py').read_text()).groups()
ah = src('src/lmcold/isochrones.py', r'Hmag\.to_numpy\(\) \+ LMC_DISTANCE_MODULUS \+ dmu \+ ([.0-9]+) \* ebv')
# IMF range, distance modulus, inclination and position angle [deg], depth [mag], E(V-I)/E(B-V), G limit, median E(B-V) of the
# parent (true and carton), subdivisions, regions, rings, dmu range and A_H / A_G.
M.update(ImfLo=imf[0], ImfHi=imf[1], DM=num(LMC_DISTANCE_MODULUS), Inc=num(LMC['inc']), Theta=num(LMC['theta']),
         Depth=num(LMC_DEPTH_MAG), EviEbv=f'{EVI_PER_EBV:.3f}', GLim=num(selection.PG_EDGES[-1]),
         EbvTrue=f"{gm['parent_median_ebv']['true']:.3f}", EbvCarton=f"{gm['parent_median_ebv']['carton']:.3f}",
         NSub=num(default(isochrones.selected_count, 'subdivisions')), NReg=num(len(reg)), Rings=listing([num(x) for x in bgm['RINGS']]),
         DmuLo=f2(reg.dmu.min()), DmuHi='+' + f2(reg.dmu.max()), AhAg=f2(ah / (magcloud.A_OVER_AV['G'] * float(av[0]) * float(av[1]))))
mc = (ROOT / 'src/lmcold/mass_conversion.py').read_text()
# The two coefficients of the [M/H]-[Fe/H]-[alpha/Fe] relation written in lmcold.mass_conversion (log10(A 10^al + B)).
sal = re.search(r'np\.log10\(([.0-9]+)\*10\*\*al\+([.0-9]+)\)', mc).groups()
at = pd.read_csv(R / 'mass_alpha_trend.csv')   # measured [alpha/Fe] against [Fe/H]
amr = LMC_AMR_POVICK24   # age-metallicity relation of lmcold.constants: feh, age [Gyr], sigma per point
# Relation coefficients, AMR end points and scatter range, and the oldest age and lowest [M/H] of the PARSEC grid.
M.update(SalA=num(float(sal[0])), SalB=num(float(sal[1])), AmrNow=f2(amr['feh'][0]), AmrOld=f2(amr['feh'][-1]), AmrAge=num(amr['age'][-1]), AmrSigLo=f2(min(amr['sigma'])),
         AmrSigHi=f2(max(amr['sigma'])), GridAge=f2(gm['grid_age_max_gyr']), GridMh=f2(gm['grid_mh_min']))
for k, i in zip(('One', 'Two', 'Three', 'Four'), (0, 2, 4, len(at) - 1)):   # four rows of the trend, from metal-poor to metal-rich
    M.update({f'Al{k}': f2(at.alpha[i]), f'AlFeh{k}': f1(at.feh[i])})
# Window A against the reference weighting: change of the masses below -2, -1.5, -1 and of the total (reference conversion,
# measured alpha, reference foreground, full coverage)
v = pd.read_csv(R / 'giant_mass_variants.csv')
vr = v[(v.conversion == 'reference') & (v.alpha_model == 'measured') & (v.foreground == 1) & v.filled].pivot(
    index='feh', columns='prior', values='mass').sort_index().cumsum()
# Fractional change, window A over AMR weighting, of the cumulative masses at rows 0, 2, 4 (upper edges -2, -1.5, -1) and the total.
ca = (vr.A / vr.AMR).iloc[[0, 2, 4, -1]] - 1
M.update(AWinLo=num(round(100 * ca.min())), AWinHi=num(round(100 * ca.max())))
# Efficiency eta against age: ratio between the grid ages nearest ETA_AGES at the lowest and highest [M/H] of figure (a)
ETA_AGES = (2., 12.)   # Gyr
gc_ = pd.read_csv(R / 'giant_conversion.csv').query("survey == 'BOSS'")   # n_box per (mh, age) of the BOSS grid
fac = {}   # [M/H] -> n_box at 2 Gyr / n_box at 12 Gyr
for mh, q in gc_.groupby('mh'):
    # n_box at the grid age nearest each of ETA_AGES.
    n = [q.n_box.iloc[np.argmin(abs(q.age - a))] for a in ETA_AGES]
    fac[mh] = n[0] / n[1]
M.update(EtaAgeLo=num(ETA_AGES[0]), EtaAgeHi=num(ETA_AGES[1]), EtaMhLo=num(min(fac)), EtaMhHi=num(max(fac)),
         EtaFacLo=f1(fac[min(fac)]), EtaFacHi=f1(fac[max(fac)]))
```

#### BOSS metallicity response and hot giants
The response of the BOSS [Fe/H] and the offsets of hot giants.

```python
# The BOSS metallicity response (results/boss_response.json)
bias = pd.DataFrame(br['bias'])   # per reporting bin: feh, n, median BOSS - APOGEE [dex]
mid = (bias.feh > FEH_EDGES[1]) & (bias.feh < FEH_EDGES[3])   # -2 < [Fe/H]_APOGEE < -1.5
c = br['closure']['all']   # closure on all stars
b2 = c['below_-2.0']   # its totals below [Fe/H] = -2
gr = pd.read_csv(R / 'giant_mass_response.csv')
# Distinct fitted / direct factors of the response correction, in table order.
fd = gr.drop_duplicates('fitted_over_direct').fitted_over_direct
# Overlap size, bias outside -2..-1.5 (rounded up to 0.01) and inside it (sign reversed), response bin settings, closure chi2,
# BOSS observed / APOGEE below -2, the measured-bin width, overlap stars below -1.5 and the four response factors.
M.update(NOverlap=f"{br['overlap']:,}".replace(',', '\\,'), BiasOk=f2(np.ceil(100 * bias['median'][~mid].abs().max()) / 100),
         BiasLo=f2(-bias['median'][mid].max()), BiasHi=f2(-bias['median'][mid].min()),
         GEdges=listing([num(x) for x in mass_samples.RESPONSE_G_EDGES[1:-1]]), SnrEdge=num(mass_samples.RESPONSE_SNR_EDGES[0]),
         RespSplit=num(src('src/lmcold/mass_samples.py', r'o\.feh_a\.to_numpy\(\) < (-[.0-9]+)')),
         RespChi=f1(c['chi2']), RespDof=num(c['dof']), DirectRatio=f2(b2['observed'] / b2['apogee']),
         DirectRatioOne=f1(b2['observed'] / b2['apogee']),
         MeasW=num(mass_samples.MEASURED_EDGES[2] - mass_samples.MEASURED_EDGES[1]),
         NPoor=num(sum(n for f, n in zip(bias.feh, bias.n) if f < FEH_EDGES[3])),
         FacOne=f2(fd.iloc[0]), FacTwo=f2(fd.iloc[1]), FacThree=f2(fd.iloc[2]), FacFour=f2(fd.iloc[3]))
# Observed / predicted below -2 with its distances to the 16th and 84th percentiles.
M.update(zip(('RespTwo', 'RespTwoUp', 'RespTwoDn'), bounds(b2['ratio'], *b2['ratio_p16_p84'])))
# Hot giants (results/boss_hot_bias.json): offsets of calibrated BOSS from APOGEE [Fe/H] per BOSS Teff bin, low and high S/N
te = hb['t_edges']   # K, Teff bin edges
k3 = te.index(4300)   # index of the bin starting at 4300 K
# Offsets [dex] (low and high S/N) in the 4300-4500 K bin (warm), the 4500-4700 K bin (hot), and the largest below 4300 K.
warm = (hb['offset_low'][k3], hb['offset_high'][k3])
hot = (hb['offset_low'][k3 + 1], hb['offset_high'][k3 + 1])
cool = max(abs(x) for x in hb['offset_low'][:k3] + hb['offset_high'][:k3])
# Teff edges, offset ranges, S/N split, hot low-S/N weight fractions [%], and the settings of the metal-rich split
# (rings [deg], G and colour edges [mag], minimum stars per cell).
M.update(TeffA=num(te[k3]), TeffB=num(te[k3 + 1]), TeffC=num(te[k3 + 2]), OffWarmLo=f2(min(warm)), OffWarmHi=f2(max(warm)),
         OffHotLo=f2(min(hot)), OffHotHi=f2(max(hot)), OffCool=f2(cool), SnrSplit=num(hb['snr_split']),
         HotWeight=pct(hb['target_weight_hot_lowsnr']), HotOverlap=pct(hb['overlap_fraction_hot_lowsnr']),
         HotAbove=pct(hb['boss_frac_above_m05']['hot_lowsnr']),
         SplitRings=', '.join(f'{a:g}–{b:g}' for a, b in zip([0] + mass_samples.SPLIT_RING_EDGES,
                                                              mass_samples.SPLIT_RING_EDGES + [gm['full_radius_deg']])),
         SplitG=listing([num(x) for x in mass_samples.SPLIT_G_EDGES]), SplitC=listing([num(x) for x in mass_samples.SPLIT_C_EDGES]),
         NMinSplit=num(mass_samples.NMIN_SPLIT))
```

#### Mass: present-day mass, uncertainties and result
The present-day mass, the uncertainty budget, the masses per bin and the APOGEE check in the field cores.

```python
# Present-day mass (results/giant_present_mass.json)
# Present over formed mass: all stars and [Fe/H] < -1.5.
M.update(PresAll=f2(gp['total']['present_over_formed']), PresPoor=f2(gp['below_m15']['present_over_formed']))

# Uncertainties (results/giant_mass.json; scripts/build_giant_mass.py)
alt = gm['conversion_alternatives']   # full mass under alternative conversions (E+-, i+-)
alphas = sorted(float(a) for a in v.alpha_model.unique() if a != 'measured')
# Bootstrap draws and nside, alpha models, alpha cap, E(B-V) change, the largest mass change from E(B-V) [%], the inclination
# change [deg] and the largest mass change from it [%, rounded up to 0.1].
M.update(NBoot=num(bgm['NBOOT']), Nside=re.search(r'nside(\d+)', gm['bootstrap']).group(1), Alphas=listing([num(a) for a in alphas]),
         AlphaCap=num(src('src/lmcold/mass_conversion.py', r'min\(float\(alpha\),([.0-9]+)\)')),
         DEvi=num(src('scripts/build_giant_mass.py', r'DEBV=([.0-9]+)/EVI_PER_EBV')),
         EChange=num(round(100 * max(abs(alt[k]['full'] / gm['full_mass'] - 1) for k in ('E+', 'E-')))),
         DInc=num(bgm['DINC']),
         IncChange=f1(np.ceil(1000 * max(abs(alt[k]['full'] / gm['full_mass'] - 1) for k in ('i+', 'i-'))) / 10))

# Result (results/giant_present_mass.csv, giant_mass_bootstrap.csv, giant_mass_variants.csv)
tot = t.iloc[-1]   # last row: cumulative over all bins
# Total formed mass [1e9 Msun] with its distances to the 16th and 84th percentiles.
M.update(zip(('Tot', 'TotUp', 'TotDn'), (f2(x / 1e9) for x in (tot.cumulative, tot.cumulative_p84 - tot.cumulative,
                                                                tot.cumulative - tot.cumulative_p16))))
# Range of the total over the model variants, present-day total and its bootstrap range [1e9 Msun].
M.update(TotMLo=f2(tot.cumulative_model_min / 1e9), TotMHi=f2(tot.cumulative_model_max / 1e9), Pres=f2(tot.present_cumulative / 1e9),
         PresBootLo=f2(tot.present_cumulative_p16 / 1e9), PresBootHi=f2(tot.present_cumulative_p84 / 1e9))
i2, i15 = 0, 2   # rows whose upper edge is -2 and -1.5
# Mass below -2 and -1.5 [1e7 Msun] and as a percentage of the total.
for k, i in (('Two', i2), ('OneFive', i15)):
    r = t.iloc[i]
    M.update(zip((f'Below{k}', f'Below{k}Up', f'Below{k}Dn'), sci2(r.cumulative, r.cumulative_p16, r.cumulative_p84, 7)))
    M[f'Pct{k}'] = f1(100 * r.cumulative / tot.cumulative)
rich = t.feh.to_numpy() > -.5   # bins centred above [Fe/H] = -0.5
boot = pd.read_csv(R / 'giant_mass_bootstrap.csv').to_numpy()   # (draws, bins) formed mass
fb = boot[:, rich].sum(1) / boot.sum(1)   # metal-rich fraction per draw
# Metal-rich fraction for each model variant (conversion, prior, alpha model, foreground).
fm = v[v.filled].groupby(['conversion', 'prior', 'alpha_model', 'foreground']).apply(
    lambda d: d.mass[d.feh > -.5].sum() / d.mass.sum(), include_groups=False)   # as scripts/paper_table_mass_summary.py
f0 = t.formed_mass[rich].sum() / t.formed_mass.sum()   # adopted metal-rich fraction
# Fractions [%] (adopted, bootstrap range, model range), and the masses below -2 and -1.5 without the response [1e7 Msun].
M.update(FracRich=pct(f0), FracRichLo=pct(np.percentile(fb, 16)), FracRichHi=pct(np.percentile(fb, 84)), FracRichMLo=pct(fm.min()),
         FracRichMHi=pct(fm.max()), NoRespTwo=f1(gm['direct_below_m2'] / 1e7), NoRespOneFive=f1(gm['direct_below_m15'] / 1e7))
peak = int(np.argmax(t.formed_mass))   # bin of the largest formed mass
# Mean factor of rise per bin from bin 1 to the peak, and the peak bin's range.
M.update(Rise=f1((t.formed_mass[peak] / t.formed_mass[1]) ** (1 / (peak - 1))), PeakBin=binname(peak))
# APOGEE check in the field cores: radius from the area of the caps, 2 pi (1 - cos r) per core
rcore = np.degrees(np.arccos(1 - gm['common_area_deg2'] * (np.pi / 180) ** 2 / (2 * np.pi * gm['fields'])))
# Number of fields, core radius [deg], coverage threshold [%], common area [deg^2].
M.update(NCore=num(gm['fields']), RCore=f1(rcore), CoreCov=num(100 * src('scripts/build_giant_mass.py', r'coverage>=([.0-9]+)')),
         CoreArea=f1(gm['common_area_deg2']))
# BOSS / APOGEE mass ratio in the cores with its distances to the percentiles.
M.update(zip(('CoreRatio', 'CoreRatioUp', 'CoreRatioDn'), bounds(gm['common_boss_apogee_ratio'], *gm['common_ratio_p16_p84'])))
```

#### Comparisons
Published masses, the Gaia XP and cluster comparisons of panel (c), and the published field-star distributions.

```python
# Published masses (lmcold.constants.LMC_MASS_LITERATURE)
lit = {r['key']: r['mass'] / 1e9 for r in LMC_MASS_LITERATURE}   # key -> mass [1e9 Msun]
# Present-day masses, van der Marel (2002) excluded from the range (quoted separately as LitVdM).
pres = [r['mass'] / 1e9 for r in LMC_MASS_LITERATURE if r['kind'] == 'present' and r['key'] != 'vanderMarel2002']
M.update(LitHZ=f1(lit['harris_zaritsky2009']), LitMazzi=f1(lit['mazzi2021']), LitPresLo=f1(min(pres)), LitPresHi=f1(max(pres)),
         LitVdM=f1(lit['vanderMarel2002']))
# Panel (c) comparisons: Gaia XP [M/H] of the BOSS parent as mass, and the clusters (as scripts/paper_mass.py)
# Conversion efficiency eta per [Fe/H] bin: whole field, reference prior, measured alpha.
conv = pd.read_csv(R / 'mass_conversion_factors.csv').query(
    "field == 'full' and prior == @mass_conversion.REFERENCE_PRIOR and alpha_model == 'measured'").sort_values('feh')
# XP [M/H] [dex] of the eligible BOSS parent stars (Andrae et al. 2023 catalogue).
xp = pd.read_parquet(DATA / 'boss_parent.parquet', columns=['source_id', 'eligible']).query('eligible').merge(
    pd.read_parquet(DATA / 'xp_andrae.parquet', columns=['source_id', 'mh_xp']), on='source_id').mh_xp.dropna().to_numpy()
# [M/H] to [Fe/H] with the measured alpha trend and the relation of lmcold.mass_conversion.
feh_xp = xp - np.log10(float(sal[0]) * 10 ** np.interp(xp, at.feh, at.alpha) + float(sal[1]))
k = np.digitize(feh_xp, FEH_EDGES) - 1   # reporting bin of each star
mx = np.bincount(k, 1 / conv.eta.to_numpy()[k], len(FEH_CENTRES))   # formed mass per reporting bin, up to a constant
cl = pd.read_csv(DATA / 'clusters_grocholski06.csv').feh.dropna()   # cluster [Fe/H] [dex]
# KDE bandwidths of panel (c), read from scripts/paper_mass.py.
kde = re.search(r'KDE_BW,NBOOT_KDE=dict\(xp=([.0-9]+),clusters=([.0-9]+)\)', (ROOT / 'scripts/paper_mass.py').read_text()).groups()
M.update(KdeXp=num(float(kde[0])), KdeCl=num(float(kde[1])), CoreMinN=num(src('scripts/paper_mass.py', r'\(c\.boss_raw>=(\d+)\)')),
         PeMinN=num(src('scripts/paper_parent_extinction.py', r'binned\.n_g >= (\d+)')),
         FgRingW=num(foreground.RINGS[1] - foreground.RINGS[0]))
# XP sample size, its mass fractions below -1.5 and -2 and above -0.5, its peak bin; clusters: number, median, fraction above -0.5.
M.update(NXp=f'{len(xp):,}'.replace(',', '\\,'), XpOneFive=f1(100 * mx[:i15 + 1].sum() / mx.sum()),
         XpTwo=f1(100 * mx[0] / mx.sum()), XpRich=pct(mx[rich].sum() / mx.sum()), XpPeakBin=binname(int(np.argmax(mx))),
         NCl=num(len(cl)), ClMed=f2(cl.median()), ClRich=pct((cl > -.5).mean()))
# Published field-star distributions (results/mdf_literature.json, from scripts/paper_mdf_literature.py)
md = json.loads((R / 'mdf_literature.json').read_text())
# Panel (a) (bar) and panel (b) of the literature figure.
pa, pb = md['(a) Bar'], next(v for k, v in md.items() if k.startswith('(b)'))
off = (pa['Cole+05, Ca II triplet']['median'] - pa['BOSS, as measured']['median'],
       pb['Carrera+08, Ca II triplet']['median'] - pb['BOSS, as measured']['median'])
sc = md['scale_check']   # BOSS minus literature [Fe/H] of matched stars
# Offset range, mean scale difference (sign reversed, 1 significant figure), and the matched-star counts.
M.update(MdfOffLo=f2(min(off)), MdfOffHi=f2(max(off)),
         MdfScale=sigfig(-np.mean([sc['BOSS - Carrera+08']['median'], sc['BOSS - Cole+05']['median']])),
         NCarMatch=num(sc['BOSS - Carrera+08']['n']), NColeMatch=num(sc['BOSS - Cole+05']['n']))
```

#### Appendix
The response by APOGEE field type, the field cores bin by bin, and dust.

```python
# Appendix: response by APOGEE field type (results/boss_response.json)
# Observed / predicted below -1.5 per APOGEE field type, with distances to the percentiles.
for k, key in (('Late', 'LMC18-35'), ('Early', 'LMC1-17 (W+D)')):
    x = br['closure'][key]['below_-1.5']
    M.update(zip((f'Resp{k}', f'Resp{k}Up', f'Resp{k}Dn'), bounds(x['ratio'], *x['ratio_p16_p84'])))
M.update(NBootResp=num(bresp['NBOOT']))
# Appendix: the field cores, bin by bin (results/giant_mass_common.csv, giant_mass_fields.csv)
cm = pd.read_csv(R / 'giant_mass_common.csv')   # BOSS / APOGEE in the field cores per bin
# Ratios of bin 0 (below -2) and bin 2.
for k, i in (('CoreTwo', 0), ('CoreMid', 2)):
    r = cm.iloc[i]
    M.update(zip((k, k + 'Up', k + 'Dn'), bounds(r.ratio, r.ratio_p16, r.ratio_p84)))
fields = pd.read_csv(R / 'giant_mass_fields.csv').field.unique()   # APOGEE field names 'LMC<n>'
up = cm.feh > FEH_EDGES[5]   # bins above [Fe/H] = -1
# APOGEE stars below -2 in the cores, bin-2 range, fields LMC1-17, BOSS / APOGEE mass above -1, H limit and lower mass edge.
M.update(CoreTwoN=num(cm.apogee_raw[0]), CoreMidBin=binname(2), NCoreWD=num(sum(int(f[3:]) <= 17 for f in fields)),
         CoreRich=f2(cm.boss_mass[up].sum() / cm.apogee_mass[up].sum()), HLim=num(default(isochrones.count_grid, 'h_limit')),
         HLo=num(src('src/lmcold/isochrones.py', r'\(m >= ([.0-9]+)\)')))
# Appendix: dust (results/parent_extinction.csv, scripts/paper_parent_extinction.py)
pe = pd.read_csv(R / 'parent_extinction.csv')
# Ratio q = observed / predicted parent counts per E(V-I) bin, for the region and the cell models.
rg, ce = (pe[pe.model == m].reset_index(drop=True) for m in ('region', 'cell'))
GROUPS = dict(A=[2], B=[3, 4, 5], C=[6], D=[7])   # rows (E(V-I) bins) of the region model quoted in the text, alone or pooled
dusty = rg.iloc[GROUPS['B'][0]:]   # cells above the lower edge of group B


def pooled(s):
    """q of several E(V-I) bins together: sum N_G over the sum of the predicted counts N_G / q."""
    return s.n_g.sum() / (s.n_g / s.q).sum()


# Pooled q and the E(V-I) range of each group.
for k, rows in GROUPS.items():
    s = rg.iloc[rows]
    M.update({f'Q{k}': f2(pooled(s)), f'Q{k}Lo': f2(s.evi_lo.iloc[0]), f'Q{k}Hi': f2(s.evi_hi.iloc[-1])})
    if k in 'CD':   # half the bootstrap 16th-84th range, quoted for the two dustiest bins
        M[f'Q{k}Err'] = f2((s.q_p84 - s.q_p16).iloc[0] / 2)
# Cell size [arcmin], radius, ages and [M/H] of the extinction check, low-E(V-I) edge, share of parent stars in dusty cells [%],
# stars missing there [%], and the largest excess of q above 1 [%].
M.update(Cell=f1(hp.nside2resol(ppe['NSIDE'], arcmin=True)), RMax=num(ppe['RMAX']), AgeLo=num(ppe['AGES'][0]),
         AgeHi=num(ppe['AGES'][1]), MhLo=f1(min(ppe['MH'])), MhHi=f1(max(ppe['MH'])), Low=f2(ce.evi_hi[1]),
         DustyShare=f1(100 * dusty.n_g.sum() / rg.n_g.sum()), Missing=f1(100 * (dusty.n_g / dusty.q - dusty.n_g).sum() / rg.n_g.sum()),
         Overshoot=sigfig(100 * (ce.q[:GROUPS['D'][0] + 1].max() - 1)))   # over the bins quoted, to the end of group D
```

#### Write the macros
One \newcommand per entry of M, with the prefix \ms, into results/macros_mass.tex; the text shows them as macros, so no number
is typed by hand.

```python
# Header comment, then one \newcommand line per macro.
lines = ['% Numbers of the KB chapter Stellar mass per metallicity (scripts/paper_text_mass.py); do not edit.']
lines += [rf'\newcommand{{\ms{k}}}{{{x}}}' for k, x in M.items()]
(R / 'macros_mass.tex').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
```

### Metallicity

The numbers of the chapter on the metallicity structure (`results/macros_feh.tex`, prefix `\fe`).

<details><summary>About <code>scripts/paper_text_feh.py</code></summary>

```text
Knowledge-base text, chapter 'Metallicity structure' (extended draft Section 3 and its appendix on published gradients and
distributions): every number of the text as a LaTeX macro (prefix \\fe), so that none is typed by hand.

As scripts/paper_text_shape.py: settings are read from the code (scripts/textnum.const, lmcold by import, or the source line where a
setting is written inline), results from the tables the figures are drawn from. Published values are read from
lmcold.constants (LMC_FEH_GRADIENT_LITERATURE, LMC_FEH_GRADIENT_DEG_LITERATURE, LMC_FEH_LEVEL_LITERATURE); other literature values
quoted with their citation stay in the text. The number of fields of Cole et al. (2005) is counted from their catalogue; the APOGEE
weight share at bright G and the BOSS giants in the 3-deg field of Carrera et al. (2008) are counted from the mass samples.
Inputs: results/{feh_profile, feh_gradients, mass_conversion_factors, kin_mcmc_data_ellipsoid, rrab_independence}.csv and
{feh_profile, boss_response, boss_selection, boss_mock, snr_threshold_sensitivity, mdf_literature}.json; data/{mdf_cole05,
mdf_carrera08}.csv and the BOSS and APOGEE giants of lmcold.mass_samples.samples() (with BOSS parent photometry); settings of
scripts/{paper_feh_profile, paper_feh_literature, paper_mdf_literature}.py, lmcold.{constants, footprint, mass_samples, rrlyrae,
selection}.
Writes results/macros_feh.tex.
Run: .venv/bin/python scripts/manifest.py run paper_text_feh.py
```
</details>

**Data read by `scripts/paper_text_feh.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/apogee_weights.parquet` | Zenodo bundle 1 | APOGEE selection weights per candidate star (selection groups, coverage, w = N_cand / N_quality x pass factor in H x XP [M/H]) |
| `data/boss_parent.parquet` | Zenodo bundle 1 | Gaia DR3 stars within 12 deg of the LMC passing the rough cuts of mwm_magcloud_rgb_boss, with SFD E(B-V) and `eligible` |
| `data/boss_weights.parquet` | Zenodo bundle 1 | BOSS selection weights per eligible parent star (selection groups, w = N_elig / N_good x pass factor in G x XP [M/H] x ring) |
| `data/mdf_carrera08.csv` | public archive | Ca II triplet index, V, I and radial velocity of stars in four LMC fields 3-8 deg north of the bar (Carrera et al. 2008, 2008AJ....135..836C; 635 rows in VizieR) |
| `data/mdf_cole05.csv` | public archive | Ca II triplet [Fe/H] (CG97 scale) of 373 LMC bar red giants (Cole et al. 2005, 2005AJ....129.1465C) |
| `data/raw/skowron21/table1.dat.gz` | public archive | OGLE-IV red-clump E(V-I) map of the Magellanic Clouds, 81,684 boxes of 3.4-27.5 arcmin; Skowron et al. 2021, ApJS 252, 23 (2021ApJS..252...23S, doi:10.3847/1538-4365/abcb81) |
| `data/spec_giants.parquet` | Zenodo bundle 1 | APOGEE DR17 and SDSS-V DR20 BOSS spectra of LMC giant candidates before the Gaia match |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |
| `results/boss_mock.json` | repository | mock tests of the BOSS selection function; written by `python scripts/boss_mock.py` |
| `results/boss_response.json` | repository | written by `python scripts/boss_response.py` |
| `results/boss_selection.json` | repository | fixed input |
| `results/feh_gradients.csv` | repository | written by `python scripts/paper_feh_literature.py` |
| `results/feh_profile.csv` | repository | written by `python scripts/paper_feh_profile.py` |
| `results/feh_profile.json` | repository | written by `python scripts/paper_feh_profile.py` |
| `results/giant_mass.json` | repository | written by `python scripts/build_giant_mass.py` |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/mass_conversion_factors.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/mdf_literature.json` | repository | written by `python scripts/paper_mdf_literature.py` |
| `results/rrab_independence.csv` | repository | completeness of the OGLE-IV and Gaia RR Lyrae catalogues and their union; written by `python scripts/check_rrab_independence.py` |
| `results/snr_threshold_sensitivity.json` | repository | fixed input |

*Shown, not run here: the result tables in results/ and the catalogues of Zenodo bundle 1. The cells below are the script itself; run it from the repository with* `python scripts/paper_text_feh.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
Imports, the dictionary M of macros (macro name without its prefix: the text it stands for) and the functions that format
numbers as the text writes them. The inputs listed in the header are read in the step that uses them.

```python
import json
import re

import healpy as hp
import numpy as np
import pandas as pd

from lmcold import footprint, mass_samples, selection
from lmcold.constants import LMC, LMC_FEH_GRADIENT_DEG_LITERATURE, LMC_FEH_GRADIENT_LITERATURE, LMC_FEH_LEVEL_LITERATURE
from lmcold.mass_samples import FEH_EDGES
from lmcold.sources import DATA, ROOT
from textnum import const, num, sigfig

R = ROOT / 'results'   # directory of the input tables and of the output macros
M = {}                 # macro name (without the prefix \fe) -> text


def f1(x):
    """x with one decimal."""
    return f'{x:.1f}'


def f2(x):
    """x with two decimals."""
    return f'{x:.2f}'


def f3(x):
    """x with three decimals."""
    return f'{x:.3f}'


def src(path, pattern):
    """The number captured by `pattern` in the source file `path` (a setting written inline)."""
    return float(re.search(pattern, (ROOT / path).read_text()).group(1))


# Literal module constants of the three figure scripts (read without running them).
fp, fl, mdf = const('paper_feh_profile'), const('paper_feh_literature'), const('paper_mdf_literature')
t = pd.read_csv(R / 'feh_profile.csv')                          # radial profiles: sample, R_lo, R_hi [kpc], median [dex], sigma_*
js = json.loads((R / 'feh_profile.json').read_text())          # fit settings, scale lengths, bootstrap counts
br = json.loads((R / 'boss_response.json').read_text())        # BOSS [Fe/H] response: bias table, overlap count
# Median [Fe/H] [dex] per radial bin, indexed by the bin's inner radius R_lo [kpc], one Series per sample.
med = {s: d.set_index('R_lo')['median'] for s, d in t.groupby('sample')}
BRIGHT = fl['BRIGHT']   # sample name of the BOSS giants with G < 16.5
```

#### Settings of the profiles
The disc plane, the radial bins and the bootstrap of scripts/paper_feh_profile.py.

```python
# Disc plane, bins and bootstrap (lmcold.constants.LMC; scripts/paper_feh_profile.py)
km = pd.read_csv(R / 'kin_mcmc_data_ellipsoid.csv')   # posterior summaries of the ellipsoid kinematic fits
inc_kin = km[(km.stage == 2) & (km.param == 'inc')]['median'].iloc[0]   # the joint (shared-geometry) fit
bias = pd.DataFrame(br['bias'])   # BOSS minus APOGEE [Fe/H] bias per [Fe/H] bin (columns feh, median)
M.update(Inc=num(LMC['inc']), Theta=num(LMC['theta']), DZero=num(LMC['D0']), IncKin=num(round(inc_kin)),
         MinorPct=num(round(100 * (1 - np.cos(np.radians(LMC['inc'])) / np.cos(np.radians(inc_kin))))),
         RBin=num(t.R_hi[0] - t.R_lo[0]), RMax=num(t.R_hi.max()), Rad=num(json.loads((R / 'giant_mass.json').read_text())['full_radius_deg']),RhoRr=num(footprint.RHO_MAX),
         TrueA=num(mass_samples.TRUE_EDGES[2]), TrueB=num(mass_samples.TRUE_EDGES[3]),
         MeasW=num(mass_samples.MEASURED_EDGES[2] - mass_samples.MEASURED_EDGES[1]),
         BiasRich=f2(np.ceil(100 * bias['median'][bias.feh > FEH_EDGES[3]].abs().max()) / 100),
         NBoot=num(js['nboot']), Nside=num(src('scripts/paper_feh_profile.py', r"ang2pix\((\d+), s\.ra")),
         NOverlap=f"{br['overlap']:,}".replace(',', '\\,'), NSkip=num(js['skipped_bins']), NFits=f"{js['nboot'] * len(med['BOSS']):,}".replace(',', '\\,'))
M['Pix'] = f1(hp.nside2resol(int(M['Nside']), arcmin=True) / 60)   # HEALPix pixel size [deg] at Nside
```

#### Profiles of the giants
Median [Fe/H] against radius, the fitted gradients, surface densities and scale lengths, and the same quantities in the mock
of the BOSS selection.

```python
# Medians (results/feh_profile.csv)
boss, apo, bright, rr = med['BOSS'], med['APOGEE'], med[BRIGHT], med['RRab']
BREAK, AGREE = 6., 8.   # kpc: the radius of the break, and the radius out to which the bright BOSS and APOGEE medians are compared
inner = boss.index < BREAK
M.update(BossIn=f2(boss.iloc[0]), BossMid=f2(boss[BREAK - 1]), BossOut=f2(boss.iloc[-1]), BossMidLo=num(BREAK - 1), Break=num(BREAK),
         ApoOffLo=f2((boss - apo).min()), ApoOffHi=f2((boss - apo).max()), GBright=num(fp['G_BRIGHT']), GLim=num(selection.PG_EDGES[-1]),
         BrightDropLo=f2((boss - bright)[inner].min()), BrightDropHi=f2((boss - bright)[inner].max()),
         BrightAgree=f3((bright - apo)[bright.index < AGREE].abs().max()), Agree=num(AGREE))
# Gradients, ours (results/feh_gradients.csv, rows 'this work')
g = pd.read_csv(R / 'feh_gradients.csv')
ours = g[g.label == 'this work'].set_index(['sample', 'r_lo', 'r_hi'])
for k, s in (('Bright', BRIGHT), ('Apo', 'APOGEE'), ('Boss', 'BOSS')):
    for part, rng in (('In', (0., BREAK)), ('Out', (BREAK, t.R_hi.max()))):
        r = ours.loc[(s, *rng)]
        M.update({f'Sl{k}{part}': f3(r.slope), f'Sl{k}{part}Err': f3(r.err)})
r = ours.loc[('RRab', 0., BREAK)]
M.update(SlRrIn=f3(r.slope), SlRrInErr=f3(r.err))
# Surface densities and scale lengths (results/feh_profile.csv, feh_profile.json)
q = t[t['sample'] == 'BOSS'].reset_index(drop=True)
M.update(FitLo=num(js['fit_range_kpc'][0]), FitHi=num(js['fit_range_kpc'][1]))
for k, key in (('Poor', 'lt15'), ('Mid', '15to10'), ('Rich', 'gt10')):
    h = js[key]
    M.update({f'H{k}': f2(h['h']), f'H{k}Up': f2(h['h_p84'] - h['h']), f'H{k}Dn': f2(h['h'] - h['h_p16']),
              f'H{k}Zero': f2(h['h_foreground_x0']), f'H{k}Double': f2(h['h_foreground_x2']), f'H{k}One': f1(h['h'])})
mid_r = .5 * (q.R_lo + q.R_hi)
fall = (q.sigma_gt10 / q.sigma_gt10.shift(-1))[mid_r > BREAK].dropna()   # factor by which the density falls from each bin to the next
M.update(FallLo=f1(fall.min()), FallHi=f1(fall.max()), FallFit=f1(np.exp(1 / js['gt10']['h'])),
         SigIn=num(round(q.sigma_lt15[0])), SigInTwo=num(round(q.sigma_lt15[1])), SigRef=num(round(q.sigma_lt15[2])),
         SigRefHi=num(q.R_hi[2]), OutLo=num(t.R_lo.max()))
sel = json.loads((R / 'boss_selection.json').read_text())['R_ring_closure']['final']
ring = next(iter(sel))   # innermost ring, e.g. '0-1.5'
M.update(RecIn=f2(sel[ring]['<-1.5']), RecRho=ring.split('-')[1])
# The same, per ring in the mock of the BOSS selection (results/boss_mock.json: central failure model, standard coverage, ring fill;
# ring0 = within the first edge of lmcold.selection.RING_EDGES)
mk = json.loads((R / 'boss_mock.json').read_text())['fill']['central']['standard/ring']['ring0']
M.update(MockIn=f2(mk['mean'][0]), MockInErr=f2(mk['sd'][0]), MockRho=num(selection.RING_EDGES[0]))
```

#### Completeness and weights
The lowest completeness of the RR Lyrae union and the share of the APOGEE selection weight in bright stars.

```python
# Completeness of the RR Lyrae union (OGLE-IV + Gaia): its minimum over the radial bins of results/rrab_independence.csv
M['UnionMin'] = num(round(100 * pd.read_csv(R / 'rrab_independence.csv').C_union_by_mag.min()))
# Share of the APOGEE selection weight at G < G_BRIGHT (Gaia G from the BOSS parent photometry, lmcold.mass_samples.with_colour)
bg, ag, _ = mass_samples.samples()
ag = mass_samples.with_colour(ag)
M['ApoBright'] = num(round(100 * ag.w[ag.g < fp['G_BRIGHT']].sum() / ag.w.sum()))
sn = json.loads((R / 'snr_threshold_sensitivity.json').read_text())
lo_snr, hi_snr = sorted((k for k in sn if k.isdigit()), key=int)   # S/N thresholds: adopted, alternative
dh = max(abs(sn[hi_snr]['h'][k] - sn[lo_snr]['h'][k]) for k in sn[lo_snr]['h'])
dm = np.abs(np.subtract(sn[f'median_feh_by_R_{hi_snr}'], sn[f'median_feh_by_R_{lo_snr}']))[1:].max()   # outside the first 1-kpc bin
M.update(SnrLo=lo_snr, SnrHi=hi_snr, SnrDh=f1(np.ceil(10 * dh) / 10), SnrDm=f3(dm))
```

#### Profiles of the RR Lyrae
Median [Fe/H] and surface density of the RR Lyrae per ring.

```python
# RR Lyrae: medians and surface densities (counts per 1-kpc ring in results/feh_profile.json)
FLAT = (4., 7.)   # kpc, the stretch of the RRab profile quoted as flat
flat = rr[(rr.index >= FLAT[0]) & (rr.index < FLAT[1])]
M.update(RrIn=f2(rr.iloc[0]), RrOut=f2(rr.iloc[-1]), FlatLo=num(FLAT[0]), FlatHi=num(FLAT[1]), FlatMedLo=f2(flat.max()),
         FlatMedHi=f2(flat.min()))
rrp = js['rr_lt15']
edges = np.arange(len(rrp['n']) + 1) * (t.R_hi[0] - t.R_lo[0])
dens = np.array(rrp['n']) / (np.pi * np.diff(edges ** 2))   # RRab below -1.5 per kpc^2
ratio = dens / q.sigma_lt15[:len(dens)].to_numpy()
fit = (.5 * (edges[1:] + edges[:-1]) > rrp['fit_range_kpc'][0]) & (.5 * (edges[1:] + edges[:-1]) < rrp['fit_range_kpc'][1])
M.update(RrRMax=num(rrp['fit_range_kpc'][1]), RrFitLo=num(rrp['fit_range_kpc'][0]), RatioMean=f2(ratio[fit].mean()),
         RatioLo=f2(ratio[fit].min()), RatioHi=f2(ratio[fit].max()), RatioSd=f2(ratio[fit].std()), HRrPoor=f2(rrp['h']),
         HRrRich=f2(js['rr_gt15']['h']), RrCentre=num(round(dens[0])), SigInLo=num(round(q.sigma_lt15_lo[0])),
         SigInHi=num(round(q.sigma_lt15_hi[0])), RatioIn=f1(ratio[0]), RatioInLo=f1(dens[0] / q.sigma_lt15_hi[0]),
         RatioInHi=num(round(dens[0] / q.sigma_lt15_lo[0])), RatioTwo=f1(ratio[1]), RatioTwoLo=f1(dens[1] / q.sigma_lt15_hi[1]),
         RatioTwoHi=f1(dens[1] / q.sigma_lt15_lo[1]))
```

#### Giants per unit mass
The number of BOSS giants per unit formed mass (eta) in the metal-poor and metal-rich bins.

```python
# Giants per unit mass (results/mass_conversion_factors.csv, BOSS, whole parent, reference weighting, measured alpha)
cv = pd.read_csv(R / 'mass_conversion_factors.csv').query("field == 'full' and prior == 'AMR' and alpha_model == 'measured'")
cv = cv[cv.survey == 'BOSS'].set_index('feh').eta
poor, rich = cv[(cv.index > -2) & (cv.index < -1.5)], cv[(cv.index > -.75) & (cv.index < -.25)]
M.update(EtaPoor=f1(poor.mean() * 1e4), EtaRichLo=f1(rich.min() * 1e4), EtaRichHi=f1(rich.max() * 1e4))
```

#### Published gradients
The published gradients (lmcold.constants) against ours over the same radial ranges.

```python
# Published gradients (lmcold.constants) against ours over the same ranges (results/feh_gradients.csv)
lit = {L['label']: L for L in LMC_FEH_GRADIENT_LITERATURE}   # published gradient [dex/kpc] by label
row = {(r.label, r['sample']): r for _, r in g.iterrows()}   # our fits over each published range, by (label, sample)


# Published slopes and errors as given (format g).
for k, label in (('Povick', 'Povick+25'), ('Grady', 'Grady+21'), ('Cioni', 'Cioni 09'), ('ChTwo', 'Choudhury+21'), ('Sk', 'Skowron+16 inner')):
    L = lit[label]
    M.update({f'Lit{k}': f'{L["slope"]:g}', f'Lit{k}Err': f'{L["err"]:g}'})
M.update(LitChTwoR=num(lit['Choudhury+21']['r_hi']), LitSkR=num(lit['Skowron+16 inner']['r_hi']))
# Choudhury et al. (2016): a range of slopes (slope to slope_hi).
c16 = lit['Choudhury+16']
M.update(LitChOneLo=f3(c16['slope']), LitChOneHi=f3(c16['slope_hi']), LitChOneR=num(c16['r_hi']))
# Our slopes [dex/kpc] over the same ranges, for the bright BOSS and the APOGEE giants.
for k, label in (('Povick', 'Povick+25'), ('ChOne', 'Choudhury+16'), ('ChTwo', 'Choudhury+21')):
    for s, sk in ((BRIGHT, 'Bright'), ('APOGEE', 'Apo')):
        r = row[label, s]
        M.update({f'Our{k}{sk}': f3(r.slope), f'Our{k}{sk}Err': f3(r.err)})
# Grady+21 and Cioni 09: range of the ratio published / our slope, and range of our slopes over the same ranges.
steep = [row[l, s] for l in ('Grady+21', 'Cioni 09') for s in (BRIGHT, 'APOGEE')]
M.update(SteepLo=f1(min(r.slope_lit / r.slope for r in steep)), SteepHi=f1(max(r.slope_lit / r.slope for r in steep)),
         OurSteepLo=f3(max(r.slope for r in steep)), OurSteepHi=f3(min(r.slope for r in steep)))
# Munoz et al. (2023) gradient per degree and our fits over its range (weighted, and unweighted for the bright giants).
mu = LMC_FEH_GRADIENT_DEG_LITERATURE[0]
mrow = {s: row[next(l for l, x in row if l.startswith('Muñoz')), s] for s in (BRIGHT, 'APOGEE')}
M.update(LitMunoz=f'{mu["slope"]:g}', LitMunozR=num(mu['rho_hi']), OurMunozApo=f3(mrow['APOGEE'].slope),
         OurMunozApoErr=f3(mrow['APOGEE'].err), OurMunozBright=f3(mrow[BRIGHT].slope), OurMunozBrightErr=f3(mrow[BRIGHT].err),
         OurMunozUnw=f3(mrow[BRIGHT].slope_unweighted))
# Carrera et al. (2008) [Fe/H] levels: inner field and the outer field at the mean of its radial range [deg].
car = [L for L in LMC_FEH_LEVEL_LITERATURE if L['label'] == 'Carrera+08']
M.update(CarIn=f1(car[0]['feh']), CarInR=num(car[0]['rho_hi']), CarOut=f1(car[1]['feh']), CarOutR=num(np.mean([car[1]['rho_lo'], car[1]['rho_hi']])))
# Skowron et al. (2016) RR Lyrae ranges: our RRab slopes over their middle and inner ranges.
skm = row['Skowron+16 middle', 'RRab']
M.update(SkMidLo=num(skm.r_lo), SkMidHi=num(skm.r_hi), OurSkMid=f3(skm.slope), OurSkMidErr=f3(skm.err),
         OurSkIn=f3(row['Skowron+16 inner', 'RRab'].slope), OurSkInErr=f3(row['Skowron+16 inner', 'RRab'].err),
         SkOutLo=num(lit['Skowron+16 outer']['r_lo']))
```

#### Appendix: fit settings and RR Lyrae metallicity scale
Settings of the fits and of the RR Lyrae metallicities.

```python
# Appendix: fit settings; scale of the RR Lyrae metallicities
SK16_ZW = (0.88, 1.431)   # Skowron et al. (2016): [Fe/H]_ZW = ([Fe/H]_J95 - 0.88) / 1.431
zw_c09 = src('src/lmcold/rrlyrae.py', r'return ([.0-9]+) \* feh_zw')   # [Fe/H]_C09 = 1.105 [Fe/H]_ZW + 0.160 (Carretta et al. 2009)
M.update(Floor=f3(src('scripts/paper_feh_literature.py', r"\.clip\(([.0-9]+)\)")),
         MinBins=num(src('scripts/paper_feh_literature.py', r'if len\(q\) < (\d+)')),
         KpcDeg=f3(LMC['D0'] * np.pi / 180), CrestOff=f2(src('src/lmcold/rrlyrae.py', r'feh_zw \+ [.0-9]+ \+ ([.0-9]+)')),
         ZwSlope=f2(zw_c09 / SK16_ZW[1]), ZwInner=f3(lit['Skowron+16 inner']['slope'] * zw_c09 / SK16_ZW[1]))
```

#### Appendix: published metallicity distributions
Comparison with the field-star distributions of Cole et al. (2005), Carrera et al. (2008) and Omkumar et al. (2025).

```python
# Appendix: metallicity distributions (results/mdf_literature.json)
md = json.loads((R / 'mdf_literature.json').read_text())
pa, pb, pc, pd_ = (next(v for k, v in md.items() if k.startswith(f'({x})')) for x in 'abcd')
cole = pd.read_csv(DATA / 'mdf_cole05.csv').Field.str.split('/').str[0]
sc = md['scale_check']
# BOSS giants within FIELD_R of the 3-deg field of Carrera et al. (2008) (their radial-velocity members within 4 deg of the centre, as
# scripts/paper_mdf_literature.py, which leaves the field out)
car = pd.read_csv(DATA / 'mdf_carrera08.csv').query('170 <= rv <= 380')
car = car[footprint.angular_distance(car.ra.to_numpy(), car.dec.to_numpy(), LMC['ra0'], LMC['dec0']) < 4]
M['NCarIn'] = num((footprint.angular_distance(bg.ra.to_numpy(), bg.dec.to_numpy(), car.ra.mean(), car.dec.mean()) < mdf['FIELD_R']).sum())


def stat(d, key, f='median'):
    """Statistic f of sample key in panel dict d: the median [dex] with two decimals, a fraction as a rounded percentage."""
    x = d[key][f]
    return f2(x) if f == 'median' else num(round(100 * x))


M.update(FieldR=num(mdf['FIELD_R']), NColeF=num(cole.nunique()), NColeMatch=num(sc['BOSS - Cole+05']['n']),
         ColeMed=stat(pa, 'Cole+05, Ca II triplet'), BarBoss=stat(pa, 'BOSS, as measured'), BarApo=stat(pa, 'APOGEE'),
         ColeRich=stat(pa, 'Cole+05, Ca II triplet', 'frac_above_m05'),
         CarMed=stat(pb, 'Carrera+08, Ca II triplet'), DiscBoss=stat(pb, 'BOSS, as measured'), DiscApo=stat(pb, 'APOGEE'),
         CarPoor=stat(pb, 'Carrera+08, Ca II triplet', 'frac_below_m1'), DiscBossPoor=stat(pb, 'BOSS, as measured', 'frac_below_m1'),
         DiscApoPoor=stat(pb, 'APOGEE', 'frac_below_m1'),
         ScCar=f2(sc['BOSS - Carrera+08']['median']), NScCar=num(sc['BOSS - Carrera+08']['n']), SdCar=f2(sc['BOSS - Carrera+08']['robust_sd']),
         ScApo=f2(sc['APOGEE - Carrera+08']['median']), NScApo=num(sc['APOGEE - Carrera+08']['n']),
         SdApo=f2(sc['APOGEE - Carrera+08']['robust_sd']), ScCole=f2(sc['BOSS - Cole+05']['median']),
         ApoTgt=stat(pc, 'APOGEE, as targeted'), ApoW=stat(pc, 'APOGEE, weighted'),
         ApoTgtRich=stat(pc, 'APOGEE, as targeted', 'frac_above_m05'), ApoWRich=stat(pc, 'APOGEE, weighted', 'frac_above_m05'),
         ApoTgtPoor=stat(pc, 'APOGEE, as targeted', 'frac_below_m1'), ApoWPoor=stat(pc, 'APOGEE, weighted', 'frac_below_m1'),
         BossAll=stat(pc, 'BOSS, weighted, as measured'), BossAllRich=stat(pc, 'BOSS, weighted, as measured', 'frac_above_m05'),
         NXp=f"{pd_['BOSS, as measured']['n']:,}".replace(',', '\\,'), OmkMed=stat(pd_, 'Omkumar+25, XP Strömgren'),
         XpBossMed=stat(pd_, 'BOSS, as measured'), OmkRich=stat(pd_, 'Omkumar+25, XP Strömgren', 'frac_above_m05'),
         OmkPoor=stat(pd_, 'Omkumar+25, XP Strömgren', 'frac_below_m1'), XpBossRich=stat(pd_, 'BOSS, as measured', 'frac_above_m05'),
         XpBossPoor=stat(pd_, 'BOSS, as measured', 'frac_below_m1'), AndMed=stat(pd_, 'Andrae+23, XP'),
         AndShift=sigfig(pd_['BOSS, as measured']['median'] - pd_['Andrae+23, XP']['median']),
         ScaleCaT=sigfig(-np.mean([sc['BOSS - Carrera+08']['median'], sc['BOSS - Cole+05']['median']])))
```

#### Write the macros
One \newcommand per entry of M, with the prefix \fe, into results/macros_feh.tex; the page text shows them through
{glue:text}, so no number is typed by hand.

```python
# Header line, then one \newcommand per macro; written to results/macros_feh.tex and printed.
lines = ['% Numbers of the KB chapter Metallicity structure (scripts/paper_text_feh.py); do not edit.']
lines += [rf'\newcommand{{\fe{k}}}{{{x}}}' for k, x in M.items()]
(R / 'macros_feh.tex').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
```

### The kinematic model

The numbers of the chapter on the velocity field and the disc model (`results/macros_kinmodel.tex`, prefix `\km`).

<details><summary>About <code>scripts/paper_text_kin_model.py</code></summary>

```text
Knowledge-base text, chapter 'Kinematics: velocity field and disc model' (extended draft Sections 4.1-4.3 and the appendix on the
ellipsoid mocks and the sky-aligned model): every number of the text as a LaTeX macro (prefix \\km), so that none is typed by hand.

As scripts/paper_text_shape.py: settings are read from the code (scripts/textnum.const, lmcold by import), results from the tables
the figures and tables are made from, and the posterior samples of the velocity-ellipsoid fits (for the rotation speed at R_sigma of
the joint fit, which no table holds), and the release samples (for the Gaia proper-motion errors in km/s). Literature values
quoted with their citation stay in the text.
Inputs: results/{kin_mcmc_data_ellipsoid, kin_mcmc_data_ellipsoid_outer, kin_mcmc_data_stream, kin_mcmc_data_stream_outer,
kin_support_ellipsoid, kin_support, kin_model_comparison, kin_ellipsoid_mock, kin_mock_physical, kin_tilted_ring, kin_footprint,
kin_residual_profiles_ellipsoid, kin_residual_cells_ellipsoid, kin_residual_radial_ellipsoid, kin_residual_radial}.csv;
data/kin_mcmc_data_ellipsoid{,_outer}_samples.npz; scripts/{fit_kinematics_mcmc, paper_kin_residuals, paper_kin_ellipsoid_mock,
fig_kin_mock_physical}.py; lmcold.kinfit (bounds, R_SIG, S_OUT), lmcold.constants, lmcold.samples (FEH_BINS, giants, rrab).
Writes results/macros_kinmodel.tex.
Run: .venv/bin/python scripts/manifest.py run paper_text_kin_model.py
```
</details>

**Data read by `scripts/paper_text_kin_model.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/giants_apogee.parquet` | Zenodo bundle 1 | APOGEE DR17 giants with Gaia DR3 |
| `data/giants_boss.parquet` | Zenodo bundle 1 | SDSS-V DR20 BOSS giants with Gaia DR3, on the APOGEE scale |
| `data/kin_mcmc_data_ellipsoid_outer_samples.npz` | Zenodo bundle 1 | Posterior samples of the ellipsoid kinematic fit, RR Lyrae 4-9 deg (NUTS) |
| `data/kin_mcmc_data_ellipsoid_samples.npz` | Zenodo bundle 1 | Posterior samples of the ellipsoid kinematic fit, giants 2-9 deg (NUTS) |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `results/kin_ellipsoid_mock.csv` | repository | written by `python scripts/paper_kin_ellipsoid_mock.py` |
| `results/kin_footprint.csv` | repository | fixed input |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_mcmc_data_ellipsoid_outer.csv` | repository | the same fit at 4-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid outer` |
| `results/kin_mcmc_data_stream.csv` | repository | posterior summary of the sky-aligned dispersion model with m = 2 streaming, 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data stream` |
| `results/kin_mcmc_data_stream_outer.csv` | repository | the same fit at 4-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data stream outer` |
| `results/kin_mock_physical.csv` | repository | fixed input |
| `results/kin_model_comparison.csv` | repository | fixed input |
| `results/kin_residual_cells_ellipsoid.csv` | repository | written by `python scripts/paper_kin_residuals.py ellipsoid` |
| `results/kin_residual_profiles_ellipsoid.csv` | repository | written by `python scripts/paper_kin_residuals.py ellipsoid` |
| `results/kin_residual_radial.csv` | repository | written by `python scripts/paper_kin_residuals.py` |
| `results/kin_residual_radial_ellipsoid.csv` | repository | written by `python scripts/paper_kin_residuals.py ellipsoid` |
| `results/kin_support.csv` | repository | written by `python scripts/paper_kin_support.py` |
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |
| `results/kin_tilted_ring.csv` | repository | tilted-ring fits per radius; written by `python scripts/fig_kin_tilted_ring.py` |

*Shown, not run here: the result tables in results/, the posterior samples and the catalogues of Zenodo bundle 1. The cells below are the script itself; run it from the repository with* `python scripts/paper_text_kin_model.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
Imports, the dictionary M of macros (macro name without its prefix: the text it stands for) and the functions that format
numbers as the text writes them. The inputs listed in the header are read in the step that uses them.

```python
import numpy as np
import pandas as pd
from scipy.stats import norm

from fig_kin_mock_physical import COLD, E_VLOS, H_THICK, HOT, N_STARS, RD, TRUTH
from fit_kinematics_mcmc import CHAINS, DRAWS, OUTER_INNER, RING, WARMUP, WARMUP_ELLIPSOID, ring
from lmcold import kinfit, samples
from lmcold.constants import KAPPA, LMC, LMC_DISTANCE_MODULUS
from lmcold.samples import FEH_BINS
from lmcold.sources import DATA, ROOT
from textnum import const, num, sigfig

R = ROOT / 'results'  # folder of the result tables
M = {}  # macro name (without the \km prefix) -> text
B = kinfit.BOUNDS  # prior bounds of every kinfit parameter: name -> (lo, hi)
res = const('paper_kin_residuals')  # literal module constants of scripts/paper_kin_residuals.py
mock = const('paper_kin_ellipsoid_mock')  # literal module constants of scripts/paper_kin_ellipsoid_mock.py
DISC_MOCKS = ['cold thin', 'hot', 'hot, no drift', 'hot thick']   # the four disc mocks; the 'hot spheroid' is quoted on its own


def f1(x):
    """x with one decimal."""
    return f'{x:.1f}'


def f2(x):
    """x with two decimals."""
    return f'{x:.2f}'


def f3(x):
    """x with three decimals."""
    return f'{x:.3f}'


def two(x):
    """A velocity or amplitude to two significant figures (11, 4.9, 0.48)."""
    return sigfig(x, 2)


def signed(x, fmt=f3):
    """x formatted by fmt, with an explicit '+' when not negative."""
    s = fmt(x)
    return s if s.startswith('-') else '+' + s


def count(n):
    """Integer n with thin spaces between thousands (LaTeX '\\,')."""
    return f'{int(n):,}'.replace(',', '\\,')


def rng(s, fmt=two, key=''):
    """Macros <key>Lo and <key>Hi: the minimum and maximum of s."""
    M.update({key + 'Lo': fmt(min(s)), key + 'Hi': fmt(max(s))})


def stage(t, st, pop, p):
    """Posterior median of parameter p of population pop at stage st (1: each group alone; 2: joint) in table t."""
    r = t[(t.stage == st) & (t.population == pop) & (t.param == p)]
    return r['median'].iloc[0]
```

#### Settings of the fits
The [Fe/H] bins, radial ranges, distance, model constants and priors.

```python
# Settings: [Fe/H] bins, radial ranges, distance, model constants and priors
# FehA-FehD: upper edges of the first four [Fe/H] bins [dex]; RhoIn/RhoOut: the fitted ring [deg]; RhoOuter: inner edge of the
# RR Lyrae fit [deg]; profile ring, clipping, cell size and minimum count of the residual maps (paper_kin_residuals.py)
M.update(FehA=num(FEH_BINS[0][1]), FehB=num(FEH_BINS[1][1]), FehC=num(FEH_BINS[2][1]), FehD=num(FEH_BINS[3][1]),
         RhoIn=num(RING[0]), RhoOut=num(RING[1]), RhoOuter=num(OUTER_INNER), ProfIn=num(res['PROFILE_RING'][0]),
         ProfOut=num(res['PROFILE_RING'][1]), Clip=num(res['CLIP']), Cell=num(res['CELL']), NMin=num(res['NMIN']),
         PaBin=num(30), Field=num(2 * RING[1]))   # 30 deg: the position-angle bins of paper_kin_residuals.py (PA_EDGES)
# distance [kpc] implied by the adopted distance modulus: 10^(mu/5 + 1) pc
d_mod = 10 ** (LMC_DISTANCE_MODULUS / 5 + 1) / 1e3   # kpc
# adopted distance D0 [kpc], modulus, its distance and their per-cent difference; KAPPA (km/s per mas/yr per kpc), R_sigma [kpc],
# outlier width S_OUT [mas/yr] and in km/s at D0; the vdMK14 systemic velocity and centre; prior bounds (P...) of the fits
M.update(DZero=num(LMC['D0']), DistMod=f'{LMC_DISTANCE_MODULUS:.3f}', DMod=f1(d_mod), DPct=num(round(100 * (1 - d_mod / LMC['D0']))),
         Kappa=f'{KAPPA:.5f}', RSig=num(kinfit.R_SIG), SOut=num(kinfit.S_OUT), SOutKms=sigfig(KAPPA * LMC['D0'] * kinfit.S_OUT, 2),
         VlosVdm=num(LMC['vlos0']), RaVdm=f2(LMC['ra0']), DecVdm=f2(LMC['dec0']),
         PDx=num(B['dx'][1]), PIncLo=num(B['inc'][0]), PIncHi=num(B['inc'][1]), PPm=num(B['pmra0'][1] - LMC['pmra0']),
         PVlos=num(B['vlos0'][1] - LMC['vlos0']), PVZero=num(B['v0'][1]), PVZeroLo=num(B['v0'][0]), PThetaLo=num(B['theta'][0]), PThetaHi=num(B['theta'][1]), PRZeroLo=num(B['R0'][0]), PRZeroHi=num(B['R0'][1]),
         PSlope=num(B['vslope'][1]), PStream=num(B['m2c'][1]), PSigLo=num(B['sigma_R'][0]), PSigHi=num(B['sigma_R'][1]),
         PGam=num(B['dlnsig_dR'][1]), PKLo=num(B['k'][0]), PKHi=num(B['k'][1]), PLogitLo=num(B['logit_fout'][0]),
         PLogitHi=num(B['logit_fout'][1]))
```

#### Velocity-ellipsoid fits
Posterior values of the stage-1 (each group alone) and stage-2 (joint) fits, the Gaia proper-motion errors in km/s and their
effect on sigma_phi.

```python
# Velocity-ellipsoid fits (results/kin_mcmc_data_ellipsoid*.csv): stage 1 (each group alone) and stage 2 (joint)
# E: posterior summaries of the 2-9 deg fits; EO: of the 4-9 deg fits (one row per stage, population, parameter)
E = pd.read_csv(R / 'kin_mcmc_data_ellipsoid.csv')
EO = pd.read_csv(R / 'kin_mcmc_data_ellipsoid_outer.csv')
# populations other than the shared parameters of stage 2; giants: the five giant bins
pops = [p for p in E.population.unique() if p != 'shared']
giants = [p for p in pops if p.startswith('giants')]
# stage-1 posterior medians: one row per population, one column per parameter
s1 = E[E.stage == 1].pivot_table(index='population', columns='param', values='median')
g1 = s1.loc[giants]
# stage-2 shared parameters (centre, orientation, systemic motion, streaming), indexed by parameter name
sh = E[(E.stage == 2) & (E.population == 'shared')].set_index('param')
# numbers of parameters of one stage-1 fit (giants, RR Lyrae) and of the shared set of stage 2
M.update(NPar=num(len(E[(E.stage == 1) & (E.population == giants[0])])), NParRrl=num(len(E[(E.stage == 1) & (E.population == 'RR Lyrae')])),
         NShared=num(len(sh)))
# largest offset [deg] of a giant bin's stage-1 centre from the joint centre, and largest inclination difference [deg]
dcen = np.hypot(g1.dx - sh.loc['dx', 'median'], g1.dy - sh.loc['dy', 'median'])
M.update(DCen=f1(dcen.max()), DInc=f1((g1.inc - sh.loc['inc', 'median']).abs().max()))
# ranges over the giant bins of v0 [km/s] and R0 [kpc]
rng(g1.v0, lambda x: num(round(x)), 'VZero')
rng(g1.R0, f1, 'RZero')
Rg = np.linspace(2, 10, 8001)   # kpc: radius at which each giant group's posterior-median curve peaks
peak = [Rg[np.argmax(r.v0 * np.tanh(Rg / r.R0) * (1 + r.vslope * (Rg - kinfit.R_SIG)))] for _, r in g1.iterrows()]
rng(peak, f1, 'RPeak')
# v_rot(4 kpc) posterior percentiles per population (results/kin_support_ellipsoid.csv); range of its half-width over the giant bins
ks = pd.read_csv(R / 'kin_support_ellipsoid.csv').set_index('population')
rng(((ks.vrot4_p84 - ks.vrot4_p16) / 2).loc[giants], f1, 'VrotErr')
# error scale factor k of the Gaia proper-motion errors: range over the giant bins, and the RR Lyrae at 4-9 deg and 2-9 deg
rng(g1.k, f2, 'KG')
M.update(KRrlOut=f2(stage(EO, 1, 'RR Lyrae', 'k')), KRrl=f2(s1.loc['RR Lyrae', 'k']))
# joint inclination [deg] and its cos^2, sin^2
inc = sh.loc['inc', 'median']
M.update(IncJ=num(round(inc)), CosSq=f2(np.cos(np.deg2rad(inc)) ** 2), SinSq=f2(np.sin(np.deg2rad(inc)) ** 2))
# stars per giant bin (thin-space thousands)
M.update(NG=count(ks.n.iloc[0]), NGb=count(ks.n.iloc[1]), NGc=count(ks.n.iloc[2]), NGd=count(ks.n.iloc[3]), NGe=count(ks.n.iloc[4]))

# Gaia proper-motion errors in km/s (release samples, lmcold.samples): per star the mean of the two components' errors times
# kappa D0; giants at RING, RR Lyrae at OUTER_INNER < rho < RING[1] (5th-95th percentiles) and at RING (median)


def err_kms(c):
    """Gaia proper-motion error [km/s] per star of catalogue c: mean of the two components [mas/yr] x KAPPA x D0."""
    return 0.5 * (c['pmra_error'] + c['pmdec_error']) * KAPPA * LMC['D0']


# errors per giant bin at 2-9 deg; RR Lyrae at 4-9 deg (er_out) and 2-9 deg (er)
eg = {f'giants {lab}': err_kms(ring(c)) for lab, c in samples.giants()}
er_out, er = err_kms(ring(samples.rrab(), OUTER_INNER)), err_kms(ring(samples.rrab()))
pg = np.array([np.percentile(eg[p], [5, 50, 95]) for p in giants])   # rows: giant bins in the order of giants
# 5th and 95th percentiles of the RR Lyrae errors at 4-9 deg [km/s]
pr = np.percentile(er_out, [5, 95])
# extremes over the giant bins and the RR Lyrae spread and median [km/s]
M.update(ErrGLo=num(round(pg[:, 0].min())), ErrGHi=num(round(pg[:, 2].max())), ErrRrlLo=num(round(pr[0])), ErrRrlHi=num(round(pr[1])),
         ErrFacRrl=f1(pr[1] / pr[0]), ErrMedRrl=num(round(np.median(er))))
# ratio of 95th to 5th percentile per giant bin; medians; medians times the fitted k [km/s]
rng(pg[:, 2] / pg[:, 0], f1, 'ErrFacG')
rng(pg[:, 1], lambda x: num(round(x)), 'ErrMedG')
rng(pg[:, 1] * g1.k.loc[giants].to_numpy(), lambda x: num(round(x)), 'ErrKG')
# A change DK of k moves sigma_phi^2 of the most metal-rich bin by about 2 k e^2 DK (e: its median error), and sigma_phi by that
# divided by 2 sigma_phi
DK = 0.1  # change of k considered
# 2 k e^2 DK [km^2/s^2] for the most metal-rich bin (e: its median error)
kterm = 2 * g1.k[giants[-1]] * pg[-1, 1] ** 2 * DK
# its sigma_phi [km/s] (results/kin_support_ellipsoid.csv)
sphi = ks.sigma_phi[giants[-1]]
M.update(DK=num(DK), KTerm=sigfig(kterm, 2), SigPhiSq=sigfig(sphi ** 2), DSigPhiK=sigfig(kterm / (2 * sphi)))
# model comparison table: stars per population at the inner radius of each fit (2 deg and OUTER_INNER)
mc = pd.read_csv(R / 'kin_model_comparison.csv')
mc_in, mc_out = mc[mc.rho_inner == RING[0]].set_index('population'), mc[mc.rho_inner == OUTER_INNER].set_index('population')
M.update(NRrl=count(mc_in.n['RR Lyrae']), NRrlOut=count(mc_out.n['RR Lyrae']))
```

#### Stage 1 against stage 2
The rotation speed at R_sigma from the posterior draws of both stages.

```python
# Stage 1 against stage 2: rotation speed at R_sigma from the posterior draws (data/kin_mcmc_data_ellipsoid*_samples.npz)


def vrot4(npz):
    """{(stage, population): median of v0 tanh(R_sigma / R0) over the draws}."""
    s = np.load(npz)
    names = list(s['names'])
    out = {}
    for i, n in enumerate(names):
        # sample keys: stage 1 '1|<i>|v0_0' (population i alone), stage 2 '2|all|v0_<i>' (population i of the joint fit)
        for st, key in ((1, f'1|{i}|{{}}_0'), (2, f'2|all|{{}}_{i}')):
            out[st, n] = np.median(s[key.format('v0')] * np.tanh(kinfit.R_SIG / s[key.format('R0')]))
    return out


# both fits: 2-9 deg (giants) and 4-9 deg (RR Lyrae)
v4 = vrot4(DATA / 'kin_mcmc_data_ellipsoid_samples.npz')
v4o = vrot4(DATA / 'kin_mcmc_data_ellipsoid_outer_samples.npz')
# stage-2 posterior medians per population, and the three dispersions sigma_R, sigma_phi, sigma_z
s2 = E[E.stage == 2].pivot_table(index='population', columns='param', values='median')
disp = list(kinfit.ELLIPSOID[:3])
# largest stage-1 minus stage-2 differences over the giant bins: v_rot(R_sigma) and dispersions [km/s]
M.update(DVrotStage=f1(max(abs(v4[1, p] - v4[2, p]) for p in giants)),
         DSigStage=f1((s1.loc[giants, disp] - s2.loc[giants, disp]).abs().values.max()))
# the 4-9 deg fit: RR Lyrae stage 1, all populations stage 2, and the shared stage-2 parameters
o1 = EO[EO.stage == 1].pivot_table(index='population', columns='param', values='median').loc['RR Lyrae']
o2 = EO[EO.stage == 2].pivot_table(index='population', columns='param', values='median')
osh = EO[(EO.stage == 2) & (EO.population == 'shared')].set_index('param')['median']
# joint orientation of the 4-9 deg fit [deg]; RR Lyrae stage-1 against stage-2 differences [km/s]
M.update(IncOut=f1(osh.inc), ThetaOut=f1(osh.theta), DVrotRrl=f1(abs(v4o[1, 'RR Lyrae'] - v4o[2, 'RR Lyrae'])),
         DSigRRrl=f1(abs(o2.loc['RR Lyrae', 'sigma_R'] - o1.sigma_R)), DSigPhiRrl=f1(abs(o2.loc['RR Lyrae', 'sigma_phi'] - o1.sigma_phi)))
half = EO[(EO.stage == 1) & (EO.population == 'RR Lyrae')].set_index('param')
half = (half.p84 - half.p16) / 2   # stage-1 posterior half-widths of the RR Lyrae at 4-9 deg
M.update(SigRRrlErr=f1(half.sigma_R), SigPhiRrlErr=f1(half.sigma_phi))
# spread of the stage-1 line of nodes, inclination and centre offset over the giant bins [deg]; joint streaming m2c
M.update(DTheta=f1(g1.theta.max() - g1.theta.min()), IncGLo=f1(g1.inc.min()), IncGHi=f1(g1.inc.max()), DxGLo=f2(g1.dx.min()),
         DxGHi=f2(g1.dx.max()), MTwoC=f3(sh.loc['m2c', 'median']))
```

#### Sampling diagnostics
Chains, draws and the convergence diagnostics of the fits.

```python
# NUTS sampling (scripts/fit_kinematics_mcmc.py settings; diagnostics of the result tables, as scripts/paper_text_mcmc.py)


def diagnostics(*runs):
    """Worst convergence over the given runs: max r_hat (rounded up to 3 decimals), min n_eff and the fits with divergences."""
    # all stages and populations of the runs; divergences per fit
    t = pd.concat([pd.read_csv(R / f'kin_mcmc_data_{r}.csv').assign(run=r) for r in runs])
    d = t.groupby(['run', 'stage', 'population']).divergences.max()
    return np.ceil(t.r_hat.max() * 1000) / 1000, np.floor(t.n_eff.min()), d[d > 0]


# velocity-ellipsoid fits (2-9 and 4-9 deg)
r_hat, n_eff, div = diagnostics('ellipsoid', 'ellipsoid_outer')
assert len(div) == 1, div   # the text names one fit with divergent transitions
# stage and population of the one fit with divergent transitions, and their number
(_, div_stage, div_pop), div_n = next(iter(div.items()))
# chains, draws, warm-up lengths, worst r_hat and n_eff; the divergent fit's stage, count and [Fe/H] bin
M.update(NChain=num(CHAINS), NDraw=count(DRAWS), NWarm=count(WARMUP_ELLIPSOID), NWarmOuter=count(WARMUP), RHat=f3(r_hat),
         NEff=count(n_eff), DivStage=num(div_stage), DivN=num(div_n), NTot=count(CHAINS * DRAWS),
         DivLo=num(FEH_BINS[[p for p in giants].index(div_pop)][0]), DivHi=num(FEH_BINS[[p for p in giants].index(div_pop)][1]))
# the sky-aligned (stream) fits
r_hat, n_eff, div = diagnostics('stream', 'stream_outer')
assert len(div) == 1 and div.index[0][2] == 'RR Lyrae', div   # the text names the RR Lyrae fit at 4-9 deg
M.update(RHatS=f3(r_hat), NEffS=count(n_eff), DivNS=num(div.iloc[0]))
```

#### The measured velocity field and residuals
Harmonic fits to the tangential velocity against position angle and the residuals about the models.

```python
# The measured velocity field and the residuals (results/kin_residual_*_ellipsoid.csv, written by paper_kin_residuals.py ellipsoid)
# harmonic fits of the residual velocity profiles against position angle, one row per population, model and component
h = pd.read_csv(R / 'kin_residual_profiles_ellipsoid.csv')
hg = h[h.population.str.startswith('giants')]


def harm(model, comp, col, who=hg):
    """Column col of the harmonic-fit rows of `model` and velocity component `comp` (default: giant bins only)."""
    return who[(who.model == model) & (who.component == comp)][col]


# Range of v_t over position angle: minimum and maximum over psi of each group's harmonic fit c0 + A1 cos(psi - psi1) +
# A2 cos 2(psi - psi2) to the stars at PROFILE_RING, extremes over the groups (giant bins and RR Lyrae)
psi = np.deg2rad(np.arange(360))
vt = [r['mean'] + r.A1 * np.cos(psi - np.deg2rad(r.phase1)) + r.A2 * np.cos(2 * (psi - np.deg2rad(r.phase2)))
      for _, r in h[(h.model == 'data') & (h.component == 'v_t')].iterrows()]
# slowest and fastest rotation [km/s] over position angle: v_t is negative, so slowest = -max(v_t), fastest = -min(v_t)
M.update(VtSlow=num(round(-max(v.max() for v in vt))), VtFast=num(round(-min(v.min() for v in vt))))
# ranges of the data's m = 2 and m = 1 amplitudes [km/s]
rng(pd.concat([harm('data', 'v_t', 'A2', h), harm('data', 'v_r', 'A2', h)]), lambda x: num(round(x)), 'DataATwo')
rng(harm('data', 'v_t', 'A1', h), lambda x: num(round(x)), 'DataAOne')
# m = 1 amplitudes of the vdMK14 model in each component
for c, k in (('v_t', 'Vt'), ('v_r', 'Vr'), ('v_los', 'Vl')):
    rng(harm('vdMK14', c, 'A1'), two, f'Vdm{k}')
# Level-1 model (L1): m = 1 and m = 2 amplitudes, and the m = 2 phase [deg] (giants, RR Lyrae)
rng(pd.concat([harm('L1', c, 'A1') for c in ('v_t', 'v_r', 'v_los')]), lambda x: num(round(x)), 'LOneAOne')
rng(harm('L1', 'v_t', 'A2'), two, 'LOneVt')
rng(harm('L1', 'v_r', 'A2'), two, 'LOneVr')
rng(harm('L1', 'v_t', 'phase2'), lambda x: num(round(x)), 'PhaseG')
M.update(PhaseRrl=num(round(h[(h.model == 'L1') & (h.component == 'v_t') & (h.population == 'RR Lyrae')].phase2.iloc[0])))
# Level-2 model (L2): m = 2 and m = 1 amplitudes [km/s]
rng(harm('L2', 'v_t', 'A2'), two, 'LTwoVt')
rng(harm('L2', 'v_r', 'A2'), two, 'LTwoVr')
rng(harm('L2', 'v_t', 'A1'), two, 'LTwoAOne')
# residual maps in sky cells: z-score spread (z_sd) and fraction with |z| > 3, for metal-poor (first three bins + RR Lyrae) and rich
cells = pd.read_csv(R / 'kin_residual_cells_ellipsoid.csv')
poor = cells[cells.population.isin(giants[:3] + ['RR Lyrae'])]
rich = cells[cells.population.isin(giants[3:])]
rng(poor.z_sd, f2, 'ZsdPoor')
rng(rich.z_sd, f1, 'ZsdRich')
rng(100 * rich.z_gt3, lambda x: num(round(x)), 'ZgtRich')
# ZgtNorm: per cent of a unit Gaussian beyond |z| = 3
M.update(ZgtPoor=num(round(100 * poor.z_gt3.max())), ZgtNorm=sigfig(100 * 2 * norm.sf(3)))
# radial profiles of the residuals (median per ring); rt: tangential component
rad = pd.read_csv(R / 'kin_residual_radial_ellipsoid.csv')
rt = rad[rad.component == 'v_t']
four = giants[:4]   # the four giant bins below [Fe/H] = -0.5
RINGS = dict(Fast=(3, 5), Agree=5, Out=6, Tanh=7)   # deg: ring boundaries of the statements on the radial residuals
M.update({f'Ring{k}': num(v) for k, v in RINGS.items() if k != 'Fast'}, RingFastLo=num(RINGS['Fast'][0]),
         RingFastHi=num(RINGS['Fast'][1]))
# tangential residuals of the four metal-poor bins between 3 and 5 deg
mid = rt[rt.population.isin(four) & (rt.rho_lo >= RINGS['Fast'][0]) & (rt.rho_hi <= RINGS['Fast'][1])]['median']
assert (mid < 0).all()   # faster rotation than the model in every ring and bin
rng(-mid, lambda x: num(round(x)), 'RadFast')
# largest tangential residual inside 3 deg; largest |residual| of any giant bin beyond 5 deg [km/s]
inner = rt[rt.population.isin(four) & (rt.rho_hi <= RINGS['Fast'][0])]['median']
M.update(RadSlow=num(round(inner.max())), RadOuter=f1(rt[rt.population.isin(giants) & (rt.rho_lo >= RINGS['Agree'])]['median'].abs().max()))
# radial component: inward motion inside OUTER_INNER (giants from the third bin up); largest residual beyond 6 deg (from the second)
rr = rad[rad.component == 'v_r']
rin = rr[rr.population.isin(giants[2:]) & (rr.rho_hi <= OUTER_INNER)]['median']
assert (rin < 0).all()   # inward motion in every ring and bin
rng(-rin, lambda x: num(round(x)), 'RadIn')
rout = rr[rr.population.isin(giants[1:]) & (rr.rho_lo >= RINGS['Out'])].groupby('population')['median'].max()
rng(rout, lambda x: num(round(x)), 'RadOut')
# RR Lyrae rings beyond OUTER_INNER with positive tangential residual
rrl = rt[(rt.population == 'RR Lyrae') & (rt.rho_lo >= OUTER_INNER) & (rt['median'] > 0)]['median']
rng(rrl, lambda x: num(round(x)), 'RadRrl')
M.update(NRadRrl=num(len(rrl)), NRingRrl=num(len(rt[(rt.population == 'RR Lyrae') & (rt.rho_lo >= OUTER_INNER)])))
```

#### Mocks and footprint
Fits to mocks drawn from symmetric discs, and the effect of the footprint.

```python
# Mocks drawn from symmetric discs (results/kin_mock_physical.csv, scripts/fig_kin_mock_physical.py)
# fits to mocks drawn from symmetric discs: with proper motions and line-of-sight velocities (pl), with distances (pdist)
mp = pd.read_csv(R / 'kin_mock_physical.csv')
pl = mp[mp.data.isin(['PM + LOS', 'PM + LOS stream'])]
pdist = mp[mp.data.isin(['PM + distances', 'PM + distances stream']) & mp.mock.str.contains('thick')]
pls = mp[mp.data == 'PM + LOS stream']
# largest centre offset [deg] and streaming amplitude in the mocks; sign of m2s; number of realisations
M.update(MockCen=f2(np.hypot(pl.dx, pl.dy).max()), MockCenDist=f2(np.hypot(pdist.dx, pdist.dy).max()),
         MockMTwo=f2(np.hypot(pls.m2c, pls.m2s).max()), MockMsLo=f3(pls.m2s.min()), MockMsHi=signed(pls.m2s.max()),
         MockNPos=num((pls.m2s > 0).sum()), MockNReal=num(len(pls)), NReal=num(mp[mp.mock == 'correlated PM'].realisation.nunique()))
# streaming amplitude |m2| of the data: tilted-ring 'stream' fits and stage-1 ellipsoid fits
tr = pd.read_csv(R / 'kin_tilted_ring.csv')
trg = tr[(tr.model == 'stream') & tr['sample'].str.startswith('giants')]
rng(np.hypot(trg.m2c, trg.m2s), f2, 'DataMTwo')
rng(np.hypot(g1.m2c, g1.m2s), f2, 'DataMTwoE')

# Footprint (results/kin_footprint.csv, scripts/fig_kin_footprint.py): symmetric disc, sky-aligned dispersions, PM + LOS
# footprint test: centre offset east dx [deg] with each bin's own footprint and the matched (common) footprint
fp = pd.read_csv(R / 'kin_footprint.csv')
fp = fp[fp.fit == 'PM + LOS, centre']
own = fp[fp.footprint == 'own'].set_index('sample').dx
mat = fp[fp.footprint == 'matched'].groupby('sample').dx.mean()
rng(own.loc[giants], f2, 'FpOwn')
rng(mat.loc[giants], f2, 'FpMat')
# shift of the two most metal-rich bins between footprints, mean error, and the shift in the mock
rng((mat - own).loc[giants[3:]], f2, 'FpShift')
M.update(FpErr=sigfig(fp[(fp.footprint == 'matched') & fp['sample'].isin(giants[3:])].dx_err.mean()),
         FpMock=f2(mat['mock'] - own['mock']))
```

#### Appendix: mocks of the velocity ellipsoid
Recovery of the input parameters from mock catalogues.

```python
# Appendix: recovery of the velocity ellipsoid from mocks (results/kin_ellipsoid_mock.csv, scripts/paper_kin_ellipsoid_mock.py)
# ellipsoid mocks: fitted against true dispersions with PM + LOS, PM + LOS with a tanh curve, and proper motions only
em = pd.read_csv(R / 'kin_ellipsoid_mock.csv')
C3 = ('sigma_R', 'sigma_phi', 'sigma_z')
both = em[em.data == 'PM + LOS'].set_index('mock')
tanh = em[em.data == 'PM + LOS, tanh'].set_index('mock')
pm = em[em.data == 'PM'].set_index('mock')
# fitted minus true dispersion per mock [km/s]
dev = {c: both[c] - both[f'truth_{c}'] for c in C3}
# mock settings (stars, line-of-sight error, thickness, scale length, spheroid flattening) and true dispersions; recovery
# errors and posterior errors of the PM + LOS fits; largest change with the tanh curve
M.update(NMock=count(N_STARS), EVlos=num(E_VLOS), HThick=num(H_THICK), RD=num(RD), QSph=num(mock['Q_SPH']), RmsSph=f1(mock['Q_SPH'] * np.sqrt(3) * RD),
         SigHotR=num(HOT[0]), SigHotPhi=num(HOT[1]), SigHotZ=num(HOT[2]), SigCold=num(COLD[0]),
         DSigRLo=f1(dev['sigma_R'].min()), DSigRHi=signed(dev['sigma_R'].max(), f1), DSigPhi=f1(dev['sigma_phi'].abs().max()),
         DSigZ=f1(dev['sigma_z'].abs().max()), DSigRZ=num(np.ceil(max(dev['sigma_R'].abs().max(), dev['sigma_z'].abs().max()))),
         ErrLo=f1(both[[c + '_err' for c in C3]].values.min()), ErrHi=f1(both[[c + '_err' for c in C3]].values.max()),
         DTanh=f1(np.ceil(10 * (tanh[list(C3)] - both[list(C3)]).abs().values.max()) / 10))
# proper motions only, disc mocks: in-plane errors, sigma_phi of the thick mock, and sigma_z
pmd = pm.loc[DISC_MOCKS]
inplane = pd.concat([pmd.sigma_R - pmd.truth_sigma_R, pmd.sigma_phi - pmd.truth_sigma_phi], axis=1)
thick_phi = pmd.sigma_phi['hot thick'] - pmd.truth_sigma_phi['hot thick']
M.update(PmInplane=f1(inplane.abs().values[inplane.abs().values < abs(thick_phi)].max()), PmThickPhi=f1(thick_phi),
         PmSzLo=num(round(pmd.sigma_z.min())), PmSzHi=num(round(pmd.sigma_z.max())), PmTzLo=num(pmd.truth_sigma_z.min()),
         PmTzHi=num(pmd.truth_sigma_z.max()), PmDzLo=num(round((pmd.sigma_z - pmd.truth_sigma_z).min())),
         PmDzHi=num(round((pmd.sigma_z - pmd.truth_sigma_z).max())), PmEzLo=f1(pmd.sigma_z_err.min()), PmEzHi=f1(pmd.sigma_z_err.max()))
# the hot-spheroid mock: dispersion errors and orientation with PM + LOS, and sigma_z, inc with proper motions only
sph, sphp = both.loc['hot spheroid'], pm.loc['hot spheroid']
M.update(SphDSig=f1(np.ceil(10 * max(abs(sph[c] - sph[f'truth_{c}']) for c in C3)) / 10), SphInc=f1(sph.inc),
         TruthInc=num(TRUTH['inc']), SphTheta=num(round(sph.theta)), TruthTheta=num(TRUTH['theta']),
         SphPmSz=num(round(sphp.sigma_z)), SphPmSzErr=num(round(sphp.sigma_z_err)), SphPmInc=num(round(sphp.inc)))
# model comparison: likelihood gain of the velocity ellipsoid at 2-9 deg (giants) and 4-9 deg (RR Lyrae), and its extra k
M.update(DlnLLo=num(round(mc_in.dlnL_max.loc[giants].min())), DlnLHi=count(round(mc_in.dlnL_max.loc[giants].max())),
         DlnLRrlOut=num(round(mc_out.dlnL_max['RR Lyrae'])), DkG=num(mc_in.dk.loc[giants].iloc[0]), DkRrl=num(mc_in.dk['RR Lyrae']))
# likelihood gain per population (letters A-F in the order of pops) for the inner and outer fits
for t, k in ((mc_in, 'In'), (mc_out, 'Out')):
    M.update({f'DlnL{k}{"ABCDEF"[i]}': count(round(v)) for i, v in enumerate(t.dlnL_max.loc[pops])})
M.update(NDraws=num(300))   # posterior draws per model in scripts/kin_model_comparison.py
```

#### Appendix: the sky-aligned dispersion model
Numbers of the appendix on the sky-aligned model.

```python
# Appendix: the sky-aligned dispersion model
# the hot mock fitted with the sky-aligned model: its sigma (proper motions) and sigma_los
hot = mp[(mp.mock == 'hot') & (mp.data == 'PM + LOS')].iloc[0]
# true inclination [rad] and dispersions (sigma_R, sigma_phi, sigma_z) [km/s] of the hot mock
i0 = np.deg2rad(TRUTH['inc'])
sR, sP, sZ = HOT
# AvgSig, AvgSigLos: proper-motion (per component) and line-of-sight dispersions of the hot ellipsoid at inclination i0,
# averaged over azimuth
M.update(HotSig=f1(hot.sigma), HotSigLos=f1(hot.sigma_los),
         AvgSig=f1(np.sqrt(0.5 * (1 - 0.5 * np.sin(i0) ** 2) * (sR ** 2 + sP ** 2) + 0.5 * sZ ** 2 * np.sin(i0) ** 2)),
         AvgSigLos=f1(np.sqrt(sZ ** 2 * np.cos(i0) ** 2 + 0.5 * np.sin(i0) ** 2 * (sR ** 2 + sP ** 2))))
# the sky-aligned (stream) joint fit at 2-9 deg: centre (SkyDy with the sign flipped), orientation, m2c, and differences
# from the velocity-ellipsoid joint fit
S = pd.read_csv(R / 'kin_mcmc_data_stream.csv')
ssh = S[(S.stage == 2) & (S.population == 'shared')].set_index('param')['median']
M.update(SkyDx=f3(ssh.dx), SkyDy=f3(-ssh.dy), SkyInc=f1(ssh.inc), SkyTheta=f1(ssh.theta), SkyMTwoC=f3(ssh.m2c),
         SkyDDx=f2(abs(ssh.dx - sh.loc['dx', 'median'])), SkyDTheta=f1(abs(ssh.theta - sh.loc['theta', 'median'])))
# v_rot / sigma_los of the first and last giant bins, and the largest APOGEE-only ratio (results/kin_support.csv)
ksup = pd.read_csv(R / 'kin_support.csv').set_index('population')
M.update(RatioLo=f2(ksup.ratio.loc[giants].iloc[0]), RatioHi=f2(ksup.ratio.loc[giants].iloc[-1]), RatioApo=f2(ksup.ratio_apogee.max()))
# tangential residuals of the sky-aligned model beyond 7 deg: metal-rich bins + RR Lyrae, and metal-poor bins
rs = pd.read_csv(R / 'kin_residual_radial.csv')
rs = rs[(rs.component == 'v_t') & (rs.rho_lo >= RINGS['Tanh'])]
rng(rs[rs.population.isin(giants[2:] + ['RR Lyrae'])]['median'], lambda x: num(round(x)), 'TanhRich')
rng(rs[rs.population.isin(giants[:2])]['median'], f1, 'TanhPoor')
# v0 [km/s] of each population in the 4-9 deg sky-aligned fit, and stars per giant bin at 4-9 deg
SO = pd.read_csv(R / 'kin_mcmc_data_stream_outer.csv')
M.update({f'VZeroOut{"ABCDEF"[i]}': f1(stage(SO, 1, p, 'v0')) for i, p in enumerate(pops)})
M.update({f'NOut{"ABCDE"[i]}': count(mc_out.n[p]) for i, p in enumerate(giants)})
```

#### Captions
The centres of the joint fits and the contour levels of the corner plot.

```python
# Centres of the joint fits [deg] (map captions)
# joint-fit centre (RA, Dec) [deg] of the ellipsoid (E) and sky-aligned (S) models
for t, k in ((sh['median'], 'E'), (ssh, 'S')):
    ra, dec = (float(np.rad2deg(v)) for v in kinfit.centre(t['dx'], t['dy']))
    M.update({f'Ra{k}': f2(ra), f'Dec{k}': f2(dec)})

# Corner-plot contour levels: mass inside 1 and 2 sigma of a 2D Gaussian (paper_kin_corner.py LEVELS)
M.update(LevOne=num(round(100 * (1 - np.exp(-0.5)))), LevTwo=num(round(100 * (1 - np.exp(-2)))))
```

#### Write the macros
One \newcommand per entry of M, with the prefix \km, into results/macros_kinmodel.tex; the page text shows them through
{glue:text}, so no number is typed by hand.

```python
# header line, then one \newcommand per macro
lines = ['% Numbers of the KB chapter Kinematics: velocity field and disc model (scripts/paper_text_kin_model.py); do not edit.']
lines += [rf'\newcommand{{\km{k}}}{{{x}}}' for k, x in M.items()]
(R / 'macros_kinmodel.tex').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
```

### Rotational support

The numbers of the page on rotational support, the centre and the sensitivity tests (`results/macros_kinsupport.tex`, prefix `\ks`).

<details><summary>About <code>scripts/paper_text_kin_support.py</code></summary>

```text
Knowledge-base text, page 'Rotational support, centre and sensitivity' of the kinematics chapter (extended draft Sections 4.4-4.8
and the further checks of its kinematics appendix): every number of the text as a LaTeX macro (prefix \\ks), so that none is typed by
hand.

As scripts/paper_text_shape.py: settings are read from the code (scripts/textnum.const, lmcold by import), results from the tables
the figures are drawn from; the stage-1 posterior draws (data/kin_mcmc_data_ellipsoid*_samples.npz, read only) give the quantities
the tables do not hold (v_rot(4 kpc) of the outer run, the RR Lyrae correlations, both stages of the joint fit); the release
samples give the Gaia proper-motion errors in km/s and, on the stars in both surveys, the [Fe/H] leakage and the BOSS velocity-error
test. No fit is made.
Inputs: results/{kin_support_ellipsoid, kin_support, kin_geometry_feh, kin_mcmc_data_ellipsoid, kin_mcmc_data_ellipsoid_outer,
kin_mcmc_data_stream, kin_mcmc_data_centre, density_centre, kin_feh_contamination, kin_feh_forward,
kin_tilted_ring, kin_los_scale, kin_turning, kin_ellipsoid_checks, kin_dispersion_cut, kin_data_split, kin_mock_physical, kin_footprint,
kin_residual_profiles_ellipsoid}.csv, data/giants_{apogee,boss}.parquet and data/rrab.parquet (lmcold.samples),
results/boss_calibration.json; scripts/{fit_kinematics_mcmc, paper_kin_support_ellipsoid, paper_kin_feh_mixing,
paper_kin_dispersion_cut, paper_kin_centre, fig_boss_velocity_errors}.py (settings); lmcold.constants (LMC distance), lmcold.kinfit (R_SIG, RHO_TILT, centre).
Writes results/macros_kinsupport.tex.
Run: .venv/bin/python scripts/manifest.py run paper_text_kin_support.py
```
</details>

**Data read by `scripts/paper_text_kin_support.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/giants_apogee.parquet` | Zenodo bundle 1 | APOGEE DR17 giants with Gaia DR3 |
| `data/giants_boss.parquet` | Zenodo bundle 1 | SDSS-V DR20 BOSS giants with Gaia DR3, on the APOGEE scale |
| `data/kin_mcmc_data_ellipsoid_outer_samples.npz` | Zenodo bundle 1 | Posterior samples of the ellipsoid kinematic fit, RR Lyrae 4-9 deg (NUTS) |
| `data/kin_mcmc_data_ellipsoid_samples.npz` | Zenodo bundle 1 | Posterior samples of the ellipsoid kinematic fit, giants 2-9 deg (NUTS) |
| `data/rrab.parquet` | Zenodo bundle 1 | RRab within 20 deg: OGLE-IV + Gaia DR3, [Fe/H], dmu, membership |
| `results/boss_calibration.json` | repository | calibration of the BOSS [Fe/H] and velocities to APOGEE on the stars observed by both; written by `python scripts/build_giants.py` |
| `results/density_centre.csv` | repository | density centres of the giant samples; written by `python scripts/fig_density_centre.py` |
| `results/kin_data_split.csv` | repository | fixed input |
| `results/kin_dispersion_cut.csv` | repository | written by `python scripts/paper_kin_dispersion_cut.py` |
| `results/kin_ellipsoid_checks.csv` | repository | fixed input |
| `results/kin_feh_contamination.csv` | repository | written by `python scripts/paper_kin_feh_mixing.py` |
| `results/kin_feh_forward.csv` | repository | written by `python scripts/paper_kin_feh_mixing.py` |
| `results/kin_footprint.csv` | repository | fixed input |
| `results/kin_geometry_feh.csv` | repository | written by `python scripts/paper_kin_geometry_feh.py` |
| `results/kin_los_scale.csv` | repository | fits with a free scale of the line-of-sight velocities; written by `python scripts/fig_kin_los_scale.py` |
| `results/kin_mcmc_data_centre.csv` | repository | posterior summary of the free-centre model; MCMC; written by `python scripts/fit_kinematics_mcmc.py data centre` |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_mcmc_data_ellipsoid_outer.csv` | repository | the same fit at 4-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid outer` |
| `results/kin_mcmc_data_stream.csv` | repository | posterior summary of the sky-aligned dispersion model with m = 2 streaming, 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data stream` |
| `results/kin_mock_physical.csv` | repository | fixed input |
| `results/kin_residual_profiles_ellipsoid.csv` | repository | written by `python scripts/paper_kin_residuals.py ellipsoid` |
| `results/kin_support.csv` | repository | written by `python scripts/paper_kin_support.py` |
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |
| `results/kin_tilted_ring.csv` | repository | tilted-ring fits per radius; written by `python scripts/fig_kin_tilted_ring.py` |
| `results/kin_turning.csv` | repository | fits of the turning-disc model; written by `python scripts/fig_kin_turning.py` |

*Shown, not run here: the result tables in results/, the posterior samples and the catalogues of Zenodo bundle 1. The cells below are the script itself; run it from the repository with* `python scripts/paper_text_kin_support.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
Imports, the dictionary M of macros (macro name without its prefix: the text it stands for) and the functions that format
numbers as the text writes them. The inputs listed in the header are read in the step that uses them.

```python
import ast
import json

import numpy as np
import pandas as pd

from fit_kinematics_mcmc import ring
from lmcold import kinfit, samples
from lmcold.constants import KAPPA, LMC
from lmcold.footprint import lmc_xy
from lmcold.samples import FEH_BINS
from lmcold.sources import DATA, ROOT, path
from textnum import const, num, sigfig

R = ROOT / 'results'   # folder of the result tables
M = {}   # macro name (without the \ks prefix): text it stands for
BINS = 'ABCDE'   # giant [Fe/H] bins, metal-poor first (macro suffixes)
SIGZ_BAND = (20, 35)   # km/s: sigma_z range of the giants, for the RR Lyrae sigma_phi quoted where their sigma_z is giant-like


def f1(x):
    """x with one decimal."""
    return f'{x:.1f}'


def f2(x):
    """x with two decimals."""
    return f'{x:.2f}'


def pm(med, lo, hi, fmt=f1):
    """'med ± e' if the two half-widths agree when rounded, else 'med (+u/−l)'."""
    # Upper and lower half-widths, formatted as the value is.
    u, d = fmt(hi - med), fmt(med - lo)
    return f'{fmt(med)} ± {u}' if u == d else f'{fmt(med)} (+{u}/−{d})'


def rng(x, fmt=f1):
    """(min, max) of x as formatted strings."""
    return fmt(min(x)), fmt(max(x))


def put(name, pair):
    """Store a (low, high) pair as the macros <name>Lo and <name>Hi."""
    M[name + 'Lo'], M[name + 'Hi'] = pair


def pct_per_kpc(gamma):
    """Per cent decline of the dispersions per kpc for the logarithmic gradient gamma [1/kpc]."""
    return 100 * (1 - np.exp(gamma))
```

#### Settings
The radial ranges, [Fe/H] bin edges and other settings, read from the fit and figure scripts.

```python
# Settings
# Literal constants of the fit and figure scripts, read from their source (textnum.const) without running them.
fk = const('fit_kinematics_mcmc')
mix, cut, cen = const('paper_kin_feh_mixing'), const('paper_kin_dispersion_cut'), const('paper_kin_centre')
# Radial ranges [deg], R_SIG [kpc], the hot-giant Teff limit [K], the HEALPix cell size [deg] (sqrt of 4 pi / (12 NSIDE^2) sr),
# and the inner edges and mock values of scripts/paper_kin_dispersion_cut.py.
M.update(RhoIn=num(fk['RING'][0]), RhoOut=num(fk['RING'][1]), RhoOuter=num(fk['OUTER_INNER']), RSig=num(kinfit.R_SIG),
         THot=num(mix['T_HOT']), NsideDeg=f1(np.rad2deg(np.sqrt(4 * np.pi / (12 * mix['NSIDE'] ** 2)))),
         MinApogee=num(mix['MIN_APOGEE']), RhoTilt=num(kinfit.RHO_TILT), InnerLo=num(min(cut['INNER'])),
         InnerHi=num(max(cut['INNER'])), MockSigma=num(cut['MOCK_SIGMA']), MockKIn=f2(cut['MOCK_K'][0]),
         MockKOut=f2(cut['MOCK_K'][1]), SigzBandLo=num(SIGZ_BAND[0]), SigzBandHi=num(SIGZ_BAND[1]))
M.update({f'Feh{BINS[i]}': num(FEH_BINS[i][1]) for i in range(4)})   # bin edges -1.5, -1.1, -0.8, -0.5
M.update(BinWidth=f1(max(hi - lo for lo, hi in FEH_BINS[2:4])), InnerTwo=num(cut['INNER'][1]))   # closed bins above -1.1
M['ApogeeFrom'] = num(const('paper_kin_support_ellipsoid')['APOGEE_FROM'])
```

#### Rotational support
v_rot/sigma of each group from the stage-1 fits at 2-9 deg and 4-9 deg, the joint fit, and the RR Lyrae.

```python
# Rotational support (results/kin_support_ellipsoid.csv: stage-1 posteriors of the velocity-ellipsoid model)
# One row per group: g the five giant bins (metal-poor first), rr the RR Lyrae; v_rot(4 kpc) and dispersions [km/s], ratios.
S = pd.read_csv(R / 'kin_support_ellipsoid.csv').set_index('population')
g, rr = S.iloc[:5], S.loc['RR Lyrae']
# Values of selected bins (indices into BINS) quoted in the text: column q, macro stem n, format fmt.
for q, n, fmt, bins in (('vrot4', 'Vrot', f1, (0, 4)), ('sigma_R', 'SigR', f1, (0, 2, 3, 4)), ('sigma_phi', 'SigPhi', f1, (0, 4)),
                        ('sigma_z', 'Sigz', f1, (0, 2, 3, 4)), ('ratio_R', 'RatR', f2, (0, 4)), ('ratio_z', 'Ratz', f2, (0, 4))):
    M.update({f'{n}{BINS[i]}': fmt(g[q].iloc[i]) for i in bins})
SIG = ['sigma_R', 'sigma_phi', 'sigma_z']
# Differences between bins [km/s]: A-B (rotation and largest dispersion difference), C-E and A-E (rounded to integers).
M.update(DVrotAB=f1(g.vrot4.iloc[1] - g.vrot4.iloc[0]), DSigAB=f1((g[SIG].iloc[0] - g[SIG].iloc[1]).abs().max()),
         DVrotCE=num(round(g.vrot4.iloc[4] - g.vrot4.iloc[2])), DSigRCE=num(round(g.sigma_R.iloc[2] - g.sigma_R.iloc[4])),
         DSigzCE=num(round(g.sigma_z.iloc[2] - g.sigma_z.iloc[4])),
         DVrotAE=num(round(g.vrot4.iloc[4] - g.vrot4.iloc[0])), DSigRAE=num(round(g.sigma_R.iloc[0] - g.sigma_R.iloc[4])),
         DSigzAE=num(round(g.sigma_z.iloc[0] - g.sigma_z.iloc[4])))
# Range over the bins of how much larger v_rot/sigma_z is than v_rot/sigma_R [per cent].
put('SuppEx', rng(100 * (g.ratio_z / g.ratio_R - 1), lambda x: num(round(x))))
# Geometry and support ratios per giant bin: g1 from stage 1 (own fits), g2 from stage 2 (joint fit), with p16 and p84.
geo = pd.read_csv(R / 'kin_geometry_feh.csv')
g1, g2 = (geo[(geo.stage == s) & (geo.population != 'RR Lyrae')].reset_index(drop=True) for s in (1, 2))
put('RatPhi', rng(g1.ratio_phi, f2))
put('RatZ', rng(g1.ratio_z, f2))
half = lambda t, q: (t[f'{q}_p84'] - t[f'{q}_p16']) / 2  # noqa: E731  half-width of the 16th-84th percentile range
# Bins D and E: v_rot/sigma_z ± half the 16th-84th range; bin E from the APOGEE giants alone: sigma_z / sigma_R.
M.update(RatZD=f'{f2(g1.ratio_z[3])} ± {f2(half(g1, "ratio_z")[3])}', RatZE=f'{f2(g1.ratio_z[4])} ± {f2(half(g1, "ratio_z")[4])}',
         RatZApoE=f2(S.sigma_z_apogee.iloc[4] / S.sigma_R_apogee.iloc[4]))
# Posterior summaries of the ellipsoid model at 2-9 deg (E) and 4-9 deg (O); e1, o1: stage-1 medians, one row per group.
E = pd.read_csv(R / 'kin_mcmc_data_ellipsoid.csv')
O = pd.read_csv(R / 'kin_mcmc_data_ellipsoid_outer.csv')
e1 = E[E.stage == 1].pivot_table(index='population', columns='param', values='median').loc[g.index]
o1 = O[O.stage == 1].pivot_table(index='population', columns='param', values='median')
# Decline of the dispersions [per cent per kpc]: the two metal-poor bins, then the three metal-rich ones.
put('GamPctMP', rng(pct_per_kpc(e1.dlnsig_dR.iloc[:2])))
put('GamPctMR', rng(pct_per_kpc(e1.dlnsig_dR.iloc[2:])))
# Outer slopes vslope [1/kpc] of the giant bins (2-9 deg) and the RR Lyrae (4-9 deg); Lo is the largest, Hi the most negative.
slopes = list(e1.vslope) + [o1.vslope['RR Lyrae']]
M.update(SlopeLo=f2(max(slopes)), SlopeHi=f2(min(slopes)))
Rg = np.linspace(0.5, 9, 1701)   # kpc: peak of the stage-1 median rotation curve v0 tanh(R/R0) [1 + vslope (R - R_SIG)]
peaks = [Rg[np.argmax(r.v0 * np.tanh(Rg / r.R0) * (1 + r.vslope * (Rg - kinfit.R_SIG)))] for _, r in e1.iterrows()]
put('Peak', rng(peaks))
# Same fits at 4 < rho < 9 deg (run 'ellipsoid_outer'), from the paired posterior draws


def draws(run, i, stage=1):
    """Posterior draws (flattened) of group i of run `run` (stage 1: its own fit; stage 2: the joint fit): v0, R0 [kpc], the three
    dispersions [km/s], k, and v_rot(4 kpc) = v0 tanh(R_SIG / R0) [km/s]."""
    z = np.load(DATA / f'kin_mcmc_data_{run}_samples.npz')
    # Keys of the npz: '1|<group>|<param>_0' in stage 1, '2|all|<param>_<group>' in stage 2.
    k = (lambda n: f'1|{i}|{n}_0') if stage == 1 else (lambda n: f'2|all|{n}_{i}')
    d = {n: z[k(n)].reshape(-1) for n in ('v0', 'R0', 'sigma_R', 'sigma_phi', 'sigma_z', 'k')}
    d['vrot4'] = d['v0'] * np.tanh(kinfit.R_SIG / d['R0'])
    return d


# The five giant bins at 4-9 deg: median v_rot(4 kpc) [km/s] and the changes from 2-9 deg of rotation, dispersions and both ratios.
out = [draws('ellipsoid_outer', i) for i in range(5)]
vout = np.array([np.median(d['vrot4']) for d in out])
dvo = vout - g.vrot4.to_numpy()
dsig = {s: o1.loc[g.index, s].to_numpy() - g[s].to_numpy() for s in SIG}
drat = [max(abs(np.median(d['vrot4'] / d['sigma_R']) - g.ratio_R.iloc[i]), abs(np.median(d['vrot4'] / d['sigma_z']) - g.ratio_z.iloc[i]))
        for i, d in enumerate(out)]
put('DVrotOut', rng(dvo))
put('DSigPhiOut', rng(-dsig['sigma_phi']))
# Largest change of sigma_R or sigma_z [km/s], of the ratios, and v_rot(4 kpc) at 4-9 deg of bins A and B.
M.update(DSigRzOut=f1(max(np.abs(dsig['sigma_R']).max(), np.abs(dsig['sigma_z']).max())), DRatOut=f1(max(drat)),
         VrotOutA=f1(vout[0]), VrotOutB=f1(vout[1]))
# Stage 1 against stage 2 (joint fit) of the giants: v_rot(4 kpc) and dispersions
st2 = [draws('ellipsoid', i, 2) for i in range(5)]
# Largest difference [km/s] between the joint-fit median and the stage-1 value of v_rot(4 kpc) or a dispersion.
M['DStage'] = f1(max(max(abs(np.median(st2[i][q]) - g[q].iloc[i]) for q in ['vrot4'] + SIG) for i in range(5)))
# RR Lyrae
# RR Lyrae stage-1 summaries at 2-9 deg (rin) and 4-9 deg (rou), indexed by parameter.
rin = E[(E.stage == 1) & (E.population == 'RR Lyrae')].set_index('param')
rou = O[(O.stage == 1) & (O.population == 'RR Lyrae')].set_index('param')
# RR Lyrae: Gaia error scale k and dispersion gradient [1/kpc] in both runs; v_rot(4 kpc), dispersions and ratio with their errors;
# [Fe/H] median and spread [dex]; DVrotRrlOut: largest difference from bins A and B at 4-9 deg [km/s], rounded up.
M.update(KRrlIn=f2(rin.loc['k', 'median']), GamRrlIn=f2(rin.loc['dlnsig_dR', 'median']),
         GamPctRrlIn=num(round(pct_per_kpc(rin.loc['dlnsig_dR', 'median']))), KRrlOut=f2(rou.loc['k', 'median']),
         GamRrlOut=f2(rou.loc['dlnsig_dR', 'median']),
         VrotRrl=pm(rr.vrot4, rr.vrot4_p16, rr.vrot4_p84), SigRRrl=pm(rr.sigma_R, rr.sigma_R_p16, rr.sigma_R_p84),
         SigPhiRrl=pm(rr.sigma_phi, rr.sigma_phi_p16, rr.sigma_phi_p84), RatRRrl=pm(rr.ratio_R, rr.ratio_R_p16, rr.ratio_R_p84, f2),
         FehRrl=f2(rr.feh_med), FehRrlLo=f2(rr.feh_p16), FehRrlHi=f2(rr.feh_p84),
         DVrotRrlOut=num(np.ceil(np.abs(rr.vrot4 - vout[:2]).max())))
# Ranges over the two metal-poor bins.
put('SigRMP', rng(g.sigma_R.iloc[:2]))
put('RatRMP', rng(g.ratio_R.iloc[:2], f2))
put('SigPhiMP', rng(g.sigma_phi.iloc[:2]))
# RR Lyrae draws at 4-9 deg (group 5): correlation of sigma_phi with k, and median sigma_phi of the draws with sigma_z in SIGZ_BAND.
d = draws('ellipsoid_outer', 5)
band = (d['sigma_z'] > SIGZ_BAND[0]) & (d['sigma_z'] < SIGZ_BAND[1])
M.update(CorrRrl=sigfig(np.corrcoef(d['sigma_phi'], d['k'])[0, 1]), SigPhiRrlBand=num(round(np.median(d['sigma_phi'][band]))))
```

#### Rotation-curve parameters
Shape parameters of the rotation curves quoted in the table caption.

```python
# Table caption: shape parameters of the rotation curves
# Turnover radius R0 [kpc] range of the giants, outer slope [1/kpc]; RR Lyrae R0 and slope from the 4-9 deg run.
put('RNullG', rng(e1.R0))
M.update(SlopeGLo=f2(e1.vslope.max()), SlopeGHi=f2(e1.vslope.min()),
         RNullRrl=pm(rou.loc['R0', 'median'], rou.loc['R0', 'p16'], rou.loc['R0', 'p84']),
         SlopeRrl=pm(rou.loc['vslope', 'median'], rou.loc['vslope', 'p16'], rou.loc['vslope', 'p84'], f2))
```

#### Metallicity errors and the two surveys
Leakage of stars between [Fe/H] bins and the BOSS velocity errors, on the stars in both surveys.

```python
# Metallicity errors and the two surveys (results/kin_feh_*.csv; boss_calibration.json)
# cal: BOSS [Fe/H] calibration (held-out scatter); ct: sigma_los per bin and subset; fw: forward-modelled sigma_los; labels: bin
# names.
cal = json.loads((R / 'boss_calibration.json').read_text())
ct = pd.read_csv(R / 'kin_feh_contamination.csv')
fw = pd.read_csv(R / 'kin_feh_forward.csv')
labels = list(ct.bin.unique())
# Stars in both release catalogues (as scripts/paper_kin_feh_mixing.py and fig_boss_velocity_errors.py): of the cool BOSS giants
# measured above FEH_BINS[4][0], the fraction whose APOGEE [Fe/H] lies in the neighbouring bin FEH_BINS[3]
m = pd.read_parquet(path('release_giants_apogee')).merge(pd.read_parquet(path('release_giants_boss')), on='source_id', suffixes=('_a', '_b'))
# _a: APOGEE columns, _b: BOSS columns of the stars in both catalogues; top: cool BOSS giants measured in the most metal-rich bin.
top = m[(m.teff_b < mix['T_HOT']) & (m.feh_b >= FEH_BINS[4][0])]
M.update(BossScatter=sigfig(cal['held_out_test']['dfeh_rsd']),
         LeakE=num(round(100 * np.mean((top.feh_a >= FEH_BINS[3][0]) & (top.feh_a < FEH_BINS[3][1])))),
         NApoA=num(ct[(ct.bin == labels[0]) & (ct.subset == 'APOGEE')].n.iloc[0]))
# Bins C, D, E: APOGEE-alone dispersions [km/s] and ratios, and the predicted and measured sigma_los [km/s] of the cool BOSS giants.
for i in (2, 3, 4):
    b = BINS[i]
    M.update({f'SigzApo{b}': f1(S.sigma_z_apogee.iloc[i]), f'SigRApo{b}': f1(S.sigma_R_apogee.iloc[i]),
              f'RatRApo{b}': f2(S.ratio_R_apogee.iloc[i]), f'RatzApo{b}': f2(S.ratio_z_apogee.iloc[i]),
              f'Pred{b}': f1(fw[(fw.bin == labels[i]) & (fw.cls == 'cool')].sigma_los_pred.iloc[0]),
              f'Meas{b}': f1(ct[(ct.bin == labels[i]) & (ct.subset == 'BOSS cool')].sigma_los.iloc[0])})
# Measured minus predicted in bin C [km/s], rounded; bin E: sigma_los of the hot and of the cool BOSS giants.
M['ShortC'] = num(round(float(M['MeasC']) - float(M['PredC'])))
M.update(HotE=f1(ct[(ct.bin == labels[4]) & (ct.subset == 'BOSS hot')].sigma_los.iloc[0]), CoolE=M['MeasE'])
# BOSS velocity errors on the same stars: robust standard deviation (1.4826 MAD) of z = (v_BOSS - v_APOGEE) / sqrt(e_BOSS^2 + e_APOGEE^2)
# in the bins of S/N, G and Teff of scripts/fig_boss_velocity_errors.py (its BINS, read from the source; bins with >= 20 stars)
# vbins: {column: bin edges} from the BINS = dict(...) assignment of that script.
tree = ast.parse((ROOT / 'scripts/fig_boss_velocity_errors.py').read_text())
vbins = next({k.arg: ast.literal_eval(k.value) for k in n.value.keywords} for n in tree.body
             if isinstance(n, ast.Assign) and getattr(n.targets[0], 'id', '') == 'BINS')
z = (m.vlos_b - m.vlos_a) / np.hypot(m.evlos_a, m.evlos_b)
# Robust standard deviation of z in every bin of every column; the range is quoted; NBoth: stars in both, with a LaTeX thin space.
rsd = [1.4826 * np.median(np.abs(x - np.median(x))) for col, e in vbins.items() for lo, hi in zip(e[:-1], e[1:])
       for x in [z[(m[col] >= lo) & (m[col] < hi)]] if len(x) >= 20]
put('Zsd', rng(rsd, f2))
M['NBoth'] = f'{len(m):,}'.replace(',', '\\,')
```

#### Centre and geometry
The fitted centre, its shift with metallicity and with the sky coverage.

```python
# Centre and geometry (results/kin_mcmc_data_*.csv stage 2 'shared'; density_centre.csv; kin_data_split.csv; kin_tilted_ring.csv)
shared = lambda t: t[(t.stage == 2) & (t.population == 'shared')].set_index('param')  # noqa: E731
# Shared (stage-2) parameters: j the ellipsoid model, j1 the Level-1 'centre' run, js the sky-aligned 'stream' run.
j, j1, js = shared(E), shared(pd.read_csv(R / 'kin_mcmc_data_centre.csv')), shared(pd.read_csv(R / 'kin_mcmc_data_stream.csv'))
# Fitted centre [deg]; offsets dx (east) and dy (north) [deg] on the tangent plane (dy negated where the text says 'south').
ra0, dec0 = (float(np.rad2deg(a)) for a in kinfit.centre(j.loc['dx', 'median'], j.loc['dy', 'median']))
M.update(CenDx=f2(j.loc['dx', 'median']), CenRa=f2(ra0), CenDec=f2(dec0), LoneDx=f2(j1.loc['dx', 'median']),
         LoneDy=f2(-j1.loc['dy', 'median']), SkyDDx=f2(js.loc['dx', 'median'] - j.loc['dx', 'median']),
         SkyDx=f2(js.loc['dx', 'median']), SkyDy=f2(-js.loc['dy', 'median']), SkyTheta=f1(js.loc['theta', 'median']),
         SkyInc=f1(js.loc['inc', 'median']), SkyMtwo=f2(js.loc['m2c', 'median']),
         MoveL=f2(j1.loc['dx', 'median'] - j.loc['dx', 'median']))   # east offset between the Level-1 and Level-2 joint centres
# ... and with the sky coverage: east shift of the two metal-rich bins resampled onto the positions of the most metal-poor bin
# (results/kin_footprint.csv, scripts/fig_kin_footprint.py; symmetric disc, PM + LOS)
fp = pd.read_csv(R / 'kin_footprint.csv')
fp = fp[fp.fit == 'PM + LOS, centre']
M['MoveSky'] = f2((fp[fp.footprint == 'matched'].groupby('sample').dx.mean() - fp[fp.footprint == 'own'].set_index('sample').dx)
                  .loc[list(g.index[3:])].max())
# Each group's own stage-1 centre: east offsets range, largest southward offset [deg]; RR Lyrae offsets at 4-9 deg and 2-9 deg.
put('OwnDx', rng(e1.dx, f2))
M['OwnDySouth'] = f2(-e1.dy.min())
M.update(RrlOutDx=f'{f2(rou.loc["dx", "median"])} ± {f2((rou.loc["dx", "p84"] - rou.loc["dx", "p16"]) / 2)}',
         RrlInDx=f2(rin.loc['dx', 'median']))
# Density centres (dx, dy [deg]) of the RR Lyrae and the red giants in several apertures [deg].
dens = pd.read_csv(R / 'density_centre.csv')
GIANTS = next(p for p in dens.population.unique() if p.startswith('red giants'))   # 'red giants, Gaia G<...'
dens = dens[dens.population.isin(['RR Lyrae', GIANTS])]
M['GLim'] = GIANTS.split('<')[1]
put('DensDx', rng(dens.dx, f2))
M['Apertures'] = ', '.join(num(a) for a in sorted(dens.aperture.unique())[:-1]) + ' and ' + num(dens.aperture.max())
# Distance of the density centres from the joint kinematic centre [deg], and in kpc at D0; then from each group's own centre.
dist = np.hypot(dens.dx - j.loc['dx', 'median'], dens.dy - j.loc['dy', 'median'])
put('DistDens', rng(dist, f2))
put('DistDensKpc', rng(np.deg2rad(dist) * LMC['D0'], f2))
own_dist = [np.hypot(dens.dx - r.dx, dens.dy - r.dy) for _, r in e1.iterrows()]
put('OwnDistDens', rng(np.concatenate(own_dist)))
M.update(OwnDxRoundLo=f1(e1.dx.min()), OwnDxRoundHi=f1(e1.dx.max()))
pub = {k: v for k, v in cen['PUBLISHED'].items() if 'PM' in k}   # published proper-motion centres (Gaia, Gaia/VMC)
# Published centres projected on the tangent plane, x (east), y (north) [deg].
xy = np.array([lmc_xy(ra, dec) for v in pub.values() for ra, dec in v[0]])
put('PubDx', rng(xy[:, 0]))
put('PubDy', rng(xy[:, 1]))
# Largest centre offset [deg] of the PM + LOS fits of the symmetric mocks, rounded up to 0.1 deg.
mp = pd.read_csv(R / 'kin_mock_physical.csv')
mp = mp[mp.data.isin(['PM + LOS', 'PM + LOS stream'])]
M['MockOffMax'] = f1(np.ceil(10 * np.hypot(mp.dx, mp.dy).max()) / 10)   # symmetric (centred) mocks
# Joint inclination and line of nodes [deg] with half the 16th-84th range (stage-2 row of bin A; the geometry is shared in the joint
# fit).
M.update(IncJoint=f1(j.loc['inc', 'median']), ThetaJoint=f1(j.loc['theta', 'median']),
         IncJointErr=f1(half(g2.iloc[[0]], 'inc').iloc[0]), ThetaJointErr=f1(half(g2.iloc[[0]], 'theta').iloc[0]))
# Inclination [deg] of the giant fits on proper motions alone and with line-of-sight velocities.
sp = pd.read_csv(R / 'kin_data_split.csv')
sp = sp[sp['sample'].str.startswith('giants')]
put('IncPM', rng(sp[sp.data == 'PM only'].inc, lambda x: num(round(x))))
put('IncBoth', rng(sp[sp.data == 'PM + LOS'].inc, lambda x: num(round(x))))
tr = pd.read_csv(R / 'kin_tilted_ring.csv')
# Inclination and line of nodes [deg] from the RR Lyrae proper motions and distances.
dd = tr[(tr['sample'] == 'RR Lyrae PM + distances') & (tr.model == 'stream')].iloc[0]
M.update(IncDist=num(round(dd.inc)), ThetaDist=num(round(dd.theta)))
# Each bin's own line of nodes and inclination [deg]; NSig: shift of bins A, B from stage 1 to stage 2 in units of their errors.
put('ThetaOwn', rng(g1.theta))
M.update(ThetaOwnIntLo=num(int(np.floor(g1.theta.min()))), ThetaOwnIntHi=num(int(np.ceil(g1.theta.max()))))
M.update(IncA=f'{f1(g1.inc[0])} ± {f1(half(g1, "inc")[0])}', IncB=f'{f1(g1.inc[1])} ± {f1(half(g1, "inc")[1])}',
         IncAInt=f1(g1.inc[0]), IncBInt=f1(g1.inc[1]))
put('IncRich', rng(g1.inc[2:]))
nsig = (g2.inc[:2] - g1.inc[:2]) / half(g1, 'inc')[:2]
put('NSig', rng(nsig))
# RR Lyrae geometry and v_rot/sigma_phi in stage 1 (r1) and in the joint fit (r2); largest stage-1 - stage-2 change of the ratios.
r1 = geo[(geo.stage == 1) & (geo.population == 'RR Lyrae')].iloc[0]
r2 = geo[(geo.stage == 2) & (geo.population == 'RR Lyrae')].iloc[0]
M.update(IncRrl=pm(r1.inc, r1.inc_p16, r1.inc_p84), ThetaRrl=pm(r1.theta, r1.theta_p16, r1.theta_p84, lambda x: num(round(x))),
         RatPhiRrl=pm(r1.ratio_phi, r1.ratio_phi_p16, r1.ratio_phi_p84, f2),
         RatPhiRrlJ=pm(r2.ratio_phi, r2.ratio_phi_p16, r2.ratio_phi_p84, f2),
         DRatStage=f'{max((g1.ratio_phi - g2.ratio_phi).abs().max(), (g1.ratio_z - g2.ratio_z).abs().max()):.3f}')
put('RatErr', rng(pd.concat([half(g1, 'ratio_phi'), half(g1, 'ratio_z')]), lambda x: f'{x:.3f}'))
```

#### Sensitivity to the model
Tilted rings, a free scale of the line-of-sight velocities, a turning disc and the checks of the velocity-ellipsoid model.

```python
# Sensitivity to the model, sky-aligned dispersions (maximum likelihood; results/kin_tilted_ring, kin_los_scale, kin_turning.csv)
# lnl: -ln L of each giant bin for model m of table t; col: column c of the same rows.
pops = list(g.index)
lnl = lambda t, m: t[t.model == m].set_index('sample').loc[pops].nll.to_numpy()  # noqa: E731  -ln L of each giant bin
col = lambda t, m, c: t[t.model == m].set_index('sample').loc[pops][c].to_numpy()  # noqa: E731
ls = pd.read_csv(R / 'kin_los_scale.csv')
ls = ls[ls.velocities == 'all']
tu = pd.read_csv(R / 'kin_turning.csv')
# Gains in ln L over the 'stream' model of a tilted disc and of a free line-of-sight scale qlos.
ref = lnl(tr, 'stream')
dl_tilt, dl_q = ref - lnl(tr, 'tilt + stream'), ref - lnl(ls, 'qlos + stream')
put('DIncTilt', rng(-col(tr, 'tilt + stream', 'dinc'), lambda x: num(round(x))))
# qlos of each bin; text quotes bins D, E and the largest of the others, with their ln L gains.
q = col(ls, 'qlos + stream', 'qlos')
M.update(DlnLTiltD=num(round(dl_tilt[3])), DlnLTiltE=num(round(dl_tilt[4])), DlnLTiltOther=num(round(dl_tilt[:3].max())),
         DVTilt=f1(np.abs(col(tr, 'tilt + stream', 'v0') - col(tr, 'stream', 'v0')).max()),
         QlosD=f2(q[3]), QlosE=f2(q[4]), DlnLQD=num(round(dl_q[3])), DlnLQE=num(round(dl_q[4])),
         QlosOther=f1(np.abs(q[:3]).max()), DlnLQOther=num(round(dl_q[:3].max())))
put('IncQ', rng(col(ls, 'qlos + stream', 'inc')[3:], lambda x: num(round(x))))
# Turning disc: ln L gains (negative: worse) and the turning rates dthdt; the rotation period converts them to degrees per turn.
dl_turn, dl_turns = ref - lnl(tu, 'turn'), ref - lnl(tu, 'turn + stream')
dth = col(tu, 'turn + stream', 'dthdt')
KPC_KMS_GYR = 977.79   # kpc / (km/s) in Myr x 1e3: 1 kpc / (1 km/s) = 977.79 Myr
period = round(2 * np.pi * kinfit.R_SIG / g.vrot4.max() * KPC_KMS_GYR / 1e3, 1)   # Gyr, rotation period at R_SIG of the fastest giants, as quoted
M.update(TurnWorseLo=num(round(dl_turn.max())), TurnWorseHi=num(round(dl_turn.min())), TurnBetterLo=num(round(dl_turns.min())),
         TurnBetterHi=num(round(dl_turns.max())), DthdtLo=num(round(dth.min())), DthdtHi=num(round(dth.max())), Period=f1(period),
         TurnLo=num(10 * round(dth.min() * period / 10)), TurnHi=num(10 * round(dth.max() * period / 10)))
# sigma_los and v0 [km/s] of every bin in every model, shape (6 models, 5 bins); their ratio v0/sigma_los.
models = [(tr, 'sym'), (tr, 'stream'), (tr, 'tilt + stream'), (ls, 'qlos + stream'), (tu, 'turn'), (tu, 'turn + stream')]
slos = np.array([col(t, m, 'sigma_los') for t, m in models])
v0s = np.array([col(t, m, 'v0') for t, m in models])
ratios = v0s / slos
M.update(DSigLos=f1(np.ptp(slos, axis=0).max()), DVAlt=num(round(np.abs(v0s[2:4] - v0s[1]).max())))
put('DVTurn', rng(v0s[5] - v0s[1], lambda x: num(round(x))))
# Whether v0/sigma_los rises from bin A to E in each model; the asserts stop the script if the text's statement no longer holds.
mono = [bool(np.all(np.diff(r) > 0)) for r in ratios]
assert mono == [True, True, True, True, False, True], mono   # only the turning disc alone breaks the ordering
assert np.all(np.diff(ratios[4][1:]) > 0) and ratios[4][0] > ratios[4][1]   # ... by swapping the two most metal-poor bins
M.update(TurnSwapA=f2(ratios[4][0]), TurnSwapB=f2(ratios[4][1]))
# v0/sigma_los of bins A and E (all giants) and of bin E from APOGEE alone (scripts/paper_kin_support.py).
ks = pd.read_csv(R / 'kin_support.csv').set_index('population')
M.update(RatLosA=f2(ks.ratio.iloc[0]), RatLosE=f2(ks.ratio.iloc[4]), RatLosApoE=f2(ks.ratio_apogee.iloc[4]))
# ... and with the velocity ellipsoid (results/kin_ellipsoid_checks.csv)
ec = pd.read_csv(R / 'kin_ellipsoid_checks.csv')
ecv = lambda m, c: ec[ec.model == m].set_index('population').loc[pops][c].to_numpy()  # noqa: E731
QUANT = ['vrot4', 'sigma_R', 'sigma_phi', 'sigma_z']
# Changes [km/s] of rotation and dispersions from the reference ellipsoid fit with a tilt (dt) or a free qlos (dq); their ln L
# gains.
dt = {c: ecv('tilt', c) - ecv('reference', c) for c in QUANT}
dq = {c: ecv('qlos', c) - ecv('reference', c) for c in QUANT}
et, eq = ecv('tilt', 'dlnL'), ecv('qlos', 'dlnL')
M.update(DlnLETiltD=num(round(et[3])), DlnLETiltE=num(round(et[4])), DlnLETiltOtherLo=f1(et[:3].min()),
         DlnLETiltOtherHi=num(round(et[:3].max())), DVETilt=f1(np.abs(dt['vrot4']).max()),
         DSigETilt=f1(max(np.abs(dt[c]).max() for c in SIG)))
put('DIncETilt', rng(-ecv('tilt', 'dinc')[3:], lambda x: num(round(x))))
# qlos of the ellipsoid fits; the two metal-poor bins sit at the upper bound (checked by the assert).
qe = ecv('qlos', 'qlos')
assert np.all(qe[:2] == qe[:2].max())   # the two metal-poor bins at the bound
M.update(QlosED=f2(qe[3]), DlnLQED=f1(eq[3]), QlosEE=f2(qe[4]), DlnLQEE=num(round(eq[4])), IncQEE=num(round(ecv('qlos', 'inc')[4])),
         QBound=f1(qe[0]), DlnLQEALo=f1(eq[:2].min()), DlnLQEAHi=f1(eq[:2].max()),
         DSigEQ=f1(max(np.abs(dq[c]).max() for c in SIG)), DVEQ=num(round(np.abs(dq['vrot4']).max())))
# Error scale k for the bright and faint halves, and the change of the dispersions [km/s] between them (bins A-C, then D-E).
put('KBright', rng(ecv('bright', 'k'), f2))
put('KFaint', rng(ecv('faint', 'k'), f2))
dhalf = {c: ecv('faint', c) - ecv('bright', c) for c in SIG}
M['DSigHalf'] = f1(max(np.abs(dhalf[c][:3]).max() for c in SIG))
put('DSigHalfMR', rng(np.concatenate([dhalf[c][3:] for c in SIG]), lambda x: num(round(x))))
```

#### Further checks
The inner edge of the sample, the m = 2 residual and the proper-motion errors.

```python
# Further checks: the inner edge of the sample (results/kin_dispersion_cut.csv)
dc = pd.read_csv(R / 'kin_dispersion_cut.csv')
# Spread of each giant bin's sigma [km/s] over the inner edges; rd, md: the RR Lyrae data and mock, indexed by inner edge [deg].
gd = dc[(dc['sample'] == 'data') & dc.population.str.startswith('giants')]
put('DSigGiant', rng(gd.groupby('population').sigma.agg(np.ptp)))
rd = dc[(dc['sample'] == 'data') & (dc.population == 'RR Lyrae')].set_index('inner')
md = dc[(dc['sample'] == 'mock')].set_index('inner')
# RR Lyrae k and sigma [km/s] at the 2-deg edge (where sigma sits at its bound) and at 3 deg; the mock sigma at 2 deg.
M.update(KRrlCutHi=f2(rd.k.iloc[0]), KRrlCutLo=f2(rd.k.min()), SigBound=num(rd.sigma.iloc[0]), SigRrlThree=num(round(rd.sigma[3.0])),
         MockSigTwo=num(round(md.sigma.iloc[0])))
# Ranges for inner edges of 4 deg and beyond (OUTER_INNER).
put('SigRrlOut', rng(rd.sigma[rd.index >= fk['OUTER_INNER']], lambda x: num(round(x))))
put('MockSigOut', rng(md.sigma[md.index >= fk['OUTER_INNER']], lambda x: num(round(x))))
put('MockKOutFit', rng(md.k[md.index >= fk['OUTER_INNER']], f2))

# m=2 residual about a symmetric disc: amplitude of the m=2 term of v_t about the Level-1 fit, all groups
# (results/kin_residual_profiles_ellipsoid.csv, scripts/paper_kin_residuals.py)
harm = pd.read_csv(R / 'kin_residual_profiles_ellipsoid.csv')
put('MTwoAmp', rng(harm[(harm.model == 'L1') & (harm.component == 'v_t')].A2, lambda x: sigfig(x, 2)))

# Gaia proper-motion errors in km/s (release samples): per star the mean of the two components' errors times kappa D0; giants and RR
# Lyrae at 2-9 deg (medians), and the change of the RR Lyrae sigma_PM at 4-9 deg for a change DK of k: the likelihood constrains
# sigma^2 + k^2 e^2, so sigma -> sqrt(sigma^2 -+ 2 k e^2 DK), with sigma and k of the fit at 4-9 deg (kin_dispersion_cut.csv) and e
# the median error there


def err_kms(c):
    """Mean of the two Gaia proper-motion errors of each star of catalogue c, in km/s at the LMC distance D0."""
    return 0.5 * (c['pmra_error'] + c['pmdec_error']) * KAPPA * LMC['D0']


# Median proper-motion error [km/s] of each giant bin at 2-9 deg; DK: change of k considered (dimensionless).
put('ErrMedG', rng([np.median(err_kms(ring(c))) for _, c in samples.giants()], lambda x: num(round(x))))
DK = 0.1
# RR Lyrae at 4-9 deg: median error e [km/s]; sigma and k of the 4-deg fit; change of sigma for k +- DK.
e_out = np.median(err_kms(ring(samples.rrab(), fk['OUTER_INNER'])))
s_out, k_out = rd.sigma[fk['OUTER_INNER']], rd.k[fk['OUTER_INNER']]
dsk = [abs(np.sqrt(s_out ** 2 + sgn * 2 * k_out * e_out ** 2 * DK) - s_out) for sgn in (1, -1)]
M.update(ErrMedRrl=num(round(np.median(err_kms(ring(samples.rrab()))))), ErrMedRrlOut=num(round(e_out)), DK=num(DK))
put('DSigK', rng(dsk, sigfig))
```

#### Write the macros
One \newcommand per entry of M, with the prefix \ks, into results/macros_kinsupport.tex; the page text shows them through
{glue:text}, so no number is typed by hand.

```python
# Header line, one \newcommand per macro, written and printed.
lines = ['% Numbers of the KB page Rotational support, centre and sensitivity (scripts/paper_text_kin_support.py); do not edit.']
lines += [rf'\newcommand{{\ks{k}}}{{{x}}}' for k, x in M.items()]
(R / 'macros_kinsupport.tex').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
```

### Rotation and the circular speed

The numbers of the section on the shared circular speed with asymmetric drift (`results/macros_vcdrift.tex`, prefix `\vd`).

<details><summary>About <code>scripts/paper_text_vc_drift.py</code></summary>

```text
Knowledge-base text, section 'Rotation and the circular speed' of the kinematics chapter: every number as a LaTeX macro (prefix
\\vd), from the tables of the shared-v_c + drift model.

Inputs: results/kin_vc_drift_data.csv, kin_vc_drift_mock.csv, kin_vc_drift_rrl.csv (scripts/kin_vc_drift.py),
kin_vc_drift_ext.csv, kin_vc_drift_deficit.csv (scripts/kin_vc_drift_extensions.py), kin_vc_drift_profiles.csv
(scripts/fig_kin_vc_drift.py); settings of scripts/kin_vc_drift.py; lmcold.kinfit.R_SIG.
Writes results/macros_vcdrift.tex.
Run: .venv/bin/python scripts/manifest.py run paper_text_vc_drift.py
```
</details>

**Data read by `scripts/paper_text_vc_drift.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/kin_vc_drift_data.csv` | repository | fixed input |
| `results/kin_vc_drift_deficit.csv` | repository | log-likelihood deficit of the shared-v_c + drift model per bin; written by `python scripts/kin_vc_drift_extensions.py` |
| `results/kin_vc_drift_ext.csv` | repository | fixed input |
| `results/kin_vc_drift_mock.csv` | repository | fixed input |
| `results/kin_vc_drift_profiles.csv` | repository | written by `python scripts/fig_kin_vc_drift.py` |
| `results/kin_vc_drift_rrl.csv` | repository | RR Lyrae rotation predicted by the shared-v_c + drift model and measured; written by `python scripts/kin_vc_drift.py` |

*Shown, not run here: the tables of the shared-v_c + drift model in results/. The cells below are the script itself; run it from the repository with* `python scripts/paper_text_vc_drift.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
Imports, settings of scripts/kin_vc_drift.py and the macro store M.

```python
import numpy as np
import pandas as pd

from kin_vc_drift import CURVE, H_FIT, MOCK_VC
from lmcold.kinfit import R_SIG
from lmcold.sources import ROOT
from textnum import num

R = ROOT / 'results'   # folder of the result tables
M = {}   # macro name (without the \vd prefix) -> text


def f1(x):
    """x with one decimal."""
    return f'{x:.1f}'
```

#### Free and drift fits
Rotation speeds, circular speed, lag and scale heights of the free and shared-v_c + drift fits to the data.

```python
d = pd.read_csv(R / 'kin_vc_drift_data.csv')   # one row per (model, population): vrot4, vc4 [km/s], h [kpc], ...
# Rows of the three models, each in population order (most metal-poor first).
free, drift, hfree = (d[d.model == m].reset_index(drop=True) for m in ('free', 'drift', 'drift, h free'))
lag = drift.vc4 - drift.vrot4   # km/s, circular speed minus rotation at R_SIG, per population
# R_SIG [kpc]; range of the free rotation speeds; v_c of the drift fit (measured and free h); lag of the first and last population;
# range of the exponential fits [kpc]; scale heights h [kpc]; NPar: rotation-curve parameters (CURVE) of all but one population.
M.update(RSig=num(R_SIG), VrotLo=f1(free.vrot4.min()), VrotHi=f1(free.vrot4.max()), VcFour=f1(drift.vc4.iloc[0]),
         VcHfree=f1(hfree.vc4.iloc[0]), LagMP=num(round(lag.iloc[0])), LagMR=num(round(lag.iloc[-1])),
         HFitIn=num(H_FIT[0]), HFitOut=num(H_FIT[1]),
         HMeasLo=f1(drift.h.min()), HMeasHi=f1(drift.h.max()), HFreeLo=f1(hfree.h.min()), HFreeHi=f1(hfree.h.max()),
         NPar=num(len(CURVE) * (len(free) - 1)))
```

#### Extensions of the model
Log-likelihood differences and circular speeds of the variants.

```python
e = pd.read_csv(R / 'kin_vc_drift_ext.csv')   # one row per (variant, population)
dl = e.groupby('variant', sort=False).dlnL.first()   # log-likelihood difference of each variant (same in its rows)
vc = e.groupby('variant', sort=False).vc4   # km/s, circular speed of each variant's rows
# Rounded log-likelihood differences of the variants (signs as the text quotes them), size of the matched sample, the gap between
# 'free + stream' and 'drift + stream', and the v_c range of 'drift + beta'.
M.update(DlnL=num(round(-dl['drift'])), DlnLHLo=num(round(-dl['drift, h p16'])), DlnLHHi=num(round(-dl['drift, h p84'])),
         DlnLMatched=num(round(-dl['matched: drift'])), NMatched=f"{int(e[e.variant == 'matched: drift'].n.sum()):,}".replace(',', '\\,'),
         DlnLStreamFree=num(round(dl['free + stream'])),
         GapStream=num(round(dl['free + stream'] - dl['drift + stream'])), DlnLBeta=num(round(-dl['drift + beta'])),
         VcBetaLo=f1(vc.min()['drift + beta']), VcBetaHi=f1(vc.max()['drift + beta']),
         DlnLStreamBeta=num(round(-dl['drift + stream + beta'])))
```

#### Deficit per bin
The log-likelihood deficit of the drift model per giant bin.

```python
de = pd.read_csv(R / 'kin_vc_drift_deficit.csv')
# Deficit per giant bin: dlnL summed over the bins in R (coordinate 'R'); range quoted with the sign reversed.
per_bin = de[de.coordinate == 'R'].groupby('population', sort=False).dlnL.sum()
M.update(DefLo=num(round(-per_bin.max())), DefHi=num(round(-per_bin.min())))
```

#### Mock test and RR Lyrae
Recovery of the circular speed in mocks, and the RR Lyrae rotation predicted and measured.

```python
m = pd.read_csv(R / 'kin_vc_drift_mock.csv')   # fits of the mock realisations
true_vc4 = MOCK_VC['v0'] * np.tanh(R_SIG / MOCK_VC['R0'])   # the mock curve at R_SIG (its linear term vanishes there; lmcold.kinfit.rotation)
rec = m[m.model == 'drift'].groupby('realisation').vc4.first() - true_vc4   # km/s, recovered minus true v_c per realisation
# True v_c and the largest recovery error.
M.update(MockVc=f1(true_vc4), MockDvc=f1(rec.abs().max()))
r = pd.read_csv(R / 'kin_vc_drift_rrl.csv', index_col=0)   # RR Lyrae rotation: columns predicted, measured; rows median, p16, p84
M.update(RrlPred=f1(r.predicted['median']), RrlPredLo=f1(r.predicted.p16), RrlPredHi=f1(r.predicted.p84),
         RrlMeas=f1(r.measured['median']), RrlMeasLo=f1(r.measured.p16), RrlMeasHi=f1(r.measured.p84))
```

#### Profiles
The measured and model v_phi of the most metal-poor bin in the radial ranges the text quotes.

```python
p = pd.read_csv(R / 'kin_vc_drift_profiles.csv')   # binned v_phi profiles: data_vphi, model_vphi [km/s] per bin lo-hi [kpc]
# The [Fe/H] < -1.5 giants, binned in R.
mp = p[(p.population == 'giants [Fe/H] < -1.5') & (p.coordinate == 'R')]
MID, OUT_FROM, INNER = (3, 5), 7, (2, 3)   # kpc: radial ranges of the comparison quoted (bins of kin_vc_drift_profiles.csv)
# Bins inside MID and bins beyond OUT_FROM; their data and drift-model v_phi ranges [km/s] are quoted as 'a--b'.
mid, out = mp[(mp.lo >= MID[0]) & (mp.hi <= MID[1])], mp[mp.lo >= OUT_FROM]
M.update(MidLo=num(MID[0]), MidHi=num(MID[1]), OutFrom=num(OUT_FROM), InLo=num(INNER[0]), InHi=num(INNER[1]),
         MidData=f'{mid[mid.model == "drift"].data_vphi.min():.0f}--{mid[mid.model == "drift"].data_vphi.max():.0f}',
         OutData=f'{out[out.model == "drift"].data_vphi.min():.0f}--{out[out.model == "drift"].data_vphi.max():.0f}',
         OutDrift=f'{out[out.model == "drift"].model_vphi.min():.0f}--{out[out.model == "drift"].model_vphi.max():.0f}')
```

#### Write the macros
One \newcommand per entry of M, prefix \vd, into results/macros_vcdrift.tex.

```python
# Header comment, then one \newcommand line per macro.
lines = ['% Numbers of the KB section Rotation and the circular speed (scripts/paper_text_vc_drift.py); do not edit.']
lines += [rf'\newcommand{{\vd{k}}}{{{x}}}' for k, x in M.items()]
(R / 'macros_vcdrift.tex').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
```

### Shape

The numbers of the chapter on the shape of the old population (`results/macros_shape.tex`, prefix `\sh`).

<details><summary>About <code>scripts/paper_text_shape.py</code></summary>

```text
Knowledge-base text, chapter 'Shape of the old population' (extended draft Section 5 and its appendix): every number of the text as
a LaTeX macro (prefix \\sh), so that none is typed by hand.

As scripts/paper_text_letter.py: settings are read from the code (scripts/textnum.const, lmcold by import), results from the tables
the figures are drawn from; the effect of the m=2 streaming on q is computed here from the posterior medians of the ellipsoid fits
(data/kin_mcmc_data_ellipsoid_samples.npz) with lmcold.virial (about 10 s). Literature values quoted with their citation stay in the text.
Inputs: results/{shape_virial, shape_virial_sims, kin_vc_drift_h, kin_support_ellipsoid, kin_mcmc_data_ellipsoid,
kin_mcmc_data_ellipsoid_outer, shape_rrl_cells, shape_rrl_fit_stats, shape_rrl_harmonics, shape_rrl_triaxial, shape_rrl_depth,
kin_ellipsoid_mock}.csv; scripts/{shape_virial, shape_rrl_cells, shape_rrl_triaxial, paper_kin_ellipsoid_mock, fig_kin_mock_physical}.py;
lmcold.virial (potentials), lmcold.shape (sky cells); tests/test_virial.py (accuracy of the grid potential).
Writes results/macros_shape.tex.
Run: .venv/bin/python scripts/manifest.py run paper_text_shape.py
```
</details>

**Data read by `scripts/paper_text_shape.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `data/kin_mcmc_data_ellipsoid_samples.npz` | Zenodo bundle 1 | Posterior samples of the ellipsoid kinematic fit, giants 2-9 deg (NUTS) |
| `results/kin_ellipsoid_mock.csv` | repository | written by `python scripts/paper_kin_ellipsoid_mock.py` |
| `results/kin_mcmc_data_ellipsoid.csv` | repository | posterior summary of the joint kinematic fit (velocity ellipsoid), 2-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid` |
| `results/kin_mcmc_data_ellipsoid_outer.csv` | repository | the same fit at 4-9 deg; MCMC; written by `python scripts/fit_kinematics_mcmc.py data ellipsoid outer` |
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |
| `results/kin_vc_drift_h.csv` | repository | fixed input |
| `results/shape_rrl_cells.csv` | repository | oblate-model fits of the RR Lyrae sky cells; written by `python scripts/shape_rrl_cells.py` |
| `results/shape_rrl_depth.csv` | repository | axis ratio from the RR Lyrae depth gradient at the inclination of the motions; written by `python scripts/shape_rrl_cells.py` |
| `results/shape_rrl_fit_stats.csv` | repository | written by `python scripts/paper_shape_rrl_residuals.py` |
| `results/shape_rrl_harmonics.csv` | repository | written by `python scripts/paper_shape_rrl_residuals.py` |
| `results/shape_rrl_triaxial.csv` | repository | oblate fits of mocks elongated in their plane; written by `python scripts/shape_rrl_triaxial.py` |
| `results/shape_virial.csv` | repository | axis ratio implied by the motions (tensor virial theorem) per giant bin and potential; written by `python scripts/shape_virial.py` |
| `results/shape_virial_sims.csv` | repository | fixed input |

*Shown, not run here: the result tables in results/ and the posterior samples of Zenodo bundle 1; about 10 s (script header). The cells below are the script itself; run it from the repository with* `python scripts/paper_text_shape.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
Imports, the dictionary M of macros (macro name without its prefix: the text it stands for) and the functions that format
numbers as the text writes them. The inputs listed in the header are read in the step that uses them.

```python
import ast
import re

import numpy as np
import pandas as pd

from lmcold import kinfit, shape, sims, spheroid, virial
from lmcold.samples import FEH_BINS
from lmcold.sources import ROOT
from fig_kin_mock_physical import H_THICK, HOT, RD, TRUTH
from shape_rrl_triaxial import C_ALL, CASES
from shape_virial import draws
from textnum import const, num

R = ROOT / 'results'   # directory of the input tables and of the output macros
M = {}                 # macro name (without the prefix \sh) -> text


def f2(x):
    """x with two decimals."""
    return f'{x:.2f}'


def f1(x):
    """x with one decimal."""
    return f'{x:.1f}'


def signed(x, fmt=f2):
    """x formatted by fmt with an explicit sign ('+' added if not negative)."""
    s = fmt(x)
    return s if s.startswith('-') else '+' + s
```

#### Settings
Settings read from the shape and mock scripts, and the parameters of the potentials P1-P3 from the source of lmcold.virial.

```python
# Settings
# Literal module constants of the shape and mock scripts (const reads them without running the script); the triaxial-mock grid
# and the physical-mock settings are imported.
sv, cells = const('shape_virial'), const('shape_rrl_cells')
tri = dict(C_ALL=C_ALL, CASES=CASES)
mock = const('paper_kin_ellipsoid_mock')
phys = dict(H_THICK=H_THICK, HOT=HOT, RD=RD, TRUTH=TRUTH)
pot = {}   # numbers of lmcold.virial.POTENTIALS, from its source: P1 (V, a), P2 (V, a, q_Phi), P3 (M, a, b, V, a)
# Walk the module syntax tree of lmcold.virial: the literal arguments of the calls inside the POTENTIALS dict, per potential.
for n in ast.parse((ROOT / 'src/lmcold/virial.py').read_text()).body:
    if isinstance(n, ast.Assign) and getattr(n.targets[0], 'id', '') == 'POTENTIALS':
        for k, v in zip(n.value.keys, n.value.values):
            pot[k.value] = [a.value for call in ast.walk(v) if isinstance(call, ast.Call) for a in call.args if isinstance(a, ast.Constant)]
M.update(VOne=num(pot['P1'][0]), AOne=num(pot['P1'][1]), QPhi=num(pot['P2'][2]), MMN=num(pot['P3'][0] / 1e9), AMN=num(pot['P3'][1]),
         BMN=num(pot['P3'][2]), VThree=num(pot['P3'][3]), AThree=num(pot['P3'][4]), NDraw=num(sv['NDRAW']), RFloor=num(sv['R_FLOOR']))
VC_R = (2.0, 8.0)   # kpc, radii over which the circular speeds of the three potentials are quoted
Rv = np.linspace(*VC_R, 13)   # radii [kpc] in the disc plane
vc = [np.sqrt(Rv * g(np.column_stack([Rv, 0 * Rv, 0 * Rv]))[:, 0]) for g in virial.POTENTIALS.values()]   # V_c = sqrt(R dPhi/dR)
M.update(VcLo=num(5 * round(min(v.min() for v in vc) / 5)), VcHi=num(5 * round(max(v.max() for v in vc) / 5)))   # nearest 5 km/s
# Radii, simulation and cylinder settings, [Fe/H] bin edges; Hernquist test flattening, tolerance [%] from tests/test_virial.py.
M.update(VcRIn=num(VC_R[0]), VcROut=num(VC_R[1]), RMaxSim=num(sims.R_MAX), CylR=num(spheroid.CYL[0]), CylZ=num(spheroid.CYL[1]),
         FehMP=num(FEH_BINS[1][1]), FehMR=num(FEH_BINS[-1][0]),
         HernQ=num(float(re.search(r'spheroid\.spheroid\([0-9.]+, ([0-9.]+),', (ROOT / 'tests/test_virial.py').read_text()).group(1))),
         NSrc=f'{virial.NSRC:,}'.replace(',', '\\,'),
         HernTol=num(100 * float(re.search(r'hernquist\(y\)\.T\) - 1\) < ([0-9.]+)', (ROOT / 'tests/test_virial.py').read_text()).group(1))))
```

#### Flattening implied by the motions
The axis ratio q of each bin from the tensor virial theorem, and the change of q when the m = 2 streaming is included.

```python
# Flattening implied by the motions (results/shape_virial.csv; bins 0-4 = giants by [Fe/H], RR Lyrae)
v = pd.read_csv(R / 'shape_virial.csv')
pops = list(v.population.unique())   # population names in file order
g0 = v[(v.r_floor == 0) & v.population.str.startswith('giants')]   # giants, no central floor
# Giant bin i -> its rows indexed by potential (P1, P2, P3); 0 is the most metal-poor, 4 the most metal-rich.
b = {i: g0[g0.population == p].set_index('potential') for i, p in enumerate(pops[:5])}
half = ((b[0].q_p84 - b[0].q_p16) / 2).mean()   # mean half-width of the 16th-84th interval of q, most metal-poor bin
M.update(QMPOne=f2(b[0].q.P1), QMPTwo=f2(b[0].q.P2), QMPThree=f2(b[0].q.P3), QHalf=f2(half),
         QMRLo=f2(b[4].q.min()), QMRHi=f2(b[4].q.max()), DqPotMP=f2(b[0].q.max() - b[0].q.min()),
         DqRatio=num(round((b[0].q.max() - b[0].q.min()) / half)), DqPotMR=f2(b[4].q.max() - b[4].q.min()),
         QOldLo=f2(min(b[0].q.min(), b[1].q.min())), QOldHi=f2(max(b[0].q.max(), b[1].q.max())))
# 'balance' column of shape_virial.csv: range over bins 0-3 in P1, P2; bin 4 mean in P1, P2; range over all bins in P3.
low = pd.concat([b[i] for i in range(4)])
M.update(BalLo=f2(low.loc[['P1', 'P2']].balance.min()), BalHi=f2(low.loc[['P1', 'P2']].balance.max()),
         BalMR=f2(b[4].loc[['P1', 'P2']].balance.mean()),
         BalThreeLo=f2(pd.concat([b[i] for i in range(5)]).loc['P3'].balance.min()),
         BalThreeHi=f2(pd.concat([b[i] for i in range(5)]).loc['P3'].balance.max()))
# q with and without the central floor R_FLOOR: largest change over giant bins and potentials.
fl = v[v.population.str.startswith('giants')].pivot_table(index=['population', 'potential'], columns='r_floor', values='q')
M.update(DqFloor=f'{(fl[sv["R_FLOOR"]] - fl[0.0]).abs().max():.3f}')
h = pd.read_csv(R / 'kin_vc_drift_h.csv').set_index('population')   # scale length h [kpc] per population
M.update(HMP=f1(h.h.iloc[0]))
# The m=2 streaming of the fits (m2c, m2s of v_phi; r2c, r2s of v_R, in units of the rotation speed; lmcold.kinfit), which the
# realisation (lmcold.spheroid.velocities) omits: averaged over azimuth it adds v_rot^2 (m2c^2 + m2s^2 + r2c^2 + r2s^2) / 2 to the
# in-plane v^2. Fraction f of the in-plane kinetic energy and change of q when Omega_kin is raised by (1 + f), for the
# posterior-median model of each giant bin (as scripts/shape_virial.py, no floor) in each potential (same positions for both q).
f_stream, dq_stream = [], []
for i in range(5):
    # Population name and its posterior-median parameter vector (shape_virial.draws).
    pop, med, _ = draws('kin_mcmc_data_ellipsoid_samples.npz', i, np.random.default_rng(0))
    # Realisation: NREAL positions X [kpc] of a spheroid with scale length h and axis ratio 1.0, velocities V [km/s], shape (N, 3).
    rg = np.random.default_rng(1)
    X = spheroid.spheroid(h.h[pop], 1.0, virial.NREAL, rg)
    V = spheroid.velocities(med, X, rg)
    vrot = np.asarray(kinfit.rotation(med, np.hypot(X[:, 0], X[:, 1])))   # mean rotation [km/s] at each in-plane radius
    # Azimuthal average of the squared m = 2 amplitudes / 2 (units of v_rot^2); f = added / existing in-plane kinetic energy.
    s2 = sum(med[kinfit.PARAMS.index(n)] ** 2 for n in ('m2c', 'm2s', 'r2c', 'r2s')) / 2
    f = np.sum(vrot ** 2) * s2 / np.sum(V[:, 0] ** 2 + V[:, 1] ** 2)
    om = virial.omega_kin(V)   # Omega_kin = in-plane / (2 x vertical) kinetic energy of the realisation (lmcold.virial)
    f_stream.append(f)
    # q solved with Omega_kin and with Omega_kin (1 + f), in each potential; change dq.
    for grad in virial.POTENTIALS.values():
        q0, q1 = (virial.solve_q(t, h.h[pop], grad, np.random.default_rng(2)) for t in (om, om * (1 + f)))
        dq_stream.append(q1 - q0)
M.update(StreamKE=f1(100 * max(f_stream)), DqStream=f'{max(np.abs(dq_stream)):.3f}')
```

#### Test on the simulated analogues
The virial axis ratio against the true one in the simulations.

```python
# Test on the simulated analogues (results/shape_virial_sims.csv)
s = pd.read_csv(R / 'shape_virial_sims.csv')
d = s.q_model_sim - s.q_true   # virial q (simulation's own potential) minus true q
M.update(NSimGal=num(s.name.nunique()), NSimBin=num(len(s)), QTrueLo=f2(s.q_true.min()), QTrueHi=f2(s.q_true.max()),
         DqSim=f2(d.median()), DqSimLo=f2(d.quantile(.16)), DqSimHi=f2(d.quantile(.84)), DqSimMin=f2(d.min()), DqSimMax=signed(d.max()))
# The same with each model potential P1-P3, signed.
for k, P in zip(('One', 'Two', 'Three'), ('P1', 'P2', 'P3')):
    e = s[f'q_model_{P}'] - s.q_true
    M.update({f'DqSim{k}': signed(e.median()), f'DqSim{k}Lo': signed(e.quantile(.16)), f'DqSim{k}Hi': signed(e.quantile(.84))})
```

#### Kinematic geometry
Inclinations and other geometry of the stage-1 fits quoted in the text.

```python
# Kinematic geometry quoted (results/kin_support_ellipsoid.csv, kin_mcmc_data_ellipsoid*.csv: stage-1 fits)
ks = pd.read_csv(R / 'kin_support_ellipsoid.csv').set_index('population')
km = pd.read_csv(R / 'kin_mcmc_data_ellipsoid.csv')
km = km[km.stage == 1].pivot_table(index='population', columns='param', values='median')   # population x parameter medians
old = list(cells['OLD'])   # the two metal-poor giant bins (OLD of scripts/shape_rrl_cells.py)
inc_old, theta_old = km.loc[old, 'inc'].mean(), km.loc[old, 'theta'].mean()
inc_rich = km.loc[[p for p in pops[2:5]], 'inc'].mean()
# RR Lyrae inclination from the 4-9 deg ('_outer') ellipsoid fit, stage 1.
ko = pd.read_csv(R / 'kin_mcmc_data_ellipsoid_outer.csv')
inc_rrl = ko[(ko.stage == 1) & (ko.population == 'RR Lyrae') & (ko.param == 'inc')]['median'].iloc[0]
M.update(FehRrl=f2(ks.feh_med['RR Lyrae']), SigzMP=num(round(ks.sigma_z.iloc[0])), IncOld=f1(inc_old), IncOldInt=num(round(inc_old)), ThetaOld=num(round(theta_old)),
         IncRich=num(round(inc_rich)), IncRrlPM=f1(inc_rrl), CosIncOld=f2(np.cos(np.deg2rad(inc_old))))
```

#### RR Lyrae sky cells
Fits of the RR Lyrae distances in sky cells, the triaxial mocks and the depth gradient.

```python
# RR Lyrae sky cells (results/shape_rrl_*.csv; lmcold.shape)
st = pd.read_csv(R / 'shape_rrl_fit_stats.csv').set_index('fit')            # fit statistics (star and cell counts, chi^2, ...)
fr = pd.read_csv(R / 'shape_rrl_cells.csv').query("fit == 'free'").iloc[0]  # free oblate fit: inc [deg], q, theta [deg]
t = pd.read_csv(R / 'shape_rrl_triaxial.csv')                               # oblate fits of the mocks; the data row has c NaN
data = t[t.c.isna()].iloc[0]                                                # observed q_sky and depth gradient g
M.update(NRrl=f'{int(st.n_stars.free):,}'.replace(',', '\\,'), NCells=num(st.n_cells.free), NRings=num(len(shape.CELL_RHO) - 1),
         NSect=num(shape.CELL_NPA), RhoIn=num(shape.CELL_RHO[0]), RhoOut=num(shape.CELL_RHO[-1]),
         QSky=f2(data.q_sky), G=f2(data.g), QSkyThree=f'{data.q_sky:.3f}', GThree=f'{data.g:.3f}',
         IncFree=num(round(fr.inc)), IncFreeOne=f1(fr.inc), QFree=f2(fr.q), ThetaFree=num(round(fr.theta)),
         ChiCounts=num(round(st.chi2_counts.free)), ChiMed=num(round(st.chi2_medians.free)),
         PearsonMin=f1(st.pearson_min.free), PearsonMax=signed(st.pearson_max.free, f1),
         Scatter=f'{st.rms_modulus.free:.3f}', MedErr=f'{st.median_se.free:.3f}')
hm = pd.read_csv(R / 'shape_rrl_harmonics.csv')   # m = 1 harmonic a1, pa1 [deg] and coverage per ring rho_lo-rho_hi [deg]
out, inner = hm[hm.rho_lo >= 5], hm.iloc[0]       # rings from 5 deg out, and the innermost ring
M.update(AOneOutLo=num(round(100 * out.a1.min())), AOneOutHi=num(round(100 * out.a1.max())),
         AOneOutLoF=f2(out.a1.min()), AOneOutHiF=f2(out.a1.max()), AOneIn=f2(inner.a1), RhoInHi=num(inner.rho_hi),
         RhoOutLo=num(out.rho_lo.min()), PAOneLo=num(round(out.pa1.min())), PAOneHi=num(round(out.pa1.max())),
         Coverage=f'{100 * hm.coverage.min():.1f}')
# The elongated mock shown in figure paper_shape_rrl (c/a 0.41, b/a 0.85, angle 150 deg: ARROW of scripts/paper_shape_rrl.py).
mk = t[(t.c == 0.41) & (t.ba == 0.85) & (t.phi == 150)].iloc[0]
M.update(MockC=num(mk.c), MockBA=num(mk.ba), MockPhi=num(mk.phi), MockInc=num(round(mk.inc)), MockIncOne=f1(mk.inc), MockQ=f2(mk.q),
         MockTheta=num(round(mk.theta)), MockQSky=f2(mk.q_sky), MockQSkyThree=f'{mk.q_sky:.3f}', MockG=f2(mk.g), MockGThree=f'{mk.g:.3f}',
         CsOne=num(tri['C_ALL'][0]), CsTwo=num(tri['C_ALL'][1]), BAOne=num(tri['CASES'][1][0]), BATwo=num(tri['CASES'][-1][0]),
         PhiStep=num(tri['CASES'][2][1] - tri['CASES'][1][1]), PhiMin=num(tri['CASES'][1][1]), PhiMax=num(tri['CASES'][-1][1]))
el = t[t.c.notna() & (t.ba < 1)]   # elongated (triaxial) mocks
NEAR, FAR = (0, 30, 150), (60, 90, 120)   # deg: major axis within 30 deg of the line of nodes, and 60-120 deg from it
near = el[el.phi.isin(NEAR)]
# FAR excludes the mocks with c = C_ALL[0], b/a = CASES[1][0] at 60 and 120 deg (the code gives no reason).
far = el[el.phi.isin(FAR) & ~((el.c == tri['C_ALL'][0]) & (el.ba == tri['CASES'][1][0]) & el.phi.isin([60, 120]))]
ob = t[t.c.notna() & (t.ba == 1)].set_index('c')   # oblate mocks (b/a = 1), indexed by c/a
M.update(NearIncLo=num(round(near.inc.min())), NearIncHi=num(round(near.inc.max())), NearQLo=f2(near.q.min()), NearQHi=f2(near.q.max()),
         NearQSkyLo=f2(near.q_sky.min()), NearQSkyHi=f2(near.q_sky.max()), FarIncLo=num(round(far.inc.min())),
         FarIncHi=num(round(far.inc.max())), FarQLo=f2(far.q.min()), FarQHi=f2(far.q.max()),
         ObIncOne=num(round(ob.inc[tri['C_ALL'][0]])), ObQOne=f2(ob.q[tri['C_ALL'][0]]), ObIncTwo=num(round(ob.inc[tri['C_ALL'][1]])),
         ObQTwo=f2(ob.q[tri['C_ALL'][1]]), FarPhiLo=num(min(FAR)), FarPhiHi=num(max(FAR)))
dp = pd.read_csv(R / 'shape_rrl_depth.csv')   # q from the depth gradient: case 'data', 'bootstrap' draws, 'dipole*' variants
bs = dp[dp.case == 'bootstrap'].q
M.update(QDepth=f2(dp[dp.case == 'data'].q.iloc[0]), QDepthLo=f2(bs.quantile(.16)), QDepthHi=f2(bs.quantile(.84)),
         QDipLo=f2(dp[dp.case.str.startswith('dipole')].q.min()), QDipHi=f2(dp[dp.case.str.startswith('dipole')].q.max()),
         Dipole=num(cells['DIPOLE']))
```

#### The shape and the kinematic model
Mocks of the velocity-ellipsoid model.

```python
# The shape and the kinematic model (results/kin_ellipsoid_mock.csv; scripts/paper_kin_ellipsoid_mock.py, fig_kin_mock_physical.py)
mo = pd.read_csv(R / 'kin_ellipsoid_mock.csv')
sph = mo[mo.mock == 'hot spheroid'].set_index('data')   # fits to the hot-spheroid mock, by data used
both, pm = sph.loc['PM + LOS'], sph.loc['PM']          # with proper motions and velocities, and with proper motions only
# Largest error [km/s] of the three dispersions in the PM + LOS fit.
dsig = max(abs(both[c] - both[f'truth_{c}']) for c in ('sigma_R', 'sigma_phi', 'sigma_z'))
M.update(QSph=num(mock['Q_SPH']), RD=num(phys['RD']), HThick=num(phys['H_THICK']), RmsSph=f1(mock['Q_SPH'] * np.sqrt(3) * phys['RD']),
         RmsMP=f1(0.5 * np.sqrt(3) * h.h.iloc[0]), SigR=num(phys['HOT'][0]), SigPhi=num(phys['HOT'][1]), SigZ=num(phys['HOT'][2]),
         IncMock=num(phys['TRUTH']['inc']), DSigMock=f1(np.ceil(10 * dsig) / 10), DIncMock=f1(abs(both.inc - phys['TRUTH']['inc'])),
         DThetaMock=num(round(phys['TRUTH']['theta'] - both.theta)), SigzPM=num(round(pm.sigma_z)), SigzPMErr=num(round(pm.sigma_z_err)),
         IncPM=num(round(pm.inc)))
```

#### Write the macros
One \newcommand per entry of M, with the prefix \sh, into results/macros_shape.tex; the text shows them as macros, so no
number is typed by hand.

```python
# Header line, then one \newcommand per macro; written to results/macros_shape.tex and printed.
lines = ['% Numbers of the KB chapter Shape (scripts/paper_text_shape.py); do not edit.']
lines += [rf'\newcommand{{\sh{k}}}{{{x}}}' for k, x in M.items()]
(R / 'macros_shape.tex').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
```

### Simulations

The numbers of the two simulation chapters that no other generator writes (`results/macros_sims.tex`, prefix `\sm`).

<details><summary>About <code>scripts/paper_text_sims.py</code></summary>

```text
Knowledge-base text, chapters 'Simulations: analogues and mock observation' and 'Simulations: birth, scrambling, halo settling,
SIDM' (extended draft Section 6 and its appendix): the numbers of the text that no other generator writes, as LaTeX macros (prefix
\\sm), so that none is typed by hand. The other numbers of these pages are the macros of scripts/paper_text_letter.py (\\lt),
paper_text_sims_birth.py (\\sb, \\sh) and paper_text_sims_suites.py (\\ss).

As scripts/paper_text_shape.py: settings are read from the code (scripts/textnum.const, lmcold by import), results from the tables
the figures are drawn from, with the statistics of the figure scripts. Where a figure script fixes a value inside its code (a bin
edge, an age cut), the value is repeated below with the script named. Literature values quoted with their citation stay in the text.
Inputs: results/{kin_support_ellipsoid, giant_mass, sims_observed, sims_formed_mass, sims_kinematics_summary, sims_mass_counts,
sims_definitions, sims_populations, sims_scramble, sims_halo_cohorts, sims_halo_settle, sims_heating_coherence, sims_heating_gas,
sims_sidm_shapes, auriga_scramble}.csv; scripts/{sims_observe, paper_sims_kinematics, paper_sims_settle, paper_text_sims_birth,
paper_text_sims_suites, build_sims_heating, sims_sidm_shapes}.py; results/mass_survival_grid.csv. Simulation files (sources.yaml):
the Auriga and FIRE-2 snapshot headers and stars (dark-matter particle masses, formed mass), the TNG50 progenitor cutouts (late-cohort
redshifts; one full snapshot for the force check of the multipole expansion), the Concerto Rockstar catalogues (virial masses).
Checks run here: the expansion's force against a softened direct sum, and AGAMA's coefficient normalisation on a Miyamoto-Nagai disc
(AGAMA; ~10 s in all).
Writes results/macros_sims.tex.
Run: .venv/bin/python scripts/manifest.py run paper_text_sims.py
```
</details>

**Data read by `scripts/paper_text_sims.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/auriga_scramble.csv` | repository | written by `python scripts/auriga_scramble.py` |
| `results/giant_mass.csv` | repository | written by `python scripts/build_giant_mass.py` |
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |
| `results/mass_survival_grid.csv` | repository | written by `python scripts/build_present_mass.py` |
| `results/sims_definitions.csv` | repository | written by `python scripts/paper_sims_kinematics.py` |
| `results/sims_formed_mass.csv` | repository | written by `python scripts/build_sims_formed_mass.py` |
| `results/sims_halo_cohorts.csv` | repository | written by `python scripts/sims_halo_settle.py` |
| `results/sims_halo_settle.csv` | repository | written by `python scripts/sims_halo_settle.py` |
| `results/sims_heating_coherence.csv` | repository | written by `python scripts/build_sims_heating.py` |
| `results/sims_heating_gas.csv` | repository | written by `python scripts/build_sims_heating.py` |
| `results/sims_kinematics_summary.csv` | repository | written by `python scripts/paper_sims_kinematics.py` |
| `results/sims_mass_counts.csv` | repository | written by `python scripts/paper_sims_kinematics.py` |
| `results/sims_observed.csv` | repository | written by `python scripts/sims_observe.py merge` |
| `results/sims_populations.csv` | repository | written by `python scripts/fig_sims_populations.py` |
| `results/sims_scramble.csv` | repository | written by `python scripts/sims_scramble.py` |
| `results/sims_sidm_shapes.csv` | repository | written by `python scripts/sims_sidm_shapes.py` |

*Shown, not run here: the result tables in results/, the Auriga, FIRE-2, TNG50 and Concerto files of the fetch scripts above, and AGAMA; the checks take about 10 s (script header). The cells below are the script itself; run it from the repository with* `python scripts/paper_text_sims.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
Imports, the dictionary M of macros (macro name without its prefix: the text it stands for) and the functions that format
numbers as the text writes them. The inputs listed in the header are read in the step that uses them.

```python
import ast
import glob
import re
import tempfile

import agama
import h5py
import numpy as np
import pandas as pd
from scipy.spatial.transform import Rotation
from scipy.stats import spearmanr

import sims_scramble as scr
from fetch_tng50_progenitors import GAS_SNAPS
from lmcold import sims
from lmcold.constants import G
from lmcold.sources import ROOT
from paper_sims_kinematics import NMIN_SIM, lmc_band
from sims_observe import lmc_knots
from sims_sidm_shapes import RUNS, SNAPS as SIDM_SNAPS, halos
from textnum import const, num, sigfig

R = ROOT / 'results'  # folder of the result tables
M = {}  # macro name (without the \sm prefix) -> text
# literal module constants of the scripts whose settings the text quotes
obs, settle, sbirth, suites = const('sims_observe'), const('paper_sims_settle'), const('paper_text_sims_birth'), const('paper_text_sims_suites')
heat, sidm = const('build_sims_heating'), const('sims_sidm_shapes')
# QUIET of paper_sims_settle.py is not a pure literal (it uses EPS_QUIET): its expression is evaluated with EPS_QUIET supplied
QUIET = next(eval(compile(ast.Expression(n.value), 'QUIET', 'eval'), {'EPS_QUIET': settle['EPS_QUIET']})   # paper_sims_settle.QUIET
             for n in ast.parse((ROOT / 'scripts/paper_sims_settle.py').read_text()).body
             if isinstance(n, ast.Assign) and getattr(n.targets[0], 'id', '') == 'QUIET')


def f(x, d=2):
    """x with d decimals."""
    return f'{x:.{d}f}'


def signed(x, d=2):
    """x with d decimals and an explicit '+' when not negative."""
    s = f(x, d)
    return s if s.startswith('-') else '+' + s


def sci(x, d=1):
    """x as m\\times10^e, m with d decimals (d = 0: one significant figure)."""
    e = int(np.floor(np.log10(x)))
    return rf'{x / 10 ** e:.{d}f}\times10^{{{e}}}'


def bound(x):
    """10^e with e the smallest integer such that 10^e > x (an upper bound on a small difference)."""
    return rf'10^{{{int(np.floor(np.log10(x))) + 1}}}'
```

#### LMC bins and the observed analogues
The LMC giant bins and the mock observations of the analogues.

```python
# LMC giant bins (results/kin_support_ellipsoid.csv: stage-1 ellipsoid fits at R = 4 kpc) and formed mass (results/giant_mass.csv)
# stage-1 ellipsoid summaries of the LMC populations; giant bins only, in [Fe/H] order
k = pd.read_csv(R / 'kin_support_ellipsoid.csv')
k = k[k.population.str.startswith('giants')].reset_index(drop=True)
# (x, y) knots of the LMC's cumulative formed-mass fraction against [Fe/H] (sims_observe.lmc_knots)
knots = lmc_knots()
frac = np.interp(k.feh_med, *knots)                                  # formed-mass fraction below each bin's median [Fe/H]
edges = np.interp([-1.5, -1.1, -0.8, -0.5], *knots)                   # ... below the bin edges (paper_sims_kinematics.py)
m_lmc = pd.read_csv(R / 'giant_mass.csv').cumulative.iloc[-1]         # formed mass within RHO_MASS
# settings of the mock observation; mass fractions and masses [Msun] below the bins; LMC v_rot/sigma_z of the most metal-poor
# bin and its half-width; second bin; anisotropy sigma_R/sigma_z; factor between the last and first bins (all, APOGEE-only)
M.update(RhoMass=num(obs['RHO_MASS']), TruthDR=num(obs['TRUTH_DR']), FirstFrac=f(100 * edges[0], 1), FracB=f(100 * edges[1], 1),
         FracC=num(round(100 * edges[2])), MLmc=sci(m_lmc), MassA=sci(frac[0] * m_lmc), MassB=sci(frac[1] * m_lmc),
         NRealMP=num(k.n[0]), LmcRatioMP=f(k.ratio_z[0]), LmcRatioMPErr=f((k.ratio_z_p84[0] - k.ratio_z_p16[0]) / 2),
         LmcRatioB=f(k.ratio_z[1]), LmcVrotB=num(round(k.vrot4[1])), LmcSzB=num(round(k.sigma_z[1])),
         AnisoLmc=f(k.sigma_R[0] / k.sigma_z[0]), ShapeLmc=f(k.sigma_z[0] / k.sigma_R[0]),
         FactorLo=sigfig(k.ratio_z.iloc[-1] / k.ratio_z[0]), FactorHi=sigfig(k.ratio_z_apogee.iloc[-1] / k.ratio_z[0]))

# The observed analogues (results/sims_observed.csv: one row per analogue and bin; bins of fewer than NMIN_SIM mock giants not used)
s = pd.read_csv(R / 'sims_observed.csv', dtype={'name': str})
ok = s[s.n >= NMIN_SIM].copy()
# ratio: measured v_rot(4 kpc) / sigma_z of each analogue's bin
ok['ratio'] = ok.vrot4 / ok.sigma_z
# recovery errors: dv [km/s], fractional sigma_z error, and ratio error, against the particle truths t_vphi, t_sigma_z
dv, dsz, dr = ok.vrot4 - ok.t_vphi, ok.sigma_z / ok.t_sigma_z - 1, ok.ratio - ok.t_vphi / ok.t_sigma_z
mp = ok.population == 0  # the most metal-poor bin (population 0)
# numbers of bins and analogues; median, 16th and 84th percentiles of the recovery errors (all bins, and the metal-poor bin)
M.update(NBins=num(len(ok)), NAll=num(s.groupby(['suite', 'name']).ngroups), NMockMP=num(round(s.n[s.population == 0].median())),
         RecDv=signed(dv.median(), 1), RecDvLo=signed(dv.quantile(.16), 1), RecDvHi=signed(dv.quantile(.84), 1),
         RecSig=f(100 * dsz.median(), 1), RecMPLo=signed(dv[mp].quantile(.16), 0), RecMPHi=signed(dv[mp].quantile(.84), 0),
         RecSzMPLo=signed(100 * dsz[mp].quantile(.16), 0), RecSzMPHi=signed(100 * dsz[mp].quantile(.84), 0),
         RecRatioMP=signed(dr[mp].median()), RecRatioMPLo=signed(dr[mp].quantile(.16)), RecRatioMPHi=signed(dr[mp].quantile(.84)))
# formed mass [Msun] of each analogue (within the same radius)
mass = pd.read_csv(R / 'sims_formed_mass.csv', dtype={'name': str}).m_formed
M.update(MSim=sci(mass.median()), MassRatio=f(mass.median() / m_lmc, 1),
         AnisoSim=f((ok.sigma_R / ok.sigma_z)[mp].median()), ShapeLo=f((ok.sigma_z / ok.sigma_R)[mp].quantile(.16)),
         ShapeHi=f((ok.sigma_z / ok.sigma_R)[mp].quantile(.84)))
```

#### Particle masses and formed mass
Dark-matter particle masses from the snapshot headers, and the formed mass of the Auriga stars against the conversion used for
TNG50 and FIRE-2 (needs the simulation files).

```python
# Dark-matter particle masses [Msun], from the snapshot headers: Auriga, the high-resolution type 1 (MassTable); FIRE-2, the type-1
# Masses (all equal); TNG50, a full progenitor cutout of scripts/fetch_tng50_progenitors.py (the one of the force check below)
FORCE_HALO, FORCE_SNAP = '708636', 30   # progenitor and snapshot (z = 2.3) of the force check
# the TNG50 progenitor of the force check (particles of all types at snapshot FORCE_SNAP)
prog = scr.load(FORCE_HALO, FORCE_SNAP, sims.main_branch(FORCE_HALO))
m_au = []
for d in sims.AURIGA.values():
    # first file of the z = 0 snapshot (251): MassTable[1], 1e10 Msun/h -> Msun
    with h5py.File(sorted(glob.glob(str(d / 'snapdir_251/snapshot_251.*.hdf5')))[0]) as fh:
        m_au.append(fh['Header'].attrs['MassTable'][1] * 1e10 / fh['Header'].attrs['HubbleParam'])
m_fire = []
for d in sims.FIRE.values():
    # z = 0 snapshot (600): every type-1 mass, which must all be equal
    with h5py.File(d / 'output/snapshot_600.hdf5') as fh:
        mf = np.unique(fh['PartType1/Masses'][:])
        assert len(mf) == 1, 'FIRE-2: one dark-matter particle mass'
        m_fire.append(mf[0] * 1e10 / fh['Header'].attrs['HubbleParam'])
assert np.ptp(m_fire) / np.mean(m_fire) < 1e-3, 'the text: one FIRE-2 dark-matter particle mass'
# ranges [1e4 Msun] and the factor between the TNG50 and the median Auriga/FIRE-2 dark-matter particle mass
M.update(DmAuLo=f(min(m_au) / 1e4, 1), DmAuHi=f(max(m_au) / 1e4, 1), DmFire=sci(m_fire[0]),
         DmFactor=num(round(prog['dm']['m'][0] / np.median(m_au + m_fire))))
# The formed mass that the Kroupa conversion (present mass / present_fraction, results/mass_survival_grid.csv of
# scripts/build_present_mass.py, as lmcold.sims.load for TNG50 and FIRE-2) gives for the Auriga stars, against their GFM_InitialMass
grid = pd.read_csv(R / 'mass_survival_grid.csv')
au_below = []
for name in sims.AURIGA:
    # stars of each Auriga analogue, with present mass m and GFM_InitialMass (m_formed)
    st = sims.load('Auriga', name, grid)
    # per cent by which the Kroupa conversion falls below the true initial mass
    au_below.append(100 * (1 - (st['m'] / sims.on_grid(grid, 'present_fraction', st['feh'], st['age'])).sum() / st['m_formed'].sum()))
M.update(AuFormedLo=num(round(min(au_below))), AuFormedHi=num(round(max(au_below))))
```

#### Rotation and dispersion against metallicity
Medians and ranges over the analogues per bin, and the values at the LMC's formed mass.

```python
# Rotation and dispersion against metallicity (results/sims_kinematics_summary.csv, written by paper_sims_kinematics.py)
# per suite, bin and quantity: percentiles over the analogues and counts above the LMC band
sk = pd.read_csv(R / 'sims_kinematics_summary.csv').set_index(['suite', 'population', 'quantity'])
# the LMC band of v_rot(4 kpc) per bin (paper_sims_kinematics.lmc_band): (lo, hi) arrays
lo, hi = lmc_band(k, 'vrot4')
# Auriga analogues in the most metal-rich bin (population 4) that reach the LMC band
au_mr = ok[(ok.suite == 'Auriga') & (ok.population == 4)].vrot4
fast = np.sort(au_mr[au_mr >= lo[4]].to_numpy())
# LMC sigma_z band per bin; Auriga and FIRE-2 analogues must all lie above it
lo_sz, hi_sz = lmc_band(k, 'sigma_z')
hot = ok[ok.suite.isin(['Auriga', 'FIRE-2'])]
assert (hot.sigma_z.to_numpy() > hi_sz[hot.population.to_numpy()]).all(), 'the text: all Auriga and FIRE-2 analogues are hotter'
M.update(VrotMRLo=num(round(sk.loc[('TNG50', 4, 'vrot4'), 'p16'])), VrotMRHi=num(round(sk.loc[('TNG50', 4, 'vrot4'), 'p84'])),
         NAuFast={2: 'two'}.get(len(fast), num(len(fast))), AuFast=' and '.join(num(round(x)) for x in fast))
# per suite, metal-poor bin: v_rot range (TNG50: 16th-84th percentiles; Auriga, FIRE-2: min-max), median sigma_z and ratio
for tag, suite in (('Tng', 'TNG50'), ('Au', 'Auriga'), ('Fire', 'FIRE-2')):
    g = ok[mp & (ok.suite == suite)]
    M.update({f'{tag}VrotMPLo': num(round(g.vrot4.quantile(.16) if tag == 'Tng' else g.vrot4.min())),
              f'{tag}VrotMPHi': num(round(g.vrot4.quantile(.84) if tag == 'Tng' else g.vrot4.max())),
              f'{tag}SzMP': num(round(g.sigma_z.median())), f'{tag}RatioMP': f(g.ratio.median())})
g = ok[mp & (ok.suite == 'TNG50')]
# TNG50 metal-poor medians and percentiles; coldest sigma_z and largest ratios over all analogues; bins above ratio 1
M.update(TngVrotMP=num(round(g.vrot4.median())), TngSzMPLo=num(round(g.sigma_z.quantile(.16))),
         TngSzMPHi=num(round(g.sigma_z.quantile(.84))), TngRatioMPLo=f(g.ratio.quantile(.16)), TngRatioMPHi=f(g.ratio.quantile(.84)),
         ColdSz=num(round(ok[mp].sigma_z.min())), RatioMaxMP=f(ok[mp].ratio.max()), RatioMaxMPOne=f(ok[mp].ratio.max(), 1),
         PartMaxMP=f((ok.t_vphi / ok.t_sigma_z)[mp].max()),
         NAboveOne=num((ok[ok.population == 1].ratio > 1).sum()), RatioMaxB=f(ok[ok.population == 1].ratio.max()))
# At the LMC's own formed mass below the second bin: each TNG50 analogue's values joined between its bins linearly in log mass (as
# paper_sims_kinematics.counts_at_mass), read there; median over the analogues whose bins span it.
# formed mass of each analogue, indexed by (suite, name)
X = pd.Series(mass.to_numpy(), index=pd.read_csv(R / 'sims_formed_mass.csv', dtype={'name': str}).set_index(['suite', 'name']).index)
at = []
for (suite, name), d in ok[ok.suite == 'TNG50'].groupby(['suite', 'name']):
    d = d.sort_values('population')
    # log10 formed mass below each bin of this analogue [Msun]
    x = np.log10(frac[d.population] * X[suite, name])
    # analogues whose bins span the LMC's mass below the second bin: values interpolated there
    if x.min() <= np.log10(frac[1] * m_lmc) <= x.max():
        at.append([np.interp(np.log10(frac[1] * m_lmc), x, d[q]) for q in ('vrot4', 'sigma_z', 'ratio')])
at = np.median(at, axis=0)
M.update(TngVrotAtB=num(round(at[0])), TngSzAtB=num(round(at[1])), TngRatioAtB=f(at[2]))
# counts of analogues covering, and above, the LMC's mass in each bin
mc = pd.read_csv(R / 'sims_mass_counts.csv')
M.update(NCoverA=num(mc.n_covering[0]), NAboveA=num(mc.n_above_lmc[0]))
for j, tag in zip((2, 3, 4), 'CDE'):
    M.update({f'NMass{tag}': num(mc.n_above_lmc[j]), f'NCov{tag}': num(mc.n_covering[j]),
              f'NBin{tag}': num(sum(sk.loc[(su, j, 'ratio_z'), 'n_above_lmc'] for su in ('TNG50', 'Auriga', 'FIRE-2')))})
# other population definitions (by rank in age, native, oldest): ratios and age ranks
pp = pd.read_csv(R / 'sims_populations.csv')
M.update(RankAge=f(pp.rank_age.median(), 1), OldestInRank=num(round(100 * pp.oldest_in_rank.median())),
         PartRank=f(pp.ratio_rank.median()), PartNative=f(pp.ratio_native.median()), PartOldest=f(pp.ratio_oldest.median()),
         PartRankMax=f(pp.ratio_rank.max()), PartNativeMax=f(pp.ratio_native.max()), PartOldestMax=f(pp.ratio_oldest.max()))
# alternative definitions (columns z, R, RMS and their particle versions; rows 1 and 2): counts of analogues
de = pd.read_csv(R / 'sims_definitions.csv')
six = ['z', 'z particles', 'R', 'R particles', 'RMS', 'RMS particles']
M.update(DefAny=num(de['any'][1]), DefLo=num(de.loc[1, six].min()), DefHi=num(de.loc[1, six].max()),
         DefCLo=num(de.loc[2, six].min()), DefCHi=num(de.loc[2, six].max()))
```

#### Birth, heating and scrambling
Cuts repeated from the birth and heating scripts, and the statistics of what turns the orbits.

```python
# Birth and heating (Figs sims_birth, sims_heating): cuts repeated from the scripts named
OLD_AGE = (9, 12)        # Gyr, 'the stars now 9-12 Gyr old' (paper_text_sims_birth.py)
YOUNG_AGE = 1.5          # Gyr, young-star check (paper_text_sims_birth.py)
AGE_CUTS = (10, 8, 4)    # Gyr, cohorts born more than 10, 8-10 and less than 4 Gyr ago (paper_sims_heating.py, panel b)
LATER = (2, 3)           # Gyr after birth of the later values of the oldest and youngest cohorts (paper_text_sims_birth.py)
COH_BIN = 0.1            # Gyr, time bins of panel c (paper_sims_heating.py)
COH_END = 0.5            # Gyr, cohorts followed up to 0.5 Gyr (build_sims_heating.py)
# gas and heating per snapshot of the TNG50 progenitors
gas = pd.read_csv(R / 'sims_heating_gas.csv', dtype={'halo': str})
dex = sbirth['DEX']  # [Fe/H] edges of the birth bins (paper_text_sims_birth.DEX) [dex]
M.update(OldAgeLo=num(OLD_AGE[0]), OldAgeHi=num(OLD_AGE[1]), YoungAge=num(YOUNG_AGE), AgeOld=num(AGE_CUTS[0]), AgeMid=num(AGE_CUTS[1]),
         AgeYoung=num(AGE_CUTS[2]), LaterOld=num(LATER[0]), LaterYoung=num(LATER[1]), CohBin=num(COH_BIN), CohEnd=num(COH_END),
         WindowHeat=num(1000 * heat['WINDOW']), NGasSnaps=num(len(GAS_SNAPS)), GasZFirst=num(round(gas.groupby('snap').z.median().iloc[0])),
         DexA=num(dex[0]), DexB=num(dex[1]), DexC=num(dex[-2]), DexD=num(dex[-1]))
# Auriga and FIRE-2 (macros_sims_suites.tex): cuts of paper_text_sims_suites.py
Z_EARLY = 1.5        # cohorts born at z > 1.5 (lookback > 9.4 Gyr there)
FIRE_LATE_LB = 9     # Gyr, FIRE-2 cohorts born less than 9 Gyr ago
M.update(BirthLbLo=num(suites['BIRTH_LB'][0]), BirthLbHi=num(suites['BIRTH_LB'][1]), ZEarly=num(Z_EARLY), FireLateLb=num(FIRE_LATE_LB))

# What turns the orbits (results/sims_scramble.csv; the statistics of paper_sims_scramble.py)
# scrambling experiments: coherence C per cohort, variant (which forces kept) and time t after birth [Gyr]
a = pd.read_csv(R / 'sims_scramble.csv', dtype={'halo': str})
# variant A at t > 0: simulated against measured coherence
pa = a[(a.variant == 'A') & (a.t > 0)]
d = pa.C_sim - pa.C_meas
# median cohort size; median and 16th-84th percentiles of C_sim - C_meas; correlation of C_sim with C_meas
M.update(ScrN=num(round(a.groupby(['halo', 's_birth']).n0.first().median())), ScrDC=signed(d.median(), 3),
         ScrDCLo=signed(d.quantile(.16)), ScrDCHi=signed(d.quantile(.84)), ScrR=f(np.corrcoef(pa.C_sim, pa.C_meas)[0, 1]))
# add the measured coherence as variant 'real'; C0: each cohort's coherence at birth (t = 0); loss = C0 - C
u = pd.concat([a, a[a.variant == 'A'].assign(variant='real', C_sim=lambda t: t.C_meas)])
u = u.join(u[u.t == 0].set_index(['variant', 'halo', 's_birth']).C_sim.rename('C0'), on=['variant', 'halo', 's_birth'])
u['loss'] = u.C0 - u.C_sim
# losses at 0.2 < t <= 0.35 Gyr, summed per galaxy and variant
W = u[(u.t > 0.2) & (u.t <= 0.35)].dropna(subset=['loss'])
S = W.groupby(['variant', 'halo']).loss.sum().unstack(0)
gal = S.index.values
# 1000 bootstrap resamples of the galaxies: each variant's summed loss over that of 'real', with 16th-84th percentiles
rng = np.random.default_rng(1)   # as paper_sims_scramble.py: 1000 bootstrap resamples of the galaxies
boot = [S.loc[gal[rng.integers(0, len(gal), len(gal))]].sum() for _ in range(1000)]
for v, tag in (('E', 'E'), ('D0', 'DZero'), ('B', 'B'), ('G_DM', 'GDM'), ('G_bar', 'GBar'), ('C', 'C'), ('H2', 'HTwo'),
               ('H1', 'HOne'), ('D0_fr', 'DZeroFr')):
    fr = np.array([b[v] / b['real'] for b in boot])
    M.update({f'Scr{tag}': f(S[v].sum() / S.real.sum()), f'Scr{tag}Lo': f(np.percentile(fr, 16)), f'Scr{tag}Hi': f(np.percentile(fr, 84))})
# radius of the halo-axes measurement: the default rmax of sims_scramble.halo_axes, read from its source
M.update(HaloR=num(float(re.search(r'def halo_axes\(x, rmax=([0-9.]+)', (ROOT / 'scripts/sims_scramble.py').read_text()).group(1))))
MIS = 45   # deg, the misalignment bins of panel b are 0-15-30-45-90 (paper_sims_scramble.py)
# mean loss per variant in misalignment bins [deg]; al: bins aligned to within MIS
mb = pd.cut(W.misalign, [0, 15, 30, 45, 90])
mean = W.groupby(['variant', mb], observed=True).loss.mean()
al = [iv for iv in mean.loc['real'].index if iv.right <= MIS]
# loss of E_split and H1 in the aligned bins, and in the first bin beyond MIS
M.update(Mis=num(MIS), AlELo=f(mean.loc['E_split'][al].min()), AlEHi=f(mean.loc['E_split'][al].max()),
         AlHOneLo=f(mean.loc['H1'][al].min()), AlHOneHi=f(mean.loc['H1'][al].max()),
         MisHOne=f(mean.loc['H1'][[iv for iv in mean.loc['H1'].index if iv.left >= MIS]].iloc[0]),
         MisE=f(mean.loc['E_split'][[iv for iv in mean.loc['E_split'].index if iv.left >= MIS]].iloc[0]))
L = pd.read_csv(R / 'sims_halo_cohorts.csv', dtype={'halo': str})
coh = pd.read_csv(R / 'sims_heating_coherence.csv', dtype={'halo': str})
# coherence of randomly oriented orbits: 1 / sqrt(median cohort size)
C_RANDOM = 1 / np.sqrt(coh[coh.set == 'full'].groupby(['halo', 's_birth']).n0.first().median())
z_late = []   # redshifts of the birth snapshots of the late control cohorts (set 'track'), from the cutout headers
for s_ in sorted(coh[coh.set == 'track'].s_birth.unique()):
    # scale factor a of each birth snapshot -> redshift z = 1/a - 1
    with h5py.File(sims.TNG_TRACK_DIR / coh.halo.iloc[0] / f'snap_{s_:03d}.hdf5') as fh:
        z_late.append(1 / fh['Header'].attrs['Time'] - 1)
M.update(LateZHi=f(max(z_late)))
# frac: fraction of the available coherence lost, (C0 - C1) / (C0 - C_RANDOM)
L['frac'] = L.loss / (L.C0 - C_RANDOM)
eb = L.groupby(pd.cut(L.eps2, [0, 0.003, 0.006, 0.01, 0.015, 0.1]), observed=True)   # bins of panel c (paper_sims_scramble.py)
# per eps2 bin: median eps2, median fraction lost, median lookback time [Gyr]
c = pd.DataFrame({'x': eb.eps2.median(), 'm': eb.frac.median(), 'lb': eb.lookback.median()}).reset_index(drop=True)
rk = pd.DataFrame({q: L[q].rank() for q in ('eps2', 'frac', 'z')}).corr()   # Spearman partial correlation at fixed redshift
dm = L.groupby('halo')[['eps2', 'frac']].transform(lambda t: t - t.mean())     # within galaxies: each galaxy's means subtracted
# cohort counts and redshift range; C_RANDOM; rank correlation of eps2 and frac: all, at fixed z (partial), within galaxies
M.update(NCohHalo=num(len(L)), ZCohHi=num(round(L.z.max())), ZCohLo=f(L.z.min(), 1), StepLo=f(L.step.min(), 1), StepHi=f(L.step.max(), 1),
         CRandom=f(C_RANDOM, 3),
         RankAll=f(spearmanr(L.eps2, L.frac)[0]),
         RankZ=f((rk.eps2.frac - rk.eps2.z * rk.frac.z) / np.sqrt((1 - rk.eps2.z ** 2) * (1 - rk.frac.z ** 2))),
         RankGal=f(spearmanr(dm.eps2, dm.frac)[0]))
# one set of macros per eps2 bin
for i, tag in enumerate(('One', 'Two', 'Three', 'Four', 'Five')):
    M.update({f'EpsBin{tag}': sigfig(c.x[i], 2), f'FracBin{tag}': f(c.m[i]), f'LbBin{tag}': f(c.lb[i], 1)})
# fraction lost in bins two and three, in per cent rounded to tens
M.update(PctBinTwo=num(10 * round(10 * c.m[1])), PctBinThree=num(10 * round(10 * c.m[2])))
```

#### Force check of the multipole expansion
The expansion's force against a softened direct sum on a TNG50 progenitor (needs the TNG50 progenitor cutouts).

```python
# The force of the expansion (as sims_scramble.py: all particles within RMAX, lmax LMAX, no softening) against a direct sum with the
# softening of TNG50 (cubic-spline kernel, Newtonian beyond 2.8 eps; eps = EPS_TNG comoving kpc/h at z > 1, Pillepich et al. 2019,
# Table 1), at FORCE_N random star positions within FORCE_R kpc of the centre of the progenitor read above
EPS_TNG, FORCE_N, FORCE_R = 0.39, 300, (0.3, 0.6)  # comoving kpc/h; stars; inner and outer radius [kpc]
# positions and masses of all particles (dark matter, gas, stars) in the progenitor's frame
xp, mp_ = scr.particles(prog, ('dm', 'gas', 'st'), np.eye(3))
# FORCE_N random stars between FORCE_R[0] and FORCE_R[1] kpc (fixed seed 0)
r_st = np.linalg.norm(prog['st']['x'], axis=1)
pts = prog['st']['x'][np.random.default_rng(0).choice(np.flatnonzero((r_st > FORCE_R[0]) & (r_st < FORCE_R[1])), FORCE_N, replace=False)]
# the expansion's force at those points (AGAMA, configuration written to a temporary file)
with tempfile.TemporaryDirectory() as tmp:
    f_exp = agama.Potential(scr.expansion(xp, mp_, f'{tmp}/force.ini')).force(pts)
# kernel radius 2.8 eps in physical kpc (comoving kpc/h x a / h)
h = 2.8 * EPS_TNG * prog['a'] / sims.TNG_H
# direct sum over the particles in blocks of 20000 (memory)
f_sim = np.zeros_like(pts)
for j in range(0, len(xp), 20000):
    dx = xp[j:j + 20000][None] - pts[:, None]
    rr = np.linalg.norm(dx, axis=2)
    u = rr / h
    # cubic-spline kernel: force factor g(r) per unit G m dx [kpc^-3], Newtonian r^-3 beyond h
    g = np.where(u < 0.5, (32 / 3 + u ** 2 * (32 * u - 38.4)) / h ** 3, np.where(
        u < 1, (64 / 3 - 48 * u + 38.4 * u ** 2 - 32 / 3 * u ** 3 - 1 / (15 * np.maximum(u, 1e-9) ** 3)) / h ** 3, np.maximum(rr, h) ** -3.))
    f_sim += G * (mp_[j:j + 20000][None, :, None] * dx * g[..., None]).sum(1)
# force radii and number, redshift, and median per-cent difference of the expansion's force from the direct sum
M.update(ForceRLo=num(FORCE_R[0]), ForceR=num(FORCE_R[1]), ForceN=num(FORCE_N), ForceZ=f(1 / prog['a'] - 1, 1),
         ForcePct=num(round(100 * (np.median(np.linalg.norm(f_exp, axis=1) / np.linalg.norm(f_sim, axis=1)) - 1))))
# Auriga scrambling table: redshift range and number of cohorts
au = pd.read_csv(R / 'auriga_scramble.csv')
M.update(AuZHi=f(au.z.max(), 1), AuZLo=f(au.z.min(), 2), AuN=num(au.groupby(['halo', 's_birth']).ngroups))
```

#### Auriga scrambling and the settling halo
The Auriga version of Fig. 4(a) and how the halo settles.

```python
# The Auriga version of the Letter's Fig. 4(a) (results/auriga_scramble.csv): v_phi/sigma_z about the birth spin in the TIME_EDGES bins
# of scripts/paper_letter_scramble.py, per galaxy the median, then the median over the galaxies; the bins' median times.
te = const('paper_letter_scramble')['TIME_EDGES']
# v_phi/sigma_z: measured for 'real', simulated for the variants
au['vs'] = np.where(au.variant == 'real', au.vs_meas, au.vs_sim)
# time bins (TIME_EDGES of paper_letter_scramble.py) [Gyr after birth]
au['tb'] = pd.cut(au.t, np.r_[-1e-9, te[1:-1], te[-1] + 1e-6])
# per galaxy the median in each bin, then the median over the galaxies; tm: median time of each bin
va = au.groupby(['variant', 'tb', 'halo'], observed=True).vs.median().groupby(level=[0, 1], observed=True).median()
tm = au.groupby('tb', observed=True).t.median()
# values per variant and bin; times of bins two to four
for v, tag in (('real', 'Real'), ('E_split', 'E'), ('G_DM_fr', 'GDM'), ('D0_fr', 'DZero')):
    M.update({f'AuVs{tag}{i}': f(x) for i, x in zip(('One', 'Two', 'Three', 'Four'), va.loc[v].to_numpy())})
M.update({f'AuT{i}': f(x) for i, x in zip(('Two', 'Three', 'Four'), tm.to_numpy()[1:])})

# How the halo settles (results/sims_halo_settle.csv; per galaxy the median in 0.5-Gyr lookback bins, then over the galaxies, as
# paper_sims_settle.py, bins with at least MIN_GAL galaxies)
MIN_GAL = 20
LB_EARLY_END, LB_MID = 11.5, 10.0   # Gyr: psi is quoted up to 11.5 Gyr ago and at 10 Gyr ago
# halo settling table (per progenitor and snapshot); 0.5-Gyr lookback bins
T = pd.read_csv(R / 'sims_halo_settle.csv', dtype={'halo': str})
T['tb'] = (T.lookback * 2).round() / 2
# per bin: each galaxy's medians, then median and 16th/84th percentiles over galaxies, in bins with >= MIN_GAL galaxies
gb = T.groupby(['tb', 'halo']).median(numeric_only=True).groupby(level=0)
n = T.groupby('tb').halo.nunique()
med, p16, p84 = (x[n >= MIN_GAL] for x in (gb.median(), gb.quantile(.16), gb.quantile(.84)))
# e, l_: earliest and latest lookback bins [Gyr]; pe: misalignment medians before LB_EARLY_END
e, l_ = med.index.max(), med.index.min()
pe = med.misalign[med.index >= LB_EARLY_END]
# psi early and late [rad]; fs, fc: expected change of eps_tilt from sin 2 psi and from 1 - c/a
psi_e, psi_l = np.deg2rad(med.misalign[e]), np.deg2rad(med.misalign[l_])
fs, fc = abs(np.sin(2 * psi_l) / np.sin(2 * psi_e)), (1 - med.c_a[l_]) / (1 - med.c_a[e])
# eps_tilt from early to late; lookback at which it crosses EPS_QUIET
tilt = med.eps_tilt.sort_index(ascending=False)
below = np.flatnonzero(tilt.to_numpy() < settle['EPS_QUIET'])[0]   # first bin below the line; crossing interpolated linearly
t_cross = np.interp(settle['EPS_QUIET'], tilt.to_numpy()[below - 1:below + 1][::-1], tilt.index[below - 1:below + 1][::-1])
# counts per galaxy; psi, c/a, b/a, eps2 and its tilt and elongation parts early and late; predicted against measured change;
# rank correlations of eps_tilt with (1 - c/a)|sin 2 psi| and of eps_elong with 1 - b/a; lookback of the crossings
M.update(NPerGalLo=num(T.groupby('halo').size().min()), NPerGalHi=num(T.groupby('halo').size().max()), MinGal=num(MIN_GAL),
         PsiEarlyLo=num(round(pe.min())), PsiEarlyHi=num(round(pe.max())), LbEarlyEnd=num(LB_EARLY_END), LbMid=num(LB_MID),
         PsiMid=num(round(med.misalign[LB_MID])), PsiLate=num(round(med.misalign[l_])),
         PsiEarlyLoP=num(round(p16.misalign[e])), PsiEarlyHiP=num(round(p84.misalign[e])),
         PsiLateLoP=num(round(p16.misalign[l_])), PsiLateHiP=num(round(p84.misalign[l_])), Span=num(e - l_),
         CaEarly=f(med.c_a[e]), CaLate=f(med.c_a[l_]), BaEarly=f(med.b_a[e]), BaLate=f(med.b_a[l_]),
         EpsFactor={3: 'three'}.get(round(med.eps2[e] / med.eps2[l_]), num(round(med.eps2[e] / med.eps2[l_]))),
         EpsEarly=sigfig(med.eps2[e], 2), EpsLate=sigfig(med.eps2[l_]), TiltEarly=sigfig(med.eps_tilt[e], 2),
         TiltLate=sigfig(med.eps_tilt[l_], 2), ElongEarly=sigfig(med.eps_elong[e], 2), ElongLate=sigfig(med.eps_elong[l_]),
         TiltEarlyThree=f(med.eps_tilt[e], 4), TiltLateFour=f(med.eps_tilt[l_], 4),
         TwoPsiEarly=num(round(2 * med.misalign[e])), TwoPsiLate=num(round(2 * med.misalign[l_])),
         FacSin=f(fs), FacCa=f(fc), FacPred=f(fs * fc), FacMeas=f(med.eps_tilt[l_] / med.eps_tilt[e]),
         RankTilt=f(spearmanr(T.eps_tilt, (1 - T.c_a) * np.abs(np.sin(2 * np.deg2rad(T.misalign))))[0]),
         RankElong=f(spearmanr(T.eps_elong, 1 - T.b_a)[0]), SinSq=sigfig((np.sin(psi_l) / np.sin(psi_e)) ** 2),
         TiltFactorPct=num(round(med.eps_tilt[e] / med.eps_tilt[l_])), ElongFactor=f(med.eps_elong[e] / med.eps_elong[l_], 1),
         LbTiltQuiet=num(round(t_cross)), EpsQuiet=num(settle['EPS_QUIET']),
         LbAlign=num(med.misalign.sort_index(ascending=False).lt(QUIET['misalign'][1]).idxmax()))
q = QUIET
other = []
# median fraction lost by cohorts born on the quiet side of each QUIET threshold, and by the others
for key, (a0, a1) in q.items():
    kq = (L[key] > a0) & (L[key] <= a1)
    M[{'misalign': 'QuietPsi', 'c_a': 'QuietCa', 'b_a': 'QuietBa', 'eps2': 'QuietEps'}[key]] = f(L.frac[kq].median())
    other.append(L.frac[~kq].median())
RULE = 0.1   # the threshold rule: the edge of the first bin of cohorts whose median loss is below 10 per cent
# the first eps2 bin (below 0.003) must lose less than RULE and the second at least RULE
assert c.m[0] < RULE <= c.m[1], 'the eps2 threshold no longer follows the rule'
M.update(OtherLo=f(min(other)), OtherHi=f(max(other)), Rule=num(100 * RULE), PsiQuiet=num(q['misalign'][1]), CaQuiet=num(q['c_a'][0]),
         BaQuiet=num(q['b_a'][0]))
```

#### SIDM Concerto
The radius where the SIDM halo is round, and the host's virial mass from the Rockstar catalogue.

```python
# A round halo: SIDM Concerto (results/sims_sidm_shapes.csv; shells with at least NMIN particles, as paper_sims_sidm.py)
NMIN = const('paper_sims_sidm')['NMIN']  # fewest particles per shell (paper_sims_sidm.py)
R_CDM = 5.0      # kpc, the CDM centre quoted
ROUND = 0.85     # 'round': every shell out to that major-axis radius has c/a above ROUND
sh = pd.read_csv(R / 'sims_sidm_shapes.csv')
# shells with enough particles; eps2 at radius R_EPS per run and redshift
shells = sh[sh.N >= NMIN]
e2 = sh[np.isclose(sh.r, sidm['R_EPS'])].set_index(['run', 'z']).eps2
# redshifts of the snapshots, earliest first (z4 > z3 > z2 ...)
zs = sorted(sh.z.unique(), reverse=True)
z4, z3, z2 = zs[0], zs[1], zs[2]


def round_radius(run, z):
    """(radius [kpc] out to which every shell has c/a > ROUND, min c/a, max c/a there) for one run and redshift; 0 if none."""
    w = shells[(shells.run == run) & np.isclose(shells.z, z)].sort_values('r')
    # shells inside the first one with c/a <= ROUND
    bad = np.flatnonzero(w.s.to_numpy() <= ROUND)
    w = w.iloc[:bad[0] if len(bad) else len(w)]
    return (w.r.max(), w.s.min(), w.s.max()) if len(w) else (0, np.nan, np.nan)


# round radius of SIDM from the third snapshot on; CDM / SIDM eps2 ratio (z3 excluded)
late = [round_radius('SIDM', z) for z in zs[2:]]
ratio = (e2['CDM'] / e2['SIDM']).drop(z3)
# TNG50 median eps2 per 0.5 redshift bin, and the CDM run's eps2 over it at matching redshifts
tng = T.assign(zb=(T.z / 0.5).round() * 0.5).groupby(['zb', 'halo']).eps2.median().groupby(level=0).median()
cdm_tng = [e2['CDM'][z] / tng[round(z * 2) / 2] for z in zs if round(z * 2) / 2 in tng.index]
# SIDM shells at z3; CDM shells within R_CDM
w3 = shells[(shells.run == 'SIDM') & np.isclose(shells.z, z3)]
cdm = shells[(shells.run == 'CDM') & (shells.r <= R_CDM)]
# CDM shape and eps2 range; SIDM round radius and c/a at z4 and later; eps2 at each redshift; CDM-to-TNG50 eps2 ratio
M.update(SiRCdm=num(R_CDM), SiCaCdmLo=f(cdm.s.min()), SiCaCdmHi=f(cdm.s.max()), SiCdmEpsLo=sigfig(e2['CDM'].min(), 2),
         SiCdmEpsHi=sigfig(e2['CDM'].max(), 2), SiRound=f(ROUND), SiRoundZFour=num(round(round_radius('SIDM', z4)[0])),
         SiCaZFourLo=f(round_radius('SIDM', z4)[1]), SiCaZFourHi=f(round_radius('SIDM', z4)[2]),
         SiRoundLo=num(round(min(r[0] for r in late))), SiRoundHi=num(round(max(r[0] for r in late))),
         SiCaRoundLo=f(min(r[1] for r in late)), SiCaRoundHi=f(max(r[2] for r in late)),
         SiRatioLo=num(round(ratio.min())), SiRatioHi=num(5 * round(ratio.max() / 5)),
         SiEpsZFour=f(e2['SIDM'][z4], 4), SiEpsZTwo=f(e2['SIDM'][z2], 4), SiEpsLate=f(e2['SIDM'][zs[3]], 4),
         SiCaZThree=sigfig(np.interp(sidm['R_EPS'], w3.r, w3.s)), SiEpsZThree=f(e2['SIDM'][z3], 4),
         SiTngRatio=sigfig(max(cdm_tng)))
assert np.isclose(e2['SIDM'][zs[3]], e2['SIDM'][zs[4]], atol=5e-5), 'the text: the same SIDM eps2 at z = 1.5 and today'
assert round_radius('SIDM', z3)[0] == 0, 'the text: the SIDM centre is not round at z = 3'
# The host's virial mass: the Rockstar halo (sims_sidm_shapes.halos) whose Rvir is the one of the shapes table (r / r_rvir)
# Rvir of the shapes table [kpc] per redshift and run
rvir = (sh.r / sh.r_rvir).groupby([sh.z, sh.run]).median()
for z_, tag in ((z4, 'ZFour'), (zs[3], 'ZOneHalf'), (zs[4], 'ZZero')):
    mv = []
    for run, name in zip(RUNS, ('CDM', 'SIDM')):
        # snapshot and scale factor of this redshift; central halos (upid = -1) of the Rockstar catalogue
        snap, a_ = next((k_, a_) for k_, a_ in SIDM_SNAPS.items() if np.isclose(1 / a_ - 1, z_, atol=1e-4))
        t = halos(run, snap, a_).query('upid == -1')
        # the halo whose Rvir matches the shapes table to 0.01; its Mvir to one significant figure
        j = (t.Rvir - rvir[z_, name]).abs().argmin()
        assert abs(t.Rvir.iloc[j] - rvir[z_, name]) < 0.01, 'the host is found again'
        mv.append(sci(t.Mvir.iloc[j], 0))
    assert mv[0] == mv[1], 'the text: the same virial mass in both runs'
    M[f'SiMvir{tag}'] = mv[0]
```

#### Normalisation of the multipole coefficients
A check of AGAMA's coefficient normalisation on a sampled Miyamoto-Nagai disc.

```python
# Normalisation of AGAMA's multipole coefficients (KB section 'Normalisation of the multipole coefficients'), checked on the
# Miyamoto-Nagai disc P3 of lmcold.virial sampled with MN_N particles, expanded as in sims_scramble.py, at the grid radius nearest
# MN_R: Phi_00 against the spherical average of the expansion's potential, Phi_20 against sqrt(5) <Phi P_2>, and sum_m Phi_2m^2
# before and after a rotation of the particles (Euler angles MN_ROT)
MN_N, MN_R, MN_ROT = 500000, 2.0, (30, 50, 20)
# fixed seed for the sampling
agama.setRandomSeed(1)
# MN_N particles of a Miyamoto-Nagai disc (2.7e9 Msun, a = 1.6 kpc, b = 0.3 kpc: P3 of lmcold.virial)
xm, mm = agama.Density(agama.Potential(type='MiyamotoNagai', mass=2.7e9, scaleRadius=1.6, scaleHeight=0.3)).sample(MN_N)


def coefficients(x, tmp):
    """Multipole of the particles x (masses mm): the potential, the grid radii and {'l=..,m=..': Phi_lm at the radii}."""
    # the expansion's text file: after the '#Phi' line, a header of 'l=..,m=..' names and scr.GRID rows (radius, coefficients)
    lines = open(scr.expansion(x, mm, f'{tmp}/mn.ini')).read().split('\n')
    k = lines.index('#Phi') + 1
    tab = np.array([[float(v) for v in ln.split('\t')] for ln in lines[k + 1:k + 1 + scr.GRID]])
    return agama.Potential(f'{tmp}/mn.ini'), tab[:, 0], dict(zip(lines[k].split('\t')[1:], tab[:, 1:].T))


# rotation matrix of the Euler angles MN_ROT [deg]
rot = Rotation.from_euler('zyx', MN_ROT, degrees=True).as_matrix()
with tempfile.TemporaryDirectory() as tmp:
    # coefficients of the disc and of the rotated disc; i: grid radius nearest MN_R
    pot, rg, P0 = coefficients(xm, tmp)
    P1 = coefficients(xm @ rot.T, tmp)[2]
    i = np.abs(rg - MN_R).argmin()
    # Gauss-Legendre nodes in mu = cos(theta) and uniform phi: the quadrature of the mean over a sphere
    mu, w = np.polynomial.legendre.leggauss(64)
    MU, PH = np.meshgrid(mu, np.linspace(0, 2 * np.pi, 128, endpoint=False))
    wt = np.broadcast_to(w, MU.shape) / (2 * MU.shape[0])          # weights of the mean over the sphere (sum 1)
    # the expansion's potential on the sphere of radius rg[i]
    phi = pot.potential(rg[i] * np.c_[np.sqrt(1 - MU.ravel() ** 2) * np.cos(PH.ravel()), np.sqrt(1 - MU.ravel() ** 2) * np.sin(PH.ravel()),
                                      MU.ravel()]).reshape(MU.shape)
    # largest relative deviation of Phi_00 from <Phi> and of Phi_20 from sqrt(5) <Phi P_2>
    dev_norm = max(abs(P0['l=0,m=0'][i] / (wt * phi).sum() - 1),
                   abs(P0['l=2,m=0'][i] / (np.sqrt(5) * (wt * phi * (1.5 * MU ** 2 - 0.5)).sum()) - 1))
    # sum_m Phi_2m^2 before and after the rotation (invariant if the normalisation is right)
    q2 = [sum(P[f'l=2,m={m_}'][i] ** 2 for m_ in range(-2, 3)) for P in (P0, P1)]
M.update(MNN=sci(MN_N, 0), MNR=f(rg[i], 1), MNNorm=bound(dev_norm), MNRot=bound(abs(q2[1] / q2[0] - 1)))

# macro names must be letters only (LaTeX command names)
assert all(k_.isalpha() for k_ in M), [k_ for k_ in M if not k_.isalpha()]
```

#### Write the macros
One \newcommand per entry of M, with the prefix \sm, into results/macros_sims.tex; the text shows them as macros, so no number
is typed by hand.

```python
# header line, then one \newcommand per macro
lines = ['% Numbers of the KB chapters Simulations (scripts/paper_text_sims.py); do not edit.']
lines += [rf'\newcommand{{\sm{k_}}}{{{x}}}' for k_, x in M.items()]
(R / 'macros_sims.tex').write_text('\n'.join(lines) + '\n')
print('\n'.join(lines))
```

### Simulated stars at birth

The numbers on the simulated stars at birth and soon after (`results/macros_sims_birth.tex`, prefixes `\sb` and `\sh`).

<details><summary>About <code>scripts/paper_text_sims_birth.py</code></summary>

```text
Paper text: the numbers of Section 6 on the simulated stars at birth and soon after (the paragraphs of Figs sims_birth and
sims_heating and their captions), as LaTeX macros, so that no number there is typed by hand.

Inputs: results/sims_birth.csv (scripts/build_sims_birth.py: birth cohorts of COHORT Gyr), results/sims_heating_history.csv,
sims_heating_gas.csv, sims_heating_coherence.csv (scripts/paper_sims_heating.py), results/kin_support_ellipsoid.csv (LMC giant bins,
stage-1 ellipsoid fits), results/giant_present_mass.csv.
Definitions:
- 'old stars': cohorts aged 9-12 Gyr today; per analogue the median over those cohorts of v_phi, sigma_z and v_phi/sigma_z (each
  cohort's own ratio), at birth and today; then the median and 16th/84th percentiles over the analogues (TNG50) or the median over the
  five Auriga analogues. 'Reach' counts the analogues with v_phi/sigma_z >= the LMC's [Fe/H] < -1.5 bin.
- TNG50 at birth at the LMC bins' ages: the median over analogues of each cohort's v_phi/sigma_z at birth, against age, interpolated
  at the formed-mass-weighted mean age of the three metal-poor giant bins (paper_sims_birth.lmc_ages); the script stops unless each
  LMC bin lies between the TNG50 16th percentile and median there (the text's wording).
- Heating: sigma_z^2 today minus at birth; the LMC's from the mean sigma_z^2 of its two most metal-poor bins; rounded to 100.
- Young stars: TNG50 cohorts younger than 1.5 Gyr, median over all cohorts; Auriga: the range over its analogues of each analogue's
  median sigma_z at birth.
- Fig. sims_heating (a): galaxy-weighted medians (paper_sims_heating.gal_median) in 0.5-dex bins of the progenitor's stellar mass,
  10^7-10^7.5 and 10^9-10^9.5 Msun, and the range over the 0.5-dex bins 10^7-10^9.5 for sigma_z; the gas likewise, its effective
  sound speed the median over the six snapshots, its v_phi/sigma_z at the first and last snapshot; the LMC's two most metal-poor
  bins: v_rot(4 kpc) and the present mass in stars more metal-poor than each bin's median [Fe/H]; TNG50 stars at that mass: the
  range of the 0.25-dex bins whose centres lie in it.
- Fig. sims_heating (b): the median time from birth to the first tracked snapshot; v_phi/sigma_z at birth, at the first step (0.5)
  and later, galaxy-weighted, as panel b. (c): C at birth and at 0.5 Gyr for z > 1.5, the late cohorts at birth, the median cohort
  size and 1/sqrt of it.
Writes results/macros_sims_birth.tex (copied to the paper's tables/; \\input in the preamble).
Run: .venv/bin/python scripts/manifest.py run paper_text_sims_birth.py
```
</details>

**Data read by `scripts/paper_text_sims_birth.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/giant_present_mass.csv` | repository | written by `python scripts/build_present_mass.py` |
| `results/kin_support_ellipsoid.csv` | repository | written by `python scripts/paper_kin_support_ellipsoid.py` |
| `results/sims_birth.csv` | repository | written by `python scripts/build_sims_birth.py` |
| `results/sims_heating_coherence.csv` | repository | written by `python scripts/build_sims_heating.py` |
| `results/sims_heating_gas.csv` | repository | written by `python scripts/build_sims_heating.py` |
| `results/sims_heating_history.csv` | repository | written by `python scripts/build_sims_heating.py` |

*Shown, not run here: the simulation result tables in results/. The cells below are the script itself; run it from the repository with* `python scripts/paper_text_sims_birth.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings
Imports and the settings read from other scripts.

```python
import numpy as np
import pandas as pd

from lmcold.sources import ROOT
from build_sims_birth import COHORT
from paper_sims_birth import lmc_ages
from paper_sims_heating import gal_median
```

#### Load the data
The birth cohorts of TNG50 and Auriga and the LMC giant bins; the tables of Fig. sims_heating are read in their own step
below.

```python
# One row per cohort (suite, halo, age [Gyr]); v_phi, sigma_z [km/s] at birth and of the same stars today; their ratios vs_*.
t = pd.read_csv(ROOT / 'results/sims_birth.csv', dtype={'halo': str}).dropna(subset=['vphi_birth'])
t['vs_birth'], t['vs_same'] = t.vphi_birth / t.sz_birth, t.vphi_same / t.sz_same
# The five LMC giant bins, metal-poor first (ratio_z = v_rot/sigma_z, sigma_z and vrot4 [km/s], feh_med [dex]).
k = pd.read_csv(ROOT / 'results/kin_support_ellipsoid.csv')
k = k[k.population.str.startswith('giants')].reset_index(drop=True)
```

#### Macro store
M collects the macros; put() writes a value with d decimals.

```python
M = {}   # macro name (without backslash): text


def put(name, value, d=0):
    """Store value as macro name, with d decimals."""
    M[name] = f'{value:.{d}f}'   # fixed-point text with d decimals
```

#### Old stars at birth and today
Cohorts aged 9-12 Gyr: medians per analogue, then over the analogues; the script stops if a result contradicts the wording of
the text. Then the LMC bins against TNG50 at their ages, the heating, and the young stars.

```python
Q = ['vphi_birth', 'sz_birth', 'vs_birth', 'vphi_same', 'sz_same', 'vs_same']
# Old cohorts (9-12 Gyr): median per analogue for each suite; lmc: v_rot/sigma_z of the [Fe/H] < -1.5 bin.
old = t[(t.age > 9) & (t.age < 12)]
tng, au = (old[old.suite == s].groupby('halo')[Q].median() for s in ('TNG50', 'Auriga'))
lmc = k.ratio_z[0]
put('sbCohort', COHORT, 1)
# TNG50 medians and 16th/84th percentiles over the analogues at birth, Auriga median; analogues reaching the LMC; then today.
put('sbVphiBirth', tng.vphi_birth.median()); put('sbSzBirth', tng.sz_birth.median()); put('sbVsBirth', tng.vs_birth.median(), 2)
put('sbVsBirthLo', tng.vs_birth.quantile(.16), 2); put('sbVsBirthHi', tng.vs_birth.quantile(.84), 2)
put('sbAuVsBirth', au.vs_birth.median(), 2)
put('sbNReach', (tng.vs_birth >= lmc).sum()); put('sbNGal', len(tng)); put('sbLmcVs', lmc, 2)
put('sbVphiToday', tng.vphi_same.median()); put('sbSzToday', tng.sz_same.median()); put('sbVsToday', tng.vs_same.median(), 2)
put('sbVsTodayHi', tng.vs_same.quantile(.84), 2); put('sbAuVsToday', au.vs_same.median(), 2)
if (tng.vs_same >= lmc).any():
    raise SystemExit('the text says no analogue reaches the LMC today; some do')
# TNG50 v_phi/sigma_z at birth against cohort age: median and 16th percentile over the analogues.
g = t[t.suite == 'TNG50'].groupby('age').vs_birth
curve, low = g.median(), g.quantile(.16)
for j, (name, (age, _, _)) in enumerate(zip(('One', 'Two', 'Three'), lmc_ages()[:3])):
    # Both curves interpolated at the mean age [Gyr] of LMC bin j (lmc_ages: mean, p16, p84).
    med, lo = np.interp(age, curve.index, curve.values), np.interp(age, low.index, low.values)
    if not lo <= k.ratio_z[j] <= med:   # the text: below the TNG50 median at birth, within its 16th-84th percentiles
        raise SystemExit(f'LMC bin {j}: {k.ratio_z[j]:.2f} not between the TNG50 16th percentile {lo:.2f} and median {med:.2f} at birth')
    put(f'sbLmcVs{name}', k.ratio_z[j], 2); put(f'sbTngVs{name}', med, 2); put(f'sbLmcAge{name}', age, 1)
# LMC sigma_z [km/s]: rms of its two most metal-poor bins; heating = sigma_z^2 now minus TNG50 at birth [(km/s)^2], rounded to 100.
sz_lmc = np.sqrt(np.mean(k.sigma_z[:2] ** 2))
put('sbLmcSzLo', k.sigma_z[:2].min()); put('sbLmcSzHi', k.sigma_z[:2].max())
put('sbHeatLmc', round(sz_lmc ** 2 - tng.sz_birth.median() ** 2, -2)); put('sbHeatSims', round(tng.sz_same.median() ** 2 - tng.sz_birth.median() ** 2, -2))
# Young TNG50 cohorts (< 1.5 Gyr): medians over all cohorts [km/s]; Auriga: range of each analogue's median sigma_z at birth.
young = t[(t.suite == 'TNG50') & (t.age < 1.5)]
put('sbYoungSz', young.sz_birth.median()); put('sbYoungVphiBirth', young.vphi_birth.median()); put('sbYoungVphiToday', young.vphi_same.median())
a = t[t.suite == 'Auriga'].groupby('halo').sz_birth.median()
put('sbAuSzLo', a.min()); put('sbAuSzHi', a.max())
```

#### Fig. sims_heating
The tables of scripts/paper_sims_heating.py: kinematics at birth against the progenitor's stellar mass (a), against time since
birth (b), and the coherence of the cohorts (c).

```python
h = pd.read_csv(ROOT / 'results/sims_heating_history.csv', dtype={'halo': str})
gas = pd.read_csv(ROOT / 'results/sims_heating_gas.csv', dtype={'halo': str})
coh = pd.read_csv(ROOT / 'results/sims_heating_coherence.csv', dtype={'halo': str})
# v_phi/sigma_z of the cohorts and of the gas; birth: the dt = 0 rows of the history table.
h['vs'], gas['vs'] = h.vphi / h.sz, gas.vphi / gas.sz
birth = h[h.dt == 0].copy()
# Edges [log10 Msun] of the 0.5-dex bins of the progenitor's stellar mass; galaxy-weighted medians per bin of stars and gas.
DEX = [7, 7.5, 8, 8.5, 9, 9.5]
birth['mb'], gas['mb'] = pd.cut(birth.logMs, DEX), pd.cut(gas.logMs, DEX)
sv, ss, gv, gs = (gal_median(d, 'mb', q).m for d, q in ((birth, 'vphi'), (birth, 'sz'), (gas, 'vphi'), (gas, 'sz')))
put('shVphiLow', sv.iloc[0]); put('shVphiHigh', sv.iloc[-1]); put('shSzLo', ss.min()); put('shSzHi', ss.max())
put('shGasVphiLow', gv.iloc[0]); put('shGasVphiHigh', gv.iloc[-1]); put('shGasSzLo', gs.min()); put('shGasSzHi', gs.max())
# Gas medians per snapshot: sound speed cs [km/s] (median over snapshots), v_phi/sigma_z at the first and last snapshot, last z.
g = gas.groupby('snap')[['z', 'cs', 'vs']].median()
put('shGasCs', g.cs.median()); put('shGasVsFirst', g.vs.iloc[0], 2); put('shGasVsLast', g.vs.iloc[-1], 1); put('shGasZLast', g.z.iloc[-1], 1)
# Present cumulative mass [Msun] below each bin edge (last row, the total, dropped), interpolated at the median [Fe/H] of LMC bins
# A, B.
cm = pd.read_csv(ROOT / 'results/giant_present_mass.csv').iloc[:-1]
logm = np.log10(np.interp(k.feh_med[:2], cm.feh_hi, cm.present_cumulative))
put('shLmcVrotLo', k.vrot4[:2].min()); put('shLmcVrotHi', k.vrot4[:2].max()); put('shLmcLogMLo', logm.min(), 1); put('shLmcLogMHi', logm.max(), 1)
# 0.25-dex mass bins; TNG50 v_phi at birth in the bins whose centres lie within the LMC's mass range.
birth['mq'] = pd.cut(birth.logMs, np.arange(6.5, 10.01, 0.25))
s = gal_median(birth, 'mq', 'vphi').m
s = s[[logm.min() <= i.mid <= logm.max() for i in s.index]]
put('shTngVphiLo', s.min()); put('shTngVphiHi', s.max())
# Time since birth [Gyr] rounded to 0.5 (at least 0.5 after birth); shFirstStep: median time to the first tracked snapshot.
h['dtb'] = np.where(h.dt == 0, 0, np.clip((h.dt * 2).round() / 2, 0.5, None))
put('shFirstStep', h[h.dt > 0].sort_values('dt').groupby(['halo', 'age']).dt.first().median(), 2)
# Age groups [Gyr]: galaxy-weighted v_phi/sigma_z at birth, at 0.5 Gyr, and later (2 Gyr for the old, 3 Gyr for the young).
for name, sel in (('Old', h.age > 10), ('Mid', (h.age > 8) & (h.age <= 10)), ('Young', h.age < 4)):
    m = gal_median(h[sel], 'dtb', 'vs').m
    put(f'sh{name}Birth', m[0], 2); put(f'sh{name}First', m[0.5], 2)
    if name == 'Old':
        put('shOldTwo', m[2.0], 2)
    if name == 'Young':
        put('shYoungThree', m[3.0], 1)
# Coherence C of the early cohorts in 0.1-Gyr steps; late control cohorts: at birth (dt < 0.2 Gyr) or the next snapshot (0.45).
early = coh[coh.set == 'full'].assign(dtb=lambda d: (d.dt / 0.1).round() * 0.1)
c = gal_median(early, 'dtb', 'C').m
late = coh[(coh.set == 'track') & (coh.dt < 0.75)].assign(dtb=lambda d: np.where(d.dt < 0.2, 0.0, 0.45))
# Median early-cohort size, rounded to 10, and 1/sqrt of it.
n_med = early.groupby(['halo', 's_birth']).n0.first().median()
put('shCohBirth', c.iloc[0], 2); put('shCohEnd', c.iloc[-1], 2); put('shCohLate', gal_median(late, 'dtb', 'C').m[0.0], 2)
put('shCohN', round(n_med, -1)); put('shCohRandom', 1 / np.sqrt(n_med), 2)
```

#### Write the macros
One \newcommand per entry of M into results/macros_sims_birth.tex.

```python
# One \newcommand per macro, written and printed.
text = ''.join(f'\\newcommand{{\\{n}}}{{{v}}}\n' for n, v in M.items())
(ROOT / 'results/macros_sims_birth.tex').write_text(text)
print(text)
```

### Auriga and FIRE-2

The numbers of the paragraph on Auriga and FIRE-2 (`results/macros_sims_suites.tex`, prefix `\ss`).

<details><summary>About <code>scripts/paper_text_sims_suites.py</code></summary>

```text
Paper text: the numbers of the Section 6.4 paragraph on Auriga and FIRE-2 (no figure), as LaTeX macros.

Inputs: results/sims_birth.csv (scripts/build_sims_birth.py: birth cohorts of COHORT Gyr, TNG50 and Auriga, from BirthPos/BirthVel)
and the research-repository tables in results/research/ (data/sources.yaml: research_suites_early): fire_early_birth.csv (FIRE-2
birth vectors, cohorts of 0.1 Gyr), auriga_early_hist.csv, fire_early_hist.csv (v_phi/sigma_z of 100-Myr cohorts after birth;
first point = the output after birth), auriga_early_coh.csv, fire_early_coh.csv (coherence C of 100-Myr cohorts),
auriga_stageD_rules.csv (loss of C at 0.2-0.35 Gyr per Auriga cohort and variant of the gravity, as Fig. sims_scramble b).
Definitions (as Fig. sims_heating; per galaxy the median in a bin, then the median over the galaxies):
- at birth: v_phi/sigma_z of the cohorts in 0.5-Gyr bins of lookback time at birth, 8-12 Gyr ago; the range (lowest, highest)
  of the bin medians, per suite;
- the oldest stars: cohorts born > 10 Gyr ago, v_phi/sigma_z at the first output and after 0.25 Gyr (0.25-Gyr bins of time since
  birth); coherence of the cohorts born at z > 1.5 (> 9.4 Gyr ago) at birth and after 0.5 Gyr (0.1-Gyr bins); FIRE-2 cohorts born
  < 9 Gyr ago, C at birth;
- Auriga halo tests: summed loss of a variant over the cohorts / summed loss with the full frozen gravity (E_split): dark matter made
  round (G_bar), axisymmetric about the stars' spin (D0_fr), the halo axisymmetric about its minor axis (H1) for cohorts born less and
  more than 45 deg from alignment.
Writes results/macros_sims_suites.tex (copied to the paper's tables/; \\input at the top of Section 6).
Run: .venv/bin/python scripts/manifest.py run paper_text_sims_suites.py
```
</details>

**Data read by `scripts/paper_text_sims_suites.py`** (from the run log)

| File | Where it comes from | What it is |
|---|---|---|
| `results/research/auriga_early_coh.csv` | repository | fixed input |
| `results/research/auriga_early_hist.csv` | repository | fixed input |
| `results/research/auriga_stageD_rules.csv` | repository | fixed input |
| `results/research/fire_early_birth.csv` | repository | written by `python scripts/fire2_birth.py` |
| `results/research/fire_early_coh.csv` | repository | fixed input |
| `results/research/fire_early_hist.csv` | repository | fixed input |
| `results/sims_birth.csv` | repository | written by `python scripts/build_sims_birth.py` |

*Shown, not run here: results/sims_birth.csv and the tables in results/research/. The cells below are the script itself; run it from the repository with* `python scripts/paper_text_sims_suites.py`*.*

In [ ]:
import matplotlib
matplotlib.rcdefaults()
import matplotlib.pyplot as plt
plt.close('all')

#### Settings and helpers
The time windows of the definitions, the macro store M, and the median over galaxies of the per-galaxy medians.

```python
import numpy as np
import pandas as pd

from lmcold.sources import ROOT

RES = ROOT / 'results/research'   # research-repository tables of data/sources.yaml entry research_suites_early
BIRTH_LB = (8, 12)   # Gyr: lookback times at birth of the 'at birth' ranges
OLD_LB = 10          # Gyr: 'the oldest stars' are born more than OLD_LB Gyr ago
T_AFTER = 0.25       # Gyr after birth of the 'after' value (and the width of the time bins)
M = {}   # macro store: LaTeX macro name -> formatted value (string)


def put(name, value, d=0):
    """Store value in M under macro name, formatted with d decimals."""
    M[name] = f'{value:.{d}f}'


def gal_median(d, by, q):
    """Median of column q per (bin by, galaxy halo), then the median over galaxies per bin; a Series indexed by the bin."""
    return d.groupby([by, 'halo'])[q].median().groupby(level=0).median()
```

#### At birth
v_phi/sigma_z at birth, 8-12 Gyr ago, for TNG50, Auriga and FIRE-2.

```python
# TNG50 and Auriga birth cohorts: vs = v_phi/sigma_z at birth, lb = lookback time at birth (column age) [Gyr]
b = pd.read_csv(ROOT / 'results/sims_birth.csv', dtype={'halo': str}).dropna(subset=['vphi_birth'])
b = b.assign(vs=b.vphi_birth / b.sz_birth, lb=b.age)
# FIRE-2 birth vectors, the 0.1-Gyr cohorts only, with the same ratio vs
f = pd.read_csv(RES / 'fire_early_birth.csv').query('width == 0.1')
f = f.assign(vs=f.vphi / f.sz)
for name, d in (('Tng', b[b.suite == 'TNG50']), ('Au', b[b.suite == 'Auriga']), ('Fire', f)):
    # Cohorts born 8-12 Gyr ago, binned in 0.5 Gyr of lookback (lbb = lower bin edge [Gyr])
    d = d[(d.lb >= BIRTH_LB[0]) & (d.lb < BIRTH_LB[1])].assign(lbb=lambda t: np.floor(t.lb * 2) / 2)
    # Median over galaxies per bin; the macros are the lowest and highest bin medians, 1 decimal
    m = gal_median(d, 'lbb', 'vs')
    put(f'ssBirth{name}Lo', m.min(), 1); put(f'ssBirth{name}Hi', m.max(), 1)
```

#### The oldest stars
v_phi/sigma_z and coherence of the oldest cohorts at birth and soon after, Auriga and FIRE-2.

```python
for name in ('Auriga', 'Fire'):
    # Macro tag; h: v_phi/sigma_z of 100-Myr cohorts versus time since birth dt [Gyr]
    tag = 'Au' if name == 'Auriga' else 'Fire'
    h = pd.read_csv(RES / f'{name.lower()}_early_hist.csv', dtype={'halo': str})
    # Cohorts born > OLD_LB Gyr ago; dt (negative values set to 0) rounded to 0.25-Gyr bins (dtb [Gyr])
    h = h[h.lb_birth > OLD_LB].assign(vs=lambda t: t.vphi / t.sz, dtb=lambda t: (t.dt.clip(lower=0) / T_AFTER).round() * T_AFTER)
    # Galaxy medians at the first output (dtb = 0) and T_AFTER after birth
    m = gal_median(h, 'dtb', 'vs')
    put(f'ssOld{tag}Birth', m[0.0], 2); put(f'ssOld{tag}After', m[T_AFTER], 2)
    # Coherence C of 100-Myr cohorts versus time since birth, in 0.1-Gyr bins
    c = pd.read_csv(RES / f'{name.lower()}_early_coh.csv', dtype={'halo': str})
    c = c.assign(dtb=(c.dt / 0.1).round() * 0.1)
    # Cohorts born at z > 1.5 (> 9.4 Gyr ago): C in the first bin and in the last bin up to 0.5 Gyr after birth
    m = gal_median(c[c.lb_birth > 9.4], 'dtb', 'C')
    put(f'ssCoh{tag}Birth', m.iloc[0], 2); put(f'ssCoh{tag}End', m[m.index <= 0.5 + 1e-9].iloc[-1], 2)
    # FIRE-2 only: C at birth of the cohorts born < 9 Gyr ago
    if tag == 'Fire':
        put('ssCohFireLate', gal_median(c[c.lb_birth < 9], 'dtb', 'C').iloc[0], 2)
```

#### Auriga halo tests
Loss of coherence with modified gravity, relative to the full frozen gravity.

```python
# One row per Auriga cohort: loss of C under each gravity variant (columns E_split, G_bar, D0_fr, H1) and
# the misalignment [deg]; lo, hi: cohorts born less and more than 45 deg from alignment
s = pd.read_csv(RES / 'auriga_stageD_rules.csv')
lo, hi = s[s.misalign < 45], s[s.misalign >= 45]
# Number of cohorts and the summed loss of each variant over the summed loss with the full frozen gravity
put('ssAuN', len(s)); put('ssAuRound', s.G_bar.sum() / s.E_split.sum(), 2); put('ssAuAxi', s.D0_fr.sum() / s.E_split.sum(), 2)
put('ssAuHOneAligned', lo.H1.sum() / lo.E_split.sum(), 2); put('ssAuHOneTilted', hi.H1.sum() / hi.E_split.sum(), 2)
```

#### Write the macros
One \newcommand per entry of M into results/macros_sims_suites.tex.

```python
# One \newcommand line per macro; written to results/macros_sims_suites.tex and printed
text = ''.join(f'\\newcommand{{\\{n}}}{{{v}}}\n' for n, v in M.items())
(ROOT / 'results/macros_sims_suites.tex').write_text(text)
print(text)
```